<a href="https://colab.research.google.com/github/samer-glitch/TADP-Cluster-Computing/blob/main/RQ5%2BRQ6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import io
import os

print("📤 Please upload the diabetes_130US.csv file:")
uploaded = files.upload()
DATA_FILENAME  = "diabetes_130US.csv"
DATA_CACHE_DIR = "./data_cache"
os.makedirs(DATA_CACHE_DIR, exist_ok=True)

# Get the uploaded file
for filename in uploaded.keys():
    file_data = uploaded[filename]
    print(f'✅ Uploaded: {filename} ({len(file_data)} bytes)')


    # Save to cache
    cache_path = os.path.join(DATA_CACHE_DIR, DATA_FILENAME)
    with open(cache_path, 'wb') as f:
        f.write(file_data)
    print(f'✅ Saved to cache: {cache_path}')

    # Verify the file was saved
    if os.path.exists(cache_path):
        file_size = os.path.getsize(cache_path)
        print(f'✅ Verified: {cache_path} exists ({file_size} bytes)')
    else:
        print(f'❌ Error: File was not saved properly')

📤 Please upload the diabetes_130US.csv file:


Saving diabetes_130US.csv to diabetes_130US.csv
✅ Uploaded: diabetes_130US.csv (19159383 bytes)
✅ Saved to cache: ./data_cache/diabetes_130US.csv
✅ Verified: ./data_cache/diabetes_130US.csv exists (19159383 bytes)


In [2]:
# ======================================================================================
# TADP v16.7 — REVIEWER-ALIGNED TRUSTWORTHY DATA PREPARATION EXPERIMENT CORE
# ======================================================================================
# Design guarantees:
#   1) GLOBAL holdout is created BEFORE client partitioning.
#   2) Only TRAIN is distributed to clients.
#   3) Preprocessing parameters/vocabularies use TRAIN only.
#   4) DQ, GE and TADP governance use TRAIN only.
#   5) Held-out TEST never affects preprocessing, governance, class weights,
#      client selection, model initialization, training, or matched-control budgets.
#   6) TEST is used only after training for final evaluation.
#
# v16.7 governance:
#   - 28 factors across 6 dimensions:
#       dim1=4, dim2(DQ)=8, dim3=4, dim4=3, dim5=5, dim6=4.
#   - Added Data Collection / Acquisition Lineage (dim1).
#   - Added Structural / Constraint Integrity (dim2).
#   - Documentary evidence is generated at the individual factor level.
#   - DQ evidence is machine-measured from client TRAIN partitions only.
#   - HPS remains client-specific and uses the six policy dimension weights.
#   - WAC is NOT client-specific.
#   - Each factor has a declared minimum adequate rubric rank.
#   - A domain WAC is derived once:
#         WAC_d = mean(minimum adequate ranks in dimension d) / 5
#         WAC_domain = equal mean of the six WAC_d values.
#   - Every averaged dimension must be >= 2.5/5 or the client is auto-rejected.
#   - HPS < 3.0 -> AUTO_REJECT.
#   - 3.0 <= HPS < 3.5 -> AUTOMATED REVIEW using Critical WAC_i.
#   - Review accepts iff Critical WAC_i >= the domain Global Critical WAC.
#   - HPS >= 3.5 -> DIRECT AUTO_ACCEPT when every critical factor meets its own adequacy minimum.
#   - Human reviewers verify evidence only; admission is server-automated.
#
# GX Core comparator:
#   - Separate from HPS/TADP.
#   - Uses REAL Great Expectations GX Core validation on TRAIN-only client data.
#   - Uses common technical checks: schema, datatype consistency, required ranges,
#     missingness, duplicate/ID integrity, label/domain validity, and structure.
#   - Uses GX native severity-aware validation: zero critical failures required; no ranking and no forced-K.
#
# Runtime/reporting:
#   - Every FL round prints configuration/run/scenario/round progress plus TRAIN-only diagnostic utility and operational metrics.
#   - Every completed scenario prints all predictive and operational metrics.
#   - Scenario checkpoints support restart/resume.
#   - Final result ZIP downloads automatically in Google Colab.
# ======================================================================================

import os
import sys
import gc
import math
import time
import json
import random
import hashlib
import threading
import zipfile
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# ======================================================================================
# POLICY CONSTANTS — v16.7
# ======================================================================================

GOOD_CUT = 3.0
HIGH_CUT = 3.5
MAX_FACTOR_SCORE = 5.0
GE_ACCEPT_COUNT = 6

# ----------------------------------------------------------------------
# v16.7 FINAL FULLY AUTOMATED ADMISSION POLICY
# ----------------------------------------------------------------------
# Human reviewers verify supporting evidence uploaded through the questionnaire.
# They do NOT make the admission decision. Once verified factor scores are
# available, the server applies this policy automatically.
#
# 1) EVERY averaged HPS dimension must be >= 2.5/5.
#    Any dimension < 2.5 -> AUTO-REJECT.
#
# 2) HPS must be >= 3.0.
#    HPS < 3.0 -> AUTO-REJECT.
#
# 3) DIRECT AUTO-ACCEPT:
#    HPS >= 3.5 AND every critical factor independently meets its own
#    factor-specific minimum adequacy requirement.
#
# 4) AUTOMATED REVIEW:
#    - all clients with 3.0 <= HPS < 3.5; and
#    - high-HPS clients that fail one or more individual critical-factor
#      adequacy requirements.
#
# 5) REVIEW RESOLUTION:
#    Critical WAC_i = mean(actual critical-factor scores / 5)
#    Global Critical WAC = mean(policy adequacy minima / 5 for same factors)
#
#    Critical WAC_i >= Global Critical WAC -> ACCEPT AFTER REVIEW
#    Critical WAC_i <  Global Critical WAC -> AUTO-REJECT
#
# Thus Direct Auto-Accept is strict at the individual critical-factor level,
# whereas Review intentionally allows controlled compensation across the
# critical subset.
DIMENSION_MIN_FLOOR = 2.5

CRITICAL_FACTORS_BY_DOMAIN = {
    "healthcare": {
        "dim1": [
            "data_controller",
            "data_collection_lineage",
        ],
        "dim5": [
            "regulation_coverage",
            "consent_ethics",
            "sensitivity_classification",
        ],
        "dim6": [
            "user_agreements",
        ],
    },
    "cifar10": {
        "dim1": [
            "data_controller",
            "data_collection_lineage",
        ],
        "dim6": [
            "license_terms",
            "user_agreements",
        ],
    },
}

WEIGHTS_PSCORE_DEFAULT = {
    "dim1": 0.25,  # Source Reliability
    "dim2": 0.15,  # Data Quality and Health
    "dim3": 0.10,  # Documentation Practices
    "dim4": 0.10,  # Timeliness and Refresh Rate
    "dim5": 0.30,  # Regulatory / Compliance Alignment
    "dim6": 0.10,  # Context / Usage Constraints
}

DIMENSION_NAMES = {
    "dim1": "Source Reliability",
    "dim2": "Data Quality and Health",
    "dim3": "Documentation Practices",
    "dim4": "Timeliness and Refresh Rate",
    "dim5": "Regulatory and Compliance Alignment",
    "dim6": "Context and Usage Constraints",
}

# v16.7: two reviewer-driven additions:
#   dim1: data_collection_lineage
#   dim2: structural_constraint_integrity
#
# Total = 4 + 8 + 4 + 3 + 5 + 4 = 28 factors.
FACTOR_NAMES = {
    "dim1": [
        "source_reputation",
        "data_controller",
        "data_objective",
        "data_collection_lineage",
    ],
    "dim2": [
        "completeness",
        "duplication_rate",
        "value_validity_error_rate",
        "type_consistency",
        "label_integrity",
        "feature_distribution_consistency",
        "feature_category_coverage",
        "structural_constraint_integrity",
    ],
    "dim3": [
        "data_dictionary",
        "version_logs",
        "collection_protocol",
        "definition_updates",
    ],
    "dim4": [
        "data_freshness",
        "scheduled_refresh",
        "retention_clarity",
    ],
    "dim5": [
        "regulation_coverage",
        "consent_ethics",
        "geo_restrictions",
        "sensitivity_classification",
        "audits",
    ],
    "dim6": [
        "license_terms",
        "ethical_reviews",
        "redistribution",
        "user_agreements",
    ],
}

DOCUMENTARY_DIMS = ("dim1", "dim3", "dim4", "dim5", "dim6")

# ----------------------------------------------------------------------
# COMPLETE 0--5 RUBRIC DESCRIPTORS
# ----------------------------------------------------------------------
# These descriptors reproduce the Appendix-A semantics and add the two
# v16.7 factors explicitly. Controlled evidence is sampled at the factor
# level; HPS is never generated directly.
RUBRIC_DESCRIPTORS = {
    "dim1": {
        "source_reputation": {
            0: "No info",
            1: "Poor",
            2: "Limited evidence",
            3: "Average, partially trusted",
            4: "Well-documented, reliable",
            5: "Highly reputable, verified",
        },
        "data_controller": {
            0: "No documented controller",
            1: "Unclear",
            2: "Partially clear",
            3: "Moderately clear",
            4: "Mostly clear",
            5: "Fully documented",
        },
        "data_objective": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General but unclear",
            4: "Mostly explicit",
            5: "Fully explicit, justified",
        },
        "data_collection_lineage": {
            0: "Collection origin unknown",
            1: "Informal or unverifiable origin",
            2: "Partially documented acquisition path",
            3: "Documented acquisition with limited traceability",
            4: "Well-documented and traceable acquisition path",
            5: "Fully source-linked, versioned, and auditable lineage",
        },
    },
    "dim2": {
        "completeness": {
            0: ">50% missing",
            1: "20-50% missing",
            2: "10-20% missing",
            3: "5-10% missing",
            4: "1-5% missing",
            5: "<1% missing",
        },
        "duplication_rate": {
            0: ">20% duplicates",
            1: "10-20% duplicates",
            2: "5-10% duplicates",
            3: "2-5% duplicates",
            4: "1-2% duplicates",
            5: "<1% duplicates",
        },
        "value_validity_error_rate": {
            0: ">15% invalid/error values",
            1: "10-15% invalid/error values",
            2: "5-10% invalid/error values",
            3: "2-5% invalid/error values",
            4: "1-2% invalid/error values",
            5: "<1% invalid/error values",
        },
        "type_consistency": {
            0: "Highly inconsistent",
            1: "Frequent type inconsistency",
            2: "Moderate type inconsistency",
            3: "Minor type inconsistency",
            4: "Rare type inconsistency",
            5: "Fully consistent",
        },
        "label_integrity": {
            0: ">10% missing/invalid/known erroneous labels",
            1: "5-10% missing/invalid/known erroneous labels",
            2: "2-5% missing/invalid/known erroneous labels",
            3: "1-2% missing/invalid/known erroneous labels",
            4: "0.1-1% missing/invalid/known erroneous labels",
            5: "<=0.1% missing/invalid/known erroneous labels",
        },
        "feature_distribution_consistency": {
            0: "JSD >0.20",
            1: "JSD 0.10-0.20",
            2: "JSD 0.05-0.10",
            3: "JSD 0.025-0.05",
            4: "JSD 0.01-0.025",
            5: "JSD <=0.01",
        },
        "feature_category_coverage": {
            0: "<50% reference support represented",
            1: "50-65% reference support represented",
            2: "65-75% reference support represented",
            3: "75-82.5% reference support represented",
            4: "82.5-90% reference support represented",
            5: ">=90% reference support represented",
        },
        "structural_constraint_integrity": {
            0: ">10% records/structures violate required constraints",
            1: "5-10% violate required constraints",
            2: "2-5% violate required constraints",
            3: "1-2% violate required constraints",
            4: "0.1-1% violate required constraints",
            5: "<=0.1% violate required constraints",
        },
    },
    "dim3": {
        "data_dictionary": {
            0: "None",
            1: "Minimal outline",
            2: "Partial coverage",
            3: "Moderate coverage",
            4: "Near-complete",
            5: "Fully detailed",
        },
        "version_logs": {
            0: "None",
            1: "Minimal logs",
            2: "Occasional logs",
            3: "Regular logs",
            4: "Near-complete",
            5: "Full version history",
        },
        "collection_protocol": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General methods",
            4: "Well-defined",
            5: "Fully transparent",
        },
        "definition_updates": {
            0: "None",
            1: "Rarely updated",
            2: "Occasional updates",
            3: "Regular but basic",
            4: "Frequent",
            5: "Real-time, documented",
        },
    },
    "dim4": {
        "data_freshness": {
            0: ">5 years old",
            1: "2-5 years old",
            2: "1-2 years old",
            3: "6-12 months old",
            4: "1-6 months old",
            5: "Real-time/current",
        },
        "scheduled_refresh": {
            0: "Never",
            1: "Irregular",
            2: "Annual",
            3: "Quarterly",
            4: "Monthly",
            5: "Daily/real-time",
        },
        "retention_clarity": {
            0: "None",
            1: "Minimal",
            2: "Basic guidelines",
            3: "Moderate clarity",
            4: "High clarity",
            5: "Fully documented",
        },
    },
    "dim5": {
        "regulation_coverage": {
            0: "None",
            1: "Minimal",
            2: "Partial",
            3: "Moderate",
            4: "Comprehensive but dated",
            5: "Fully documented/current",
        },
        "consent_ethics": {
            0: "None",
            1: "Minimal record",
            2: "Partial consent/ethics evidence",
            3: "Moderate logs",
            4: "Substantial",
            5: "Fully documented",
        },
        "geo_restrictions": {
            0: "None",
            1: "Basic mention",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully documented",
        },
        "sensitivity_classification": {
            0: "None",
            1: "Basic flagging",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully classified",
        },
        "audits": {
            0: "None",
            1: "Internal only",
            2: "Basic certification",
            3: "Occasional audit",
            4: "Recent audit",
            5: "Regular external audits",
        },
    },
    "dim6": {
        "license_terms": {
            0: "None",
            1: "Vague",
            2: "Basic",
            3: "Clear",
            4: "Detailed",
            5: "Industry-compliant",
        },
        "ethical_reviews": {
            0: "None",
            1: "Informal approval",
            2: "Partial",
            3: "Moderate",
            4: "Well-documented",
            5: "Certified",
        },
        "redistribution": {
            0: "No policy",
            1: "Unclear",
            2: "Partial",
            3: "Clear",
            4: "Detailed",
            5: "Fully compliant",
        },
        "user_agreements": {
            0: "Non-compliant",
            1: "Minimal adherence",
            2: "Partial",
            3: "Mostly compliant",
            4: "Fully compliant",
            5: "Audited compliance",
        },
    },
}

# ----------------------------------------------------------------------
# FACTOR-SPECIFIC ADEQUACY POLICY
# ----------------------------------------------------------------------
# The minimum adequate rank is derived factor-by-factor from the wording of
# the Appendix-A rubric. It is NOT learned from model/test outcomes.
#
# Healthcare is the primary policy. CIFAR-10 uses the same reference ranks
# for cross-domain comparability; its DQ factors are measured with image-
# specific checks, while documentary dimensions remain controlled evidence.
FACTOR_ADEQUACY_MIN_HEALTHCARE = {
    "dim1": {
        "source_reputation": 4,
        "data_controller": 4,
        "data_objective": 4,
        "data_collection_lineage": 4,
    },
    "dim2": {
        "completeness": 3,
        "duplication_rate": 3,
        "value_validity_error_rate": 3,
        "type_consistency": 3,
        "label_integrity": 3,
        "feature_distribution_consistency": 3,
        "feature_category_coverage": 3,
        "structural_constraint_integrity": 3,
    },
    "dim3": {
        "data_dictionary": 3,
        "version_logs": 3,
        "collection_protocol": 4,
        "definition_updates": 3,
    },
    "dim4": {
        "data_freshness": 3,
        "scheduled_refresh": 3,
        "retention_clarity": 4,
    },
    "dim5": {
        "regulation_coverage": 3,
        "consent_ethics": 3,
        "geo_restrictions": 3,
        "sensitivity_classification": 3,
        "audits": 4,
    },
    "dim6": {
        "license_terms": 3,
        "ethical_reviews": 4,
        "redistribution": 3,
        "user_agreements": 4,
    },
}

FACTOR_ADEQUACY_MIN_CIFAR10 = {
    dim: dict(values)
    for dim, values in FACTOR_ADEQUACY_MIN_HEALTHCARE.items()
}

def derive_domain_wac(
    factor_minima: Dict[str, Dict[str, float]]
) -> Tuple[Dict[str, float], float]:
    """
    Derive the domain policy WAC from factor-specific minimum adequate ranks.

    WAC_d = mean_k(adequate_rank_dk / 5)
    WAC_domain = equal mean across the six dimension WAC_d values.

    IMPORTANT:
      WAC is a DOMAIN POLICY value, not a client-specific score.
    """
    dimension_wac = {}
    for dim in FACTOR_NAMES:
        vals = [
            float(factor_minima[dim][factor])
            for factor in FACTOR_NAMES[dim]
        ]
        dimension_wac[dim] = float(np.mean(vals) / MAX_FACTOR_SCORE)
    global_wac = float(np.mean(list(dimension_wac.values())))
    return dimension_wac, global_wac


HEALTHCARE_DIMENSION_WAC, WAC_HEALTHCARE = derive_domain_wac(
    FACTOR_ADEQUACY_MIN_HEALTHCARE
)
CIFAR10_DIMENSION_WAC, WAC_CIFAR10 = derive_domain_wac(
    FACTOR_ADEQUACY_MIN_CIFAR10
)

DOMAIN_FACTOR_MINIMA = {
    "healthcare": FACTOR_ADEQUACY_MIN_HEALTHCARE,
    "cifar10": FACTOR_ADEQUACY_MIN_CIFAR10,
}
DOMAIN_DIMENSION_WAC = {
    "healthcare": HEALTHCARE_DIMENSION_WAC,
    "cifar10": CIFAR10_DIMENSION_WAC,
}
DOMAIN_GLOBAL_WAC = {
    "healthcare": WAC_HEALTHCARE,
    "cifar10": WAC_CIFAR10,
}

DQ_RAW_METRIC_BY_FACTOR = {
    "completeness": "missing_fraction",
    "duplication_rate": "duplicate_fraction",
    "value_validity_error_rate": "error_fraction",
    "type_consistency": "type_inconsistency_fraction",
    "label_integrity": "invalid_label_fraction",
    "feature_distribution_consistency": "max_jsd",
    "feature_category_coverage": "mean_category_coverage",
    "structural_constraint_integrity": "structural_violation_fraction",
}

assert abs(sum(WEIGHTS_PSCORE_DEFAULT.values()) - 1.0) < 1e-12
assert sum(len(v) for v in FACTOR_NAMES.values()) == 28
assert len(FACTOR_NAMES["dim1"]) == 4
assert len(FACTOR_NAMES["dim2"]) == 8
assert abs(WAC_HEALTHCARE - 0.6761111111111111) < 1e-12

# ======================================================================================
# GENERAL UTILITIES
# ======================================================================================

def seed_everything(seed: int):
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        tf.keras.utils.set_random_seed(seed)
    except Exception:
        tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def sha256_bytes(x: bytes) -> str:
    return hashlib.sha256(x).hexdigest()


def sha256_array(x: np.ndarray) -> str:
    x = np.asarray(x)
    return sha256_bytes(np.ascontiguousarray(x).view(np.uint8).tobytes())


def sha256_weights(weights: List[np.ndarray]) -> str:
    h = hashlib.sha256()
    for w in weights:
        a = np.ascontiguousarray(np.asarray(w))
        h.update(str(a.shape).encode())
        h.update(a.view(np.uint8).tobytes())
    return h.hexdigest()


def _process_rss_mb() -> float:
    try:
        import psutil
        return float(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2))
    except Exception:
        pass
    try:
        with open("/proc/self/statm", "r", encoding="utf-8") as f:
            pages = int(f.read().split()[1])
        return float(pages * int(os.sysconf("SC_PAGE_SIZE")) / (1024 ** 2))
    except Exception:
        pass
    try:
        import resource
        x = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
        return x / (1024 ** 2) if sys.platform == "darwin" else x / 1024.0
    except Exception:
        return 0.0


class RAMMonitor:
    def __init__(self, interval_s: float = 0.05):
        self.interval_s = float(interval_s)
        self.start_mb = 0.0
        self.end_mb = 0.0
        self.peak_mb = 0.0
        self._stop = threading.Event()
        self._thread = None

    def _loop(self):
        while not self._stop.wait(self.interval_s):
            self.peak_mb = max(self.peak_mb, _process_rss_mb())

    def start(self):
        self.start_mb = _process_rss_mb()
        self.peak_mb = self.start_mb
        self._stop.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def stop(self) -> Dict[str, float]:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=1.0)
        self.end_mb = _process_rss_mb()
        self.peak_mb = max(self.peak_mb, self.start_mb, self.end_mb)
        return {
            "ram_start_mb": float(self.start_mb),
            "ram_end_mb": float(self.end_mb),
            "ram_peak_mb": float(self.peak_mb),
            "ram_delta_mb": float(max(0.0, self.peak_mb - self.start_mb)),
            "ram_mb": float(self.peak_mb),
        }


def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
    return Path(path)


def append_csv(row: Dict[str, Any], path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([row]).to_csv(
        path, mode="a", header=not path.exists(), index=False
    )


def summarize_runs(perf: pd.DataFrame) -> pd.DataFrame:
    if perf.empty:
        return pd.DataFrame()
    metrics = [
        c for c in [
            "accuracy", "precision_macro", "recall_macro", "f1_macro",
            "roc_auc_ovr_macro", "runtime_s", "energy_wh", "communication_mb",
            "optimizer_steps", "participants", "ram_peak_mb", "ram_delta_mb"
        ] if c in perf.columns
    ]
    rows = []
    for scenario, d in perf.groupby("scenario", sort=False):
        row = {"scenario": scenario, "n_runs": len(d)}
        for m in metrics:
            vals = pd.to_numeric(d[m], errors="coerce")
            row[f"{m}_mean"] = float(vals.mean())
            row[f"{m}_sd"] = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
        rows.append(row)
    return pd.DataFrame(rows)


def package_and_download_results(output_dir: Path, label: str) -> Path:
    output_dir = Path(output_dir)
    manifest = []
    for p in sorted(output_dir.rglob("*")):
        if p.is_file():
            manifest.append({
                "relative_path": str(p.relative_to(output_dir)),
                "bytes": int(p.stat().st_size),
                "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
            })
    pd.DataFrame(manifest).to_csv(
        output_dir / "RESULTS_FILE_MANIFEST.csv", index=False
    )

    zip_path = output_dir.parent / f"{output_dir.name}_RESULTS.zip"
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
    ) as zf:
        for p in sorted(output_dir.rglob("*")):
            if p.is_file():
                zf.write(p, arcname=str(p.relative_to(output_dir)))

    print("\n" + "=" * 100)
    print(f"{label} COMPLETE")
    print(f"Results ZIP: {zip_path}")
    print(f"ZIP size: {zip_path.stat().st_size / (1024**2):.2f} MB")
    print("=" * 100)

    try:
        from google.colab import files as colab_files
        print("Starting automatic download to your laptop...")
        colab_files.download(str(zip_path))
    except Exception as exc:
        print("Automatic Colab download unavailable.")
        print(f"ZIP remains at: {zip_path}")
        print(f"Reason: {exc}")

    return zip_path


# ======================================================================================
# CONTROLLED DOCUMENTARY EVIDENCE — v16.7
# ======================================================================================

def rubric_descriptor(dim: str, factor: str, score: float) -> str:
    s = int(np.clip(np.rint(float(score)), 0, 5))
    return str(RUBRIC_DESCRIPTORS[dim][factor][s])


def generate_controlled_documentary_evidence(
    client_ids: List[str],
    evidence_seed: int,
    factor_minima: Dict[str, Dict[str, float]],
    domain: str,
) -> Tuple[Dict[str, Dict[str, Dict[str, float]]], pd.DataFrame]:
    """
    Generate controlled documentary evidence at the INDIVIDUAL FACTOR level.

    v16.7 uses PRE-SPECIFIED governance archetypes so the controlled experiment
    contains all three decision outcomes needed to validate the policy:

      - DIRECT_STRONG:
          designed to satisfy the strict Direct Auto-Accept route;
      - REVIEW_RECOVERABLE:
          borderline overall evidence, but sufficiently strong average evidence
          across the critical subset for Automated Review acceptance;
      - REVIEW_LIMITED:
          adequate enough to enter Automated Review, but insufficient average
          critical evidence for Review acceptance;
      - LOW_HPS_WEAK:
          every documentary dimension remains at/above the 2.5 dimension floor,
          but the overall HPS is expected to remain below 3.0;
      - DIMENSION_FLOOR_WEAK:
          contains a deliberately weak documentary dimension (<2.5).

    IMPORTANT SCIENTIFIC INTERPRETATION
    -----------------------------------
    These are controlled governance scenarios, not observed hospital-site
    provenance records and not estimates of real-world admission prevalence.
    The archetype composition is specified BEFORE model training and does not
    use predictive performance, test labels, or downstream model outcomes.

    DQ (dim2) is NEVER synthesized here; it remains measured from TRAIN data.
    The frozen evidence seed randomly assigns the pre-generated archetypes to
    client identities.
    """
    domain = str(domain).lower()
    if domain not in CRITICAL_FACTORS_BY_DOMAIN:
        raise ValueError(f"Unsupported domain: {domain!r}")

    n = len(client_ids)
    rng = np.random.default_rng(int(evidence_seed))

    critical_policy = CRITICAL_FACTORS_BY_DOMAIN[domain]
    critical_keys = {
        (dim, factor)
        for dim, factor_list in critical_policy.items()
        for factor in factor_list
    }

    def make_documentary_profile(role: str, variant: int):
        factors = {
            dim: {}
            for dim in DOCUMENTARY_DIMS
        }

        if role == "DIRECT_STRONG":
            # Strong but not uniformly perfect. Every factor is at least 4,
            # while some values reach 5 according to a deterministic variant.
            for dim in DOCUMENTARY_DIMS:
                for j, factor in enumerate(FACTOR_NAMES[dim]):
                    minimum = float(factor_minima[dim][factor])
                    base = max(4.0, minimum)
                    bonus = 1.0 if ((j + variant + len(dim)) % 3 == 0) else 0.0
                    factors[dim][factor] = float(min(5.0, base + bonus))

        elif role == "REVIEW_RECOVERABLE":
            # Start from moderate evidence: all documentary dimensions remain
            # safely above the 2.5 floor without making HPS automatically high.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = 3.0

            critical_sequence = [
                (dim, factor)
                for dim, factor_list in critical_policy.items()
                for factor in factor_list
            ]

            # Put critical factors at their policy adequacy references.
            for dim, factor in critical_sequence:
                factors[dim][factor] = float(
                    factor_minima[dim][factor]
                )

            # Intentionally allow ONE critical factor to fall one point below
            # its individual minimum, while another critical factor with room
            # is strengthened. This is exactly the compensatory situation that
            # Automated Review is intended to resolve.
            high_min = [
                (dim, factor)
                for dim, factor in critical_sequence
                if float(factor_minima[dim][factor]) >= 4.0
            ]
            lower_min = [
                (dim, factor)
                for dim, factor in critical_sequence
                if float(factor_minima[dim][factor]) <= 3.0
            ]

            if high_min and lower_min:
                weak_key = high_min[variant % len(high_min)]
                strong_key = lower_min[variant % len(lower_min)]

                weak_min = float(
                    factor_minima[weak_key[0]][weak_key[1]]
                )
                strong_min = float(
                    factor_minima[strong_key[0]][strong_key[1]]
                )

                factors[weak_key[0]][weak_key[1]] = float(
                    max(0.0, weak_min - 1.0)
                )
                factors[strong_key[0]][strong_key[1]] = float(
                    min(5.0, strong_min + 2.0)
                )

            # Small documentary variation between the two recoverable bundles.
            if variant % 2 == 1:
                factors["dim3"][FACTOR_NAMES["dim3"][0]] = 4.0

        elif role == "REVIEW_LIMITED":
            # Preserve dimensions at/above 2.5 and keep HPS in/near the review
            # region, but make the average critical evidence too weak.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    if dim in critical_policy and (dim, factor) not in critical_keys:
                        factors[dim][factor] = 4.0
                    else:
                        factors[dim][factor] = 3.0

            for dim, factor_list in critical_policy.items():
                for factor in factor_list:
                    minimum = float(factor_minima[dim][factor])
                    factors[dim][factor] = float(
                        max(2.0, minimum - 1.0)
                    )

            if variant % 2 == 1:
                factors["dim3"][FACTOR_NAMES["dim3"][0]] = 4.0

        elif role == "LOW_HPS_WEAK":
            # Each documentary dimension averages >=2.5, but remains weak.
            # This isolates the HPS<3.0 rejection path from the dimension floor.
            for dim in DOCUMENTARY_DIMS:
                names = list(FACTOR_NAMES[dim])
                values = [2.0] * len(names)
                n_three = (len(names) + 1) // 2
                for j in range(n_three):
                    values[j] = 3.0

                for factor, value in zip(names, values):
                    factors[dim][factor] = float(value)

        elif role == "DIMENSION_FLOOR_WEAK":
            # General evidence is moderate, but Timeliness is deliberately below
            # the 2.5 dimension trustworthiness floor.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = 3.0

            for factor in FACTOR_NAMES["dim4"]:
                factors["dim4"][factor] = 2.0

        else:
            raise ValueError(f"Unknown evidence role: {role!r}")

        return factors

    # For the fixed K=10 experiments, pre-specify a balanced governance
    # branch-coverage set:
    #   4 direct-strong
    #   2 review-recoverable
    #   2 review-limited
    #   1 low-HPS weak
    #   1 dimension-floor weak
    #
    # This is not a post-hoc selection by client identity. The ten bundles are
    # generated first and then randomly assigned to clients using evidence_seed.
    if n == 10:
        bundle_specs = [
            ("DIRECT_STRONG", 0),
            ("DIRECT_STRONG", 1),
            ("DIRECT_STRONG", 2),
            ("DIRECT_STRONG", 3),
            ("REVIEW_RECOVERABLE", 0),
            ("REVIEW_RECOVERABLE", 1),
            ("REVIEW_LIMITED", 0),
            ("REVIEW_LIMITED", 1),
            ("LOW_HPS_WEAK", 0),
            ("DIMENSION_FLOOR_WEAK", 0),
        ]
    else:
        # Generic fallback for non-10-client studies: cycle through the same
        # archetypes without conditioning on client data or model performance.
        archetypes = [
            "DIRECT_STRONG",
            "REVIEW_RECOVERABLE",
            "REVIEW_LIMITED",
            "LOW_HPS_WEAK",
            "DIMENSION_FLOOR_WEAK",
        ]
        bundle_specs = [
            (archetypes[j % len(archetypes)], j // len(archetypes))
            for j in range(n)
        ]

    bundles = []
    for b, (role, variant) in enumerate(bundle_specs):
        factors = make_documentary_profile(role, int(variant))

        bundles.append({
            "bundle_id": f"E{b+1:02d}",
            "profile": role,
            "scenario_role": role,
            "profile_variant": int(variant),
            "factors": factors,
        })

    # Randomly assign already-generated governance bundles to client identities.
    assignment = rng.permutation(n)

    evidence_by_client = {}
    rows = []

    for client_pos, cid in enumerate(client_ids):
        bundle = bundles[int(assignment[client_pos])]

        evidence_by_client[cid] = {
            dim: dict(bundle["factors"][dim])
            for dim in DOCUMENTARY_DIMS
        }

        for dim in DOCUMENTARY_DIMS:
            for factor, score in bundle["factors"][dim].items():
                min_rank = float(
                    factor_minima[dim][factor]
                )

                rows.append({
                    "client":
                        str(cid),
                    "bundle_id":
                        bundle["bundle_id"],
                    "evidence_profile":
                        bundle["profile"],
                    "scenario_role":
                        bundle["scenario_role"],
                    "profile_variant":
                        int(bundle["profile_variant"]),
                    "dimension":
                        dim,
                    "dimension_name":
                        DIMENSION_NAMES[dim],
                    "factor":
                        factor,
                    "evidence_source_type":
                        "CONTROLLED_DOCUMENTARY_EVIDENCE",
                    "verified_rubric_level_0_5":
                        float(score),
                    "rubric_score_0_5":
                        float(score),
                    "server_mapped_score_0_5":
                        float(score),
                    "rubric_descriptor":
                        rubric_descriptor(
                            dim,
                            factor,
                            score,
                        ),
                    "human_role":
                        "VERIFY_EVIDENCE_ONLY",
                    "score_assignment":
                        "DETERMINISTIC_SERVER_MAPPING_FROM_VERIFIED_RUBRIC_LEVEL",
                    "admission_decision_by":
                        "SERVER_POLICY",
                    "adequacy_min_rank":
                        min_rank,
                    "adequacy_min_normalized":
                        min_rank / MAX_FACTOR_SCORE,
                    "meets_factor_adequacy":
                        bool(float(score) >= min_rank),
                    "evidence_artifact_id": (
                        f"{cid}-{bundle['bundle_id']}-{dim}-{factor}"
                    ),
                    "validation_status":
                        "CONTROLLED_SCENARIO_EVIDENCE",
                    "evidence_seed":
                        int(evidence_seed),
                    "domain":
                        domain,
                })

    return evidence_by_client, pd.DataFrame(rows)

def dimension_scores_from_factors(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, float]:
    """
    Client-specific HPS dimension scores (0..5):
    mean of the observed factor rubric scores within each dimension.
    """
    out = {}
    for dim in FACTOR_NAMES:
        vals = [
            float(v)
            for v in factors.get(dim, {}).values()
            if np.isfinite(float(v))
        ]
        out[dim] = float(np.mean(vals)) if vals else 0.0
    return out


def compute_hps(
    dimensions: Dict[str, float],
    weights: Dict[str, float] = WEIGHTS_PSCORE_DEFAULT,
) -> float:
    return float(
        sum(float(weights[d]) * float(dimensions[d]) for d in weights)
    )


def client_dimension_adequacy(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, float]:
    """
    Client ACHIEVED adequacy, not WAC.

    A_i,d = mean(client factor ranks in dimension d) / 5.
    """
    scores = {}
    for dim in FACTOR_NAMES:
        vals = [
            float(v)
            for v in factors.get(dim, {}).values()
            if np.isfinite(float(v))
        ]
        scores[dim] = (
            float(np.mean(vals) / MAX_FACTOR_SCORE)
            if vals else 0.0
        )
    return scores


def client_adequacy_score(
    factors: Dict[str, Dict[str, float]]
) -> Tuple[Dict[str, float], float]:
    """
    Legacy audit helper retained for backward comparability only.

    v16.7 does NOT use this value for admission. Review decisions use the
    client Critical WAC_i versus the domain Global Critical WAC.
    """
    by_dim = client_dimension_adequacy(factors)
    cas = float(np.mean(list(by_dim.values())))
    return by_dim, cas


def all_zero_score_factors(
    factors: Dict[str, Dict[str, float]]
) -> List[str]:
    """Audit-only zero list. Zero is not a separate admission rule in v16.7."""
    zeros = []
    for dim in FACTOR_NAMES:
        for factor, value in factors.get(dim, {}).items():
            if np.isfinite(float(value)) and float(value) <= 0.0:
                zeros.append(f"{dim}.{factor}")
    return zeros


def derive_global_critical_wac(domain: str) -> float:
    """
    Domain Global Critical WAC = mean(minimum critical adequacy rank / 5).

    This is a policy reference used only to resolve the HPS Review band.
    """
    domain = str(domain).lower()
    vals = []
    for dim, factors_ in CRITICAL_FACTORS_BY_DOMAIN[domain].items():
        for factor in factors_:
            vals.append(
                float(DOMAIN_FACTOR_MINIMA[domain][dim][factor])
                / MAX_FACTOR_SCORE
            )
    if not vals:
        raise ValueError(f"No critical factors for domain={domain!r}")
    return float(np.mean(vals))


def critical_factor_audit(
    factors: Dict[str, Dict[str, float]],
    domain: str,
) -> Dict[str, Any]:
    """
    Compute the client Critical WAC_i and the strict individual critical gate.

    Direct Auto-Accept:
        every critical factor >= its own factor-specific adequacy minimum.

    Automated Review:
        uses the overall average Critical WAC_i, allowing compensation across
        critical factors while preserving the dimension and HPS floors.
    """
    domain = str(domain).lower()

    scores = {}
    minima = {}
    missing = []
    below_adequacy = []

    for dim, factor_list in CRITICAL_FACTORS_BY_DOMAIN[domain].items():
        for factor in factor_list:
            key = f"{dim}.{factor}"
            minimum = float(
                DOMAIN_FACTOR_MINIMA[domain][dim][factor]
            )
            minima[key] = minimum

            if factor not in factors.get(dim, {}):
                missing.append(key)
                continue

            value = float(factors[dim][factor])

            if not np.isfinite(value):
                missing.append(key)
                continue

            scores[key] = value

            if value < minimum:
                below_adequacy.append(
                    f"{key}:{value:.1f}<{minimum:.1f}"
                )

    expected_n = sum(
        len(v)
        for v in CRITICAL_FACTORS_BY_DOMAIN[domain].values()
    )

    if missing or len(scores) != expected_n:
        return {
            "scores": scores,
            "minima": minima,
            "missing": missing,
            "critical_count": expected_n,
            "critical_wac_i": np.nan,
            "critical_mean_score_0_5": np.nan,
            "global_critical_wac":
                float(derive_global_critical_wac(domain)),
            "all_critical_meet_adequacy": False,
            "critical_below_adequacy": below_adequacy,
        }

    values = np.array(
        list(scores.values()),
        dtype=float,
    )

    return {
        "scores": scores,
        "minima": minima,
        "missing": [],
        "critical_count": expected_n,
        "critical_wac_i":
            float(np.mean(values / MAX_FACTOR_SCORE)),
        "critical_mean_score_0_5":
            float(np.mean(values)),
        "global_critical_wac":
            float(derive_global_critical_wac(domain)),
        "all_critical_meet_adequacy":
            bool(len(below_adequacy) == 0),
        "critical_below_adequacy":
            below_adequacy,
    }

def dimension_floor_failures(
    dimensions: Dict[str, float],
    floor: float = DIMENSION_MIN_FLOOR,
) -> List[str]:
    """Return averaged HPS dimensions below the minimum floor."""
    return [
        dim for dim, value in dimensions.items()
        if (not np.isfinite(float(value))) or float(value) < float(floor)
    ]

def factor_adequacy_attainment(
    factors: Dict[str, Dict[str, float]],
    factor_minima: Dict[str, Dict[str, float]],
) -> Dict[str, Any]:
    total = 0
    passed = 0
    per_dim = {}

    for dim in FACTOR_NAMES:
        dim_total = 0
        dim_passed = 0

        for factor in FACTOR_NAMES[dim]:
            total += 1
            dim_total += 1

            score = float(factors[dim][factor])
            threshold = float(factor_minima[dim][factor])

            if score >= threshold:
                passed += 1
                dim_passed += 1

        per_dim[dim] = {
            "passed": dim_passed,
            "total": dim_total,
            "fraction": float(dim_passed / max(1, dim_total)),
        }

    return {
        "passed": passed,
        "total": total,
        "fraction": float(passed / max(1, total)),
        "per_dim": per_dim,
    }


def tadp_decision(
    factors: Dict[str, Dict[str, float]],
    domain: str,
    good_cut: float = GOOD_CUT,
    high_cut: float = HIGH_CUT,
) -> Dict[str, Any]:
    """
    v16.7 final fully automated admission policy.

    Flow:
      EVERY dimension >= 2.5?
          NO -> AUTO-REJECT

      HPS >= 3.0?
          NO -> AUTO-REJECT

      HPS >= 3.5?
          YES:
              all critical factors >= own adequacy minima?
                  YES -> DIRECT AUTO-ACCEPT
                  NO  -> AUTOMATED REVIEW
          NO (3.0 <= HPS < 3.5):
              -> AUTOMATED REVIEW

      AUTOMATED REVIEW:
          Critical WAC_i >= Global Critical WAC
              -> ACCEPT AFTER REVIEW
          otherwise
              -> AUTO-REJECT
    """
    domain = str(domain).lower()

    factor_minima = DOMAIN_FACTOR_MINIMA[domain]
    dimension_wac = DOMAIN_DIMENSION_WAC[domain]
    global_wac = float(DOMAIN_GLOBAL_WAC[domain])
    global_critical_wac = float(
        derive_global_critical_wac(domain)
    )

    dims = dimension_scores_from_factors(factors)
    hps = compute_hps(dims)

    attainment = factor_adequacy_attainment(
        factors,
        factor_minima,
    )

    zeros = all_zero_score_factors(factors)

    dim_floor_failed = dimension_floor_failures(
        dims,
        floor=DIMENSION_MIN_FLOOR,
    )

    critical = critical_factor_audit(
        factors,
        domain,
    )

    critical_score_string = ";".join(
        (
            f"{key}={value:.1f}"
            f"(min={critical['minima'][key]:.1f})"
        )
        for key, value in critical["scores"].items()
    )

    base = {
        "hps": float(hps),
        "policy_dimension_wac": dimension_wac,
        "global_domain_wac": global_wac,
        "global_critical_wac": global_critical_wac,
        "critical_wac_i": critical["critical_wac_i"],
        "critical_wac_margin": (
            float(critical["critical_wac_i"])
            - global_critical_wac
            if np.isfinite(critical["critical_wac_i"])
            else np.nan
        ),
        "critical_mean_score_0_5":
            critical["critical_mean_score_0_5"],
        "critical_factor_count":
            int(critical["critical_count"]),
        "critical_scores":
            critical_score_string,
        "critical_missing":
            ";".join(critical["missing"]),
        "all_critical_meet_adequacy":
            bool(critical["all_critical_meet_adequacy"]),
        "critical_below_adequacy":
            ";".join(critical["critical_below_adequacy"]),
        "factor_adequacy_passed":
            int(attainment["passed"]),
        "factor_adequacy_total":
            int(attainment["total"]),
        "factor_adequacy_fraction":
            float(attainment["fraction"]),
        "dimensions":
            dims,
        "dimension_min_floor":
            float(DIMENSION_MIN_FLOOR),
        "dimension_floor_failures":
            ";".join(dim_floor_failed),
        # Audit only; zero is not a separate decision rule.
        "all_zero_factors":
            ";".join(zeros),
    }

    # Gate 1: every complete trustworthiness dimension must clear 2.5/5.
    if dim_floor_failed:
        return {
            **base,
            "decision_path":
                "DIMENSION_FLOOR",
            "initial_action":
                "AUTO_REJECT",
            "final_action":
                "REJECT",
            "status":
                "AUTO_REJECTED_DIMENSION_FLOOR",
            "reason": (
                f"At least one averaged dimension is below "
                f"{DIMENSION_MIN_FLOOR:.1f}/5: "
                + ";".join(
                    f"{dim}={dims[dim]:.3f}"
                    for dim in dim_floor_failed
                )
            ),
        }

    # Missing/non-finite critical evidence is fail-closed in this experiment.
    if critical["missing"]:
        return {
            **base,
            "decision_path":
                "MISSING_CRITICAL_EVIDENCE",
            "initial_action":
                "AUTO_REJECT",
            "final_action":
                "REJECT",
            "status":
                "AUTO_REJECTED_MISSING_CRITICAL_EVIDENCE",
            "reason": (
                "Missing/non-finite verified critical evidence: "
                + ";".join(critical["missing"])
            ),
        }

    # Gate 2: overall HPS lower bound.
    if hps < float(good_cut):
        return {
            **base,
            "decision_path":
                "LOW_HPS",
            "initial_action":
                "AUTO_REJECT",
            "final_action":
                "REJECT",
            "status":
                "AUTO_REJECTED_LOW_HPS",
            "reason":
                f"HPS {hps:.3f} < {good_cut:.3f}",
        }

    # High-HPS strict direct route.
    if hps >= float(high_cut):
        if critical["all_critical_meet_adequacy"]:
            return {
                **base,
                "decision_path":
                    "DIRECT_AUTO_ACCEPT",
                "initial_action":
                    "AUTO_ACCEPT",
                "final_action":
                    "ACCEPT",
                "status":
                    "DIRECT_AUTO_ACCEPTED",
                "reason": (
                    f"HPS {hps:.3f} >= {high_cut:.3f}; all critical "
                    "factors meet their individual adequacy requirements"
                ),
            }

        # High HPS but failed strict critical gate:
        # fall back to the same automated review rather than immediate rejection.
        review_origin = (
            "HIGH_HPS_CRITICAL_FALLBACK"
        )

    else:
        # 3.0 <= HPS < 3.5
        review_origin = (
            "HPS_REVIEW_BAND"
        )

    # Automated Review for both origins.
    review_pass = bool(
        float(critical["critical_wac_i"])
        >= global_critical_wac
    )

    if review_pass:
        return {
            **base,
            "decision_path":
                review_origin,
            "initial_action":
                "AUTOMATED_REVIEW",
            "final_action":
                "ACCEPT",
            "status":
                "ACCEPTED_AFTER_AUTOMATED_REVIEW",
            "reason": (
                f"{review_origin}: Critical WAC_i "
                f"{critical['critical_wac_i']:.3f} >= "
                f"Global Critical WAC {global_critical_wac:.3f}"
            ),
        }

    return {
        **base,
        "decision_path":
            review_origin,
        "initial_action":
            "AUTOMATED_REVIEW",
        "final_action":
            "REJECT",
        "status":
            "AUTO_REJECTED_REVIEW_CRITICAL_WAC",
        "reason": (
            f"{review_origin}: Critical WAC_i "
            f"{critical['critical_wac_i']:.3f} < "
            f"Global Critical WAC {global_critical_wac:.3f}"
        ),
    }

def build_tadp_governance(
    client_ids: List[str],
    documentary_evidence: Dict[str, Dict[str, Dict[str, float]]],
    dq_scores: Dict[str, Dict[str, float]],
    run: int,
    evidence_seed: int,
    domain: str,
) -> pd.DataFrame:
    domain = str(domain).lower()
    rows = []

    for cid in client_ids:
        factors = {
            dim: dict(documentary_evidence[cid][dim])
            for dim in DOCUMENTARY_DIMS
        }

        factors["dim2"] = {
            name: float(dq_scores[cid][name])
            for name in FACTOR_NAMES["dim2"]
        }

        d = tadp_decision(
            factors,
            domain=domain,
        )

        row = {
            "run": int(run),
            "domain": domain,
            "evidence_seed": int(evidence_seed),
            "client": str(cid),
            "hps": float(d["hps"]),
            "global_domain_wac":
                float(d["global_domain_wac"]),
            "global_critical_wac":
                float(d["global_critical_wac"]),
            "critical_wac_i": (
                float(d["critical_wac_i"])
                if np.isfinite(d["critical_wac_i"])
                else np.nan
            ),
            "critical_wac_margin": (
                float(d["critical_wac_margin"])
                if np.isfinite(d["critical_wac_margin"])
                else np.nan
            ),
            "critical_mean_score_0_5": (
                float(d["critical_mean_score_0_5"])
                if np.isfinite(d["critical_mean_score_0_5"])
                else np.nan
            ),
            "critical_factor_count":
                int(d["critical_factor_count"]),
            "critical_scores":
                d["critical_scores"],
            "critical_missing":
                d["critical_missing"],
            "all_critical_meet_adequacy":
                bool(d["all_critical_meet_adequacy"]),
            "critical_below_adequacy":
                d["critical_below_adequacy"],
            "dimension_min_floor":
                float(d["dimension_min_floor"]),
            "dimension_floor_failures":
                d["dimension_floor_failures"],
            "factor_adequacy_passed":
                int(d["factor_adequacy_passed"]),
            "factor_adequacy_total":
                int(d["factor_adequacy_total"]),
            "factor_adequacy_fraction":
                float(d["factor_adequacy_fraction"]),
            "decision_path":
                d["decision_path"],
            "initial_action":
                d["initial_action"],
            "final_action":
                d["final_action"],
            "status":
                d["status"],
            "reason":
                d["reason"],
            "all_zero_factors":
                d["all_zero_factors"],
        }

        for dim, value in d["dimensions"].items():
            row[
                f"{dim}_score_0_5"
            ] = float(value)

        for dim, value in d["policy_dimension_wac"].items():
            row[
                f"{dim}_policy_wac"
            ] = float(value)

        rows.append(row)

    return pd.DataFrame(rows)

def accepted_tadp_vr(governance_df: pd.DataFrame) -> List[str]:
    return governance_df.loc[
        governance_df["final_action"].eq("ACCEPT"), "client"
    ].astype(str).tolist()


def accepted_tadp_sda(
    governance_df: pd.DataFrame
) -> List[str]:
    """
    Select one best TADP-eligible client for SDA from already accepted clients.
    """
    eligible = governance_df[
        governance_df["final_action"].eq("ACCEPT")
    ].copy()

    if eligible.empty:
        raise RuntimeError(
            "TADP-SDA cannot select a client: no TADP-eligible client."
        )

    eligible["direct_accept_priority"] = (
        eligible["status"]
        .eq("DIRECT_AUTO_ACCEPTED")
        .astype(int)
    )

    eligible = eligible.sort_values(
        [
            "hps",
            "direct_accept_priority",
            "critical_wac_i",
            "client",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    )

    return [
        str(
            eligible.iloc[0]["client"]
        )
    ]

def build_full_factor_evidence_table(
    client_ids: List[str],
    documentary_evidence: Dict[str, Dict[str, Dict[str, float]]],
    dq_scores: Dict[str, Dict[str, float]],
    dq_audit: pd.DataFrame,
    evidence_seed: int,
    domain: str,
) -> pd.DataFrame:
    """
    Full 28-factor evidence audit.

    Critical factors are flagged with their own policy adequacy minima.
    Direct Auto-Accept uses these individual minima; Automated Review uses the
    aggregate Critical WAC_i.
    """
    domain = str(domain).lower()
    factor_minima = DOMAIN_FACTOR_MINIMA[domain]
    critical_policy = CRITICAL_FACTORS_BY_DOMAIN[domain]

    dq_index = dq_audit.copy()
    dq_index["client"] = dq_index["client"].astype(str)
    dq_index = dq_index.set_index(
        "client",
        drop=False,
    )

    def policy_fields(dim, factor):
        is_critical = bool(
            factor in critical_policy.get(
                dim,
                [],
            )
        )

        return {
            "is_critical_factor":
                is_critical,
            "critical_adequacy_min_rank": (
                float(factor_minima[dim][factor])
                if is_critical
                else np.nan
            ),
            "critical_adequacy_min_normalized": (
                float(factor_minima[dim][factor])
                / MAX_FACTOR_SCORE
                if is_critical
                else np.nan
            ),
        }

    rows = []

    for cid in client_ids:
        cid = str(cid)

        for dim in DOCUMENTARY_DIMS:
            for factor in FACTOR_NAMES[dim]:
                score = float(
                    documentary_evidence[cid][dim][factor]
                )
                min_rank = float(
                    factor_minima[dim][factor]
                )

                rows.append({
                    "client": cid,
                    "domain": domain,
                    "dimension": dim,
                    "dimension_name":
                        DIMENSION_NAMES[dim],
                    "factor": factor,
                    "factor_source":
                        "CONTROLLED_DOCUMENTARY_EVIDENCE",
                    "raw_measured_value":
                        np.nan,
                    "rubric_score_0_5":
                        score,
                    "rubric_descriptor":
                        rubric_descriptor(
                            dim,
                            factor,
                            score,
                        ),
                    "adequacy_min_rank":
                        min_rank,
                    "adequacy_min_normalized":
                        min_rank / MAX_FACTOR_SCORE,
                    "meets_factor_adequacy":
                        bool(score >= min_rank),
                    **policy_fields(
                        dim,
                        factor,
                    ),
                    "evidence_seed":
                        int(evidence_seed),
                })

        for factor in FACTOR_NAMES["dim2"]:
            score = float(
                dq_scores[cid][factor]
            )
            min_rank = float(
                factor_minima["dim2"][factor]
            )
            raw_col = DQ_RAW_METRIC_BY_FACTOR[
                factor
            ]

            raw_value = (
                float(
                    dq_index.loc[
                        cid,
                        raw_col,
                    ]
                )
                if raw_col in dq_index.columns
                else np.nan
            )

            rows.append({
                "client": cid,
                "domain": domain,
                "dimension": "dim2",
                "dimension_name":
                    DIMENSION_NAMES["dim2"],
                "factor": factor,
                "factor_source":
                    "MACHINE_MEASURED_TRAIN_ONLY",
                "raw_measured_value":
                    raw_value,
                "rubric_score_0_5":
                    score,
                "rubric_descriptor":
                    rubric_descriptor(
                        "dim2",
                        factor,
                        score,
                    ),
                "adequacy_min_rank":
                    min_rank,
                "adequacy_min_normalized":
                    min_rank / MAX_FACTOR_SCORE,
                "meets_factor_adequacy":
                    bool(score >= min_rank),
                **policy_fields(
                    "dim2",
                    factor,
                ),
                "evidence_seed":
                    int(evidence_seed),
            })

    out = pd.DataFrame(
        rows
    )

    expected_rows = len(client_ids) * 28

    if len(out) != expected_rows:
        raise RuntimeError(
            f"Full evidence matrix should contain "
            f"{expected_rows} rows; found {len(out)}."
        )

    return out

# ======================================================================================
# GREAT EXPECTATIONS (GX CORE) — NATIVE VALIDATOR BASELINE
# ======================================================================================
# GX is used here in its native role: validate each client's TRAIN-only data against
# a predefined Expectation Suite and use the suite-level success flag as PASS/FAIL.
# There is NO ranking, NO forced-K selection, and NO HPS/WAC information in this
# baseline. Great Expectations reports suite success=True only when all configured
# Expectations pass. The suite deliberately uses common technical checks only:
#   1) schema, 2) datatype consistency, 3) required-value ranges,
#   4) missingness, 5) duplicates / unique IDs, 6) label/domain validity,
#   7) structural integrity.
# Distribution checks are intentionally omitted because non-IID client distributions
# are expected in federated learning and should not by themselves constitute failure.
GX_CORE_VERSION = "1.23.0"
GX_VALUE_MOSTLY = 0.99
GX_NONNULL_REFERENCE_MIN = 0.80
GX_NONNULL_TOLERANCE = 0.15


def ensure_great_expectations():
    """Import pinned GX Core; install once in Colab if unavailable."""
    try:
        import great_expectations as gx
        return gx
    except ImportError:
        import subprocess
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            f"great_expectations=={GX_CORE_VERSION}",
        ])
        import great_expectations as gx
        return gx


def _gx_meta(category: str, name: str) -> Dict[str, Any]:
    return {
        "dq_category": str(category),
        "check_name": str(name),
        "baseline": "GX_NATIVE_VALIDATOR",
    }


def _gx_add(suite, specs: List[Dict[str, Any]], expectation, category: str, name: str):
    suite.add_expectation(expectation)
    specs.append({"category": str(category), "name": str(name)})


def _aggregate_nonnull_reference(
    client_frames: Dict[str, pd.DataFrame],
    columns: List[str],
) -> Dict[str, float]:
    total_rows = float(sum(len(df) for df in client_frames.values()))
    out = {}
    for c in columns:
        nonnull = sum(int(df[c].notna().sum()) for df in client_frames.values() if c in df.columns)
        out[c] = float(nonnull / max(1.0, total_rows))
    return out


def build_gx_healthcare_reference(client_frames: Dict[str, pd.DataFrame], preprocessor: Any) -> Dict[str, Any]:
    first = next(iter(client_frames.values()))

    # GX datatype validation is intentionally independent of TADP's model
    # preprocessing type inference.  The preprocessor labels a feature numeric
    # when >=95% of pooled TRAIN non-missing values are parseable; reusing that
    # inferred list inside GX and then requiring 99% parseability per client
    # creates an artificial contradiction.  GX therefore validates datatype
    # consistency only for fields whose numeric meaning is explicit in the
    # Diabetes data schema.
    known_numeric_fields = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]
    gx_numeric_cols = [c for c in known_numeric_fields if c in first.columns]

    return {
        "original_columns": list(first.columns),
        "numeric_cols": gx_numeric_cols,
        "feature_cols": list(preprocessor.feature_cols),
        "nonnull_reference": _aggregate_nonnull_reference(
            client_frames, list(preprocessor.feature_cols)
        ),
    }


def build_gx_healthcare_validation_frame(df: pd.DataFrame, preprocessor: Any) -> pd.DataFrame:
    """GX validation view; raw TRAIN records remain the source of all checks."""
    out = df.copy()
    for c in preprocessor.numeric_cols:
        raw = df[c]
        parsed = pd.to_numeric(raw, errors="coerce")
        type_ok = raw.isna() | parsed.notna()
        out[c] = parsed.astype(float)
        out[f"__gx_type_ok__{c}"] = type_ok.astype(np.int8)
    return out


def build_gx_healthcare_suite(gx, reference: Dict[str, Any]):
    """Simple native GX technical-validation suite for the Diabetes TRAIN shards."""
    gxe = gx.expectations
    suite = gx.ExpectationSuite(name="tadp_healthcare_gx_native_suite")
    specs = []
    cols = reference["original_columns"]

    # 1) Schema.
    _gx_add(
        suite, specs,
        gxe.ExpectTableColumnsToMatchSet(
            column_set=cols,
            exact_match=False,
            severity="critical",
            meta=_gx_meta("Schema", "required_column_set"),
        ),
        "Schema", "required_column_set",
    )

    # 2) Datatype consistency for columns inferred as numeric from TRAIN only.
    for c in reference["numeric_cols"]:
        diag = f"__gx_type_ok__{c}"
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=diag,
                value_set=[1],
                mostly=GX_VALUE_MOSTLY,
                severity="critical",
                meta=_gx_meta("Datatype Consistency", f"{c}_numeric_parseability"),
            ),
            "Datatype Consistency", f"{c}_numeric_parseability",
        )

    # 3) Required-value ranges for known count/duration fields.
    nonnegative_fields = [
        "num_lab_procedures", "num_procedures", "num_medications",
        "number_outpatient", "number_emergency", "number_inpatient",
        "number_diagnoses",
    ]
    for c in nonnegative_fields:
        if c in cols:
            _gx_add(
                suite, specs,
                gxe.ExpectColumnValuesToBeBetween(
                    column=c, min_value=0.0, mostly=GX_VALUE_MOSTLY,
                    severity="critical",
                    meta=_gx_meta("Required Value Ranges", f"{c}_nonnegative"),
                ),
                "Required Value Ranges", f"{c}_nonnegative",
            )
    if "time_in_hospital" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeBetween(
                column="time_in_hospital", min_value=1.0, max_value=14.0,
                mostly=GX_VALUE_MOSTLY,
                severity="critical",
                meta=_gx_meta("Required Value Ranges", "time_in_hospital_range"),
            ),
            "Required Value Ranges", "time_in_hospital_range",
        )

    # 4) Missingness. Only columns that are substantially populated in the frozen
    # TRAIN reference are treated as required-enough for a missingness expectation.
    for c, global_nonnull in reference["nonnull_reference"].items():
        if c not in cols or float(global_nonnull) < GX_NONNULL_REFERENCE_MIN:
            continue
        minimum = max(0.70, float(global_nonnull) - GX_NONNULL_TOLERANCE)
        _gx_add(
            suite, specs,
            gxe.ExpectColumnProportionOfNonNullValuesToBeBetween(
                column=c, min_value=float(minimum), max_value=1.0,
                severity="warning",
                meta=_gx_meta("Missingness", f"{c}_nonnull"),
            ),
            "Missingness", f"{c}_nonnull",
        )

    # 5) Duplicates / unique identifiers.
    if "encounter_id" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeUnique(
                column="encounter_id",
                severity="critical",
                meta=_gx_meta("Duplicates / Unique IDs", "encounter_id_unique"),
            ),
            "Duplicates / Unique IDs", "encounter_id_unique",
        )

    # 6) Labels / domain validity.
    if "_target" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column="_target", value_set=[0, 1, 2],
                severity="critical",
                meta=_gx_meta("Labels / Domain Validity", "target_domain"),
            ),
            "Labels / Domain Validity", "target_domain",
        )

    # 7) Structural integrity.
    if "_row_id" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeUnique(
                column="_row_id",
                severity="critical",
                meta=_gx_meta("Structural Integrity", "row_id_unique"),
            ),
            "Structural Integrity", "row_id_unique",
        )
    _gx_add(
        suite, specs,
        gxe.ExpectTableRowCountToBeBetween(
            min_value=100,
            severity="warning",
            meta=_gx_meta("Structural Integrity", "minimum_client_rows"),
        ),
        "Structural Integrity", "minimum_client_rows",
    )
    return suite, specs


def build_gx_cifar_metadata(X: np.ndarray, y: np.ndarray) -> pd.DataFrame:
    X = np.asarray(X)
    y = np.asarray(y).reshape(-1)
    if X.ndim != 4:
        raise RuntimeError(f"Expected CIFAR tensor [N,H,W,C], got shape={X.shape}")
    finite = np.isfinite(X.astype(np.float32)).reshape(len(X), -1).all(axis=1)
    flat = X.reshape(len(X), -1)
    return pd.DataFrame({
        "sample_id": np.arange(len(X), dtype=np.int64),
        "label": y.astype(np.int32),
        "height": np.full(len(X), X.shape[1], dtype=np.int32),
        "width": np.full(len(X), X.shape[2], dtype=np.int32),
        "channels": np.full(len(X), X.shape[3], dtype=np.int32),
        "dtype_ok": np.full(len(X), int(X.dtype == np.uint8), dtype=np.int8),
        "finite": finite.astype(np.int8),
        "pixel_min": flat.min(axis=1).astype(float),
        "pixel_max": flat.max(axis=1).astype(float),
    })


def build_gx_cifar_reference(client_raw: Dict[str, Tuple[np.ndarray, np.ndarray]]) -> Dict[str, Any]:
    # Native validator uses fixed technical constraints; no distribution reference needed.
    return {}


def build_gx_cifar_suite(gx, reference: Dict[str, Any]):
    """Simple native GX technical-validation suite for CIFAR-10 client metadata."""
    gxe = gx.expectations
    suite = gx.ExpectationSuite(name="tadp_cifar10_gx_native_suite")
    specs = []
    columns = [
        "sample_id", "label", "height", "width", "channels",
        "dtype_ok", "finite", "pixel_min", "pixel_max",
    ]

    _gx_add(
        suite, specs,
        gxe.ExpectTableColumnsToMatchSet(
            column_set=columns, exact_match=True,
            meta=_gx_meta("Schema", "image_metadata_schema"),
        ),
        "Schema", "image_metadata_schema",
    )
    for c in columns:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToNotBeNull(
                column=c,
                severity="warning",
                meta=_gx_meta("Missingness", f"{c}_not_null"),
            ),
            "Missingness", f"{c}_not_null",
        )
    for c, value in [("height", 32), ("width", 32), ("channels", 3), ("dtype_ok", 1), ("finite", 1)]:
        category = "Datatype Consistency" if c in {"dtype_ok", "finite"} else "Structural Integrity"
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=c, value_set=[value],
                meta=_gx_meta(category, f"{c}_constraint"),
            ),
            category, f"{c}_constraint",
        )
    for c in ["pixel_min", "pixel_max"]:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeBetween(
                column=c, min_value=0.0, max_value=255.0,
                meta=_gx_meta("Required Value Ranges", f"{c}_valid_range"),
            ),
            "Required Value Ranges", f"{c}_valid_range",
        )
    _gx_add(
        suite, specs,
        gxe.ExpectColumnValuesToBeInSet(
            column="label", value_set=list(range(10)),
            meta=_gx_meta("Labels / Domain Validity", "label_domain"),
        ),
        "Labels / Domain Validity", "label_domain",
    )
    _gx_add(
        suite, specs,
        gxe.ExpectColumnValuesToBeUnique(
            column="sample_id",
            meta=_gx_meta("Duplicates / Unique IDs", "sample_id_unique"),
        ),
        "Duplicates / Unique IDs", "sample_id_unique",
    )
    _gx_add(
        suite, specs,
        gxe.ExpectTableRowCountToBeBetween(
            min_value=100,
            severity="warning",
            meta=_gx_meta("Structural Integrity", "minimum_client_images"),
        ),
        "Structural Integrity", "minimum_client_images",
    )
    return suite, specs


def ge_governance(
    client_data: Dict[str, Any],
    client_ids: List[str],
    accept_count: Optional[int] = None,  # ignored; retained only for call compatibility
    domain: str = "healthcare",
    preprocessor: Optional[Any] = None,
    dq_scores: Optional[Dict[str, Dict[str, float]]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run native GX suite validation and PASS only clients whose entire suite succeeds."""
    domain = str(domain).lower()
    gx = ensure_great_expectations()
    context = gx.get_context(mode="ephemeral")
    datasource = context.data_sources.add_pandas(name=f"tadp_gx_native_{domain}_datasource")
    asset = datasource.add_dataframe_asset(name=f"tadp_gx_native_{domain}_asset")
    batch_definition = asset.add_batch_definition_whole_dataframe(name="whole_client_train_partition")

    if domain == "healthcare":
        if preprocessor is None:
            raise ValueError("Healthcare GX native baseline requires TRAIN-only preprocessor metadata.")
        reference = build_gx_healthcare_reference(client_data, preprocessor)
        suite, specs = build_gx_healthcare_suite(gx, reference)
        make_frame = lambda cid: build_gx_healthcare_validation_frame(client_data[cid], preprocessor)
    elif domain == "cifar10":
        reference = build_gx_cifar_reference(client_data)
        suite, specs = build_gx_cifar_suite(gx, reference)
        def make_frame(cid):
            X, y = client_data[cid]
            return build_gx_cifar_metadata(X, y)
    else:
        raise ValueError(f"Unsupported GX domain: {domain!r}")

    summary_rows, detail_rows = [], []
    for cid in client_ids:
        frame = make_frame(cid)
        batch = batch_definition.get_batch(batch_parameters={"dataframe": frame})
        validation = batch.validate(suite)
        results = list(validation.results)
        if len(results) != len(specs):
            raise RuntimeError(
                f"GX result/spec mismatch for client {cid}: {len(results)} vs {len(specs)}"
            )

        passed = 0
        critical_failures = 0
        warning_failures = 0
        info_failures = 0
        category_totals, category_passed = {}, {}
        observed_names = []

        for result in results:
            success = bool(result.success)
            passed += int(success)
            cfg = result.expectation_config
            meta = getattr(cfg, "meta", None) or {}
            name = str(meta.get("check_name", getattr(cfg, "type", "GX_EXPECTATION")))
            cat = str(meta.get("dq_category", "Technical Validation"))

            sev_obj = getattr(cfg, "severity", None)
            sev = getattr(sev_obj, "value", sev_obj)
            sev = str(sev if sev is not None else "critical").lower()
            if "." in sev:
                sev = sev.split(".")[-1]
            if sev not in {"critical", "warning", "info"}:
                sev = "critical"

            if not success:
                if sev == "critical":
                    critical_failures += 1
                elif sev == "warning":
                    warning_failures += 1
                else:
                    info_failures += 1

            observed_names.append(name)
            category_totals[cat] = category_totals.get(cat, 0) + 1
            category_passed[cat] = category_passed.get(cat, 0) + int(success)

            detail_rows.append({
                "client": str(cid),
                "domain": domain,
                "gx_version": GX_CORE_VERSION,
                "expectation_name": name,
                "dq_category": cat,
                "severity": sev,
                "success": success,
            })

        expected_names = sorted(str(x["name"]) for x in specs)
        if sorted(observed_names) != expected_names:
            raise RuntimeError(
                f"GX expectation identity mismatch for client {cid}: "
                f"expected={expected_names}, observed={sorted(observed_names)}"
            )

        total = len(specs)
        stats = getattr(validation, "statistics", {}) or {}
        suite_success = bool(validation.success)

        # GX-native severity-aware operational gate.
        # GX itself exposes the maximum failed severity for a Validation Result.
        # We use that native result rather than reconstructing the gate from a
        # custom ranking.  A failed Expectation execution is also treated by GX
        # as CRITICAL.
        max_failed_severity_obj = validation.get_max_severity_failure()
        if max_failed_severity_obj is None:
            max_failed_severity = "none"
        else:
            max_failed_severity = str(
                getattr(max_failed_severity_obj, "value", max_failed_severity_obj)
            ).lower()
            if "." in max_failed_severity:
                max_failed_severity = max_failed_severity.split(".")[-1]

        operational_valid = (max_failed_severity != "critical")

        row = {
            "client": str(cid),
            "gx_version": GX_CORE_VERSION,
            "gx_native_suite_success": suite_success,
            "gx_operational_valid": bool(operational_valid),
            "gx_critical_failures": int(critical_failures),
            "gx_warning_failures": int(warning_failures),
            "gx_info_failures": int(info_failures),
            "gx_max_failed_severity": max_failed_severity,
            "ge_expectations_passed": int(stats.get("successful_expectations", passed)),
            "ge_expectations_total": int(stats.get("evaluated_expectations", total)),
            "ge_pass_rate": float(
                stats.get("success_percent", 100.0 * passed / max(1, total))
            ) / 100.0,
            "ge_native_validation_class": (
                "GX_SUITE_PASS"
                if suite_success
                else (
                    "GX_WARNING_ONLY"
                    if operational_valid
                    else "GX_CRITICAL_FAILURE"
                )
            ),
            "ge_final_action": "ACCEPT" if operational_valid else "REJECT",
            "ge_policy": (
                "GX Core severity-aware validation; ACCEPT requires zero critical "
                "Expectation failures; warning/info failures are reported but do not "
                "exclude; no ranking and no forced-K selection"
            ),
        }

        for cat in sorted(category_totals):
            safe = re.sub(r"[^a-z0-9]+", "_", cat.lower()).strip("_")
            row[f"ge_{safe}_passed"] = int(category_passed.get(cat, 0))
            row[f"ge_{safe}_total"] = int(category_totals[cat])

        summary_rows.append(row)

    return pd.DataFrame(summary_rows), pd.DataFrame(detail_rows)


def score_lower_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [best_upper, score4_upper, score3_upper, score2_upper, score1_upper]
    value <= cuts[0] => 5; ... value <= cuts[4] => 1; else 0.
    """
    v = float(value)
    for score, upper in zip([5, 4, 3, 2, 1], cuts):
        if v <= float(upper):
            return float(score)
    return 0.0


def score_higher_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [score5_lower, score4_lower, score3_lower, score2_lower, score1_lower]
    """
    v = float(value)
    for score, lower in zip([5, 4, 3, 2, 1], cuts):
        if v >= float(lower):
            return float(score)
    return 0.0


def js_divergence(p, q, eps=1e-12) -> float:
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / max(eps, p.sum())
    q = q / max(eps, q.sum())
    m = 0.5 * (p + q)
    kl_pm = np.sum(np.where(p > 0, p * np.log((p + eps) / (m + eps)), 0.0))
    kl_qm = np.sum(np.where(q > 0, q * np.log((q + eps) / (m + eps)), 0.0))
    return float(0.5 * (kl_pm + kl_qm))


# ======================================================================================
# MODEL / METRICS / FL TRAINING
# ======================================================================================

def class_weight_dict(y: np.ndarray) -> Dict[int, float]:
    y = np.asarray(y, dtype=np.int32)
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(w) for c, w in zip(classes, weights)}


def model_parameter_bytes(model: keras.Model) -> int:
    return int(sum(np.asarray(w).nbytes for w in model.get_weights()))


def evaluate_model(
    model: keras.Model, X: np.ndarray, y: np.ndarray, n_classes: int
) -> Dict[str, float]:
    p = model.predict(X, batch_size=512, verbose=0)
    pred = np.argmax(p, axis=1)
    out = {
        "accuracy": float(accuracy_score(y, pred)),
        "precision_macro": float(
            precision_score(y, pred, average="macro", zero_division=0)
        ),
        "recall_macro": float(
            recall_score(y, pred, average="macro", zero_division=0)
        ),
        "f1_macro": float(
            f1_score(y, pred, average="macro", zero_division=0)
        ),
    }
    try:
        out["roc_auc_ovr_macro"] = float(
            roc_auc_score(y, p, multi_class="ovr", average="macro")
        )
    except Exception:
        out["roc_auc_ovr_macro"] = float("nan")
    return out



ROUND_PROGRESS_MONITOR_MAX_SAMPLES = 4096
ROUND_PROGRESS_POWER_W = 12.0


def build_train_monitor_subset(
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    max_samples: int = ROUND_PROGRESS_MONITOR_MAX_SAMPLES,
    seed: int = 99117,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Build a deterministic lightweight monitoring subset from TRAIN client arrays only.

    This subset is used solely for console progress after FL rounds. It never affects
    preprocessing, governance, client selection, optimizer budgets, checkpoint
    decisions, or final reporting. The held-out TEST set remains final-evaluation-only.
    """
    rng = np.random.default_rng(int(seed))
    client_ids = sorted(client_arrays.keys())
    sizes = {cid: int(len(client_arrays[cid][1])) for cid in client_ids}
    total = int(sum(sizes.values()))
    target = int(min(max_samples, total))

    # Proportional allocation across clients, then distribute rounding remainder.
    raw = {cid: target * sizes[cid] / max(1, total) for cid in client_ids}
    take = {cid: min(sizes[cid], int(np.floor(raw[cid]))) for cid in client_ids}

    while sum(take.values()) < target:
        candidates = [c for c in client_ids if take[c] < sizes[c]]
        if not candidates:
            break
        cid = max(candidates, key=lambda c: (raw[c] - take[c], sizes[c], c))
        take[cid] += 1

    xs, ys = [], []
    for cid in client_ids:
        n_take = int(take[cid])
        if n_take <= 0:
            continue
        Xc, yc = client_arrays[cid]
        if n_take >= len(yc):
            idx = np.arange(len(yc))
        else:
            idx = rng.choice(len(yc), size=n_take, replace=False)
        xs.append(np.asarray(Xc[idx]))
        ys.append(np.asarray(yc[idx], dtype=np.int32))

    Xmon = np.concatenate(xs, axis=0)
    ymon = np.concatenate(ys, axis=0)

    # Deterministic shuffle so monitoring batches do not follow client order.
    order = rng.permutation(len(ymon))
    return Xmon[order], ymon[order]


def _fmt_metric(x: float) -> str:
    return "nan" if not np.isfinite(float(x)) else f"{float(x):.4f}"


def print_round_progress(
    progress_context: Optional[Dict[str, Any]],
    round_idx: int,
    round_total: int,
    selected_ids: List[str],
    round_steps: int,
    cumulative_steps: int,
    monitor_metrics: Dict[str, float],
    cumulative_runtime_s: float,
    cumulative_communication_mb: float,
    ram_start_mb: float,
    ram_peak_mb: float,
    extra: str = "",
) -> None:
    """Compact, human-readable progress block after each federated round."""
    ctx = progress_context or {}
    scenario = str(ctx.get("scenario", "Federated scenario"))
    run_idx = int(ctx.get("run_idx", 0))
    run_total = int(ctx.get("run_total", 0))
    scenario_idx = int(ctx.get("scenario_idx", 0))
    scenario_total = int(ctx.get("scenario_total", 0))
    overall_idx = int(ctx.get("overall_idx", 0))
    overall_total = int(ctx.get("overall_total", 0))

    energy_wh = float(
        ROUND_PROGRESS_POWER_W * float(cumulative_runtime_s) / 3600.0
    )
    ram_delta = max(0.0, float(ram_peak_mb) - float(ram_start_mb))

    print("\n" + "-" * 112)
    print(
        f"PROGRESS | overall configuration {overall_idx}/{overall_total} | "
        f"run {run_idx}/{run_total} | scenario {scenario_idx}/{scenario_total}"
    )
    print(f"SCENARIO | {scenario}")
    print(
        f"ROUND    | {round_idx}/{round_total} | "
        f"selected={len(selected_ids)} [{','.join(map(str, selected_ids))}] | "
        f"steps={round_steps} | cumulative_steps={cumulative_steps}"
    )
    if extra:
        print(f"DETAIL   | {extra}")
    print(
        "TRAIN-MONITOR (diagnostic only; TEST untouched) | "
        f"Accuracy={_fmt_metric(monitor_metrics.get('accuracy', np.nan))} | "
        f"F1={_fmt_metric(monitor_metrics.get('f1_macro', np.nan))} | "
        f"AUC={_fmt_metric(monitor_metrics.get('roc_auc_ovr_macro', np.nan))} | "
        f"Precision={_fmt_metric(monitor_metrics.get('precision_macro', np.nan))} | "
        f"Recall={_fmt_metric(monitor_metrics.get('recall_macro', np.nan))}"
    )
    print(
        f"CUMULATIVE OPERATIONAL | runtime={cumulative_runtime_s:.2f}s | "
        f"energy≈{energy_wh:.5f}Wh | communication={cumulative_communication_mb:.3f}MB | "
        f"RAM peak={ram_peak_mb:.1f}MB | RAM Δ={ram_delta:.1f}MB"
    )
    print("-" * 112)


def train_exact_steps(
    model: keras.Model,
    X: np.ndarray,
    y: np.ndarray,
    steps: int,
    batch_size: int,
    seed: int,
    class_weights: Optional[Dict[int, float]] = None,
    prox_reference: Optional[List[np.ndarray]] = None,
    prox_mu: float = 0.0,
):
    """
    Exact mini-batch update count. Used for step-parity audits.
    """
    steps = int(max(1, steps))
    rng = np.random.default_rng(int(seed))
    n = len(y)
    if n == 0:
        raise RuntimeError("Cannot train on an empty dataset.")

    loss_fn = keras.losses.SparseCategoricalCrossentropy(
        reduction=keras.losses.Reduction.NONE
    )

    order = rng.permutation(n)
    cursor = 0

    prox_tensors = None
    if prox_reference is not None and prox_mu > 0:
        prox_tensors = [tf.convert_to_tensor(w) for w in prox_reference]

    for _ in range(steps):
        if cursor + batch_size > n:
            order = rng.permutation(n)
            cursor = 0

        idx = order[cursor:cursor + batch_size]
        cursor += batch_size

        xb = tf.convert_to_tensor(np.asarray(X[idx]), dtype=tf.float32)
        yb_np = np.asarray(y[idx], dtype=np.int32)
        yb = tf.convert_to_tensor(yb_np, dtype=tf.int32)

        with tf.GradientTape() as tape:
            probs = model(xb, training=True)
            per_loss = loss_fn(yb, probs)

            if class_weights:
                sw = np.array(
                    [class_weights.get(int(v), 1.0) for v in yb_np],
                    dtype=np.float32,
                )
                sw_t = tf.convert_to_tensor(sw)
                data_loss = tf.reduce_sum(per_loss * sw_t) / tf.reduce_sum(sw_t)
            else:
                data_loss = tf.reduce_mean(per_loss)

            loss = data_loss

            if prox_tensors is not None:
                prox = tf.constant(0.0, dtype=tf.float32)
                for var, ref in zip(model.trainable_variables, prox_tensors):
                    prox += tf.reduce_sum(tf.square(var - tf.cast(ref, var.dtype)))
                loss = loss + 0.5 * float(prox_mu) * prox

        grads = tape.gradient(loss, model.trainable_variables)
        model.optimizer.apply_gradients(zip(grads, model.trainable_variables))


def natural_steps(n_records: int, batch_size: int, local_epochs: int = 1) -> int:
    return int(max(1, math.ceil(int(n_records) / int(batch_size)) * int(local_epochs)))


def allocate_exact_step_budget(
    selected: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    target_total: int,
    batch_size: int,
    local_epochs: int = 1,
) -> Dict[str, int]:
    natural = {
        cid: natural_steps(len(client_arrays[cid][1]), batch_size, local_epochs)
        for cid in selected
    }
    total_nat = max(1, sum(natural.values()))
    raw = {cid: target_total * natural[cid] / total_nat for cid in selected}
    alloc = {cid: max(1, int(math.floor(raw[cid]))) for cid in selected}

    # Adjust to exact target.
    while sum(alloc.values()) < target_total:
        cid = max(selected, key=lambda c: raw[c] - alloc[c])
        alloc[cid] += 1
    while sum(alloc.values()) > target_total:
        candidates = [c for c in selected if alloc[c] > 1]
        if not candidates:
            break
        cid = min(candidates, key=lambda c: raw[c] - alloc[c])
        alloc[cid] -= 1

    if sum(alloc.values()) != int(target_total):
        raise RuntimeError("Exact step-budget allocation failed.")
    return alloc


def aggregate_weights(
    local_weights: List[List[np.ndarray]],
    sample_sizes: List[int],
    equal_weight: bool = False,
) -> List[np.ndarray]:
    if not local_weights:
        raise RuntimeError("No local weights to aggregate.")
    if equal_weight:
        alpha = np.ones(len(local_weights), dtype=float) / len(local_weights)
    else:
        sizes = np.asarray(sample_sizes, dtype=float)
        alpha = sizes / sizes.sum()

    out = []
    for layer_idx in range(len(local_weights[0])):
        x = sum(alpha[j] * np.asarray(local_weights[j][layer_idx])
                for j in range(len(local_weights)))
        out.append(np.asarray(x))
    return out


def federated_train(
    build_model_fn,
    initial_weights: List[np.ndarray],
    selected_per_round: List[List[str]],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    local_epochs: int,
    class_weights: Dict[int, float],
    run_seed: int,
    exact_step_maps: Optional[List[Dict[str, int]]] = None,
    equal_weight: bool = False,
    fedprox_mu: float = 0.0,
    train_monitor: Optional[Tuple[np.ndarray, np.ndarray]] = None,
    progress_context: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    monitor = RAMMonitor().start()
    start = time.perf_counter()

    global_model = build_model_fn()
    global_model.set_weights([np.array(w, copy=True) for w in initial_weights])

    round_rows = []
    total_steps = 0
    total_selected = 0
    monitor_eval_s = 0.0
    cumulative_comm_raw_b = 0
    param_b = model_parameter_bytes(global_model)

    for r, selected in enumerate(selected_per_round, start=1):
        global_weights = [np.array(w, copy=True) for w in global_model.get_weights()]
        local_weights = []
        local_sizes = []
        round_steps = 0

        if exact_step_maps is None:
            step_map = {
                cid: natural_steps(
                    len(client_arrays[cid][1]), batch_size, local_epochs
                )
                for cid in selected
            }
        else:
            step_map = exact_step_maps[r - 1]

        for j, cid in enumerate(selected):
            Xc, yc = client_arrays[cid]
            local_model = build_model_fn()
            local_model.set_weights(global_weights)
            train_exact_steps(
                local_model, Xc, yc,
                steps=int(step_map[cid]),
                batch_size=batch_size,
                seed=int(run_seed + 1000 * r + 17 * j),
                class_weights=class_weights,
                prox_reference=global_weights if fedprox_mu > 0 else None,
                prox_mu=float(fedprox_mu),
            )
            local_weights.append(
                [np.array(w, copy=True) for w in local_model.get_weights()]
            )
            local_sizes.append(len(yc))
            round_steps += int(step_map[cid])

            del local_model
            tf.keras.backend.clear_session()

        agg = aggregate_weights(
            local_weights, local_sizes, equal_weight=equal_weight
        )
        global_model = build_model_fn()
        global_model.set_weights(agg)

        total_steps += round_steps
        total_selected += len(selected)

        # Cumulative model traffic through this round. Same accounting as the
        # final experiment metric: download + upload + 12% protocol overhead.
        cumulative_comm_raw_b += int(len(selected)) * 2 * int(param_b)
        cumulative_comm_mb = float(
            (cumulative_comm_raw_b * 1.12) / (1024 ** 2)
        )

        # Measure training runtime BEFORE this round's diagnostic evaluation.
        # Previous diagnostic-evaluation time is subtracted so progress printing
        # does not inflate the experiment's runtime metric.
        cumulative_runtime_s = float(
            max(0.0, (time.perf_counter() - start) - monitor_eval_s)
        )

        if train_monitor is not None:
            Xmon, ymon = train_monitor
            t_mon = time.perf_counter()
            monitor_metrics = evaluate_model(global_model, Xmon, ymon, n_classes)
            monitor_eval_s += float(time.perf_counter() - t_mon)
        else:
            monitor_metrics = {
                "accuracy": float("nan"),
                "precision_macro": float("nan"),
                "recall_macro": float("nan"),
                "f1_macro": float("nan"),
                "roc_auc_ovr_macro": float("nan"),
            }

        live_peak = float(monitor.peak_mb)
        round_row = {
            "round": r,
            "selected_clients": len(selected),
            "selected_ids": ";".join(selected),
            "optimizer_steps": int(round_steps),
            "cumulative_optimizer_steps": int(total_steps),
            "train_monitor_accuracy": float(monitor_metrics["accuracy"]),
            "train_monitor_precision_macro": float(monitor_metrics["precision_macro"]),
            "train_monitor_recall_macro": float(monitor_metrics["recall_macro"]),
            "train_monitor_f1_macro": float(monitor_metrics["f1_macro"]),
            "train_monitor_auc_ovr_macro": float(monitor_metrics["roc_auc_ovr_macro"]),
            "cumulative_runtime_s": float(cumulative_runtime_s),
            "cumulative_energy_wh_est": float(
                ROUND_PROGRESS_POWER_W * cumulative_runtime_s / 3600.0
            ),
            "cumulative_communication_mb": float(cumulative_comm_mb),
            "ram_peak_mb_live": float(live_peak),
            "ram_delta_mb_live": float(max(0.0, live_peak - monitor.start_mb)),
        }
        round_rows.append(round_row)

        print_round_progress(
            progress_context=progress_context,
            round_idx=r,
            round_total=len(selected_per_round),
            selected_ids=list(selected),
            round_steps=int(round_steps),
            cumulative_steps=int(total_steps),
            monitor_metrics=monitor_metrics,
            cumulative_runtime_s=cumulative_runtime_s,
            cumulative_communication_mb=cumulative_comm_mb,
            ram_start_mb=float(monitor.start_mb),
            ram_peak_mb=live_peak,
        )

    runtime = float(max(0.0, (time.perf_counter() - start) - monitor_eval_s))
    metrics = evaluate_model(global_model, X_test, y_test, n_classes)
    ram = monitor.stop()

    communication_b = int(cumulative_comm_raw_b * 1.12)

    return {
        "model": global_model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(total_steps),
        "communication_mb": float(communication_b / (1024 ** 2)),
        "round_audit": round_rows,
        "participants_mean_per_round": float(
            total_selected / max(1, len(round_rows))
        ),
        **ram,
    }



def select_dq_only_clients(
    dq_scores: Dict[str, Dict[str, float]],
    k: int,
) -> Tuple[List[str], pd.DataFrame]:
    """Select top-K clients by the machine-measured TADP Data Quality dimension only."""
    rows = []
    for cid, scores in dq_scores.items():
        vals = [float(scores[f]) for f in FACTOR_NAMES["dim2"]]
        rows.append({
            "client": str(cid),
            "dq_only_score": float(np.mean(vals)),
            "dq_factor_count": int(len(vals)),
        })
    audit = pd.DataFrame(rows).sort_values(
        ["dq_only_score", "client"], ascending=[False, True]
    ).reset_index(drop=True)
    audit["dq_only_rank"] = np.arange(1, len(audit) + 1)
    selected = audit.head(int(k))["client"].astype(str).tolist()
    audit["dq_only_selected"] = audit["client"].isin(selected)
    return selected, audit


def local_training_loss(model: keras.Model, X: np.ndarray, y: np.ndarray, batch_size: int = 512) -> float:
    """Mean sparse cross-entropy on client TRAIN data only; used by Power-of-Choice."""
    p = model.predict(X, batch_size=batch_size, verbose=0)
    y = np.asarray(y, dtype=np.int32)
    idx = np.arange(len(y))
    probs = np.clip(p[idx, y], 1e-12, 1.0)
    return float(-np.mean(np.log(probs)))


def federated_train_power_of_choice(
    build_model_fn,
    initial_weights: List[np.ndarray],
    candidate_clients: List[str],
    select_k: int,
    target_steps_per_round: int,
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    local_epochs: int,
    class_weights: Dict[int, float],
    run_seed: int,
    candidate_multiplier: int = 2,
    train_monitor: Optional[Tuple[np.ndarray, np.ndarray]] = None,
    progress_context: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Power-of-Choice baseline: each round samples d candidates and selects the K
    clients with largest current local TRAIN loss. K, rounds, initialisation, and
    total optimizer steps per round are matched to TADP-VR. No TEST information is used.
    """
    monitor = RAMMonitor().start()
    start = time.perf_counter()
    global_model = build_model_fn()
    global_model.set_weights([np.array(w, copy=True) for w in initial_weights])
    rng = np.random.default_rng(int(run_seed) + 880000)
    round_rows = []
    total_steps = 0
    total_selected = 0
    communication_b = 0
    monitor_eval_s = 0.0

    all_candidates = list(candidate_clients)
    if int(select_k) < 1 or int(select_k) > len(all_candidates):
        raise RuntimeError("Invalid Power-of-Choice K.")

    for r in range(1, NUM_ROUNDS_FL + 1):
        d = min(len(all_candidates), max(int(select_k), int(candidate_multiplier) * int(select_k)))
        if d == len(all_candidates):
            candidate_pool = list(all_candidates)
        else:
            # Canonical pow-d samples candidate clients without replacement
            # according to p_k, the client's fraction of total TRAIN data.
            sizes = np.asarray(
                [len(client_arrays[c][1]) for c in all_candidates], dtype=float
            )
            probs = sizes / sizes.sum()
            candidate_pool = rng.choice(
                all_candidates, size=d, replace=False, p=probs
            ).tolist()

        losses = []
        for cid in candidate_pool:
            Xc, yc = client_arrays[cid]
            losses.append((str(cid), local_training_loss(global_model, Xc, yc)))
        losses.sort(key=lambda x: (-x[1], x[0]))
        selected = [cid for cid, _ in losses[:int(select_k)]]
        step_map = allocate_exact_step_budget(
            selected, client_arrays, int(target_steps_per_round), batch_size, local_epochs
        )

        global_weights = [np.array(w, copy=True) for w in global_model.get_weights()]
        local_weights, local_sizes = [], []
        for j, cid in enumerate(selected):
            Xc, yc = client_arrays[cid]
            local_model = build_model_fn()
            local_model.set_weights(global_weights)
            train_exact_steps(
                local_model, Xc, yc,
                steps=int(step_map[cid]), batch_size=batch_size,
                seed=int(run_seed + 1000 * r + 17 * j),
                class_weights=class_weights,
            )
            local_weights.append([np.array(w, copy=True) for w in local_model.get_weights()])
            local_sizes.append(len(yc))
            del local_model
            tf.keras.backend.clear_session()

        agg = aggregate_weights(local_weights, local_sizes, equal_weight=False)
        global_model = build_model_fn()
        global_model.set_weights(agg)
        round_steps = int(sum(step_map.values()))
        total_steps += round_steps
        total_selected += len(selected)
        param_b = model_parameter_bytes(global_model)
        # Candidate clients receive the current model to evaluate local loss;
        # selected clients return one model update. Scalar loss uploads are negligible.
        communication_b += (len(candidate_pool) + len(selected)) * param_b
        cumulative_comm_mb = float(
            (communication_b * 1.12) / (1024 ** 2)
        )
        cumulative_runtime_s = float(
            max(0.0, (time.perf_counter() - start) - monitor_eval_s)
        )

        if train_monitor is not None:
            Xmon, ymon = train_monitor
            t_mon = time.perf_counter()
            monitor_metrics = evaluate_model(global_model, Xmon, ymon, n_classes)
            monitor_eval_s += float(time.perf_counter() - t_mon)
        else:
            monitor_metrics = {
                "accuracy": float("nan"),
                "precision_macro": float("nan"),
                "recall_macro": float("nan"),
                "f1_macro": float("nan"),
                "roc_auc_ovr_macro": float("nan"),
            }

        live_peak = float(monitor.peak_mb)
        round_rows.append({
            "round": r,
            "candidate_count": len(candidate_pool),
            "candidate_ids": ";".join(candidate_pool),
            "selected_clients": len(selected),
            "selected_ids": ";".join(selected),
            "optimizer_steps": round_steps,
            "cumulative_optimizer_steps": int(total_steps),
            "local_losses": json.dumps({cid: loss for cid, loss in losses}, sort_keys=True),
            "train_monitor_accuracy": float(monitor_metrics["accuracy"]),
            "train_monitor_precision_macro": float(monitor_metrics["precision_macro"]),
            "train_monitor_recall_macro": float(monitor_metrics["recall_macro"]),
            "train_monitor_f1_macro": float(monitor_metrics["f1_macro"]),
            "train_monitor_auc_ovr_macro": float(monitor_metrics["roc_auc_ovr_macro"]),
            "cumulative_runtime_s": float(cumulative_runtime_s),
            "cumulative_energy_wh_est": float(
                ROUND_PROGRESS_POWER_W * cumulative_runtime_s / 3600.0
            ),
            "cumulative_communication_mb": float(cumulative_comm_mb),
            "ram_peak_mb_live": float(live_peak),
            "ram_delta_mb_live": float(max(0.0, live_peak - monitor.start_mb)),
        })

        loss_preview = ", ".join(
            f"{cid}:{loss:.3f}" for cid, loss in losses[:min(5, len(losses))]
        )
        print_round_progress(
            progress_context=progress_context,
            round_idx=r,
            round_total=NUM_ROUNDS_FL,
            selected_ids=list(selected),
            round_steps=int(round_steps),
            cumulative_steps=int(total_steps),
            monitor_metrics=monitor_metrics,
            cumulative_runtime_s=cumulative_runtime_s,
            cumulative_communication_mb=cumulative_comm_mb,
            ram_start_mb=float(monitor.start_mb),
            ram_peak_mb=live_peak,
            extra=f"PoC candidates={len(candidate_pool)} | highest TRAIN losses: {loss_preview}",
        )

    runtime = float(max(0.0, (time.perf_counter() - start) - monitor_eval_s))
    metrics = evaluate_model(global_model, X_test, y_test, n_classes)
    ram = monitor.stop()
    communication_b = int(communication_b * 1.12)
    return {
        "model": global_model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(total_steps),
        "communication_mb": float(communication_b / (1024 ** 2)),
        "round_audit": round_rows,
        "participants_mean_per_round": float(total_selected / max(1, NUM_ROUNDS_FL)),
        **ram,
    }


def centralized_train(
    build_model_fn,
    initial_weights: List[np.ndarray],
    selected_clients: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    exact_steps: int,
    class_weights: Dict[int, float],
    seed: int,
) -> Dict[str, Any]:
    if not selected_clients:
        raise RuntimeError("Centralized scenario has no TRAIN clients.")

    monitor = RAMMonitor().start()
    start = time.perf_counter()

    # Pool ACCEPTED TRAIN partitions only. TEST is not present here.
    X = np.concatenate([client_arrays[c][0] for c in selected_clients], axis=0)
    y = np.concatenate([client_arrays[c][1] for c in selected_clients], axis=0)

    model = build_model_fn()
    model.set_weights([np.array(w, copy=True) for w in initial_weights])
    train_exact_steps(
        model, X, y,
        steps=int(exact_steps),
        batch_size=batch_size,
        seed=int(seed),
        class_weights=class_weights,
    )

    runtime = time.perf_counter() - start
    metrics = evaluate_model(model, X_test, y_test, n_classes)
    ram = monitor.stop()

    return {
        "model": model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(exact_steps),
        "communication_mb": 0.0,
        "participants_mean_per_round": float(len(selected_clients)),
        **ram,
    }


def result_row(
    run: int,
    seed: int,
    scenario: str,
    result: Dict[str, Any],
    initial_hash: str,
    power_w: float = 12.0,
) -> Dict[str, Any]:
    row = {
        "run": int(run),
        "seed": int(seed),
        "scenario": str(scenario),
        **result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "participants": float(result.get("participants_mean_per_round", 0.0)),
        "ram_start_mb": float(result.get("ram_start_mb", 0.0)),
        "ram_end_mb": float(result.get("ram_end_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
        "ram_delta_mb": float(result.get("ram_delta_mb", 0.0)),
        "ram_mb": float(result.get("ram_mb", result.get("ram_peak_mb", 0.0))),
        "initial_weights_sha256": initial_hash,
    }
    row["energy_wh"] = float(power_w * row["runtime_s"] / 3600.0)
    row["energy_kwh"] = float(row["energy_wh"] / 1000.0)
    row["co2_kg"] = float(row["energy_kwh"] * 0.430)
    row["energy_cost_usd"] = float(row["energy_kwh"] * 0.20)
    row["communication_cost_usd"] = float(row["communication_mb"] * 0.005)
    row["total_estimated_cost_usd"] = float(
        row["energy_cost_usd"] + row["communication_cost_usd"]
    )
    return row


# ======================================================================================
# LEAKAGE AUDIT
# ======================================================================================

def write_leakage_audit(
    out_dir: Path,
    train_ids,
    test_ids,
    client_train_ids: Dict[str, np.ndarray],
    extra: Optional[Dict[str, Any]] = None,
):
    train_set = set(map(str, train_ids))
    test_set = set(map(str, test_ids))
    overlap = train_set & test_set

    client_union = set()
    duplicates_across_clients = 0
    for cid, ids in client_train_ids.items():
        s = set(map(str, ids))
        duplicates_across_clients += len(client_union & s)
        client_union |= s

    test_in_clients = len(test_set & client_union)
    missing_train = len(train_set - client_union)
    extra_client_rows = len(client_union - train_set)

    row = {
        "train_test_overlap": len(overlap),
        "test_rows_in_any_client": test_in_clients,
        "train_rows_missing_from_clients": missing_train,
        "client_rows_not_in_global_train": extra_client_rows,
        "duplicate_train_rows_across_clients": duplicates_across_clients,
        "pass": (
            len(overlap) == 0
            and test_in_clients == 0
            and missing_train == 0
            and extra_client_rows == 0
            and duplicates_across_clients == 0
        ),
    }
    if extra:
        row.update(extra)

    pd.DataFrame([row]).to_csv(
        Path(out_dir) / "leakage_audit.csv", index=False
    )

    if not bool(row["pass"]):
        raise RuntimeError(f"FAIL-CLOSED leakage audit failed: {row}")
    return row


# ======================================================================================
# DIABETES 130-US — LEAKAGE-SAFE DATA PREPARATION
# ======================================================================================

TARGET = "readmitted"
ID_COLUMNS = ["encounter_id", "patient_nbr"]


def locate_diabetes_csv() -> str:
    candidates = [
        os.environ.get("DIABETES_CSV", ""),
        "/content/diabetes_130US.csv",
        "./diabetes_130US.csv",
        "/content/drive/MyDrive/diabetes_130US.csv",
    ]
    for p in candidates:
        if p and os.path.exists(p):
            return p

    try:
        from google.colab import files as colab_files
        print("Please upload the Diabetes 130-US CSV file.")
        uploaded = colab_files.upload()
        csvs = [name for name in uploaded if str(name).lower().endswith(".csv")]
        if not csvs:
            raise RuntimeError("No CSV file was uploaded.")
        return str(csvs[0])
    except ImportError:
        pass

    raise FileNotFoundError(
        "Diabetes CSV not found. Set DIABETES_CSV or upload the CSV in Colab."
    )


def load_diabetes(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df = df.replace("?", np.nan)

    if TARGET not in df.columns:
        raise RuntimeError(f"Missing target column: {TARGET}")

    valid_target = {"NO": 0, ">30": 1, "<30": 2}
    df = df[df[TARGET].isin(valid_target)].copy()
    df["_target"] = df[TARGET].map(valid_target).astype(np.int32)
    df["_row_id"] = np.arange(len(df), dtype=np.int64)

    return df


def global_patient_grouped_split(
    df: pd.DataFrame, seed: int, test_fraction: float = 0.20
):
    """
    Patient-grouped and stratified whenever patient_nbr is available.
    The split happens before any client partitioning or data-dependent preprocessing.
    """
    if "patient_nbr" in df.columns:
        splitter = StratifiedGroupKFold(
            n_splits=5, shuffle=True, random_state=int(seed)
        )
        train_idx, test_idx = next(
            splitter.split(
                np.zeros(len(df)),
                y=df["_target"].to_numpy(),
                groups=df["patient_nbr"].astype(str).to_numpy(),
            )
        )
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        patient_overlap = len(
            set(train_df["patient_nbr"].astype(str))
            & set(test_df["patient_nbr"].astype(str))
        )
        if patient_overlap != 0:
            raise RuntimeError("Patient leakage detected across TRAIN/TEST.")
    else:
        train_df, test_df = train_test_split(
            df, test_size=float(test_fraction),
            stratify=df["_target"], random_state=int(seed)
        )
        patient_overlap = 0

    return train_df.reset_index(drop=True), test_df.reset_index(drop=True), patient_overlap


def dirichlet_partition_dataframe(
    train_df: pd.DataFrame,
    n_clients: int,
    alpha: float,
    seed: int,
    min_client_records: int = 100,
) -> Dict[str, pd.DataFrame]:
    y = train_df["_target"].to_numpy()
    client_ids = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")[:n_clients]

    for attempt in range(100):
        rng = np.random.default_rng(int(seed + attempt))
        buckets = [[] for _ in range(n_clients)]

        for cls in sorted(np.unique(y)):
            idx = np.where(y == cls)[0]
            rng.shuffle(idx)
            props = rng.dirichlet(np.full(n_clients, float(alpha)))
            cuts = (np.cumsum(props)[:-1] * len(idx)).astype(int)
            splits = np.split(idx, cuts)
            for k, s in enumerate(splits):
                buckets[k].extend(s.tolist())

        sizes = [len(b) for b in buckets]
        if min(sizes) >= int(min_client_records):
            out = {}
            for cid, idxs in zip(client_ids, buckets):
                out[cid] = train_df.iloc[np.array(idxs, dtype=int)].copy()
            return out

    raise RuntimeError("Could not obtain a valid Dirichlet client partition.")


@dataclass
class TabularPreprocessor:
    feature_cols: List[str]
    numeric_cols: List[str]
    categorical_cols: List[str]
    mean: Dict[str, float]
    std: Dict[str, float]
    categories: Dict[str, List[str]]
    encoder: Any

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        pieces = []

        if self.numeric_cols:
            x_num = []
            for c in self.numeric_cols:
                s = pd.to_numeric(df[c], errors="coerce").astype(float)
                a = s.fillna(self.mean[c]).to_numpy(dtype=np.float32)
                a = (a - self.mean[c]) / self.std[c]
                x_num.append(a[:, None])
            pieces.append(np.concatenate(x_num, axis=1).astype(np.float32))

        if self.categorical_cols:
            cat = pd.DataFrame({
                c: df[c].astype("string").fillna("__MISSING__").astype(str)
                for c in self.categorical_cols
            })
            x_cat = self.encoder.transform(cat)
            pieces.append(np.asarray(x_cat, dtype=np.float32))

        if not pieces:
            raise RuntimeError("No predictor columns remained.")
        return np.concatenate(pieces, axis=1).astype(np.float32)


def fit_federated_train_only_preprocessor(
    client_frames: Dict[str, pd.DataFrame]
) -> TabularPreprocessor:
    """
    No raw TRAIN pooling is used to ESTIMATE numeric parameters.
    Numeric mean/std comes from aggregated local count/sum/sum-of-squares.
    Categorical vocabulary comes from union of local TRAIN category sets.
    """
    any_df = next(iter(client_frames.values()))
    feature_cols = [
        c for c in any_df.columns
        if c not in {TARGET, "_target", "_row_id", *ID_COLUMNS}
    ]

    # Infer expected type from TRAIN only.
    numeric_cols = []
    categorical_cols = []
    for c in feature_cols:
        total_nonmissing = 0
        numeric_valid = 0
        for df in client_frames.values():
            raw = df[c]
            nm = raw.notna()
            total_nonmissing += int(nm.sum())
            if nm.any():
                numeric_valid += int(
                    pd.to_numeric(raw[nm], errors="coerce").notna().sum()
                )
        ratio = numeric_valid / max(1, total_nonmissing)
        if ratio >= 0.95:
            numeric_cols.append(c)
        else:
            categorical_cols.append(c)

    mean = {}
    std = {}
    for c in numeric_cols:
        count = 0
        sum_ = 0.0
        sumsq = 0.0
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            count += len(a)
            sum_ += float(a.sum())
            sumsq += float(np.square(a).sum())
        mu = sum_ / max(1, count)
        var = max(1e-12, sumsq / max(1, count) - mu * mu)
        mean[c] = float(mu)
        std[c] = float(math.sqrt(var))

    categories = {}
    for c in categorical_cols:
        values = set()
        for df in client_frames.values():
            s = df[c].astype("string").fillna("__MISSING__").astype(str)
            values.update(s.unique().tolist())
        categories[c] = sorted(values)

    encoder = OneHotEncoder(
        categories=[categories[c] for c in categorical_cols],
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float32,
    )

    # Fit only metadata-shaped dummy rows; the vocabulary is already frozen from TRAIN.
    if categorical_cols:
        max_len = max(len(categories[c]) for c in categorical_cols)
        dummy = {}
        for c in categorical_cols:
            vals = categories[c]
            dummy[c] = [vals[i % len(vals)] for i in range(max_len)]
        encoder.fit(pd.DataFrame(dummy))

    return TabularPreprocessor(
        feature_cols=feature_cols,
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        mean=mean,
        std=std,
        categories=categories,
        encoder=encoder,
    )


def build_tabular_reference(
    client_frames: Dict[str, pd.DataFrame],
    preprocessor: TabularPreprocessor,
) -> Dict[str, Any]:
    """
    TRAIN-only DQ reference built from client-local summaries only.

    Numerical reference histograms are constructed by combining local
    min/max summaries and then summing client-local histogram counts.
    Categorical reference support is the union of local TRAIN category sets.
    Raw client records are not concatenated to construct the reference.
    """
    ref = {"num_hist": {}, "cat_values": {}}

    chosen_num = preprocessor.numeric_cols[:12]
    for c in chosen_num:
        local_min, local_max = [], []
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                local_min.append(float(np.min(a)))
                local_max.append(float(np.max(a)))
        if not local_min:
            continue
        gmin, gmax = float(min(local_min)), float(max(local_max))
        if gmax <= gmin:
            edges = np.array([gmin - 1e-6, gmax + 1e-6], dtype=float)
        else:
            edges = np.linspace(gmin, gmax, 11, dtype=float)
        global_hist = np.zeros(len(edges)-1, dtype=np.float64)
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                h, _ = np.histogram(a, bins=edges)
                global_hist += h.astype(np.float64)
        ref["num_hist"][c] = {
            "edges": edges.tolist(),
            "hist": global_hist.tolist(),
            "construction": "aggregated_client_local_histograms_only",
        }

    for c in preprocessor.categorical_cols:
        values = set()
        for df in client_frames.values():
            values.update(
                df[c].astype("string").fillna("__MISSING__")
                .astype(str).unique().tolist()
            )
        ref["cat_values"][c] = sorted(values)
    return ref

def tabular_dq_scores(
    df: pd.DataFrame,
    preprocessor: TabularPreprocessor,
    reference: Dict[str, Any],
) -> Tuple[Dict[str, float], Dict[str, float]]:
    """
    Eight machine-measured TRAIN-only DQ factors for the healthcare experiment.
    """
    feature_df = df[preprocessor.feature_cols]

    # 1) Completeness.
    missing_fraction = float(feature_df.isna().mean().mean())
    completeness = score_lower_is_better(
        missing_fraction,
        [0.01, 0.05, 0.10, 0.20, 0.50],
    )

    # 2) Duplication rate.
    duplicate_fraction = float(feature_df.duplicated().mean())
    duplication = score_lower_is_better(
        duplicate_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 3) Value validity / error rate.
    bad = 0
    observed = 0

    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())

        if nm.any():
            conv = pd.to_numeric(
                raw[nm], errors="coerce"
            ).to_numpy(dtype=float)
            bad += int(np.sum(~np.isfinite(conv)))

    for c in preprocessor.categorical_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())

        if nm.any():
            bad += int(
                np.sum(raw[nm].astype(str).str.strip().eq(""))
            )

    error_fraction = float(bad / max(1, observed))
    value_validity_error_rate = score_lower_is_better(
        error_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.15],
    )

    # 4) Type consistency.
    type_bad = 0
    type_obs = 0

    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        type_obs += int(nm.sum())

        if nm.any():
            type_bad += int(
                pd.to_numeric(
                    raw[nm], errors="coerce"
                ).isna().sum()
            )

    type_inconsistency = float(type_bad / max(1, type_obs))
    type_consistency = score_lower_is_better(
        type_inconsistency,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 5) Label integrity.
    invalid_label_fraction = float(
        (~df["_target"].isin([0, 1, 2])).mean()
    )
    label_integrity = score_lower_is_better(
        invalid_label_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    # 6) Feature-distribution consistency — TRAIN-only reference.
    jsds = []

    for c, spec in reference["num_hist"].items():
        a = pd.to_numeric(
            df[c], errors="coerce"
        ).to_numpy(dtype=float)
        a = a[np.isfinite(a)]

        if len(a):
            hist, _ = np.histogram(
                a,
                bins=np.array(spec["edges"], dtype=float),
            )
            jsds.append(
                js_divergence(
                    hist,
                    np.array(spec["hist"], dtype=float),
                )
            )

    max_jsd = float(max(jsds)) if jsds else 0.0
    distribution_consistency = score_lower_is_better(
        max_jsd,
        [0.01, 0.025, 0.05, 0.10, 0.20],
    )

    # 7) Feature/category coverage.
    coverage_vals = []

    for c, ref_vals in reference["cat_values"].items():
        ref_set = set(ref_vals)

        if ref_set:
            client_set = set(
                df[c]
                .astype("string")
                .fillna("__MISSING__")
                .astype(str)
                .unique()
            )
            coverage_vals.append(
                len(client_set & ref_set) / len(ref_set)
            )

    mean_coverage = (
        float(np.mean(coverage_vals))
        if coverage_vals else 1.0
    )
    feature_coverage = score_higher_is_better(
        mean_coverage,
        [0.90, 0.825, 0.75, 0.65, 0.50],
    )

    # 8) Structural / constraint integrity.
    # Uses only TRAIN records. It checks:
    #   - key presence / encounter uniqueness;
    #   - non-negative count-like clinical fields;
    #   - finite numeric values where a numeric value is expected.
    n = len(df)
    record_violation = np.zeros(n, dtype=bool)

    if "encounter_id" not in df.columns:
        record_violation[:] = True
    else:
        encounter = df["encounter_id"]
        record_violation |= encounter.isna().to_numpy()
        record_violation |= encounter.duplicated(keep=False).to_numpy()

    nonnegative_fields = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]

    for c in nonnegative_fields:
        if c in df.columns:
            a = pd.to_numeric(
                df[c], errors="coerce"
            ).to_numpy(dtype=float)
            bad_c = (~np.isfinite(a)) | (a < 0)
            record_violation |= bad_c

    structural_violation_fraction = float(
        np.mean(record_violation)
    ) if n else 1.0

    structural_integrity = score_lower_is_better(
        structural_violation_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    scores = {
        "completeness": completeness,
        "duplication_rate": duplication,
        "value_validity_error_rate": value_validity_error_rate,
        "type_consistency": type_consistency,
        "label_integrity": label_integrity,
        "feature_distribution_consistency": distribution_consistency,
        "feature_category_coverage": feature_coverage,
        "structural_constraint_integrity": structural_integrity,
    }

    raw = {
        "missing_fraction": missing_fraction,
        "duplicate_fraction": duplicate_fraction,
        "error_fraction": error_fraction,
        "type_inconsistency_fraction": type_inconsistency,
        "invalid_label_fraction": invalid_label_fraction,
        "max_jsd": max_jsd,
        "mean_category_coverage": mean_coverage,
        "structural_violation_fraction": structural_violation_fraction,
    }

    return scores, raw

def prepare_diabetes_no_leakage(
    csv_path: str,
    split_seed: int,
    partition_seed: int,
    n_clients: int = 10,
    alpha: float = 1.0,
):
    raw = load_diabetes(csv_path)
    train_df, test_df, patient_overlap = global_patient_grouped_split(
        raw, split_seed
    )

    clients = dirichlet_partition_dataframe(
        train_df, n_clients=n_clients, alpha=alpha, seed=partition_seed
    )
    client_ids = list(clients.keys())

    # Hard row-level leakage audit.
    client_train_ids = {
        cid: df["_row_id"].astype(str).to_numpy()
        for cid, df in clients.items()
    }

    pre = fit_federated_train_only_preprocessor(clients)
    reference = build_tabular_reference(clients, pre)

    dq_scores = {}
    dq_raw_rows = []
    client_arrays = {}

    for cid, df in clients.items():
        scores, raw_metrics = tabular_dq_scores(df, pre, reference)
        dq_scores[cid] = scores

        row = {"client": cid, **scores, **raw_metrics}
        dq_raw_rows.append(row)

        X = pre.transform(df)
        y = df["_target"].to_numpy(dtype=np.int32)
        client_arrays[cid] = (X, y)

    X_test = pre.transform(test_df)
    y_test = test_df["_target"].to_numpy(dtype=np.int32)

    y_train_all = np.concatenate(
        [client_arrays[c][1] for c in client_ids]
    )
    cw = class_weight_dict(y_train_all)

    meta = {
        "raw_rows": len(raw),
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "patient_overlap": int(patient_overlap),
        "input_dim": int(X_test.shape[1]),
        "n_clients": int(n_clients),
        "numeric_features": len(pre.numeric_cols),
        "categorical_features": len(pre.categorical_cols),
    }

    return {
        "raw": raw,
        "train_df": train_df,
        "test_df": test_df,
        "clients_raw": clients,
        "client_arrays": client_arrays,
        "client_ids": client_ids,
        "X_test": X_test,
        "y_test": y_test,
        "dq_scores": dq_scores,
        "dq_audit": pd.DataFrame(dq_raw_rows),
        "class_weights": cw,
        "meta": meta,
        "client_train_ids": client_train_ids,
        "global_train_ids": train_df["_row_id"].astype(str).to_numpy(),
        "global_test_ids": test_df["_row_id"].astype(str).to_numpy(),
        "preprocessor": pre,
        "dq_reference": reference,
    }


def build_diabetes_model(input_dim: int, lr: float = 1e-3) -> keras.Model:
    inp = keras.Input(shape=(int(input_dim),), dtype=tf.float32)
    x = layers.Dense(128, activation="relu")(inp)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(3, activation="softmax", dtype=tf.float32)(x)
    model = keras.Model(inp, out)
    model.optimizer = keras.optimizers.Adam(learning_rate=float(lr))
    return model



# ======================================================================================
# FULL EXPERIMENT-A REPORTING, CHECKPOINTING, LEDGER, AND STATISTICS
# ======================================================================================
from datetime import datetime, timezone
import re

# =============================================================================
# MAIN REPEATED TRAINING SET
#
# These eight configurations are genuinely distinct and are repeated over all
# five training seeds for mean ± SD / CI reporting.
#
# NOT repeated here:
#   - Great Expectations Centralized
#   - TADP-AA Centralized
#   - Great Expectations Federated
#   - TADP-AA Federated
# Those all-client equivalences are verified independently by the companion
# one-seed equivalence-audit script.
#
# TADP-SDA is retained as a boundary/stress condition but is run once only,
# outside the five-seed headline statistical comparison.
# =============================================================================
MANUSCRIPT_SCENARIOS = [
    "Naïve Centralized",
    "TADP-VR Centralized",
    "Vanilla FedAvg",
    "FedProx",
    "Random-K",
    "DQ-only Federated",
    "Power-of-Choice",
    "TADP-VR Federated",
]
assert len(MANUSCRIPT_SCENARIOS) == 8

BOUNDARY_SCENARIOS = [
    "TADP-SDA Centralized",
    "TADP-SDA Federated",
]
BOUNDARY_SEED = 42



def print_banner(title: str, width: int = 108):
    print("\n" + "=" * width)
    print(title)
    print("=" * width)


def client_partition_table(clients_raw):
    rows = []
    for cid, df in clients_raw.items():
        counts = df["_target"].value_counts().to_dict()
        rows.append({
            "client": cid,
            "records": int(len(df)),
            "class_0_NO": int(counts.get(0, 0)),
            "class_1_GT30": int(counts.get(1, 0)),
            "class_2_LT30": int(counts.get(2, 0)),
        })
    return pd.DataFrame(rows)


def evidence_assignment_summary(
    evidence_df: pd.DataFrame
) -> pd.DataFrame:
    """
    Summarize the frozen controlled documentary-evidence assignment.

    v16.7 reports the pre-specified governance archetype for each evidence
    bundle. These archetypes are branch-coverage scenarios, not observed
    real-world prevalence classes.
    """
    required = {
        "client",
        "bundle_id",
        "evidence_profile",
        "scenario_role",
        "profile_variant",
        "factor",
        "rubric_score_0_5",
        "meets_factor_adequacy",
        "evidence_seed",
    }

    missing = sorted(
        required - set(evidence_df.columns)
    )

    if missing:
        raise RuntimeError(
            "Controlled documentary-evidence table is missing required "
            f"column(s): {missing}. Available columns: "
            f"{sorted(evidence_df.columns.tolist())}"
        )

    work = evidence_df.copy()
    work["meets_factor_adequacy"] = (
        work["meets_factor_adequacy"]
        .astype(bool)
    )

    summary = (
        work
        .groupby(
            ["client", "bundle_id"],
            as_index=False,
        )
        .agg(
            evidence_profile=(
                "evidence_profile",
                "first",
            ),
            scenario_role=(
                "scenario_role",
                "first",
            ),
            profile_variant=(
                "profile_variant",
                "first",
            ),
            documentary_factor_count=(
                "factor",
                "count",
            ),
            documentary_adequate_factor_count=(
                "meets_factor_adequacy",
                "sum",
            ),
            documentary_mean_score=(
                "rubric_score_0_5",
                "mean",
            ),
            documentary_min_score=(
                "rubric_score_0_5",
                "min",
            ),
            documentary_max_score=(
                "rubric_score_0_5",
                "max",
            ),
            evidence_seed=(
                "evidence_seed",
                "first",
            ),
        )
        .sort_values("client")
        .reset_index(drop=True)
    )

    summary["documentary_adequacy_fraction"] = (
        summary["documentary_adequate_factor_count"]
        / summary["documentary_factor_count"].clip(lower=1)
    )

    expected_documentary_factors = sum(
        len(FACTOR_NAMES[d])
        for d in DOCUMENTARY_DIMS
    )

    count_ok = summary[
        "documentary_factor_count"
    ].eq(
        expected_documentary_factors
    )

    if not count_ok.all():
        bad = summary.loc[
            ~count_ok,
            [
                "client",
                "bundle_id",
                "documentary_factor_count",
            ],
        ]

        raise RuntimeError(
            "Unexpected controlled-evidence factor count. "
            f"Expected {expected_documentary_factors} documentary factors "
            "per client. Offending rows:\n"
            + bad.to_string(index=False)
        )

    return summary

def print_governance_details(
    gov: pd.DataFrame,
    ge: pd.DataFrame,
    dq: pd.DataFrame,
    vr_clients: List[str],
    sda_clients: List[str],
    ge_clients: List[str],
    domain: str,
):
    domain = str(domain).lower()

    print_banner(
        "DOMAIN ADEQUACY POLICY — FULL WAC + CRITICAL WAC"
    )

    policy_rows = []
    for dim in FACTOR_NAMES:
        policy_rows.append({
            "dimension":
                dim,
            "dimension_name":
                DIMENSION_NAMES[dim],
            "n_factors":
                len(FACTOR_NAMES[dim]),
            "minimum_adequacy_ranks":
                ",".join(
                    str(
                        DOMAIN_FACTOR_MINIMA[
                            domain
                        ][dim][f]
                    )
                    for f in FACTOR_NAMES[
                        dim
                    ]
                ),
            "dimension_policy_wac":
                DOMAIN_DIMENSION_WAC[
                    domain
                ][dim],
        })

    print(
        pd.DataFrame(
            policy_rows
        ).to_string(
            index=False
        )
    )

    global_critical_wac = derive_global_critical_wac(
        domain
    )

    print(
        f"\nGLOBAL {domain.upper()} WAC "
        f"(descriptive full-policy summary) = "
        f"{DOMAIN_GLOBAL_WAC[domain]:.6f}"
    )
    print(
        f"GLOBAL {domain.upper()} CRITICAL WAC "
        f"(automated Review threshold) = "
        f"{global_critical_wac:.6f}"
    )

    print_banner(
        "CRITICAL FACTORS — INDIVIDUAL ADEQUACY REQUIREMENTS"
    )

    critical_rows = []

    for dim, factor_list in (
        CRITICAL_FACTORS_BY_DOMAIN[
            domain
        ].items()
    ):
        for factor in factor_list:
            minimum = float(
                DOMAIN_FACTOR_MINIMA[
                    domain
                ][dim][factor]
            )
            critical_rows.append({
                "dimension":
                    dim,
                "factor":
                    factor,
                "individual_adequacy_min_0_5":
                    minimum,
                "normalized_reference":
                    minimum / MAX_FACTOR_SCORE,
                "direct_auto_accept_rule":
                    f"score >= {minimum:.1f}",
            })

    print(
        pd.DataFrame(
            critical_rows
        ).to_string(
            index=False
        )
    )

    print(
        f"\nMinimum dimension floor: EVERY averaged dimension "
        f"must be >= {DIMENSION_MIN_FLOOR:.1f}/5."
    )

    print(
        "Human reviewer role: verify uploaded questionnaire evidence only. "
        "The server makes the admission decision automatically."
    )

    print_banner(
        "TRAIN-ONLY DATA-QUALITY FACTORS — 8 FACTORS"
    )

    dq_cols = [
        "client",
        "completeness",
        "duplication_rate",
        "value_validity_error_rate",
        "type_consistency",
        "label_integrity",
        "feature_distribution_consistency",
        "feature_category_coverage",
        "structural_constraint_integrity",
    ]

    print(
        dq[dq_cols].to_string(
            index=False
        )
    )

    print_banner(
        "FROZEN TADP GOVERNANCE — HPS + CRITICAL WAC"
    )

    display_cols = [
        "client",
        "hps",
        "critical_wac_i",
        "global_critical_wac",
        "critical_wac_margin",
        "all_critical_meet_adequacy",
        "critical_below_adequacy",
        "dimension_floor_failures",
        "dim1_score_0_5",
        "dim2_score_0_5",
        "dim3_score_0_5",
        "dim4_score_0_5",
        "dim5_score_0_5",
        "dim6_score_0_5",
        "decision_path",
        "initial_action",
        "final_action",
        "status",
        "reason",
    ]

    print(
        gov[
            display_cols
        ].to_string(
            index=False
        )
    )

    print(
        "\nTADP v16.7 final decision order:"
    )
    print(
        f"  1) ANY averaged dimension < "
        f"{DIMENSION_MIN_FLOOR:.1f}/5 -> AUTO-REJECT"
    )
    print(
        f"  2) HPS < {GOOD_CUT:.1f} -> AUTO-REJECT"
    )
    print(
        f"  3) HPS >= {HIGH_CUT:.1f}:"
    )
    print(
        "       all critical factors meet their own adequacy minima "
        "-> DIRECT AUTO-ACCEPT"
    )
    print(
        "       otherwise -> AUTOMATED REVIEW fallback"
    )
    print(
        f"  4) {GOOD_CUT:.1f} <= HPS < "
        f"{HIGH_CUT:.1f} -> AUTOMATED REVIEW"
    )
    print(
        "  5) AUTOMATED REVIEW:"
    )
    print(
        f"       Critical WAC_i >= Global Critical WAC "
        f"({global_critical_wac:.6f}) -> ACCEPT AFTER REVIEW"
    )
    print(
        "       otherwise -> AUTO-REJECT"
    )

    print(
        f"\nFrozen TADP-VR cohort: "
        f"{len(vr_clients)}/10 -> "
        f"{vr_clients}"
    )
    print(
        f"Frozen TADP-SDA cohort: "
        f"{len(sda_clients)}/10 -> "
        f"{sda_clients}"
    )

    print_banner("GX CORE NATIVE TRAIN-ONLY VALIDATOR BASELINE")
    gx_base_cols = [
        "client", "gx_native_suite_success", "ge_expectations_passed",
        "ge_expectations_total", "ge_pass_rate", "ge_native_validation_class",
        "ge_final_action",
    ]
    gx_category_cols = [
        c for c in ge.columns
        if c.startswith("ge_") and (c.endswith("_passed") or c.endswith("_total"))
        and c not in {"ge_expectations_passed", "ge_expectations_total"}
    ]
    print(ge[gx_base_cols + sorted(gx_category_cols)].to_string(index=False))
    print(f"\nGX operationally valid clients (zero critical failures): {len(ge_clients)}/{len(ge)} -> {ge_clients}")
    print(
        "GX is used only as a native rule-based data validator. PASS/FAIL is the "
        "suite-level GX result; no ranking, no forced-K selection, and no TADP signal is used."
    )





def build_hash_chained_governance_ledger(
    gov,
    ge,
    output_path,
):
    rows = []
    prev_hash = "GENESIS"
    seq = 0

    for _, r in (
        gov.sort_values(
            "client"
        ).iterrows()
    ):
        seq += 1

        payload = {
            "sequence":
                seq,
            "governance_system":
                "TADP",
            "client":
                str(r["client"]),
            "hps":
                float(r["hps"]),
            "global_domain_wac":
                float(r["global_domain_wac"]),
            "global_critical_wac":
                float(r["global_critical_wac"]),
            "critical_wac_i": (
                float(r["critical_wac_i"])
                if np.isfinite(r["critical_wac_i"])
                else None
            ),
            "critical_wac_margin": (
                float(r["critical_wac_margin"])
                if np.isfinite(r["critical_wac_margin"])
                else None
            ),
            "all_critical_meet_adequacy":
                bool(r["all_critical_meet_adequacy"]),
            "critical_below_adequacy":
                str(r["critical_below_adequacy"]),
            "dimension_floor_failures":
                str(r["dimension_floor_failures"]),
            "decision_path":
                str(r["decision_path"]),
            "initial_action":
                str(r["initial_action"]),
            "final_action":
                str(r["final_action"]),
            "status":
                str(r["status"]),
            "reason":
                str(r["reason"]),
            "previous_hash":
                prev_hash,
        }

        canonical = json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
        )

        entry_hash = hashlib.sha256(
            canonical.encode("utf-8")
        ).hexdigest()

        payload["entry_hash"] = entry_hash

        rows.append(payload)
        prev_hash = entry_hash

    for _, r in (
        ge.sort_values(
            "client"
        ).iterrows()
    ):
        seq += 1

        payload = {
            "sequence":
                seq,
            "governance_system":
                "Great Expectations GX Core native validator",
            "client":
                str(r["client"]),
            "hps":
                None,
            "global_domain_wac":
                None,
            "global_critical_wac":
                None,
            "critical_wac_i":
                None,
            "critical_wac_margin":
                None,
            "all_critical_meet_adequacy":
                None,
            "critical_below_adequacy":
                None,
            "dimension_floor_failures":
                None,
            "decision_path":
                "RULE_BASED_VALIDATION",
            "initial_action":
                "RULE_BASED_VALIDATION",
            "final_action":
                str(r["ge_final_action"]),
            "status":
                "GE_CONTROLLED_TRAIN_ONLY",
            "reason": (
                f"GX native suite {'PASSED' if bool(r['gx_native_suite_success']) else 'FAILED'}; "
                f"{int(r['ge_expectations_passed'])}/{int(r['ge_expectations_total'])} "
                "configured technical Expectations passed"
            ),
            "previous_hash":
                prev_hash,
        }

        canonical = json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
        )

        entry_hash = hashlib.sha256(
            canonical.encode("utf-8")
        ).hexdigest()

        payload["entry_hash"] = entry_hash

        rows.append(payload)
        prev_hash = entry_hash

    out = pd.DataFrame(rows)

    out.to_csv(
        output_path,
        index=False,
    )

    return out

def choose_experiment_root(experiment_name, use_drive=True):
    if use_drive:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            root = Path("/content/drive/MyDrive/TADP_CHECKPOINTS") / experiment_name
            root.mkdir(parents=True, exist_ok=True)
            print(f"Persistent checkpoint root: {root}")
            return root
        except Exception as exc:
            print(f"Google Drive checkpoint mount unavailable: {exc}")
    root = Path("/content") / experiment_name
    root.mkdir(parents=True, exist_ok=True)
    print(f"Local checkpoint root: {root}")
    return root


def atomic_write_json(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")
    tmp.replace(path)


def load_checkpoint_state(path):
    path = Path(path)
    if not path.exists():
        return {"completed": [], "last_completed": None, "updated_utc": None}
    return json.loads(path.read_text(encoding="utf-8"))


def mark_checkpoint_complete(state_path, key, extra=None):
    state = load_checkpoint_state(state_path)
    completed = list(state.get("completed", []))
    if key not in completed:
        completed.append(key)
    state["completed"] = completed
    state["last_completed"] = key
    state["updated_utc"] = datetime.now(timezone.utc).isoformat()
    if extra:
        state.update(extra)
    atomic_write_json(state, state_path)


def upsert_csv(row, path, key_cols):
    path = Path(path)
    new = pd.DataFrame([row])
    if path.exists():
        old = pd.read_csv(path)
        if not old.empty:
            mask = pd.Series(True, index=old.index)
            for c in key_cols:
                mask &= old[c].astype(str).eq(str(row[c]))
            old = old.loc[~mask].copy()
            new = pd.concat([old, new], ignore_index=True)
    new.to_csv(path, index=False)


def write_scenario_checkpoint(root, run_idx, scenario, result):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", scenario).strip("_")
    cdir = ensure_dir(root / "scenario_checkpoints" / f"run_{run_idx:02d}")
    payload = {
        "run": int(run_idx),
        "scenario": scenario,
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "metrics": result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
    }
    atomic_write_json(payload, cdir / f"{safe}.json")
    if "round_audit" in result:
        pd.DataFrame(result["round_audit"]).to_csv(
            cdir / f"{safe}_rounds.csv", index=False
        )


def ci95_mean(values):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan
    if len(x) == 1:
        return float(x[0]), float(x[0])
    mean = float(np.mean(x))
    sd = float(np.std(x, ddof=1))
    try:
        from scipy.stats import t
        crit = float(t.ppf(0.975, df=len(x)-1))
    except Exception:
        crit = 1.96
    half = crit * sd / math.sqrt(len(x))
    return mean-half, mean+half


def summarize_runs_with_ci(perf):
    metrics = [
        "accuracy", "precision_macro", "recall_macro", "f1_macro",
        "roc_auc_ovr_macro", "runtime_s", "energy_wh", "energy_kwh",
        "co2_kg", "communication_mb", "ram_peak_mb", "ram_delta_mb",
        "optimizer_steps", "participants", "energy_cost_usd",
        "communication_cost_usd", "total_estimated_cost_usd",
    ]
    rows = []
    for scenario, d in perf.groupby("scenario", sort=False):
        row = {"scenario": scenario, "n_runs": int(len(d))}
        for metric in metrics:
            if metric not in d.columns:
                continue
            vals = pd.to_numeric(d[metric], errors="coerce")
            vals = vals[np.isfinite(vals)]
            row[f"{metric}_mean"] = float(vals.mean()) if len(vals) else np.nan
            row[f"{metric}_sd"] = float(vals.std(ddof=1)) if len(vals)>1 else 0.0
            lo, hi = ci95_mean(vals)
            row[f"{metric}_ci95_low"] = lo
            row[f"{metric}_ci95_high"] = hi
        rows.append(row)
    return pd.DataFrame(rows)


def paired_vr_randomk_statistics(perf):
    vr = perf[perf["scenario"].eq("TADP-VR Federated")].copy()
    rk = perf[perf["scenario"].eq("Random-K")].copy()
    merged = vr.merge(rk, on=["run", "seed"], suffixes=("_vr", "_randomk"), validate="one_to_one")
    rows = []
    for metric in ["accuracy", "f1_macro", "roc_auc_ovr_macro"]:
        a = merged[f"{metric}_vr"].to_numpy(dtype=float)
        b = merged[f"{metric}_randomk"].to_numpy(dtype=float)
        diff = a-b
        mean = float(np.mean(diff))
        sd = float(np.std(diff, ddof=1)) if len(diff)>1 else 0.0
        lo, hi = ci95_mean(diff)
        t_stat=t_p=wil_stat=wil_p=np.nan
        try:
            from scipy.stats import ttest_rel, wilcoxon
            tr = ttest_rel(a,b,nan_policy="omit")
            t_stat, t_p = float(tr.statistic), float(tr.pvalue)
            if np.any(np.abs(diff)>0):
                wr = wilcoxon(a,b)
                wil_stat, wil_p = float(wr.statistic), float(wr.pvalue)
        except Exception:
            pass
        rows.append({
            "metric": metric,
            "n_pairs": len(diff),
            "mean_difference_vr_minus_randomk": mean,
            "sd_difference": sd,
            "ci95_low": lo,
            "ci95_high": hi,
            "cohens_dz": float(mean/sd) if sd>0 else np.nan,
            "paired_t_stat": t_stat,
            "paired_t_p": t_p,
            "wilcoxon_stat": wil_stat,
            "wilcoxon_p": wil_p,
            "vr_wins": int(np.sum(diff>0)),
            "ties": int(np.sum(np.isclose(diff,0))),
            "vr_losses": int(np.sum(diff<0)),
        })
    return pd.DataFrame(rows)


def scenario_method_table():
    return pd.DataFrame([
        ["Naïve Centralized","centralized","baseline","all clients","5-seed headline"],
        ["Great Expectations Centralized","centralized","GX native validator","GX clients with zero critical failures","equivalence audit only"],
        ["TADP-AA Centralized","centralized","TADP-AA","all clients","equivalence audit only"],
        ["TADP-VR Centralized","centralized","TADP-VR","frozen TADP-VR cohort","5-seed headline"],
        ["TADP-SDA Centralized","centralized","TADP-SDA","frozen best eligible client","one-seed boundary"],
        ["Vanilla FedAvg","federated","FedAvg","all clients","5-seed headline"],
        ["FedProx","federated","FedProx","all clients","5-seed headline"],
        ["Random-K","federated","matched random control","same K/rounds/steps as TADP-VR","5-seed headline"],
        ["Great Expectations Federated","federated","GX native validator","GX clients with zero critical failures","equivalence audit only"],
        ["DQ-only Federated","federated","DQ-only","top-K by machine-measured DQ; same K/rounds/steps as TADP-VR","5-seed headline"],
        ["Power-of-Choice","federated","Power-of-Choice","dynamic loss-based selection; same K/rounds/steps as TADP-VR","5-seed headline"],
        ["TADP-AA Federated","federated","TADP-AA","all clients","equivalence audit only"],
        ["TADP-VR Federated","federated","TADP-VR","frozen TADP-VR cohort","5-seed headline"],
        ["TADP-SDA Federated","federated","TADP-SDA","frozen best eligible client","one-seed boundary"],
    ], columns=["scenario","paradigm","method","participation","execution_role"])



def governance_only_monte_carlo(
    client_ids,
    dq_scores,
    base_seed,
    n_realizations=1000,
    domain="healthcare",
):
    domain = str(domain).lower()
    rows=[]
    for j in range(int(n_realizations)):
        seed=int(base_seed+j)
        evidence,_=generate_controlled_documentary_evidence(
            client_ids,
            seed,
            DOMAIN_FACTOR_MINIMA[domain],
            domain=domain,
        )
        gov=build_tadp_governance(
            client_ids,
            evidence,
            dq_scores,
            run=0,
            evidence_seed=seed,
            domain=domain,
        )
        accepted=accepted_tadp_vr(gov)
        rows.append({
            "realization":j+1,
            "evidence_seed":seed,
            "accepted_count":len(accepted),
            "accepted_clients":";".join(accepted),
            "mean_hps":float(gov["hps"].mean()),
            "mean_critical_wac_i":float(gov["critical_wac_i"].mean()),
            "dimension_floor_rejects":int(gov["status"].eq("AUTO_REJECTED_DIMENSION_FLOOR").sum()),
            "low_hps_rejects":int(gov["status"].eq("AUTO_REJECTED_LOW_HPS").sum()),
            "review_accepts":int(
                gov["status"].eq("ACCEPTED_AFTER_AUTOMATED_REVIEW").sum()
            ),
            "direct_auto_accepts":int(
                gov["status"].eq("DIRECT_AUTO_ACCEPTED").sum()
            ),
        })
    return pd.DataFrame(rows)


# ======================================================================================
# EXPERIMENT B3 — PREDICTIVE UTILITY UNDER CLIENT SCALING
# ======================================================================================
#
# PURPOSE
# -------
# Complement Experiment B1/B2 (governance-only scalability) with downstream
# predictive-utility scalability.
#
# Tested client counts:
#     K = {20, 50, 100}
#
# K=10 is intentionally not rerun here because it is already evaluated in the
# final Experiment-A protocol. For cross-K manuscript plots, use only the
# corresponding 4-round Experiment-A K=10 rows for seeds {42,142,242}.
#
# Three scenarios are intentionally retained:
#
#   1) Vanilla FedAvg
#      - all submitted clients participate
#      - full-participation predictive-utility reference
#
#   2) Random-K
#      - same number of participating clients as TADP-VR
#      - exact same total optimizer-step budget per round as TADP-VR
#      - frozen random cohort within each K, across all rounds and seeds
#
#   3) TADP-VR Federated
#      - clients admitted by the frozen TADP governance policy
#      - natural one-local-epoch step budget
#
# FAIRNESS / INTERPRETATION
# -------------------------
# TADP-VR vs Random-K is the matched-compute selection comparison:
#   same K_selected, same FL rounds, same total optimizer steps per round,
#   same model architecture, same initial weights within seed, same batch size,
#   same optimizer, same global TEST set.
#
# Vanilla FedAvg is NOT step-matched to TADP-VR because it is the intended
# full-participation utility reference. Its larger compute/communication budget
# is reported rather than hidden.
#
# The experiment tests whether TADP-VR retains useful predictive performance
# as the submitted contributor population grows, while using fewer clients
# than full FedAvg and while being compared fairly against a matched random
# subset.
#
# The held-out TEST set is used only after training for final evaluation.
# It is never used for governance, preprocessing, client selection, step-budget
# construction, or checkpoint decisions.
# ======================================================================================

import platform
import shutil
from datetime import datetime, timezone

EXPERIMENT_VERSION = "TADP-B3-v16.9-PREDICTIVE-SCALABILITY-K20-50-100-3SEED-4ROUND"

CLIENT_COUNTS = [20, 50, 100]
TRAINING_RUN_SEEDS = [42, 142, 242]

NUM_ROUNDS_FL = 4
LOCAL_EPOCHS = 1
BATCH_SIZE = 64
LEARNING_RATE = 1e-3

DIRICHLET_ALPHA = 1.0
MIN_CLIENT_RECORDS = 30

GLOBAL_SPLIT_SEED = 7001
PARTITION_BASE_SEED = 9101
EVIDENCE_BASE_SEED = 12042
RANDOMK_BASE_SEED = 22042

DOMAIN = "healthcare"
USE_GOOGLE_DRIVE_CHECKPOINTS = True

SCENARIOS = [
    "Vanilla FedAvg",
    "Random-K",
    "TADP-VR Federated",
]

EXPERIMENT_ROOT = choose_experiment_root(
    "TADP_EXPERIMENT_B3_" + EXPERIMENT_VERSION,
    use_drive=USE_GOOGLE_DRIVE_CHECKPOINTS,
)
CHECKPOINT_STATE = EXPERIMENT_ROOT / "checkpoint_state.json"
PERF_CHECKPOINT = EXPERIMENT_ROOT / "performance_metrics_checkpoint.csv"


# ======================================================================================
# SCALABLE CLIENT PARTITIONING / CONTROLLED EVIDENCE
# ======================================================================================

def scalable_client_ids(n_clients: int) -> List[str]:
    return [f"C{i:03d}" for i in range(1, int(n_clients) + 1)]


def dirichlet_partition_dataframe_scalable(
    train_df: pd.DataFrame,
    n_clients: int,
    alpha: float,
    seed: int,
    min_client_records: int = 30,
    max_attempts: int = 500,
) -> Dict[str, pd.DataFrame]:
    """
    Label-wise Dirichlet partition that supports K > 26 while preserving every
    global TRAIN row exactly once.
    """
    y = train_df["_target"].to_numpy()
    client_ids = scalable_client_ids(n_clients)

    for attempt in range(int(max_attempts)):
        rng = np.random.default_rng(int(seed + attempt))
        buckets = [[] for _ in range(int(n_clients))]

        for cls in sorted(np.unique(y)):
            idx = np.where(y == cls)[0]
            rng.shuffle(idx)
            props = rng.dirichlet(np.full(int(n_clients), float(alpha)))
            cuts = (np.cumsum(props)[:-1] * len(idx)).astype(int)
            splits = np.split(idx, cuts)
            for j, split in enumerate(splits):
                buckets[j].extend(split.tolist())

        sizes = [len(b) for b in buckets]
        if min(sizes) >= int(min_client_records):
            return {
                cid: train_df.iloc[np.asarray(idxs, dtype=int)].copy()
                for cid, idxs in zip(client_ids, buckets)
            }

    raise RuntimeError(
        f"Could not obtain valid K={n_clients} partition after {max_attempts} attempts "
        f"with min_client_records={min_client_records}."
    )


def partition_integrity_audit(
    train_df: pd.DataFrame,
    clients: Dict[str, pd.DataFrame],
) -> Dict[str, Any]:
    global_ids = set(train_df["_row_id"].astype(int).tolist())
    all_ids = []
    for df in clients.values():
        all_ids.extend(df["_row_id"].astype(int).tolist())
    client_set = set(all_ids)

    result = {
        "global_train_rows": int(len(train_df)),
        "client_rows_total": int(len(all_ids)),
        "unique_client_rows": int(len(client_set)),
        "rows_missing_from_clients": int(len(global_ids - client_set)),
        "rows_not_in_global_train": int(len(client_set - global_ids)),
        "duplicate_rows_across_clients": int(len(all_ids) - len(client_set)),
    }
    result["pass"] = bool(
        len(all_ids) == len(train_df)
        and client_set == global_ids
        and len(all_ids) == len(client_set)
    )
    return result


def generate_scalable_controlled_evidence(
    client_ids: List[str],
    evidence_seed: int,
    domain: str = "healthcare",
):
    """
    Preserve the audited Experiment-A governance archetype composition.

    Every block of ten contributors receives the same 4/2/2/1/1 archetype mix:
      - 4 DIRECT_STRONG
      - 2 REVIEW_RECOVERABLE
      - 2 REVIEW_LIMITED
      - 1 LOW_HPS_WEAK
      - 1 DIMENSION_FLOOR_WEAK

    Assignment to identities is seeded before training and never uses TEST or
    downstream predictive performance.
    """
    if len(client_ids) % 10 != 0:
        raise ValueError(
            "B3 uses K values divisible by 10 so the controlled evidence-profile "
            "composition remains exactly proportional."
        )

    combined = {}
    frames = []
    minima = DOMAIN_FACTOR_MINIMA[str(domain).lower()]

    for block_idx, start in enumerate(range(0, len(client_ids), 10), start=1):
        block = client_ids[start:start + 10]
        block_seed = int(evidence_seed + 1009 * (block_idx - 1))
        evidence, df = generate_controlled_documentary_evidence(
            block,
            block_seed,
            minima,
            domain=domain,
        )
        combined.update(evidence)
        df = df.copy()
        df["scalability_block"] = int(block_idx)
        df["block_evidence_seed"] = int(block_seed)
        df["bundle_id"] = df["bundle_id"].map(
            lambda x: f"B{block_idx:02d}-{x}"
        )
        frames.append(df)

    return combined, pd.concat(frames, ignore_index=True)


def validate_expected_profile_mix(evidence_df: pd.DataFrame, k: int):
    one = evidence_df[["client", "scenario_role"]].drop_duplicates("client")
    got = one["scenario_role"].value_counts().to_dict()
    scale = int(k // 10)
    expected = {
        "DIRECT_STRONG": 4 * scale,
        "REVIEW_RECOVERABLE": 2 * scale,
        "REVIEW_LIMITED": 2 * scale,
        "LOW_HPS_WEAK": 1 * scale,
        "DIMENSION_FLOOR_WEAK": 1 * scale,
    }
    if got != expected:
        raise RuntimeError(
            f"Controlled evidence composition mismatch for K={k}: "
            f"got={got}, expected={expected}"
        )
    return expected


def prepare_k_data(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    k: int,
    partition_seed: int,
):
    clients = dirichlet_partition_dataframe_scalable(
        train_df=train_df,
        n_clients=k,
        alpha=DIRICHLET_ALPHA,
        seed=partition_seed,
        min_client_records=MIN_CLIENT_RECORDS,
    )

    integrity = partition_integrity_audit(train_df, clients)
    if not integrity["pass"]:
        raise RuntimeError(f"Partition integrity failed at K={k}: {integrity}")

    pre = fit_federated_train_only_preprocessor(clients)
    reference = build_tabular_reference(clients, pre)

    dq_scores = {}
    dq_rows = []
    client_arrays = {}

    for cid, df in clients.items():
        scores, raw_metrics = tabular_dq_scores(df, pre, reference)
        dq_scores[cid] = scores
        dq_rows.append({"client": cid, **scores, **raw_metrics})

        Xc = pre.transform(df)
        yc = df["_target"].to_numpy(dtype=np.int32)
        client_arrays[cid] = (Xc, yc)

    X_test = pre.transform(test_df)
    y_test = test_df["_target"].to_numpy(dtype=np.int32)

    y_train_all = np.concatenate(
        [client_arrays[cid][1] for cid in sorted(client_arrays)]
    )
    class_weights = class_weight_dict(y_train_all)

    return {
        "clients_raw": clients,
        "client_ids": list(clients.keys()),
        "client_arrays": client_arrays,
        "preprocessor": pre,
        "dq_reference": reference,
        "dq_scores": dq_scores,
        "dq_audit": pd.DataFrame(dq_rows),
        "X_test": X_test,
        "y_test": y_test,
        "class_weights": class_weights,
        "input_dim": int(X_test.shape[1]),
        "integrity": integrity,
    }


# ======================================================================================
# SUMMARIES / CHECKPOINTS
# ======================================================================================

def summarize_b3(perf: pd.DataFrame) -> pd.DataFrame:
    metrics = [
        "accuracy",
        "precision_macro",
        "recall_macro",
        "f1_macro",
        "roc_auc_ovr_macro",
        "runtime_s",
        "communication_mb",
        "optimizer_steps",
        "participants",
        "energy_wh",
        "ram_peak_mb",
        "ram_delta_mb",
    ]
    rows = []
    for (k, scenario), d in perf.groupby(
        ["k_submissions", "scenario"], sort=True
    ):
        row = {
            "k_submissions": int(k),
            "scenario": str(scenario),
            "n_runs": int(len(d)),
        }
        for metric in metrics:
            if metric not in d.columns:
                continue
            vals = pd.to_numeric(d[metric], errors="coerce").to_numpy(dtype=float)
            vals = vals[np.isfinite(vals)]
            row[f"{metric}_mean"] = (
                float(np.mean(vals)) if len(vals) else np.nan
            )
            row[f"{metric}_sd"] = (
                float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            )
            lo, hi = ci95_mean(vals)
            row[f"{metric}_ci95_low"] = lo
            row[f"{metric}_ci95_high"] = hi
        rows.append(row)
    return pd.DataFrame(rows)


def paired_b3(perf: pd.DataFrame) -> pd.DataFrame:
    """
    Paired seed-wise differences.

    TADP-VR vs Random-K:
        matched K and matched optimizer-step budget.

    TADP-VR vs Vanilla FedAvg:
        utility comparison against full participation; NOT compute-matched.
    """
    rows = []
    comparisons = [
        ("Random-K", "MATCHED_SELECTION_CONTROL"),
        ("Vanilla FedAvg", "FULL_PARTICIPATION_REFERENCE"),
    ]

    for k in sorted(perf["k_submissions"].unique()):
        tadp = perf[
            (perf["k_submissions"].eq(k))
            & (perf["scenario"].eq("TADP-VR Federated"))
        ].copy()

        for baseline, role in comparisons:
            base = perf[
                (perf["k_submissions"].eq(k))
                & (perf["scenario"].eq(baseline))
            ].copy()

            merged = tadp.merge(
                base,
                on=["k_submissions", "seed"],
                suffixes=("_tadp", "_baseline"),
                validate="one_to_one",
            )

            for metric in [
                "accuracy",
                "precision_macro",
                "recall_macro",
                "f1_macro",
                "roc_auc_ovr_macro",
            ]:
                a = merged[f"{metric}_tadp"].to_numpy(dtype=float)
                b = merged[f"{metric}_baseline"].to_numpy(dtype=float)
                diff = a - b
                lo, hi = ci95_mean(diff)

                rows.append({
                    "k_submissions": int(k),
                    "baseline": baseline,
                    "baseline_role": role,
                    "metric": metric,
                    "n_pairs": int(len(diff)),
                    "mean_difference_tadp_minus_baseline":
                        float(np.mean(diff)),
                    "sd_difference":
                        float(np.std(diff, ddof=1)) if len(diff) > 1 else 0.0,
                    "ci95_low": lo,
                    "ci95_high": hi,
                    "tadp_wins": int(np.sum(diff > 0)),
                    "ties": int(np.sum(np.isclose(diff, 0.0))),
                    "tadp_losses": int(np.sum(diff < 0)),
                })

    return pd.DataFrame(rows)


def utility_retention_table(summary: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for k in sorted(summary["k_submissions"].unique()):
        d = summary[summary["k_submissions"].eq(k)].set_index("scenario")
        if "TADP-VR Federated" not in d.index or "Vanilla FedAvg" not in d.index:
            continue

        row = {"k_submissions": int(k)}
        for metric in ["accuracy", "f1_macro", "roc_auc_ovr_macro"]:
            tadp = float(d.loc["TADP-VR Federated", f"{metric}_mean"])
            full = float(d.loc["Vanilla FedAvg", f"{metric}_mean"])
            rand = float(d.loc["Random-K", f"{metric}_mean"])

            row[f"tadp_{metric}_mean"] = tadp
            row[f"fedavg_{metric}_mean"] = full
            row[f"randomk_{metric}_mean"] = rand
            row[f"tadp_minus_fedavg_{metric}"] = tadp - full
            row[f"tadp_minus_randomk_{metric}"] = tadp - rand
            row[f"tadp_retention_pct_of_fedavg_{metric}"] = (
                100.0 * tadp / full if np.isfinite(full) and full != 0 else np.nan
            )

        row["tadp_participants_mean"] = float(
            d.loc["TADP-VR Federated", "participants_mean"]
        )
        row["fedavg_participants_mean"] = float(
            d.loc["Vanilla FedAvg", "participants_mean"]
        )
        row["randomk_participants_mean"] = float(
            d.loc["Random-K", "participants_mean"]
        )
        row["tadp_communication_mb_mean"] = float(
            d.loc["TADP-VR Federated", "communication_mb_mean"]
        )
        row["fedavg_communication_mb_mean"] = float(
            d.loc["Vanilla FedAvg", "communication_mb_mean"]
        )
        row["communication_reduction_pct_vs_fedavg"] = (
            100.0
            * (
                float(d.loc["Vanilla FedAvg", "communication_mb_mean"])
                - float(d.loc["TADP-VR Federated", "communication_mb_mean"])
            )
            / max(
                float(d.loc["Vanilla FedAvg", "communication_mb_mean"]),
                1e-12,
            )
        )
        rows.append(row)

    return pd.DataFrame(rows)


def write_b3_scenario_checkpoint(
    root: Path,
    k: int,
    run_idx: int,
    scenario: str,
    result: Dict[str, Any],
):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", scenario).strip("_")
    cdir = ensure_dir(
        root
        / "scenario_checkpoints"
        / f"K_{int(k):03d}"
        / f"run_{int(run_idx):02d}"
    )
    payload = {
        "k_submissions": int(k),
        "run": int(run_idx),
        "scenario": str(scenario),
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "metrics": result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
    }
    atomic_write_json(payload, cdir / f"{safe}.json")
    if "round_audit" in result:
        pd.DataFrame(result["round_audit"]).to_csv(
            cdir / f"{safe}_rounds.csv",
            index=False,
        )


def save_environment_metadata():
    meta = {
        "experiment_version": EXPERIMENT_VERSION,
        "python": sys.version,
        "platform": platform.platform(),
        "tensorflow": tf.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "client_counts": CLIENT_COUNTS,
        "training_seeds": TRAINING_RUN_SEEDS,
        "fl_rounds": NUM_ROUNDS_FL,
        "local_epochs": LOCAL_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "dirichlet_alpha": DIRICHLET_ALPHA,
        "global_split_seed": GLOBAL_SPLIT_SEED,
        "partition_seed_formula": "PARTITION_BASE_SEED + 101*K",
        "evidence_seed_formula": "EVIDENCE_BASE_SEED + 103*K",
        "randomk_seed_formula": "RANDOMK_BASE_SEED + 107*K",
        "test_semantics":
            "final evaluation only; never used for governance, selection, "
            "preprocessing, step budgets, or checkpoint decisions",
        "scenario_roles": {
            "Vanilla FedAvg":
                "full-participation predictive-utility reference; not compute matched",
            "Random-K":
                "same selected-client count and exact total optimizer-step budget per round as TADP-VR",
            "TADP-VR Federated":
                "governance-selected cohort",
        },
        "k10_note":
            "K=10 intentionally omitted here; use final Experiment-A 4-round rows "
            "for seeds 42,142,242 when constructing the manuscript K=10/20/50/100 plot.",
    }
    atomic_write_json(
        meta,
        EXPERIMENT_ROOT / "b3_experiment_design.json",
    )


# ======================================================================================
# RQ5 — TRUST-DIMENSION ABLATION (K=10, 5 seeds, 4 rounds)
# ======================================================================================
#
# Research question:
#   How does removing individual trust dimensions affect governance decisions and
#   downstream predictive utility?
#
# IMPORTANT DESIGN NOTE
# ---------------------
# This rerun does NOT change the experiment because of a preferred outcome. It reports
# the full-policy leave-one-dimension-out (LODO) result even if the effect is small or
# concentrated in one dimension. A second, clearly labelled HPS-only ablation is added
# to explain whether a dimension matters through the weighted HPS itself or through the
# wider policy gates (dimension floor / critical-factor checks).
#
# Primary analysis:
#   FULL_POLICY_LODO
#     - remove one dimension from HPS;
#     - renormalize remaining HPS weights;
#     - remove that dimension from the dimension-floor gate;
#     - remove critical factors belonging to that dimension.
#
# Secondary diagnostic:
#   HPS_ONLY_ZERO_WEIGHT
#     - set one dimension's HPS weight to zero;
#     - renormalize remaining HPS weights;
#     - KEEP all dimension-floor and critical-factor requirements unchanged.
#
# Frozen main-experiment setting:
#   K=10, evidence seed=1042, partition seed=7101, split seed=7001,
#   training seeds=[42,142,242,342,442], 4 FL rounds, 1 local epoch, batch=64.
# ======================================================================================

import matplotlib.pyplot as plt

EXPERIMENT_VERSION = "TADP-RQ5-v17.1-K10-DIMENSION-ABLATION-5SEED-4ROUND"

K_SUBMISSIONS = 10
TRAINING_RUN_SEEDS = [42, 142, 242, 342, 442]
NUM_ROUNDS_FL = 4
LOCAL_EPOCHS = 1
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
DIRICHLET_ALPHA = 1.0

GLOBAL_SPLIT_SEED = 7001
CLIENT_PARTITION_SEED = 7101
FROZEN_EVIDENCE_ASSIGNMENT_SEED = 1042
DOMAIN = "healthcare"

REFERENCE_LOWER_CUT = 3.0
REFERENCE_UPPER_CUT = 3.5
REFERENCE_DIMENSION_FLOOR = 2.5

USE_GOOGLE_DRIVE_CHECKPOINTS = True
EXPERIMENT_ROOT = choose_experiment_root(
    "TADP_EXPERIMENT_RQ5_" + EXPERIMENT_VERSION,
    use_drive=USE_GOOGLE_DRIVE_CHECKPOINTS,
)
CHECKPOINT_STATE = EXPERIMENT_ROOT / "checkpoint_state.json"
UNIQUE_COHORT_PERF = EXPERIMENT_ROOT / "unique_cohort_performance_checkpoint.csv"

DIMS = [f"dim{i}" for i in range(1, 7)]


def _bool_series_rq5(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s
    return s.astype(str).str.lower().isin(["true", "1", "yes"])


def _jaccard_rq5(a, b) -> float:
    a = set(map(str, a))
    b = set(map(str, b))
    u = a | b
    return 1.0 if not u else float(len(a & b) / len(u))


def build_policy_tables_rq5(full_factor_df: pd.DataFrame):
    f = full_factor_df.copy()
    f["client"] = f["client"].astype(str)
    f["dimension"] = f["dimension"].astype(str)
    f["rubric_score_0_5"] = pd.to_numeric(f["rubric_score_0_5"], errors="raise")

    dim_scores = (
        f.groupby(["client", "dimension"])["rubric_score_0_5"]
        .mean()
        .unstack("dimension")
        .sort_index()
        .reindex(columns=DIMS)
    )

    crit = f.loc[_bool_series_rq5(f["is_critical_factor"])].copy()
    crit["critical_adequacy_min_rank"] = pd.to_numeric(
        crit["critical_adequacy_min_rank"], errors="coerce"
    )
    crit = crit.loc[crit["critical_adequacy_min_rank"].notna()].copy()
    return f, dim_scores, crit


def _renormalize_without_dim(removed_dim: str) -> dict:
    active = [d for d in DIMS if d != removed_dim]
    denom = float(sum(float(WEIGHTS_PSCORE_DEFAULT[d]) for d in active))
    return {
        d: (0.0 if d == removed_dim else float(WEIGHTS_PSCORE_DEFAULT[d]) / denom)
        for d in DIMS
    }


def evaluate_rq5_policy(
    dim_scores: pd.DataFrame,
    crit: pd.DataFrame,
    mode: str = "REFERENCE",
    removed_dim: str | None = None,
) -> tuple[pd.DataFrame, dict]:
    """Evaluate reference, HPS-only, or full-policy dimension ablation."""
    if mode not in {"REFERENCE", "HPS_ONLY_ZERO_WEIGHT", "FULL_POLICY_LODO"}:
        raise ValueError(f"Unknown RQ5 ablation mode: {mode}")
    if mode != "REFERENCE" and removed_dim not in DIMS:
        raise ValueError(f"Invalid removed_dim={removed_dim}")

    if mode == "REFERENCE":
        weights = {d: float(WEIGHTS_PSCORE_DEFAULT[d]) for d in DIMS}
        active_floor_dims = list(DIMS)
        active_critical_dims = list(DIMS)
    elif mode == "HPS_ONLY_ZERO_WEIGHT":
        weights = _renormalize_without_dim(removed_dim)
        # Diagnostic only: isolate the HPS contribution while retaining policy gates.
        active_floor_dims = list(DIMS)
        active_critical_dims = list(DIMS)
    else:  # FULL_POLICY_LODO
        weights = _renormalize_without_dim(removed_dim)
        active_floor_dims = [d for d in DIMS if d != removed_dim]
        active_critical_dims = [d for d in DIMS if d != removed_dim]

    active_crit = crit.loc[crit["dimension"].isin(active_critical_dims)].copy()
    if active_crit.empty:
        global_critical_wac = 0.0
    else:
        global_critical_wac = float(
            np.mean(
                active_crit["critical_adequacy_min_rank"].astype(float).to_numpy()
                / MAX_FACTOR_SCORE
            )
        )

    rows = []
    for client in dim_scores.index.astype(str):
        ds = dim_scores.loc[client]
        hps = float(sum(float(ds[d]) * float(weights[d]) for d in DIMS))

        floor_failures = [
            d for d in active_floor_dims
            if (not np.isfinite(float(ds[d])))
            or float(ds[d]) < REFERENCE_DIMENSION_FLOOR
        ]

        cdf = active_crit.loc[active_crit["client"].astype(str).eq(client)].copy()
        if cdf.empty:
            critical_wac_i = 1.0
            all_critical_meet = True
            below = []
        else:
            scores = cdf["rubric_score_0_5"].astype(float).to_numpy()
            minima = cdf["critical_adequacy_min_rank"].astype(float).to_numpy()
            critical_wac_i = float(np.mean(scores / MAX_FACTOR_SCORE))
            all_critical_meet = bool(np.all(scores >= minima))
            below = [
                f"{r.dimension}.{r.factor}:{float(r.rubric_score_0_5):.1f}"
                f"<{float(r.critical_adequacy_min_rank):.1f}"
                for r in cdf.itertuples()
                if float(r.rubric_score_0_5) < float(r.critical_adequacy_min_rank)
            ]

        if floor_failures:
            action = "REJECT"
            path = "DIMENSION_FLOOR"
        elif hps < REFERENCE_LOWER_CUT:
            action = "REJECT"
            path = "LOW_HPS"
        elif hps >= REFERENCE_UPPER_CUT and all_critical_meet:
            action = "ACCEPT"
            path = "DIRECT_AUTO_ACCEPT"
        elif critical_wac_i >= global_critical_wac:
            action = "ACCEPT"
            path = "ACCEPTED_AFTER_AUTOMATED_REVIEW"
        else:
            action = "REJECT"
            path = "AUTO_REJECTED_REVIEW_CRITICAL_WAC"

        rows.append({
            "client": client,
            "hps": hps,
            "final_action": action,
            "decision_path": path,
            "dimension_floor_failures": ";".join(floor_failures),
            "critical_wac_i": critical_wac_i,
            "global_critical_wac": global_critical_wac,
            "all_critical_meet_adequacy": bool(all_critical_meet),
            "critical_below_adequacy": ";".join(below),
            "ablation_mode": mode,
            "removed_dimension": removed_dim or "",
        })

    return pd.DataFrame(rows), {
        "weights": weights,
        "active_floor_dimensions": active_floor_dims,
        "active_critical_dimensions": active_critical_dims,
        "global_critical_wac": global_critical_wac,
    }


def build_rq5_configurations() -> pd.DataFrame:
    rows = [{
        "configuration": "REFERENCE",
        "ablation_mode": "REFERENCE",
        "removed_dimension": "",
    }]
    for dim in DIMS:
        rows.append({
            "configuration": f"HPS_ONLY_ZERO_{dim}",
            "ablation_mode": "HPS_ONLY_ZERO_WEIGHT",
            "removed_dimension": dim,
        })
    for dim in DIMS:
        rows.append({
            "configuration": f"FULL_POLICY_LODO_{dim}",
            "ablation_mode": "FULL_POLICY_LODO",
            "removed_dimension": dim,
        })
    return pd.DataFrame(rows)


def summarize_rq5_governance(configs, dim_scores, crit):
    ref, _ = evaluate_rq5_policy(dim_scores, crit, "REFERENCE", None)
    ref = ref.sort_values("client").reset_index(drop=True)
    ref_accept = ref.loc[ref["final_action"].eq("ACCEPT"), "client"].astype(str).tolist()
    ref_rate = len(ref_accept) / len(ref)

    gov_by_config = {}
    rows = []
    for cfg in configs.itertuples(index=False):
        removed = str(cfg.removed_dimension).strip() or None
        cur, meta = evaluate_rq5_policy(
            dim_scores, crit, cfg.ablation_mode, removed
        )
        cur = cur.sort_values("client").reset_index(drop=True)
        gov_by_config[cfg.configuration] = cur
        admitted = cur.loc[cur["final_action"].eq("ACCEPT"), "client"].astype(str).tolist()

        comp = ref[["client", "final_action", "hps"]].merge(
            cur[["client", "final_action", "hps"]],
            on="client", suffixes=("_reference", "_current"), validate="one_to_one"
        )
        flips = comp["final_action_reference"].ne(comp["final_action_current"])
        cur_rate = len(admitted) / len(cur)

        rows.append({
            "configuration": cfg.configuration,
            "ablation_mode": cfg.ablation_mode,
            "removed_dimension": removed or "",
            "dimension_name": DIMENSION_NAMES.get(removed, "Reference"),
            "accepted_count": int(len(admitted)),
            "accepted_rate": float(cur_rate),
            "accepted_rate_pct": float(100.0 * cur_rate),
            "accepted_count_change_vs_reference": int(len(admitted) - len(ref_accept)),
            "accepted_rate_change_pp_vs_reference": float(100.0 * (cur_rate - ref_rate)),
            "accepted_rate_relative_change_pct_vs_reference": (
                float(100.0 * (cur_rate - ref_rate) / ref_rate) if ref_rate > 0 else np.nan
            ),
            "decision_flip_count": int(flips.sum()),
            "decision_flip_rate": float(flips.mean()),
            "accepted_set_jaccard_vs_reference": _jaccard_rq5(ref_accept, admitted),
            "mean_abs_hps_change": float(np.mean(np.abs(
                comp["hps_current"].to_numpy(float) - comp["hps_reference"].to_numpy(float)
            ))),
            "admitted_clients": ";".join(admitted),
            "cohort_key": ";".join(sorted(admitted)),
            "weights_json": json.dumps(meta["weights"], sort_keys=True),
        })

    return pd.DataFrame(rows), gov_by_config, ref


def assign_rq5_cohort_ids(gov_summary: pd.DataFrame):
    keys = sorted(gov_summary["cohort_key"].astype(str).unique().tolist())
    mapping = {key: f"COHORT_{i+1:02d}" for i, key in enumerate(keys)}
    out = gov_summary.copy()
    out["cohort_id"] = out["cohort_key"].map(mapping)
    return out, mapping


def train_rq5_unique_cohorts(cohort_mapping, gov_summary, data):
    cohort_clients = {
        cohort_id: [x for x in str(key).split(";") if x]
        for key, cohort_id in cohort_mapping.items()
    }
    train_monitor = build_train_monitor_subset(
        data["client_arrays"],
        max_samples=ROUND_PROGRESS_MONITOR_MAX_SAMPLES,
        seed=99137,
    )
    build_model_fn = lambda: build_diabetes_model(data["meta"]["input_dim"], lr=LEARNING_RATE)
    completed = set(load_checkpoint_state(CHECKPOINT_STATE).get("completed", []))
    total_jobs = len(TRAINING_RUN_SEEDS) * len(cohort_clients)
    job_idx = 0

    for training_seed in TRAINING_RUN_SEEDS:
        seed_everything(training_seed)
        base_model = build_model_fn()
        initial_weights = [np.array(w, copy=True) for w in base_model.get_weights()]
        initial_hash = sha256_weights(initial_weights)
        del base_model
        tf.keras.backend.clear_session(); gc.collect()

        for cohort_id, selected in cohort_clients.items():
            job_idx += 1
            key = f"seed{training_seed}|{cohort_id}"
            if key in completed:
                print(f"CHECKPOINT FOUND -> SKIP: {key}")
                continue
            if not selected:
                raise RuntimeError(f"Empty cohort {cohort_id}")

            print_banner(
                f"RQ5 UNIQUE COHORT {job_idx}/{total_jobs} | seed={training_seed} | {cohort_id}"
            )
            print(f"Selected clients ({len(selected)}): {selected}")

            natural_map = {
                cid: natural_steps(len(data["client_arrays"][cid][1]), BATCH_SIZE, LOCAL_EPOCHS)
                for cid in selected
            }
            result = federated_train(
                build_model_fn=build_model_fn,
                initial_weights=initial_weights,
                selected_per_round=[list(selected) for _ in range(NUM_ROUNDS_FL)],
                client_arrays=data["client_arrays"],
                X_test=data["X_test"], y_test=data["y_test"], n_classes=3,
                batch_size=BATCH_SIZE, local_epochs=LOCAL_EPOCHS,
                class_weights=data["class_weights"], run_seed=training_seed,
                exact_step_maps=[dict(natural_map) for _ in range(NUM_ROUNDS_FL)],
                equal_weight=False, fedprox_mu=0.0, train_monitor=train_monitor,
                progress_context={
                    "scenario": f"RQ5 | {cohort_id}",
                    "run_idx": TRAINING_RUN_SEEDS.index(training_seed)+1,
                    "run_total": len(TRAINING_RUN_SEEDS),
                    "scenario_idx": job_idx, "scenario_total": total_jobs,
                    "overall_idx": job_idx, "overall_total": total_jobs,
                },
            )
            row = result_row(
                run=TRAINING_RUN_SEEDS.index(training_seed)+1,
                seed=training_seed, scenario=cohort_id, result=result,
                initial_hash=initial_hash,
            )
            row.update({
                "training_seed": int(training_seed),
                "cohort_id": cohort_id,
                "selected_k": int(len(selected)),
                "selected_clients": ";".join(selected),
                "steps_per_round": int(sum(natural_map.values())),
                "fl_rounds": int(NUM_ROUNDS_FL),
            })
            upsert_csv(row, UNIQUE_COHORT_PERF, ["training_seed", "cohort_id"])
            mark_checkpoint_complete(CHECKPOINT_STATE, key, extra={"last_job": key})
            completed.add(key)
            del result
            tf.keras.backend.clear_session(); gc.collect()

    perf = pd.read_csv(UNIQUE_COHORT_PERF)
    expected = len(TRAINING_RUN_SEEDS) * len(cohort_clients)
    if len(perf) != expected:
        raise RuntimeError(f"Expected {expected} performance rows; found {len(perf)}")

    audits = []
    for seed in TRAINING_RUN_SEEDS:
        d = perf[perf["training_seed"].eq(seed)]
        hashes = d["initial_weights_sha256"].astype(str).unique()
        passed = len(hashes) == 1
        audits.append({
            "training_seed": seed,
            "n_unique_cohorts": len(d),
            "unique_W0_hashes": len(hashes),
            "same_W0_across_cohorts": passed,
            "initial_weights_sha256": hashes[0] if len(hashes) else "",
        })
        if not passed:
            raise RuntimeError(f"W0 parity failed for seed {seed}")
    pd.DataFrame(audits).to_csv(EXPERIMENT_ROOT / "rq5_W0_parity_audit.csv", index=False)
    return perf


def summarize_rq5_performance(gov_summary, cohort_perf):
    expanded = gov_summary.merge(cohort_perf, on="cohort_id", how="left", validate="many_to_many")
    expanded.to_csv(EXPERIMENT_ROOT / "rq5_performance_all_seeds.csv", index=False)

    metrics = [
        "roc_auc_ovr_macro", "f1_macro", "accuracy", "precision_macro", "recall_macro",
        "runtime_s", "communication_mb", "optimizer_steps", "participants", "energy_wh",
    ]
    rows = []
    for cfg, d in expanded.groupby("configuration", sort=False):
        g = gov_summary[gov_summary["configuration"].eq(cfg)].iloc[0]
        row = {
            "configuration": cfg,
            "ablation_mode": g["ablation_mode"],
            "removed_dimension": g["removed_dimension"],
            "dimension_name": g["dimension_name"],
            "accepted_count": int(g["accepted_count"]),
            "accepted_rate_pct": float(g["accepted_rate_pct"]),
            "accepted_rate_change_pp_vs_reference": float(g["accepted_rate_change_pp_vs_reference"]),
            "decision_flip_count": int(g["decision_flip_count"]),
            "accepted_set_jaccard_vs_reference": float(g["accepted_set_jaccard_vs_reference"]),
            "admitted_clients": g["admitted_clients"],
            "cohort_id": g["cohort_id"],
            "n_training_seeds": int(len(d)),
        }
        for metric in metrics:
            if metric not in d.columns:
                continue
            x = pd.to_numeric(d[metric], errors="coerce").dropna().to_numpy(float)
            row[f"{metric}_mean"] = float(np.mean(x)) if len(x) else np.nan
            row[f"{metric}_sd"] = float(np.std(x, ddof=1)) if len(x)>1 else 0.0
            lo, hi = ci95_mean(x)
            row[f"{metric}_ci95_low"] = lo
            row[f"{metric}_ci95_high"] = hi
        rows.append(row)

    summary = pd.DataFrame(rows)
    ref = summary[summary["configuration"].eq("REFERENCE")].iloc[0]
    for metric in ["roc_auc_ovr_macro", "f1_macro", "accuracy"]:
        base = float(ref[f"{metric}_mean"])
        summary[f"{metric}_delta_vs_reference"] = summary[f"{metric}_mean"] - base
        summary[f"{metric}_change_pct_vs_reference"] = (
            100.0 * summary[f"{metric}_delta_vs_reference"] / base
        )
    summary.to_csv(EXPERIMENT_ROOT / "rq5_dimension_ablation_summary.csv", index=False)

    # Paired seed-level differences versus reference.
    ref_seed = expanded[expanded["configuration"].eq("REFERENCE")][
        ["training_seed", "roc_auc_ovr_macro", "f1_macro", "accuracy"]
    ].copy()
    paired_rows = []
    for cfg, d in expanded.groupby("configuration", sort=False):
        if cfg == "REFERENCE":
            continue
        m = d.merge(ref_seed, on="training_seed", suffixes=("_config", "_reference"), validate="one_to_one")
        for metric in ["roc_auc_ovr_macro", "f1_macro", "accuracy"]:
            diff = m[f"{metric}_config"].to_numpy(float) - m[f"{metric}_reference"].to_numpy(float)
            lo, hi = ci95_mean(diff)
            paired_rows.append({
                "configuration": cfg,
                "metric": metric,
                "n_pairs": len(diff),
                "mean_difference": float(np.mean(diff)),
                "sd_difference": float(np.std(diff, ddof=1)) if len(diff)>1 else 0.0,
                "ci95_low": lo, "ci95_high": hi,
                "wins": int(np.sum(diff>0)), "ties": int(np.sum(np.isclose(diff,0))),
                "losses": int(np.sum(diff<0)),
            })
    pd.DataFrame(paired_rows).to_csv(EXPERIMENT_ROOT / "rq5_paired_differences_vs_reference.csv", index=False)
    return expanded, summary


def create_rq5_figure(summary: pd.DataFrame):
    plot = summary[summary["ablation_mode"].ne("REFERENCE")].copy()
    fig = plt.figure(figsize=(14, 10))
    gs = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.28)

    modes = [
        ("HPS_ONLY_ZERO_WEIGHT", "HPS-only"),
        ("FULL_POLICY_LODO", "Full-policy LODO"),
    ]
    dims = DIMS
    names = [DIMENSION_NAMES[d] for d in dims]
    x = np.arange(len(dims), dtype=float)
    width = 0.36

    ax1 = fig.add_subplot(gs[0,0])
    for i, (mode, label) in enumerate(modes):
        vals = []
        for dim in dims:
            d = plot[(plot["ablation_mode"].eq(mode)) & (plot["removed_dimension"].eq(dim))]
            vals.append(float(d["accepted_rate_change_pp_vs_reference"].iloc[0]))
        ax1.bar(x + (i-0.5)*width, vals, width, label=label)
    ax1.axhline(0, linewidth=0.8)
    ax1.set_xticks(x); ax1.set_xticklabels(names, rotation=30, ha="right")
    ax1.set_ylabel("Change in admitted clients (percentage points)")
    ax1.set_title("A. Governance impact of dimension ablation", fontweight="bold")
    ax1.legend(fontsize=8)

    panels = [
        (gs[0,1], "roc_auc_ovr_macro_delta_vs_reference", "B. Change in Macro ROC-AUC", "Δ Macro ROC-AUC"),
        (gs[1,0], "f1_macro_delta_vs_reference", "C. Change in Macro-F1", "Δ Macro-F1"),
        (gs[1,1], "accuracy_delta_vs_reference", "D. Change in Accuracy", "Δ Accuracy"),
    ]
    for cell, metric, title, ylabel in panels:
        ax = fig.add_subplot(cell)
        for i, (mode, label) in enumerate(modes):
            vals = []
            for dim in dims:
                d = plot[(plot["ablation_mode"].eq(mode)) & (plot["removed_dimension"].eq(dim))]
                vals.append(float(d[metric].iloc[0]))
            ax.bar(x + (i-0.5)*width, vals, width, label=label)
        ax.axhline(0, linewidth=0.8)
        ax.set_xticks(x); ax.set_xticklabels(names, rotation=30, ha="right")
        ax.set_ylabel(ylabel); ax.set_title(title, fontweight="bold")

    fig.suptitle("RQ5 — Trust-Dimension Ablation under Frozen K=10 TADP-VR", fontsize=14, fontweight="bold")
    out = EXPERIMENT_ROOT / "RQ5_dimension_ablation_figure.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return out


def main():
    print_banner(EXPERIMENT_VERSION)
    print("RQ5 primary analysis: FULL_POLICY_LODO")
    print("RQ5 secondary diagnostic: HPS_ONLY_ZERO_WEIGHT")
    print(f"K={K_SUBMISSIONS} | seeds={TRAINING_RUN_SEEDS} | rounds={NUM_ROUNDS_FL}")

    atomic_write_json({
        "experiment_version": EXPERIMENT_VERSION,
        "research_question": "How does removing individual trust dimensions affect governance decisions and downstream predictive utility?",
        "primary_analysis": "FULL_POLICY_LODO",
        "secondary_diagnostic": "HPS_ONLY_ZERO_WEIGHT",
        "k_submissions": K_SUBMISSIONS,
        "training_seeds": TRAINING_RUN_SEEDS,
        "fl_rounds": NUM_ROUNDS_FL,
        "local_epochs": LOCAL_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "dirichlet_alpha": DIRICHLET_ALPHA,
        "global_split_seed": GLOBAL_SPLIT_SEED,
        "client_partition_seed": CLIENT_PARTITION_SEED,
        "frozen_evidence_assignment_seed": FROZEN_EVIDENCE_ASSIGNMENT_SEED,
        "reference_weights": WEIGHTS_PSCORE_DEFAULT,
        "reference_lower_cut": REFERENCE_LOWER_CUT,
        "reference_upper_cut": REFERENCE_UPPER_CUT,
        "reference_dimension_floor": REFERENCE_DIMENSION_FLOOR,
        "interpretation": "Pre-specified ablation. Report all dimensions and both ablation modes; do not select results based on predictive performance.",
    }, EXPERIMENT_ROOT / "rq5_experiment_design.json")

    csv_path = locate_diabetes_csv()
    data = prepare_diabetes_no_leakage(
        csv_path=csv_path, split_seed=GLOBAL_SPLIT_SEED,
        partition_seed=CLIENT_PARTITION_SEED,
        n_clients=K_SUBMISSIONS, alpha=DIRICHLET_ALPHA,
    )

    leakage = write_leakage_audit(
        EXPERIMENT_ROOT, data["global_train_ids"], data["global_test_ids"], data["client_train_ids"],
        extra={
            "patient_overlap": data["meta"]["patient_overlap"],
            "global_holdout_before_client_partition": True,
            "preprocessing_train_only": True,
            "dq_tadp_train_only": True,
            "class_weights_train_only": True,
            "test_used_for_final_evaluation_only": True,
            "rq5_frozen_main_setting": True,
        },
    )
    if not bool(leakage.get("pass", False)):
        raise RuntimeError(f"No-leakage audit failed: {leakage}")

    client_ids = list(data["client_ids"])
    if client_ids != list("ABCDEFGHIJ"):
        raise RuntimeError(f"Expected A-J clients, got {client_ids}")

    documentary, evidence_df = generate_scalable_controlled_evidence(
        client_ids, FROZEN_EVIDENCE_ASSIGNMENT_SEED, domain=DOMAIN
    )
    validate_expected_profile_mix(evidence_df, K_SUBMISSIONS)
    evidence_df.to_csv(EXPERIMENT_ROOT / "rq5_frozen_controlled_evidence.csv", index=False)
    data["dq_audit"].to_csv(EXPERIMENT_ROOT / "rq5_frozen_DQ_audit.csv", index=False)

    original_gov = build_tadp_governance(
        client_ids, documentary, data["dq_scores"], run=0,
        evidence_seed=FROZEN_EVIDENCE_ASSIGNMENT_SEED, domain=DOMAIN,
    ).sort_values("client").reset_index(drop=True)
    original_gov.to_csv(EXPERIMENT_ROOT / "rq5_original_frozen_governance.csv", index=False)

    full_factor_df = build_full_factor_evidence_table(
        client_ids, documentary, data["dq_scores"], data["dq_audit"],
        FROZEN_EVIDENCE_ASSIGNMENT_SEED, DOMAIN,
    )
    full_factor_df.to_csv(EXPERIMENT_ROOT / "rq5_frozen_all_28_factor_evidence.csv", index=False)
    _, dim_scores, crit = build_policy_tables_rq5(full_factor_df)

    # Fail closed: reference recomputation must exactly reproduce the frozen main governance.
    ref_recomputed, _ = evaluate_rq5_policy(dim_scores, crit, "REFERENCE", None)
    ref_recomputed = ref_recomputed.sort_values("client").reset_index(drop=True)
    audit = ref_recomputed[["client", "hps", "final_action"]].merge(
        original_gov[["client", "hps", "final_action"]], on="client",
        suffixes=("_recomputed", "_original"), validate="one_to_one"
    )
    audit["decision_match"] = audit["final_action_recomputed"].eq(audit["final_action_original"])
    audit["hps_abs_diff"] = np.abs(audit["hps_recomputed"] - audit["hps_original"])
    audit.to_csv(EXPERIMENT_ROOT / "rq5_reference_policy_reproduction_audit.csv", index=False)
    if not audit["decision_match"].all() or float(audit["hps_abs_diff"].max()) > 1e-10:
        raise RuntimeError("RQ5 reference-policy reproduction failed")

    configs = build_rq5_configurations()
    configs.to_csv(EXPERIMENT_ROOT / "rq5_requested_configurations.csv", index=False)
    gov_summary, gov_by_config, _ = summarize_rq5_governance(configs, dim_scores, crit)
    gov_summary, cohort_mapping = assign_rq5_cohort_ids(gov_summary)
    gov_summary.to_csv(EXPERIMENT_ROOT / "rq5_governance_summary.csv", index=False)

    frames = []
    for cfg, df in gov_by_config.items():
        x = df.copy(); x.insert(0, "configuration", cfg); frames.append(x)
    pd.concat(frames, ignore_index=True).to_csv(
        EXPERIMENT_ROOT / "rq5_governance_client_level.csv", index=False
    )

    print_banner("RQ5 GOVERNANCE SUMMARY")
    print(gov_summary[[
        "configuration", "ablation_mode", "removed_dimension", "dimension_name",
        "accepted_count", "accepted_rate_pct", "accepted_rate_change_pp_vs_reference",
        "decision_flip_count", "accepted_set_jaccard_vs_reference", "admitted_clients"
    ]].to_string(index=False))

    cohort_perf = train_rq5_unique_cohorts(cohort_mapping, gov_summary, data)
    _, perf_summary = summarize_rq5_performance(gov_summary, cohort_perf)
    fig = create_rq5_figure(perf_summary)

    print_banner("RQ5 PERFORMANCE SUMMARY")
    print(perf_summary[[
        "configuration", "ablation_mode", "removed_dimension", "accepted_count",
        "roc_auc_ovr_macro_mean", "roc_auc_ovr_macro_delta_vs_reference",
        "f1_macro_mean", "f1_macro_delta_vs_reference",
        "accuracy_mean", "accuracy_delta_vs_reference",
    ]].to_string(index=False))

    atomic_write_json({
        "reference_policy_reproduction": "PASS",
        "no_leakage": "PASS",
        "same_W0_within_seed": "PASS",
        "n_policy_configurations": int(len(configs)),
        "n_unique_admitted_cohorts": int(len(cohort_mapping)),
        "actual_model_training_runs": int(len(cohort_mapping) * len(TRAINING_RUN_SEEDS)),
        "figure": str(fig),
    }, EXPERIMENT_ROOT / "rq5_final_audit_summary.json")

    return EXPERIMENT_ROOT


if __name__ == "__main__":
    finished_root = main()
    package_and_download_results(finished_root, EXPERIMENT_VERSION)


Google Drive checkpoint mount unavailable: mount failed
Local checkpoint root: /content/TADP_EXPERIMENT_B3_TADP-B3-v16.9-PREDICTIVE-SCALABILITY-K20-50-100-3SEED-4ROUND
Google Drive checkpoint mount unavailable: mount failed
Local checkpoint root: /content/TADP_EXPERIMENT_RQ5_TADP-RQ5-v17.1-K10-DIMENSION-ABLATION-5SEED-4ROUND

TADP-RQ5-v17.1-K10-DIMENSION-ABLATION-5SEED-4ROUND
RQ5 primary analysis: FULL_POLICY_LODO
RQ5 secondary diagnostic: HPS_ONLY_ZERO_WEIGHT
K=10 | seeds=[42, 142, 242, 342, 442] | rounds=4

RQ5 GOVERNANCE SUMMARY
        configuration        ablation_mode removed_dimension                      dimension_name  accepted_count  accepted_rate_pct  accepted_rate_change_pp_vs_reference  decision_flip_count  accepted_set_jaccard_vs_reference admitted_clients
            REFERENCE            REFERENCE                                             Reference               6               60.0                                   0.0                    0                           1.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

RQ6


In [4]:
# ======================================================================================
# TADP v16.7 — REVIEWER-ALIGNED TRUSTWORTHY DATA PREPARATION EXPERIMENT CORE
# ======================================================================================
# Design guarantees:
#   1) GLOBAL holdout is created BEFORE client partitioning.
#   2) Only TRAIN is distributed to clients.
#   3) Preprocessing parameters/vocabularies use TRAIN only.
#   4) DQ, GE and TADP governance use TRAIN only.
#   5) Held-out TEST never affects preprocessing, governance, class weights,
#      client selection, model initialization, training, or matched-control budgets.
#   6) TEST is used only after training for final evaluation.
#
# v16.7 governance:
#   - 28 factors across 6 dimensions:
#       dim1=4, dim2(DQ)=8, dim3=4, dim4=3, dim5=5, dim6=4.
#   - Added Data Collection / Acquisition Lineage (dim1).
#   - Added Structural / Constraint Integrity (dim2).
#   - Documentary evidence is generated at the individual factor level.
#   - DQ evidence is machine-measured from client TRAIN partitions only.
#   - HPS remains client-specific and uses the six policy dimension weights.
#   - WAC is NOT client-specific.
#   - Each factor has a declared minimum adequate rubric rank.
#   - A domain WAC is derived once:
#         WAC_d = mean(minimum adequate ranks in dimension d) / 5
#         WAC_domain = equal mean of the six WAC_d values.
#   - Every averaged dimension must be >= 2.5/5 or the client is auto-rejected.
#   - HPS < 3.0 -> AUTO_REJECT.
#   - 3.0 <= HPS < 3.5 -> AUTOMATED REVIEW using Critical WAC_i.
#   - Review accepts iff Critical WAC_i >= the domain Global Critical WAC.
#   - HPS >= 3.5 -> DIRECT AUTO_ACCEPT when every critical factor meets its own adequacy minimum.
#   - Human reviewers verify evidence only; admission is server-automated.
#
# GX Core comparator:
#   - Separate from HPS/TADP.
#   - Uses REAL Great Expectations GX Core validation on TRAIN-only client data.
#   - Uses common technical checks: schema, datatype consistency, required ranges,
#     missingness, duplicate/ID integrity, label/domain validity, and structure.
#   - Uses GX native severity-aware validation: zero critical failures required; no ranking and no forced-K.
#
# Runtime/reporting:
#   - Every FL round prints configuration/run/scenario/round progress plus TRAIN-only diagnostic utility and operational metrics.
#   - Every completed scenario prints all predictive and operational metrics.
#   - Scenario checkpoints support restart/resume.
#   - Final result ZIP downloads automatically in Google Colab.
# ======================================================================================

import os
import sys
import gc
import math
import time
import json
import random
import hashlib
import threading
import zipfile
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# ======================================================================================
# POLICY CONSTANTS — v16.7
# ======================================================================================

GOOD_CUT = 3.0
HIGH_CUT = 3.5
MAX_FACTOR_SCORE = 5.0
GE_ACCEPT_COUNT = 6

# ----------------------------------------------------------------------
# v16.7 FINAL FULLY AUTOMATED ADMISSION POLICY
# ----------------------------------------------------------------------
# Human reviewers verify supporting evidence uploaded through the questionnaire.
# They do NOT make the admission decision. Once verified factor scores are
# available, the server applies this policy automatically.
#
# 1) EVERY averaged HPS dimension must be >= 2.5/5.
#    Any dimension < 2.5 -> AUTO-REJECT.
#
# 2) HPS must be >= 3.0.
#    HPS < 3.0 -> AUTO-REJECT.
#
# 3) DIRECT AUTO-ACCEPT:
#    HPS >= 3.5 AND every critical factor independently meets its own
#    factor-specific minimum adequacy requirement.
#
# 4) AUTOMATED REVIEW:
#    - all clients with 3.0 <= HPS < 3.5; and
#    - high-HPS clients that fail one or more individual critical-factor
#      adequacy requirements.
#
# 5) REVIEW RESOLUTION:
#    Critical WAC_i = mean(actual critical-factor scores / 5)
#    Global Critical WAC = mean(policy adequacy minima / 5 for same factors)
#
#    Critical WAC_i >= Global Critical WAC -> ACCEPT AFTER REVIEW
#    Critical WAC_i <  Global Critical WAC -> AUTO-REJECT
#
# Thus Direct Auto-Accept is strict at the individual critical-factor level,
# whereas Review intentionally allows controlled compensation across the
# critical subset.
DIMENSION_MIN_FLOOR = 2.5

CRITICAL_FACTORS_BY_DOMAIN = {
    "healthcare": {
        "dim1": [
            "data_controller",
            "data_collection_lineage",
        ],
        "dim5": [
            "regulation_coverage",
            "consent_ethics",
            "sensitivity_classification",
        ],
        "dim6": [
            "user_agreements",
        ],
    },
    "cifar10": {
        "dim1": [
            "data_controller",
            "data_collection_lineage",
        ],
        "dim6": [
            "license_terms",
            "user_agreements",
        ],
    },
}

WEIGHTS_PSCORE_DEFAULT = {
    "dim1": 0.25,  # Source Reliability
    "dim2": 0.15,  # Data Quality and Health
    "dim3": 0.10,  # Documentation Practices
    "dim4": 0.10,  # Timeliness and Refresh Rate
    "dim5": 0.30,  # Regulatory / Compliance Alignment
    "dim6": 0.10,  # Context / Usage Constraints
}

DIMENSION_NAMES = {
    "dim1": "Source Reliability",
    "dim2": "Data Quality and Health",
    "dim3": "Documentation Practices",
    "dim4": "Timeliness and Refresh Rate",
    "dim5": "Regulatory and Compliance Alignment",
    "dim6": "Context and Usage Constraints",
}

# v16.7: two reviewer-driven additions:
#   dim1: data_collection_lineage
#   dim2: structural_constraint_integrity
#
# Total = 4 + 8 + 4 + 3 + 5 + 4 = 28 factors.
FACTOR_NAMES = {
    "dim1": [
        "source_reputation",
        "data_controller",
        "data_objective",
        "data_collection_lineage",
    ],
    "dim2": [
        "completeness",
        "duplication_rate",
        "value_validity_error_rate",
        "type_consistency",
        "label_integrity",
        "feature_distribution_consistency",
        "feature_category_coverage",
        "structural_constraint_integrity",
    ],
    "dim3": [
        "data_dictionary",
        "version_logs",
        "collection_protocol",
        "definition_updates",
    ],
    "dim4": [
        "data_freshness",
        "scheduled_refresh",
        "retention_clarity",
    ],
    "dim5": [
        "regulation_coverage",
        "consent_ethics",
        "geo_restrictions",
        "sensitivity_classification",
        "audits",
    ],
    "dim6": [
        "license_terms",
        "ethical_reviews",
        "redistribution",
        "user_agreements",
    ],
}

DOCUMENTARY_DIMS = ("dim1", "dim3", "dim4", "dim5", "dim6")

# ----------------------------------------------------------------------
# COMPLETE 0--5 RUBRIC DESCRIPTORS
# ----------------------------------------------------------------------
# These descriptors reproduce the Appendix-A semantics and add the two
# v16.7 factors explicitly. Controlled evidence is sampled at the factor
# level; HPS is never generated directly.
RUBRIC_DESCRIPTORS = {
    "dim1": {
        "source_reputation": {
            0: "No info",
            1: "Poor",
            2: "Limited evidence",
            3: "Average, partially trusted",
            4: "Well-documented, reliable",
            5: "Highly reputable, verified",
        },
        "data_controller": {
            0: "No documented controller",
            1: "Unclear",
            2: "Partially clear",
            3: "Moderately clear",
            4: "Mostly clear",
            5: "Fully documented",
        },
        "data_objective": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General but unclear",
            4: "Mostly explicit",
            5: "Fully explicit, justified",
        },
        "data_collection_lineage": {
            0: "Collection origin unknown",
            1: "Informal or unverifiable origin",
            2: "Partially documented acquisition path",
            3: "Documented acquisition with limited traceability",
            4: "Well-documented and traceable acquisition path",
            5: "Fully source-linked, versioned, and auditable lineage",
        },
    },
    "dim2": {
        "completeness": {
            0: ">50% missing",
            1: "20-50% missing",
            2: "10-20% missing",
            3: "5-10% missing",
            4: "1-5% missing",
            5: "<1% missing",
        },
        "duplication_rate": {
            0: ">20% duplicates",
            1: "10-20% duplicates",
            2: "5-10% duplicates",
            3: "2-5% duplicates",
            4: "1-2% duplicates",
            5: "<1% duplicates",
        },
        "value_validity_error_rate": {
            0: ">15% invalid/error values",
            1: "10-15% invalid/error values",
            2: "5-10% invalid/error values",
            3: "2-5% invalid/error values",
            4: "1-2% invalid/error values",
            5: "<1% invalid/error values",
        },
        "type_consistency": {
            0: "Highly inconsistent",
            1: "Frequent type inconsistency",
            2: "Moderate type inconsistency",
            3: "Minor type inconsistency",
            4: "Rare type inconsistency",
            5: "Fully consistent",
        },
        "label_integrity": {
            0: ">10% missing/invalid/known erroneous labels",
            1: "5-10% missing/invalid/known erroneous labels",
            2: "2-5% missing/invalid/known erroneous labels",
            3: "1-2% missing/invalid/known erroneous labels",
            4: "0.1-1% missing/invalid/known erroneous labels",
            5: "<=0.1% missing/invalid/known erroneous labels",
        },
        "feature_distribution_consistency": {
            0: "JSD >0.20",
            1: "JSD 0.10-0.20",
            2: "JSD 0.05-0.10",
            3: "JSD 0.025-0.05",
            4: "JSD 0.01-0.025",
            5: "JSD <=0.01",
        },
        "feature_category_coverage": {
            0: "<50% reference support represented",
            1: "50-65% reference support represented",
            2: "65-75% reference support represented",
            3: "75-82.5% reference support represented",
            4: "82.5-90% reference support represented",
            5: ">=90% reference support represented",
        },
        "structural_constraint_integrity": {
            0: ">10% records/structures violate required constraints",
            1: "5-10% violate required constraints",
            2: "2-5% violate required constraints",
            3: "1-2% violate required constraints",
            4: "0.1-1% violate required constraints",
            5: "<=0.1% violate required constraints",
        },
    },
    "dim3": {
        "data_dictionary": {
            0: "None",
            1: "Minimal outline",
            2: "Partial coverage",
            3: "Moderate coverage",
            4: "Near-complete",
            5: "Fully detailed",
        },
        "version_logs": {
            0: "None",
            1: "Minimal logs",
            2: "Occasional logs",
            3: "Regular logs",
            4: "Near-complete",
            5: "Full version history",
        },
        "collection_protocol": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General methods",
            4: "Well-defined",
            5: "Fully transparent",
        },
        "definition_updates": {
            0: "None",
            1: "Rarely updated",
            2: "Occasional updates",
            3: "Regular but basic",
            4: "Frequent",
            5: "Real-time, documented",
        },
    },
    "dim4": {
        "data_freshness": {
            0: ">5 years old",
            1: "2-5 years old",
            2: "1-2 years old",
            3: "6-12 months old",
            4: "1-6 months old",
            5: "Real-time/current",
        },
        "scheduled_refresh": {
            0: "Never",
            1: "Irregular",
            2: "Annual",
            3: "Quarterly",
            4: "Monthly",
            5: "Daily/real-time",
        },
        "retention_clarity": {
            0: "None",
            1: "Minimal",
            2: "Basic guidelines",
            3: "Moderate clarity",
            4: "High clarity",
            5: "Fully documented",
        },
    },
    "dim5": {
        "regulation_coverage": {
            0: "None",
            1: "Minimal",
            2: "Partial",
            3: "Moderate",
            4: "Comprehensive but dated",
            5: "Fully documented/current",
        },
        "consent_ethics": {
            0: "None",
            1: "Minimal record",
            2: "Partial consent/ethics evidence",
            3: "Moderate logs",
            4: "Substantial",
            5: "Fully documented",
        },
        "geo_restrictions": {
            0: "None",
            1: "Basic mention",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully documented",
        },
        "sensitivity_classification": {
            0: "None",
            1: "Basic flagging",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully classified",
        },
        "audits": {
            0: "None",
            1: "Internal only",
            2: "Basic certification",
            3: "Occasional audit",
            4: "Recent audit",
            5: "Regular external audits",
        },
    },
    "dim6": {
        "license_terms": {
            0: "None",
            1: "Vague",
            2: "Basic",
            3: "Clear",
            4: "Detailed",
            5: "Industry-compliant",
        },
        "ethical_reviews": {
            0: "None",
            1: "Informal approval",
            2: "Partial",
            3: "Moderate",
            4: "Well-documented",
            5: "Certified",
        },
        "redistribution": {
            0: "No policy",
            1: "Unclear",
            2: "Partial",
            3: "Clear",
            4: "Detailed",
            5: "Fully compliant",
        },
        "user_agreements": {
            0: "Non-compliant",
            1: "Minimal adherence",
            2: "Partial",
            3: "Mostly compliant",
            4: "Fully compliant",
            5: "Audited compliance",
        },
    },
}

# ----------------------------------------------------------------------
# FACTOR-SPECIFIC ADEQUACY POLICY
# ----------------------------------------------------------------------
# The minimum adequate rank is derived factor-by-factor from the wording of
# the Appendix-A rubric. It is NOT learned from model/test outcomes.
#
# Healthcare is the primary policy. CIFAR-10 uses the same reference ranks
# for cross-domain comparability; its DQ factors are measured with image-
# specific checks, while documentary dimensions remain controlled evidence.
FACTOR_ADEQUACY_MIN_HEALTHCARE = {
    "dim1": {
        "source_reputation": 4,
        "data_controller": 4,
        "data_objective": 4,
        "data_collection_lineage": 4,
    },
    "dim2": {
        "completeness": 3,
        "duplication_rate": 3,
        "value_validity_error_rate": 3,
        "type_consistency": 3,
        "label_integrity": 3,
        "feature_distribution_consistency": 3,
        "feature_category_coverage": 3,
        "structural_constraint_integrity": 3,
    },
    "dim3": {
        "data_dictionary": 3,
        "version_logs": 3,
        "collection_protocol": 4,
        "definition_updates": 3,
    },
    "dim4": {
        "data_freshness": 3,
        "scheduled_refresh": 3,
        "retention_clarity": 4,
    },
    "dim5": {
        "regulation_coverage": 3,
        "consent_ethics": 3,
        "geo_restrictions": 3,
        "sensitivity_classification": 3,
        "audits": 4,
    },
    "dim6": {
        "license_terms": 3,
        "ethical_reviews": 4,
        "redistribution": 3,
        "user_agreements": 4,
    },
}

FACTOR_ADEQUACY_MIN_CIFAR10 = {
    dim: dict(values)
    for dim, values in FACTOR_ADEQUACY_MIN_HEALTHCARE.items()
}

def derive_domain_wac(
    factor_minima: Dict[str, Dict[str, float]]
) -> Tuple[Dict[str, float], float]:
    """
    Derive the domain policy WAC from factor-specific minimum adequate ranks.

    WAC_d = mean_k(adequate_rank_dk / 5)
    WAC_domain = equal mean across the six dimension WAC_d values.

    IMPORTANT:
      WAC is a DOMAIN POLICY value, not a client-specific score.
    """
    dimension_wac = {}
    for dim in FACTOR_NAMES:
        vals = [
            float(factor_minima[dim][factor])
            for factor in FACTOR_NAMES[dim]
        ]
        dimension_wac[dim] = float(np.mean(vals) / MAX_FACTOR_SCORE)
    global_wac = float(np.mean(list(dimension_wac.values())))
    return dimension_wac, global_wac


HEALTHCARE_DIMENSION_WAC, WAC_HEALTHCARE = derive_domain_wac(
    FACTOR_ADEQUACY_MIN_HEALTHCARE
)
CIFAR10_DIMENSION_WAC, WAC_CIFAR10 = derive_domain_wac(
    FACTOR_ADEQUACY_MIN_CIFAR10
)

DOMAIN_FACTOR_MINIMA = {
    "healthcare": FACTOR_ADEQUACY_MIN_HEALTHCARE,
    "cifar10": FACTOR_ADEQUACY_MIN_CIFAR10,
}
DOMAIN_DIMENSION_WAC = {
    "healthcare": HEALTHCARE_DIMENSION_WAC,
    "cifar10": CIFAR10_DIMENSION_WAC,
}
DOMAIN_GLOBAL_WAC = {
    "healthcare": WAC_HEALTHCARE,
    "cifar10": WAC_CIFAR10,
}

DQ_RAW_METRIC_BY_FACTOR = {
    "completeness": "missing_fraction",
    "duplication_rate": "duplicate_fraction",
    "value_validity_error_rate": "error_fraction",
    "type_consistency": "type_inconsistency_fraction",
    "label_integrity": "invalid_label_fraction",
    "feature_distribution_consistency": "max_jsd",
    "feature_category_coverage": "mean_category_coverage",
    "structural_constraint_integrity": "structural_violation_fraction",
}

assert abs(sum(WEIGHTS_PSCORE_DEFAULT.values()) - 1.0) < 1e-12
assert sum(len(v) for v in FACTOR_NAMES.values()) == 28
assert len(FACTOR_NAMES["dim1"]) == 4
assert len(FACTOR_NAMES["dim2"]) == 8
assert abs(WAC_HEALTHCARE - 0.6761111111111111) < 1e-12

# ======================================================================================
# GENERAL UTILITIES
# ======================================================================================

def seed_everything(seed: int):
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        tf.keras.utils.set_random_seed(seed)
    except Exception:
        tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def sha256_bytes(x: bytes) -> str:
    return hashlib.sha256(x).hexdigest()


def sha256_array(x: np.ndarray) -> str:
    x = np.asarray(x)
    return sha256_bytes(np.ascontiguousarray(x).view(np.uint8).tobytes())


def sha256_weights(weights: List[np.ndarray]) -> str:
    h = hashlib.sha256()
    for w in weights:
        a = np.ascontiguousarray(np.asarray(w))
        h.update(str(a.shape).encode())
        h.update(a.view(np.uint8).tobytes())
    return h.hexdigest()


def _process_rss_mb() -> float:
    try:
        import psutil
        return float(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2))
    except Exception:
        pass
    try:
        with open("/proc/self/statm", "r", encoding="utf-8") as f:
            pages = int(f.read().split()[1])
        return float(pages * int(os.sysconf("SC_PAGE_SIZE")) / (1024 ** 2))
    except Exception:
        pass
    try:
        import resource
        x = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
        return x / (1024 ** 2) if sys.platform == "darwin" else x / 1024.0
    except Exception:
        return 0.0


class RAMMonitor:
    def __init__(self, interval_s: float = 0.05):
        self.interval_s = float(interval_s)
        self.start_mb = 0.0
        self.end_mb = 0.0
        self.peak_mb = 0.0
        self._stop = threading.Event()
        self._thread = None

    def _loop(self):
        while not self._stop.wait(self.interval_s):
            self.peak_mb = max(self.peak_mb, _process_rss_mb())

    def start(self):
        self.start_mb = _process_rss_mb()
        self.peak_mb = self.start_mb
        self._stop.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def stop(self) -> Dict[str, float]:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=1.0)
        self.end_mb = _process_rss_mb()
        self.peak_mb = max(self.peak_mb, self.start_mb, self.end_mb)
        return {
            "ram_start_mb": float(self.start_mb),
            "ram_end_mb": float(self.end_mb),
            "ram_peak_mb": float(self.peak_mb),
            "ram_delta_mb": float(max(0.0, self.peak_mb - self.start_mb)),
            "ram_mb": float(self.peak_mb),
        }


def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
    return Path(path)


def append_csv(row: Dict[str, Any], path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([row]).to_csv(
        path, mode="a", header=not path.exists(), index=False
    )


def summarize_runs(perf: pd.DataFrame) -> pd.DataFrame:
    if perf.empty:
        return pd.DataFrame()
    metrics = [
        c for c in [
            "accuracy", "precision_macro", "recall_macro", "f1_macro",
            "roc_auc_ovr_macro", "runtime_s", "energy_wh", "communication_mb",
            "optimizer_steps", "participants", "ram_peak_mb", "ram_delta_mb"
        ] if c in perf.columns
    ]
    rows = []
    for scenario, d in perf.groupby("scenario", sort=False):
        row = {"scenario": scenario, "n_runs": len(d)}
        for m in metrics:
            vals = pd.to_numeric(d[m], errors="coerce")
            row[f"{m}_mean"] = float(vals.mean())
            row[f"{m}_sd"] = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
        rows.append(row)
    return pd.DataFrame(rows)


def package_and_download_results(output_dir: Path, label: str) -> Path:
    output_dir = Path(output_dir)
    manifest = []
    for p in sorted(output_dir.rglob("*")):
        if p.is_file():
            manifest.append({
                "relative_path": str(p.relative_to(output_dir)),
                "bytes": int(p.stat().st_size),
                "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
            })
    pd.DataFrame(manifest).to_csv(
        output_dir / "RESULTS_FILE_MANIFEST.csv", index=False
    )

    zip_path = output_dir.parent / f"{output_dir.name}_RESULTS.zip"
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
    ) as zf:
        for p in sorted(output_dir.rglob("*")):
            if p.is_file():
                zf.write(p, arcname=str(p.relative_to(output_dir)))

    print("\n" + "=" * 100)
    print(f"{label} COMPLETE")
    print(f"Results ZIP: {zip_path}")
    print(f"ZIP size: {zip_path.stat().st_size / (1024**2):.2f} MB")
    print("=" * 100)

    try:
        from google.colab import files as colab_files
        print("Starting automatic download to your laptop...")
        colab_files.download(str(zip_path))
    except Exception as exc:
        print("Automatic Colab download unavailable.")
        print(f"ZIP remains at: {zip_path}")
        print(f"Reason: {exc}")

    return zip_path


# ======================================================================================
# CONTROLLED DOCUMENTARY EVIDENCE — v16.7
# ======================================================================================

def rubric_descriptor(dim: str, factor: str, score: float) -> str:
    s = int(np.clip(np.rint(float(score)), 0, 5))
    return str(RUBRIC_DESCRIPTORS[dim][factor][s])


def generate_controlled_documentary_evidence(
    client_ids: List[str],
    evidence_seed: int,
    factor_minima: Dict[str, Dict[str, float]],
    domain: str,
) -> Tuple[Dict[str, Dict[str, Dict[str, float]]], pd.DataFrame]:
    """
    Generate controlled documentary evidence at the INDIVIDUAL FACTOR level.

    v16.7 uses PRE-SPECIFIED governance archetypes so the controlled experiment
    contains all three decision outcomes needed to validate the policy:

      - DIRECT_STRONG:
          designed to satisfy the strict Direct Auto-Accept route;
      - REVIEW_RECOVERABLE:
          borderline overall evidence, but sufficiently strong average evidence
          across the critical subset for Automated Review acceptance;
      - REVIEW_LIMITED:
          adequate enough to enter Automated Review, but insufficient average
          critical evidence for Review acceptance;
      - LOW_HPS_WEAK:
          every documentary dimension remains at/above the 2.5 dimension floor,
          but the overall HPS is expected to remain below 3.0;
      - DIMENSION_FLOOR_WEAK:
          contains a deliberately weak documentary dimension (<2.5).

    IMPORTANT SCIENTIFIC INTERPRETATION
    -----------------------------------
    These are controlled governance scenarios, not observed hospital-site
    provenance records and not estimates of real-world admission prevalence.
    The archetype composition is specified BEFORE model training and does not
    use predictive performance, test labels, or downstream model outcomes.

    DQ (dim2) is NEVER synthesized here; it remains measured from TRAIN data.
    The frozen evidence seed randomly assigns the pre-generated archetypes to
    client identities.
    """
    domain = str(domain).lower()
    if domain not in CRITICAL_FACTORS_BY_DOMAIN:
        raise ValueError(f"Unsupported domain: {domain!r}")

    n = len(client_ids)
    rng = np.random.default_rng(int(evidence_seed))

    critical_policy = CRITICAL_FACTORS_BY_DOMAIN[domain]
    critical_keys = {
        (dim, factor)
        for dim, factor_list in critical_policy.items()
        for factor in factor_list
    }

    def make_documentary_profile(role: str, variant: int):
        factors = {
            dim: {}
            for dim in DOCUMENTARY_DIMS
        }

        if role == "DIRECT_STRONG":
            # Strong but not uniformly perfect. Every factor is at least 4,
            # while some values reach 5 according to a deterministic variant.
            for dim in DOCUMENTARY_DIMS:
                for j, factor in enumerate(FACTOR_NAMES[dim]):
                    minimum = float(factor_minima[dim][factor])
                    base = max(4.0, minimum)
                    bonus = 1.0 if ((j + variant + len(dim)) % 3 == 0) else 0.0
                    factors[dim][factor] = float(min(5.0, base + bonus))

        elif role == "REVIEW_RECOVERABLE":
            # Start from moderate evidence: all documentary dimensions remain
            # safely above the 2.5 floor without making HPS automatically high.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = 3.0

            critical_sequence = [
                (dim, factor)
                for dim, factor_list in critical_policy.items()
                for factor in factor_list
            ]

            # Put critical factors at their policy adequacy references.
            for dim, factor in critical_sequence:
                factors[dim][factor] = float(
                    factor_minima[dim][factor]
                )

            # Intentionally allow ONE critical factor to fall one point below
            # its individual minimum, while another critical factor with room
            # is strengthened. This is exactly the compensatory situation that
            # Automated Review is intended to resolve.
            high_min = [
                (dim, factor)
                for dim, factor in critical_sequence
                if float(factor_minima[dim][factor]) >= 4.0
            ]
            lower_min = [
                (dim, factor)
                for dim, factor in critical_sequence
                if float(factor_minima[dim][factor]) <= 3.0
            ]

            if high_min and lower_min:
                weak_key = high_min[variant % len(high_min)]
                strong_key = lower_min[variant % len(lower_min)]

                weak_min = float(
                    factor_minima[weak_key[0]][weak_key[1]]
                )
                strong_min = float(
                    factor_minima[strong_key[0]][strong_key[1]]
                )

                factors[weak_key[0]][weak_key[1]] = float(
                    max(0.0, weak_min - 1.0)
                )
                factors[strong_key[0]][strong_key[1]] = float(
                    min(5.0, strong_min + 2.0)
                )

            # Small documentary variation between the two recoverable bundles.
            if variant % 2 == 1:
                factors["dim3"][FACTOR_NAMES["dim3"][0]] = 4.0

        elif role == "REVIEW_LIMITED":
            # Preserve dimensions at/above 2.5 and keep HPS in/near the review
            # region, but make the average critical evidence too weak.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    if dim in critical_policy and (dim, factor) not in critical_keys:
                        factors[dim][factor] = 4.0
                    else:
                        factors[dim][factor] = 3.0

            for dim, factor_list in critical_policy.items():
                for factor in factor_list:
                    minimum = float(factor_minima[dim][factor])
                    factors[dim][factor] = float(
                        max(2.0, minimum - 1.0)
                    )

            if variant % 2 == 1:
                factors["dim3"][FACTOR_NAMES["dim3"][0]] = 4.0

        elif role == "LOW_HPS_WEAK":
            # Each documentary dimension averages >=2.5, but remains weak.
            # This isolates the HPS<3.0 rejection path from the dimension floor.
            for dim in DOCUMENTARY_DIMS:
                names = list(FACTOR_NAMES[dim])
                values = [2.0] * len(names)
                n_three = (len(names) + 1) // 2
                for j in range(n_three):
                    values[j] = 3.0

                for factor, value in zip(names, values):
                    factors[dim][factor] = float(value)

        elif role == "DIMENSION_FLOOR_WEAK":
            # General evidence is moderate, but Timeliness is deliberately below
            # the 2.5 dimension trustworthiness floor.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = 3.0

            for factor in FACTOR_NAMES["dim4"]:
                factors["dim4"][factor] = 2.0

        else:
            raise ValueError(f"Unknown evidence role: {role!r}")

        return factors

    # For the fixed K=10 experiments, pre-specify a balanced governance
    # branch-coverage set:
    #   4 direct-strong
    #   2 review-recoverable
    #   2 review-limited
    #   1 low-HPS weak
    #   1 dimension-floor weak
    #
    # This is not a post-hoc selection by client identity. The ten bundles are
    # generated first and then randomly assigned to clients using evidence_seed.
    if n == 10:
        bundle_specs = [
            ("DIRECT_STRONG", 0),
            ("DIRECT_STRONG", 1),
            ("DIRECT_STRONG", 2),
            ("DIRECT_STRONG", 3),
            ("REVIEW_RECOVERABLE", 0),
            ("REVIEW_RECOVERABLE", 1),
            ("REVIEW_LIMITED", 0),
            ("REVIEW_LIMITED", 1),
            ("LOW_HPS_WEAK", 0),
            ("DIMENSION_FLOOR_WEAK", 0),
        ]
    else:
        # Generic fallback for non-10-client studies: cycle through the same
        # archetypes without conditioning on client data or model performance.
        archetypes = [
            "DIRECT_STRONG",
            "REVIEW_RECOVERABLE",
            "REVIEW_LIMITED",
            "LOW_HPS_WEAK",
            "DIMENSION_FLOOR_WEAK",
        ]
        bundle_specs = [
            (archetypes[j % len(archetypes)], j // len(archetypes))
            for j in range(n)
        ]

    bundles = []
    for b, (role, variant) in enumerate(bundle_specs):
        factors = make_documentary_profile(role, int(variant))

        bundles.append({
            "bundle_id": f"E{b+1:02d}",
            "profile": role,
            "scenario_role": role,
            "profile_variant": int(variant),
            "factors": factors,
        })

    # Randomly assign already-generated governance bundles to client identities.
    assignment = rng.permutation(n)

    evidence_by_client = {}
    rows = []

    for client_pos, cid in enumerate(client_ids):
        bundle = bundles[int(assignment[client_pos])]

        evidence_by_client[cid] = {
            dim: dict(bundle["factors"][dim])
            for dim in DOCUMENTARY_DIMS
        }

        for dim in DOCUMENTARY_DIMS:
            for factor, score in bundle["factors"][dim].items():
                min_rank = float(
                    factor_minima[dim][factor]
                )

                rows.append({
                    "client":
                        str(cid),
                    "bundle_id":
                        bundle["bundle_id"],
                    "evidence_profile":
                        bundle["profile"],
                    "scenario_role":
                        bundle["scenario_role"],
                    "profile_variant":
                        int(bundle["profile_variant"]),
                    "dimension":
                        dim,
                    "dimension_name":
                        DIMENSION_NAMES[dim],
                    "factor":
                        factor,
                    "evidence_source_type":
                        "CONTROLLED_DOCUMENTARY_EVIDENCE",
                    "verified_rubric_level_0_5":
                        float(score),
                    "rubric_score_0_5":
                        float(score),
                    "server_mapped_score_0_5":
                        float(score),
                    "rubric_descriptor":
                        rubric_descriptor(
                            dim,
                            factor,
                            score,
                        ),
                    "human_role":
                        "VERIFY_EVIDENCE_ONLY",
                    "score_assignment":
                        "DETERMINISTIC_SERVER_MAPPING_FROM_VERIFIED_RUBRIC_LEVEL",
                    "admission_decision_by":
                        "SERVER_POLICY",
                    "adequacy_min_rank":
                        min_rank,
                    "adequacy_min_normalized":
                        min_rank / MAX_FACTOR_SCORE,
                    "meets_factor_adequacy":
                        bool(float(score) >= min_rank),
                    "evidence_artifact_id": (
                        f"{cid}-{bundle['bundle_id']}-{dim}-{factor}"
                    ),
                    "validation_status":
                        "CONTROLLED_SCENARIO_EVIDENCE",
                    "evidence_seed":
                        int(evidence_seed),
                    "domain":
                        domain,
                })

    return evidence_by_client, pd.DataFrame(rows)

def dimension_scores_from_factors(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, float]:
    """
    Client-specific HPS dimension scores (0..5):
    mean of the observed factor rubric scores within each dimension.
    """
    out = {}
    for dim in FACTOR_NAMES:
        vals = [
            float(v)
            for v in factors.get(dim, {}).values()
            if np.isfinite(float(v))
        ]
        out[dim] = float(np.mean(vals)) if vals else 0.0
    return out


def compute_hps(
    dimensions: Dict[str, float],
    weights: Dict[str, float] = WEIGHTS_PSCORE_DEFAULT,
) -> float:
    return float(
        sum(float(weights[d]) * float(dimensions[d]) for d in weights)
    )


def client_dimension_adequacy(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, float]:
    """
    Client ACHIEVED adequacy, not WAC.

    A_i,d = mean(client factor ranks in dimension d) / 5.
    """
    scores = {}
    for dim in FACTOR_NAMES:
        vals = [
            float(v)
            for v in factors.get(dim, {}).values()
            if np.isfinite(float(v))
        ]
        scores[dim] = (
            float(np.mean(vals) / MAX_FACTOR_SCORE)
            if vals else 0.0
        )
    return scores


def client_adequacy_score(
    factors: Dict[str, Dict[str, float]]
) -> Tuple[Dict[str, float], float]:
    """
    Legacy audit helper retained for backward comparability only.

    v16.7 does NOT use this value for admission. Review decisions use the
    client Critical WAC_i versus the domain Global Critical WAC.
    """
    by_dim = client_dimension_adequacy(factors)
    cas = float(np.mean(list(by_dim.values())))
    return by_dim, cas


def all_zero_score_factors(
    factors: Dict[str, Dict[str, float]]
) -> List[str]:
    """Audit-only zero list. Zero is not a separate admission rule in v16.7."""
    zeros = []
    for dim in FACTOR_NAMES:
        for factor, value in factors.get(dim, {}).items():
            if np.isfinite(float(value)) and float(value) <= 0.0:
                zeros.append(f"{dim}.{factor}")
    return zeros


def derive_global_critical_wac(domain: str) -> float:
    """
    Domain Global Critical WAC = mean(minimum critical adequacy rank / 5).

    This is a policy reference used only to resolve the HPS Review band.
    """
    domain = str(domain).lower()
    vals = []
    for dim, factors_ in CRITICAL_FACTORS_BY_DOMAIN[domain].items():
        for factor in factors_:
            vals.append(
                float(DOMAIN_FACTOR_MINIMA[domain][dim][factor])
                / MAX_FACTOR_SCORE
            )
    if not vals:
        raise ValueError(f"No critical factors for domain={domain!r}")
    return float(np.mean(vals))


def critical_factor_audit(
    factors: Dict[str, Dict[str, float]],
    domain: str,
) -> Dict[str, Any]:
    """
    Compute the client Critical WAC_i and the strict individual critical gate.

    Direct Auto-Accept:
        every critical factor >= its own factor-specific adequacy minimum.

    Automated Review:
        uses the overall average Critical WAC_i, allowing compensation across
        critical factors while preserving the dimension and HPS floors.
    """
    domain = str(domain).lower()

    scores = {}
    minima = {}
    missing = []
    below_adequacy = []

    for dim, factor_list in CRITICAL_FACTORS_BY_DOMAIN[domain].items():
        for factor in factor_list:
            key = f"{dim}.{factor}"
            minimum = float(
                DOMAIN_FACTOR_MINIMA[domain][dim][factor]
            )
            minima[key] = minimum

            if factor not in factors.get(dim, {}):
                missing.append(key)
                continue

            value = float(factors[dim][factor])

            if not np.isfinite(value):
                missing.append(key)
                continue

            scores[key] = value

            if value < minimum:
                below_adequacy.append(
                    f"{key}:{value:.1f}<{minimum:.1f}"
                )

    expected_n = sum(
        len(v)
        for v in CRITICAL_FACTORS_BY_DOMAIN[domain].values()
    )

    if missing or len(scores) != expected_n:
        return {
            "scores": scores,
            "minima": minima,
            "missing": missing,
            "critical_count": expected_n,
            "critical_wac_i": np.nan,
            "critical_mean_score_0_5": np.nan,
            "global_critical_wac":
                float(derive_global_critical_wac(domain)),
            "all_critical_meet_adequacy": False,
            "critical_below_adequacy": below_adequacy,
        }

    values = np.array(
        list(scores.values()),
        dtype=float,
    )

    return {
        "scores": scores,
        "minima": minima,
        "missing": [],
        "critical_count": expected_n,
        "critical_wac_i":
            float(np.mean(values / MAX_FACTOR_SCORE)),
        "critical_mean_score_0_5":
            float(np.mean(values)),
        "global_critical_wac":
            float(derive_global_critical_wac(domain)),
        "all_critical_meet_adequacy":
            bool(len(below_adequacy) == 0),
        "critical_below_adequacy":
            below_adequacy,
    }

def dimension_floor_failures(
    dimensions: Dict[str, float],
    floor: float = DIMENSION_MIN_FLOOR,
) -> List[str]:
    """Return averaged HPS dimensions below the minimum floor."""
    return [
        dim for dim, value in dimensions.items()
        if (not np.isfinite(float(value))) or float(value) < float(floor)
    ]

def factor_adequacy_attainment(
    factors: Dict[str, Dict[str, float]],
    factor_minima: Dict[str, Dict[str, float]],
) -> Dict[str, Any]:
    total = 0
    passed = 0
    per_dim = {}

    for dim in FACTOR_NAMES:
        dim_total = 0
        dim_passed = 0

        for factor in FACTOR_NAMES[dim]:
            total += 1
            dim_total += 1

            score = float(factors[dim][factor])
            threshold = float(factor_minima[dim][factor])

            if score >= threshold:
                passed += 1
                dim_passed += 1

        per_dim[dim] = {
            "passed": dim_passed,
            "total": dim_total,
            "fraction": float(dim_passed / max(1, dim_total)),
        }

    return {
        "passed": passed,
        "total": total,
        "fraction": float(passed / max(1, total)),
        "per_dim": per_dim,
    }


def tadp_decision(
    factors: Dict[str, Dict[str, float]],
    domain: str,
    good_cut: float = GOOD_CUT,
    high_cut: float = HIGH_CUT,
) -> Dict[str, Any]:
    """
    v16.7 final fully automated admission policy.

    Flow:
      EVERY dimension >= 2.5?
          NO -> AUTO-REJECT

      HPS >= 3.0?
          NO -> AUTO-REJECT

      HPS >= 3.5?
          YES:
              all critical factors >= own adequacy minima?
                  YES -> DIRECT AUTO-ACCEPT
                  NO  -> AUTOMATED REVIEW
          NO (3.0 <= HPS < 3.5):
              -> AUTOMATED REVIEW

      AUTOMATED REVIEW:
          Critical WAC_i >= Global Critical WAC
              -> ACCEPT AFTER REVIEW
          otherwise
              -> AUTO-REJECT
    """
    domain = str(domain).lower()

    factor_minima = DOMAIN_FACTOR_MINIMA[domain]
    dimension_wac = DOMAIN_DIMENSION_WAC[domain]
    global_wac = float(DOMAIN_GLOBAL_WAC[domain])
    global_critical_wac = float(
        derive_global_critical_wac(domain)
    )

    dims = dimension_scores_from_factors(factors)
    hps = compute_hps(dims)

    attainment = factor_adequacy_attainment(
        factors,
        factor_minima,
    )

    zeros = all_zero_score_factors(factors)

    dim_floor_failed = dimension_floor_failures(
        dims,
        floor=DIMENSION_MIN_FLOOR,
    )

    critical = critical_factor_audit(
        factors,
        domain,
    )

    critical_score_string = ";".join(
        (
            f"{key}={value:.1f}"
            f"(min={critical['minima'][key]:.1f})"
        )
        for key, value in critical["scores"].items()
    )

    base = {
        "hps": float(hps),
        "policy_dimension_wac": dimension_wac,
        "global_domain_wac": global_wac,
        "global_critical_wac": global_critical_wac,
        "critical_wac_i": critical["critical_wac_i"],
        "critical_wac_margin": (
            float(critical["critical_wac_i"])
            - global_critical_wac
            if np.isfinite(critical["critical_wac_i"])
            else np.nan
        ),
        "critical_mean_score_0_5":
            critical["critical_mean_score_0_5"],
        "critical_factor_count":
            int(critical["critical_count"]),
        "critical_scores":
            critical_score_string,
        "critical_missing":
            ";".join(critical["missing"]),
        "all_critical_meet_adequacy":
            bool(critical["all_critical_meet_adequacy"]),
        "critical_below_adequacy":
            ";".join(critical["critical_below_adequacy"]),
        "factor_adequacy_passed":
            int(attainment["passed"]),
        "factor_adequacy_total":
            int(attainment["total"]),
        "factor_adequacy_fraction":
            float(attainment["fraction"]),
        "dimensions":
            dims,
        "dimension_min_floor":
            float(DIMENSION_MIN_FLOOR),
        "dimension_floor_failures":
            ";".join(dim_floor_failed),
        # Audit only; zero is not a separate decision rule.
        "all_zero_factors":
            ";".join(zeros),
    }

    # Gate 1: every complete trustworthiness dimension must clear 2.5/5.
    if dim_floor_failed:
        return {
            **base,
            "decision_path":
                "DIMENSION_FLOOR",
            "initial_action":
                "AUTO_REJECT",
            "final_action":
                "REJECT",
            "status":
                "AUTO_REJECTED_DIMENSION_FLOOR",
            "reason": (
                f"At least one averaged dimension is below "
                f"{DIMENSION_MIN_FLOOR:.1f}/5: "
                + ";".join(
                    f"{dim}={dims[dim]:.3f}"
                    for dim in dim_floor_failed
                )
            ),
        }

    # Missing/non-finite critical evidence is fail-closed in this experiment.
    if critical["missing"]:
        return {
            **base,
            "decision_path":
                "MISSING_CRITICAL_EVIDENCE",
            "initial_action":
                "AUTO_REJECT",
            "final_action":
                "REJECT",
            "status":
                "AUTO_REJECTED_MISSING_CRITICAL_EVIDENCE",
            "reason": (
                "Missing/non-finite verified critical evidence: "
                + ";".join(critical["missing"])
            ),
        }

    # Gate 2: overall HPS lower bound.
    if hps < float(good_cut):
        return {
            **base,
            "decision_path":
                "LOW_HPS",
            "initial_action":
                "AUTO_REJECT",
            "final_action":
                "REJECT",
            "status":
                "AUTO_REJECTED_LOW_HPS",
            "reason":
                f"HPS {hps:.3f} < {good_cut:.3f}",
        }

    # High-HPS strict direct route.
    if hps >= float(high_cut):
        if critical["all_critical_meet_adequacy"]:
            return {
                **base,
                "decision_path":
                    "DIRECT_AUTO_ACCEPT",
                "initial_action":
                    "AUTO_ACCEPT",
                "final_action":
                    "ACCEPT",
                "status":
                    "DIRECT_AUTO_ACCEPTED",
                "reason": (
                    f"HPS {hps:.3f} >= {high_cut:.3f}; all critical "
                    "factors meet their individual adequacy requirements"
                ),
            }

        # High HPS but failed strict critical gate:
        # fall back to the same automated review rather than immediate rejection.
        review_origin = (
            "HIGH_HPS_CRITICAL_FALLBACK"
        )

    else:
        # 3.0 <= HPS < 3.5
        review_origin = (
            "HPS_REVIEW_BAND"
        )

    # Automated Review for both origins.
    review_pass = bool(
        float(critical["critical_wac_i"])
        >= global_critical_wac
    )

    if review_pass:
        return {
            **base,
            "decision_path":
                review_origin,
            "initial_action":
                "AUTOMATED_REVIEW",
            "final_action":
                "ACCEPT",
            "status":
                "ACCEPTED_AFTER_AUTOMATED_REVIEW",
            "reason": (
                f"{review_origin}: Critical WAC_i "
                f"{critical['critical_wac_i']:.3f} >= "
                f"Global Critical WAC {global_critical_wac:.3f}"
            ),
        }

    return {
        **base,
        "decision_path":
            review_origin,
        "initial_action":
            "AUTOMATED_REVIEW",
        "final_action":
            "REJECT",
        "status":
            "AUTO_REJECTED_REVIEW_CRITICAL_WAC",
        "reason": (
            f"{review_origin}: Critical WAC_i "
            f"{critical['critical_wac_i']:.3f} < "
            f"Global Critical WAC {global_critical_wac:.3f}"
        ),
    }

def build_tadp_governance(
    client_ids: List[str],
    documentary_evidence: Dict[str, Dict[str, Dict[str, float]]],
    dq_scores: Dict[str, Dict[str, float]],
    run: int,
    evidence_seed: int,
    domain: str,
) -> pd.DataFrame:
    domain = str(domain).lower()
    rows = []

    for cid in client_ids:
        factors = {
            dim: dict(documentary_evidence[cid][dim])
            for dim in DOCUMENTARY_DIMS
        }

        factors["dim2"] = {
            name: float(dq_scores[cid][name])
            for name in FACTOR_NAMES["dim2"]
        }

        d = tadp_decision(
            factors,
            domain=domain,
        )

        row = {
            "run": int(run),
            "domain": domain,
            "evidence_seed": int(evidence_seed),
            "client": str(cid),
            "hps": float(d["hps"]),
            "global_domain_wac":
                float(d["global_domain_wac"]),
            "global_critical_wac":
                float(d["global_critical_wac"]),
            "critical_wac_i": (
                float(d["critical_wac_i"])
                if np.isfinite(d["critical_wac_i"])
                else np.nan
            ),
            "critical_wac_margin": (
                float(d["critical_wac_margin"])
                if np.isfinite(d["critical_wac_margin"])
                else np.nan
            ),
            "critical_mean_score_0_5": (
                float(d["critical_mean_score_0_5"])
                if np.isfinite(d["critical_mean_score_0_5"])
                else np.nan
            ),
            "critical_factor_count":
                int(d["critical_factor_count"]),
            "critical_scores":
                d["critical_scores"],
            "critical_missing":
                d["critical_missing"],
            "all_critical_meet_adequacy":
                bool(d["all_critical_meet_adequacy"]),
            "critical_below_adequacy":
                d["critical_below_adequacy"],
            "dimension_min_floor":
                float(d["dimension_min_floor"]),
            "dimension_floor_failures":
                d["dimension_floor_failures"],
            "factor_adequacy_passed":
                int(d["factor_adequacy_passed"]),
            "factor_adequacy_total":
                int(d["factor_adequacy_total"]),
            "factor_adequacy_fraction":
                float(d["factor_adequacy_fraction"]),
            "decision_path":
                d["decision_path"],
            "initial_action":
                d["initial_action"],
            "final_action":
                d["final_action"],
            "status":
                d["status"],
            "reason":
                d["reason"],
            "all_zero_factors":
                d["all_zero_factors"],
        }

        for dim, value in d["dimensions"].items():
            row[
                f"{dim}_score_0_5"
            ] = float(value)

        for dim, value in d["policy_dimension_wac"].items():
            row[
                f"{dim}_policy_wac"
            ] = float(value)

        rows.append(row)

    return pd.DataFrame(rows)

def accepted_tadp_vr(governance_df: pd.DataFrame) -> List[str]:
    return governance_df.loc[
        governance_df["final_action"].eq("ACCEPT"), "client"
    ].astype(str).tolist()


def accepted_tadp_sda(
    governance_df: pd.DataFrame
) -> List[str]:
    """
    Select one best TADP-eligible client for SDA from already accepted clients.
    """
    eligible = governance_df[
        governance_df["final_action"].eq("ACCEPT")
    ].copy()

    if eligible.empty:
        raise RuntimeError(
            "TADP-SDA cannot select a client: no TADP-eligible client."
        )

    eligible["direct_accept_priority"] = (
        eligible["status"]
        .eq("DIRECT_AUTO_ACCEPTED")
        .astype(int)
    )

    eligible = eligible.sort_values(
        [
            "hps",
            "direct_accept_priority",
            "critical_wac_i",
            "client",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    )

    return [
        str(
            eligible.iloc[0]["client"]
        )
    ]

def build_full_factor_evidence_table(
    client_ids: List[str],
    documentary_evidence: Dict[str, Dict[str, Dict[str, float]]],
    dq_scores: Dict[str, Dict[str, float]],
    dq_audit: pd.DataFrame,
    evidence_seed: int,
    domain: str,
) -> pd.DataFrame:
    """
    Full 28-factor evidence audit.

    Critical factors are flagged with their own policy adequacy minima.
    Direct Auto-Accept uses these individual minima; Automated Review uses the
    aggregate Critical WAC_i.
    """
    domain = str(domain).lower()
    factor_minima = DOMAIN_FACTOR_MINIMA[domain]
    critical_policy = CRITICAL_FACTORS_BY_DOMAIN[domain]

    dq_index = dq_audit.copy()
    dq_index["client"] = dq_index["client"].astype(str)
    dq_index = dq_index.set_index(
        "client",
        drop=False,
    )

    def policy_fields(dim, factor):
        is_critical = bool(
            factor in critical_policy.get(
                dim,
                [],
            )
        )

        return {
            "is_critical_factor":
                is_critical,
            "critical_adequacy_min_rank": (
                float(factor_minima[dim][factor])
                if is_critical
                else np.nan
            ),
            "critical_adequacy_min_normalized": (
                float(factor_minima[dim][factor])
                / MAX_FACTOR_SCORE
                if is_critical
                else np.nan
            ),
        }

    rows = []

    for cid in client_ids:
        cid = str(cid)

        for dim in DOCUMENTARY_DIMS:
            for factor in FACTOR_NAMES[dim]:
                score = float(
                    documentary_evidence[cid][dim][factor]
                )
                min_rank = float(
                    factor_minima[dim][factor]
                )

                rows.append({
                    "client": cid,
                    "domain": domain,
                    "dimension": dim,
                    "dimension_name":
                        DIMENSION_NAMES[dim],
                    "factor": factor,
                    "factor_source":
                        "CONTROLLED_DOCUMENTARY_EVIDENCE",
                    "raw_measured_value":
                        np.nan,
                    "rubric_score_0_5":
                        score,
                    "rubric_descriptor":
                        rubric_descriptor(
                            dim,
                            factor,
                            score,
                        ),
                    "adequacy_min_rank":
                        min_rank,
                    "adequacy_min_normalized":
                        min_rank / MAX_FACTOR_SCORE,
                    "meets_factor_adequacy":
                        bool(score >= min_rank),
                    **policy_fields(
                        dim,
                        factor,
                    ),
                    "evidence_seed":
                        int(evidence_seed),
                })

        for factor in FACTOR_NAMES["dim2"]:
            score = float(
                dq_scores[cid][factor]
            )
            min_rank = float(
                factor_minima["dim2"][factor]
            )
            raw_col = DQ_RAW_METRIC_BY_FACTOR[
                factor
            ]

            raw_value = (
                float(
                    dq_index.loc[
                        cid,
                        raw_col,
                    ]
                )
                if raw_col in dq_index.columns
                else np.nan
            )

            rows.append({
                "client": cid,
                "domain": domain,
                "dimension": "dim2",
                "dimension_name":
                    DIMENSION_NAMES["dim2"],
                "factor": factor,
                "factor_source":
                    "MACHINE_MEASURED_TRAIN_ONLY",
                "raw_measured_value":
                    raw_value,
                "rubric_score_0_5":
                    score,
                "rubric_descriptor":
                    rubric_descriptor(
                        "dim2",
                        factor,
                        score,
                    ),
                "adequacy_min_rank":
                    min_rank,
                "adequacy_min_normalized":
                    min_rank / MAX_FACTOR_SCORE,
                "meets_factor_adequacy":
                    bool(score >= min_rank),
                **policy_fields(
                    "dim2",
                    factor,
                ),
                "evidence_seed":
                    int(evidence_seed),
            })

    out = pd.DataFrame(
        rows
    )

    expected_rows = len(client_ids) * 28

    if len(out) != expected_rows:
        raise RuntimeError(
            f"Full evidence matrix should contain "
            f"{expected_rows} rows; found {len(out)}."
        )

    return out

# ======================================================================================
# GREAT EXPECTATIONS (GX CORE) — NATIVE VALIDATOR BASELINE
# ======================================================================================
# GX is used here in its native role: validate each client's TRAIN-only data against
# a predefined Expectation Suite and use the suite-level success flag as PASS/FAIL.
# There is NO ranking, NO forced-K selection, and NO HPS/WAC information in this
# baseline. Great Expectations reports suite success=True only when all configured
# Expectations pass. The suite deliberately uses common technical checks only:
#   1) schema, 2) datatype consistency, 3) required-value ranges,
#   4) missingness, 5) duplicates / unique IDs, 6) label/domain validity,
#   7) structural integrity.
# Distribution checks are intentionally omitted because non-IID client distributions
# are expected in federated learning and should not by themselves constitute failure.
GX_CORE_VERSION = "1.23.0"
GX_VALUE_MOSTLY = 0.99
GX_NONNULL_REFERENCE_MIN = 0.80
GX_NONNULL_TOLERANCE = 0.15


def ensure_great_expectations():
    """Import pinned GX Core; install once in Colab if unavailable."""
    try:
        import great_expectations as gx
        return gx
    except ImportError:
        import subprocess
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            f"great_expectations=={GX_CORE_VERSION}",
        ])
        import great_expectations as gx
        return gx


def _gx_meta(category: str, name: str) -> Dict[str, Any]:
    return {
        "dq_category": str(category),
        "check_name": str(name),
        "baseline": "GX_NATIVE_VALIDATOR",
    }


def _gx_add(suite, specs: List[Dict[str, Any]], expectation, category: str, name: str):
    suite.add_expectation(expectation)
    specs.append({"category": str(category), "name": str(name)})


def _aggregate_nonnull_reference(
    client_frames: Dict[str, pd.DataFrame],
    columns: List[str],
) -> Dict[str, float]:
    total_rows = float(sum(len(df) for df in client_frames.values()))
    out = {}
    for c in columns:
        nonnull = sum(int(df[c].notna().sum()) for df in client_frames.values() if c in df.columns)
        out[c] = float(nonnull / max(1.0, total_rows))
    return out


def build_gx_healthcare_reference(client_frames: Dict[str, pd.DataFrame], preprocessor: Any) -> Dict[str, Any]:
    first = next(iter(client_frames.values()))

    # GX datatype validation is intentionally independent of TADP's model
    # preprocessing type inference.  The preprocessor labels a feature numeric
    # when >=95% of pooled TRAIN non-missing values are parseable; reusing that
    # inferred list inside GX and then requiring 99% parseability per client
    # creates an artificial contradiction.  GX therefore validates datatype
    # consistency only for fields whose numeric meaning is explicit in the
    # Diabetes data schema.
    known_numeric_fields = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]
    gx_numeric_cols = [c for c in known_numeric_fields if c in first.columns]

    return {
        "original_columns": list(first.columns),
        "numeric_cols": gx_numeric_cols,
        "feature_cols": list(preprocessor.feature_cols),
        "nonnull_reference": _aggregate_nonnull_reference(
            client_frames, list(preprocessor.feature_cols)
        ),
    }


def build_gx_healthcare_validation_frame(df: pd.DataFrame, preprocessor: Any) -> pd.DataFrame:
    """GX validation view; raw TRAIN records remain the source of all checks."""
    out = df.copy()
    for c in preprocessor.numeric_cols:
        raw = df[c]
        parsed = pd.to_numeric(raw, errors="coerce")
        type_ok = raw.isna() | parsed.notna()
        out[c] = parsed.astype(float)
        out[f"__gx_type_ok__{c}"] = type_ok.astype(np.int8)
    return out


def build_gx_healthcare_suite(gx, reference: Dict[str, Any]):
    """Simple native GX technical-validation suite for the Diabetes TRAIN shards."""
    gxe = gx.expectations
    suite = gx.ExpectationSuite(name="tadp_healthcare_gx_native_suite")
    specs = []
    cols = reference["original_columns"]

    # 1) Schema.
    _gx_add(
        suite, specs,
        gxe.ExpectTableColumnsToMatchSet(
            column_set=cols,
            exact_match=False,
            severity="critical",
            meta=_gx_meta("Schema", "required_column_set"),
        ),
        "Schema", "required_column_set",
    )

    # 2) Datatype consistency for columns inferred as numeric from TRAIN only.
    for c in reference["numeric_cols"]:
        diag = f"__gx_type_ok__{c}"
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=diag,
                value_set=[1],
                mostly=GX_VALUE_MOSTLY,
                severity="critical",
                meta=_gx_meta("Datatype Consistency", f"{c}_numeric_parseability"),
            ),
            "Datatype Consistency", f"{c}_numeric_parseability",
        )

    # 3) Required-value ranges for known count/duration fields.
    nonnegative_fields = [
        "num_lab_procedures", "num_procedures", "num_medications",
        "number_outpatient", "number_emergency", "number_inpatient",
        "number_diagnoses",
    ]
    for c in nonnegative_fields:
        if c in cols:
            _gx_add(
                suite, specs,
                gxe.ExpectColumnValuesToBeBetween(
                    column=c, min_value=0.0, mostly=GX_VALUE_MOSTLY,
                    severity="critical",
                    meta=_gx_meta("Required Value Ranges", f"{c}_nonnegative"),
                ),
                "Required Value Ranges", f"{c}_nonnegative",
            )
    if "time_in_hospital" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeBetween(
                column="time_in_hospital", min_value=1.0, max_value=14.0,
                mostly=GX_VALUE_MOSTLY,
                severity="critical",
                meta=_gx_meta("Required Value Ranges", "time_in_hospital_range"),
            ),
            "Required Value Ranges", "time_in_hospital_range",
        )

    # 4) Missingness. Only columns that are substantially populated in the frozen
    # TRAIN reference are treated as required-enough for a missingness expectation.
    for c, global_nonnull in reference["nonnull_reference"].items():
        if c not in cols or float(global_nonnull) < GX_NONNULL_REFERENCE_MIN:
            continue
        minimum = max(0.70, float(global_nonnull) - GX_NONNULL_TOLERANCE)
        _gx_add(
            suite, specs,
            gxe.ExpectColumnProportionOfNonNullValuesToBeBetween(
                column=c, min_value=float(minimum), max_value=1.0,
                severity="warning",
                meta=_gx_meta("Missingness", f"{c}_nonnull"),
            ),
            "Missingness", f"{c}_nonnull",
        )

    # 5) Duplicates / unique identifiers.
    if "encounter_id" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeUnique(
                column="encounter_id",
                severity="critical",
                meta=_gx_meta("Duplicates / Unique IDs", "encounter_id_unique"),
            ),
            "Duplicates / Unique IDs", "encounter_id_unique",
        )

    # 6) Labels / domain validity.
    if "_target" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column="_target", value_set=[0, 1, 2],
                severity="critical",
                meta=_gx_meta("Labels / Domain Validity", "target_domain"),
            ),
            "Labels / Domain Validity", "target_domain",
        )

    # 7) Structural integrity.
    if "_row_id" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeUnique(
                column="_row_id",
                severity="critical",
                meta=_gx_meta("Structural Integrity", "row_id_unique"),
            ),
            "Structural Integrity", "row_id_unique",
        )
    _gx_add(
        suite, specs,
        gxe.ExpectTableRowCountToBeBetween(
            min_value=100,
            severity="warning",
            meta=_gx_meta("Structural Integrity", "minimum_client_rows"),
        ),
        "Structural Integrity", "minimum_client_rows",
    )
    return suite, specs


def build_gx_cifar_metadata(X: np.ndarray, y: np.ndarray) -> pd.DataFrame:
    X = np.asarray(X)
    y = np.asarray(y).reshape(-1)
    if X.ndim != 4:
        raise RuntimeError(f"Expected CIFAR tensor [N,H,W,C], got shape={X.shape}")
    finite = np.isfinite(X.astype(np.float32)).reshape(len(X), -1).all(axis=1)
    flat = X.reshape(len(X), -1)
    return pd.DataFrame({
        "sample_id": np.arange(len(X), dtype=np.int64),
        "label": y.astype(np.int32),
        "height": np.full(len(X), X.shape[1], dtype=np.int32),
        "width": np.full(len(X), X.shape[2], dtype=np.int32),
        "channels": np.full(len(X), X.shape[3], dtype=np.int32),
        "dtype_ok": np.full(len(X), int(X.dtype == np.uint8), dtype=np.int8),
        "finite": finite.astype(np.int8),
        "pixel_min": flat.min(axis=1).astype(float),
        "pixel_max": flat.max(axis=1).astype(float),
    })


def build_gx_cifar_reference(client_raw: Dict[str, Tuple[np.ndarray, np.ndarray]]) -> Dict[str, Any]:
    # Native validator uses fixed technical constraints; no distribution reference needed.
    return {}


def build_gx_cifar_suite(gx, reference: Dict[str, Any]):
    """Simple native GX technical-validation suite for CIFAR-10 client metadata."""
    gxe = gx.expectations
    suite = gx.ExpectationSuite(name="tadp_cifar10_gx_native_suite")
    specs = []
    columns = [
        "sample_id", "label", "height", "width", "channels",
        "dtype_ok", "finite", "pixel_min", "pixel_max",
    ]

    _gx_add(
        suite, specs,
        gxe.ExpectTableColumnsToMatchSet(
            column_set=columns, exact_match=True,
            meta=_gx_meta("Schema", "image_metadata_schema"),
        ),
        "Schema", "image_metadata_schema",
    )
    for c in columns:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToNotBeNull(
                column=c,
                severity="warning",
                meta=_gx_meta("Missingness", f"{c}_not_null"),
            ),
            "Missingness", f"{c}_not_null",
        )
    for c, value in [("height", 32), ("width", 32), ("channels", 3), ("dtype_ok", 1), ("finite", 1)]:
        category = "Datatype Consistency" if c in {"dtype_ok", "finite"} else "Structural Integrity"
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=c, value_set=[value],
                meta=_gx_meta(category, f"{c}_constraint"),
            ),
            category, f"{c}_constraint",
        )
    for c in ["pixel_min", "pixel_max"]:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeBetween(
                column=c, min_value=0.0, max_value=255.0,
                meta=_gx_meta("Required Value Ranges", f"{c}_valid_range"),
            ),
            "Required Value Ranges", f"{c}_valid_range",
        )
    _gx_add(
        suite, specs,
        gxe.ExpectColumnValuesToBeInSet(
            column="label", value_set=list(range(10)),
            meta=_gx_meta("Labels / Domain Validity", "label_domain"),
        ),
        "Labels / Domain Validity", "label_domain",
    )
    _gx_add(
        suite, specs,
        gxe.ExpectColumnValuesToBeUnique(
            column="sample_id",
            meta=_gx_meta("Duplicates / Unique IDs", "sample_id_unique"),
        ),
        "Duplicates / Unique IDs", "sample_id_unique",
    )
    _gx_add(
        suite, specs,
        gxe.ExpectTableRowCountToBeBetween(
            min_value=100,
            severity="warning",
            meta=_gx_meta("Structural Integrity", "minimum_client_images"),
        ),
        "Structural Integrity", "minimum_client_images",
    )
    return suite, specs


def ge_governance(
    client_data: Dict[str, Any],
    client_ids: List[str],
    accept_count: Optional[int] = None,  # ignored; retained only for call compatibility
    domain: str = "healthcare",
    preprocessor: Optional[Any] = None,
    dq_scores: Optional[Dict[str, Dict[str, float]]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run native GX suite validation and PASS only clients whose entire suite succeeds."""
    domain = str(domain).lower()
    gx = ensure_great_expectations()
    context = gx.get_context(mode="ephemeral")
    datasource = context.data_sources.add_pandas(name=f"tadp_gx_native_{domain}_datasource")
    asset = datasource.add_dataframe_asset(name=f"tadp_gx_native_{domain}_asset")
    batch_definition = asset.add_batch_definition_whole_dataframe(name="whole_client_train_partition")

    if domain == "healthcare":
        if preprocessor is None:
            raise ValueError("Healthcare GX native baseline requires TRAIN-only preprocessor metadata.")
        reference = build_gx_healthcare_reference(client_data, preprocessor)
        suite, specs = build_gx_healthcare_suite(gx, reference)
        make_frame = lambda cid: build_gx_healthcare_validation_frame(client_data[cid], preprocessor)
    elif domain == "cifar10":
        reference = build_gx_cifar_reference(client_data)
        suite, specs = build_gx_cifar_suite(gx, reference)
        def make_frame(cid):
            X, y = client_data[cid]
            return build_gx_cifar_metadata(X, y)
    else:
        raise ValueError(f"Unsupported GX domain: {domain!r}")

    summary_rows, detail_rows = [], []
    for cid in client_ids:
        frame = make_frame(cid)
        batch = batch_definition.get_batch(batch_parameters={"dataframe": frame})
        validation = batch.validate(suite)
        results = list(validation.results)
        if len(results) != len(specs):
            raise RuntimeError(
                f"GX result/spec mismatch for client {cid}: {len(results)} vs {len(specs)}"
            )

        passed = 0
        critical_failures = 0
        warning_failures = 0
        info_failures = 0
        category_totals, category_passed = {}, {}
        observed_names = []

        for result in results:
            success = bool(result.success)
            passed += int(success)
            cfg = result.expectation_config
            meta = getattr(cfg, "meta", None) or {}
            name = str(meta.get("check_name", getattr(cfg, "type", "GX_EXPECTATION")))
            cat = str(meta.get("dq_category", "Technical Validation"))

            sev_obj = getattr(cfg, "severity", None)
            sev = getattr(sev_obj, "value", sev_obj)
            sev = str(sev if sev is not None else "critical").lower()
            if "." in sev:
                sev = sev.split(".")[-1]
            if sev not in {"critical", "warning", "info"}:
                sev = "critical"

            if not success:
                if sev == "critical":
                    critical_failures += 1
                elif sev == "warning":
                    warning_failures += 1
                else:
                    info_failures += 1

            observed_names.append(name)
            category_totals[cat] = category_totals.get(cat, 0) + 1
            category_passed[cat] = category_passed.get(cat, 0) + int(success)

            detail_rows.append({
                "client": str(cid),
                "domain": domain,
                "gx_version": GX_CORE_VERSION,
                "expectation_name": name,
                "dq_category": cat,
                "severity": sev,
                "success": success,
            })

        expected_names = sorted(str(x["name"]) for x in specs)
        if sorted(observed_names) != expected_names:
            raise RuntimeError(
                f"GX expectation identity mismatch for client {cid}: "
                f"expected={expected_names}, observed={sorted(observed_names)}"
            )

        total = len(specs)
        stats = getattr(validation, "statistics", {}) or {}
        suite_success = bool(validation.success)

        # GX-native severity-aware operational gate.
        # GX itself exposes the maximum failed severity for a Validation Result.
        # We use that native result rather than reconstructing the gate from a
        # custom ranking.  A failed Expectation execution is also treated by GX
        # as CRITICAL.
        max_failed_severity_obj = validation.get_max_severity_failure()
        if max_failed_severity_obj is None:
            max_failed_severity = "none"
        else:
            max_failed_severity = str(
                getattr(max_failed_severity_obj, "value", max_failed_severity_obj)
            ).lower()
            if "." in max_failed_severity:
                max_failed_severity = max_failed_severity.split(".")[-1]

        operational_valid = (max_failed_severity != "critical")

        row = {
            "client": str(cid),
            "gx_version": GX_CORE_VERSION,
            "gx_native_suite_success": suite_success,
            "gx_operational_valid": bool(operational_valid),
            "gx_critical_failures": int(critical_failures),
            "gx_warning_failures": int(warning_failures),
            "gx_info_failures": int(info_failures),
            "gx_max_failed_severity": max_failed_severity,
            "ge_expectations_passed": int(stats.get("successful_expectations", passed)),
            "ge_expectations_total": int(stats.get("evaluated_expectations", total)),
            "ge_pass_rate": float(
                stats.get("success_percent", 100.0 * passed / max(1, total))
            ) / 100.0,
            "ge_native_validation_class": (
                "GX_SUITE_PASS"
                if suite_success
                else (
                    "GX_WARNING_ONLY"
                    if operational_valid
                    else "GX_CRITICAL_FAILURE"
                )
            ),
            "ge_final_action": "ACCEPT" if operational_valid else "REJECT",
            "ge_policy": (
                "GX Core severity-aware validation; ACCEPT requires zero critical "
                "Expectation failures; warning/info failures are reported but do not "
                "exclude; no ranking and no forced-K selection"
            ),
        }

        for cat in sorted(category_totals):
            safe = re.sub(r"[^a-z0-9]+", "_", cat.lower()).strip("_")
            row[f"ge_{safe}_passed"] = int(category_passed.get(cat, 0))
            row[f"ge_{safe}_total"] = int(category_totals[cat])

        summary_rows.append(row)

    return pd.DataFrame(summary_rows), pd.DataFrame(detail_rows)


def score_lower_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [best_upper, score4_upper, score3_upper, score2_upper, score1_upper]
    value <= cuts[0] => 5; ... value <= cuts[4] => 1; else 0.
    """
    v = float(value)
    for score, upper in zip([5, 4, 3, 2, 1], cuts):
        if v <= float(upper):
            return float(score)
    return 0.0


def score_higher_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [score5_lower, score4_lower, score3_lower, score2_lower, score1_lower]
    """
    v = float(value)
    for score, lower in zip([5, 4, 3, 2, 1], cuts):
        if v >= float(lower):
            return float(score)
    return 0.0


def js_divergence(p, q, eps=1e-12) -> float:
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / max(eps, p.sum())
    q = q / max(eps, q.sum())
    m = 0.5 * (p + q)
    kl_pm = np.sum(np.where(p > 0, p * np.log((p + eps) / (m + eps)), 0.0))
    kl_qm = np.sum(np.where(q > 0, q * np.log((q + eps) / (m + eps)), 0.0))
    return float(0.5 * (kl_pm + kl_qm))


# ======================================================================================
# MODEL / METRICS / FL TRAINING
# ======================================================================================

def class_weight_dict(y: np.ndarray) -> Dict[int, float]:
    y = np.asarray(y, dtype=np.int32)
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(w) for c, w in zip(classes, weights)}


def model_parameter_bytes(model: keras.Model) -> int:
    return int(sum(np.asarray(w).nbytes for w in model.get_weights()))


def evaluate_model(
    model: keras.Model, X: np.ndarray, y: np.ndarray, n_classes: int
) -> Dict[str, float]:
    p = model.predict(X, batch_size=512, verbose=0)
    pred = np.argmax(p, axis=1)
    out = {
        "accuracy": float(accuracy_score(y, pred)),
        "precision_macro": float(
            precision_score(y, pred, average="macro", zero_division=0)
        ),
        "recall_macro": float(
            recall_score(y, pred, average="macro", zero_division=0)
        ),
        "f1_macro": float(
            f1_score(y, pred, average="macro", zero_division=0)
        ),
    }
    try:
        out["roc_auc_ovr_macro"] = float(
            roc_auc_score(y, p, multi_class="ovr", average="macro")
        )
    except Exception:
        out["roc_auc_ovr_macro"] = float("nan")
    return out



ROUND_PROGRESS_MONITOR_MAX_SAMPLES = 4096
ROUND_PROGRESS_POWER_W = 12.0


def build_train_monitor_subset(
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    max_samples: int = ROUND_PROGRESS_MONITOR_MAX_SAMPLES,
    seed: int = 99117,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Build a deterministic lightweight monitoring subset from TRAIN client arrays only.

    This subset is used solely for console progress after FL rounds. It never affects
    preprocessing, governance, client selection, optimizer budgets, checkpoint
    decisions, or final reporting. The held-out TEST set remains final-evaluation-only.
    """
    rng = np.random.default_rng(int(seed))
    client_ids = sorted(client_arrays.keys())
    sizes = {cid: int(len(client_arrays[cid][1])) for cid in client_ids}
    total = int(sum(sizes.values()))
    target = int(min(max_samples, total))

    # Proportional allocation across clients, then distribute rounding remainder.
    raw = {cid: target * sizes[cid] / max(1, total) for cid in client_ids}
    take = {cid: min(sizes[cid], int(np.floor(raw[cid]))) for cid in client_ids}

    while sum(take.values()) < target:
        candidates = [c for c in client_ids if take[c] < sizes[c]]
        if not candidates:
            break
        cid = max(candidates, key=lambda c: (raw[c] - take[c], sizes[c], c))
        take[cid] += 1

    xs, ys = [], []
    for cid in client_ids:
        n_take = int(take[cid])
        if n_take <= 0:
            continue
        Xc, yc = client_arrays[cid]
        if n_take >= len(yc):
            idx = np.arange(len(yc))
        else:
            idx = rng.choice(len(yc), size=n_take, replace=False)
        xs.append(np.asarray(Xc[idx]))
        ys.append(np.asarray(yc[idx], dtype=np.int32))

    Xmon = np.concatenate(xs, axis=0)
    ymon = np.concatenate(ys, axis=0)

    # Deterministic shuffle so monitoring batches do not follow client order.
    order = rng.permutation(len(ymon))
    return Xmon[order], ymon[order]


def _fmt_metric(x: float) -> str:
    return "nan" if not np.isfinite(float(x)) else f"{float(x):.4f}"


def print_round_progress(
    progress_context: Optional[Dict[str, Any]],
    round_idx: int,
    round_total: int,
    selected_ids: List[str],
    round_steps: int,
    cumulative_steps: int,
    monitor_metrics: Dict[str, float],
    cumulative_runtime_s: float,
    cumulative_communication_mb: float,
    ram_start_mb: float,
    ram_peak_mb: float,
    extra: str = "",
) -> None:
    """Compact, human-readable progress block after each federated round."""
    ctx = progress_context or {}
    scenario = str(ctx.get("scenario", "Federated scenario"))
    run_idx = int(ctx.get("run_idx", 0))
    run_total = int(ctx.get("run_total", 0))
    scenario_idx = int(ctx.get("scenario_idx", 0))
    scenario_total = int(ctx.get("scenario_total", 0))
    overall_idx = int(ctx.get("overall_idx", 0))
    overall_total = int(ctx.get("overall_total", 0))

    energy_wh = float(
        ROUND_PROGRESS_POWER_W * float(cumulative_runtime_s) / 3600.0
    )
    ram_delta = max(0.0, float(ram_peak_mb) - float(ram_start_mb))

    print("\n" + "-" * 112)
    print(
        f"PROGRESS | overall configuration {overall_idx}/{overall_total} | "
        f"run {run_idx}/{run_total} | scenario {scenario_idx}/{scenario_total}"
    )
    print(f"SCENARIO | {scenario}")
    print(
        f"ROUND    | {round_idx}/{round_total} | "
        f"selected={len(selected_ids)} [{','.join(map(str, selected_ids))}] | "
        f"steps={round_steps} | cumulative_steps={cumulative_steps}"
    )
    if extra:
        print(f"DETAIL   | {extra}")
    print(
        "TRAIN-MONITOR (diagnostic only; TEST untouched) | "
        f"Accuracy={_fmt_metric(monitor_metrics.get('accuracy', np.nan))} | "
        f"F1={_fmt_metric(monitor_metrics.get('f1_macro', np.nan))} | "
        f"AUC={_fmt_metric(monitor_metrics.get('roc_auc_ovr_macro', np.nan))} | "
        f"Precision={_fmt_metric(monitor_metrics.get('precision_macro', np.nan))} | "
        f"Recall={_fmt_metric(monitor_metrics.get('recall_macro', np.nan))}"
    )
    print(
        f"CUMULATIVE OPERATIONAL | runtime={cumulative_runtime_s:.2f}s | "
        f"energy≈{energy_wh:.5f}Wh | communication={cumulative_communication_mb:.3f}MB | "
        f"RAM peak={ram_peak_mb:.1f}MB | RAM Δ={ram_delta:.1f}MB"
    )
    print("-" * 112)


def train_exact_steps(
    model: keras.Model,
    X: np.ndarray,
    y: np.ndarray,
    steps: int,
    batch_size: int,
    seed: int,
    class_weights: Optional[Dict[int, float]] = None,
    prox_reference: Optional[List[np.ndarray]] = None,
    prox_mu: float = 0.0,
):
    """
    Exact mini-batch update count. Used for step-parity audits.
    """
    steps = int(max(1, steps))
    rng = np.random.default_rng(int(seed))
    n = len(y)
    if n == 0:
        raise RuntimeError("Cannot train on an empty dataset.")

    loss_fn = keras.losses.SparseCategoricalCrossentropy(
        reduction=keras.losses.Reduction.NONE
    )

    order = rng.permutation(n)
    cursor = 0

    prox_tensors = None
    if prox_reference is not None and prox_mu > 0:
        prox_tensors = [tf.convert_to_tensor(w) for w in prox_reference]

    for _ in range(steps):
        if cursor + batch_size > n:
            order = rng.permutation(n)
            cursor = 0

        idx = order[cursor:cursor + batch_size]
        cursor += batch_size

        xb = tf.convert_to_tensor(np.asarray(X[idx]), dtype=tf.float32)
        yb_np = np.asarray(y[idx], dtype=np.int32)
        yb = tf.convert_to_tensor(yb_np, dtype=tf.int32)

        with tf.GradientTape() as tape:
            probs = model(xb, training=True)
            per_loss = loss_fn(yb, probs)

            if class_weights:
                sw = np.array(
                    [class_weights.get(int(v), 1.0) for v in yb_np],
                    dtype=np.float32,
                )
                sw_t = tf.convert_to_tensor(sw)
                data_loss = tf.reduce_sum(per_loss * sw_t) / tf.reduce_sum(sw_t)
            else:
                data_loss = tf.reduce_mean(per_loss)

            loss = data_loss

            if prox_tensors is not None:
                prox = tf.constant(0.0, dtype=tf.float32)
                for var, ref in zip(model.trainable_variables, prox_tensors):
                    prox += tf.reduce_sum(tf.square(var - tf.cast(ref, var.dtype)))
                loss = loss + 0.5 * float(prox_mu) * prox

        grads = tape.gradient(loss, model.trainable_variables)
        model.optimizer.apply_gradients(zip(grads, model.trainable_variables))


def natural_steps(n_records: int, batch_size: int, local_epochs: int = 1) -> int:
    return int(max(1, math.ceil(int(n_records) / int(batch_size)) * int(local_epochs)))


def allocate_exact_step_budget(
    selected: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    target_total: int,
    batch_size: int,
    local_epochs: int = 1,
) -> Dict[str, int]:
    natural = {
        cid: natural_steps(len(client_arrays[cid][1]), batch_size, local_epochs)
        for cid in selected
    }
    total_nat = max(1, sum(natural.values()))
    raw = {cid: target_total * natural[cid] / total_nat for cid in selected}
    alloc = {cid: max(1, int(math.floor(raw[cid]))) for cid in selected}

    # Adjust to exact target.
    while sum(alloc.values()) < target_total:
        cid = max(selected, key=lambda c: raw[c] - alloc[c])
        alloc[cid] += 1
    while sum(alloc.values()) > target_total:
        candidates = [c for c in selected if alloc[c] > 1]
        if not candidates:
            break
        cid = min(candidates, key=lambda c: raw[c] - alloc[c])
        alloc[cid] -= 1

    if sum(alloc.values()) != int(target_total):
        raise RuntimeError("Exact step-budget allocation failed.")
    return alloc


def aggregate_weights(
    local_weights: List[List[np.ndarray]],
    sample_sizes: List[int],
    equal_weight: bool = False,
) -> List[np.ndarray]:
    if not local_weights:
        raise RuntimeError("No local weights to aggregate.")
    if equal_weight:
        alpha = np.ones(len(local_weights), dtype=float) / len(local_weights)
    else:
        sizes = np.asarray(sample_sizes, dtype=float)
        alpha = sizes / sizes.sum()

    out = []
    for layer_idx in range(len(local_weights[0])):
        x = sum(alpha[j] * np.asarray(local_weights[j][layer_idx])
                for j in range(len(local_weights)))
        out.append(np.asarray(x))
    return out


def federated_train(
    build_model_fn,
    initial_weights: List[np.ndarray],
    selected_per_round: List[List[str]],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    local_epochs: int,
    class_weights: Dict[int, float],
    run_seed: int,
    exact_step_maps: Optional[List[Dict[str, int]]] = None,
    equal_weight: bool = False,
    fedprox_mu: float = 0.0,
    train_monitor: Optional[Tuple[np.ndarray, np.ndarray]] = None,
    progress_context: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    monitor = RAMMonitor().start()
    start = time.perf_counter()

    global_model = build_model_fn()
    global_model.set_weights([np.array(w, copy=True) for w in initial_weights])

    round_rows = []
    total_steps = 0
    total_selected = 0
    monitor_eval_s = 0.0
    cumulative_comm_raw_b = 0
    param_b = model_parameter_bytes(global_model)

    for r, selected in enumerate(selected_per_round, start=1):
        global_weights = [np.array(w, copy=True) for w in global_model.get_weights()]
        local_weights = []
        local_sizes = []
        round_steps = 0

        if exact_step_maps is None:
            step_map = {
                cid: natural_steps(
                    len(client_arrays[cid][1]), batch_size, local_epochs
                )
                for cid in selected
            }
        else:
            step_map = exact_step_maps[r - 1]

        for j, cid in enumerate(selected):
            Xc, yc = client_arrays[cid]
            local_model = build_model_fn()
            local_model.set_weights(global_weights)
            train_exact_steps(
                local_model, Xc, yc,
                steps=int(step_map[cid]),
                batch_size=batch_size,
                seed=int(run_seed + 1000 * r + 17 * j),
                class_weights=class_weights,
                prox_reference=global_weights if fedprox_mu > 0 else None,
                prox_mu=float(fedprox_mu),
            )
            local_weights.append(
                [np.array(w, copy=True) for w in local_model.get_weights()]
            )
            local_sizes.append(len(yc))
            round_steps += int(step_map[cid])

            del local_model
            tf.keras.backend.clear_session()

        agg = aggregate_weights(
            local_weights, local_sizes, equal_weight=equal_weight
        )
        global_model = build_model_fn()
        global_model.set_weights(agg)

        total_steps += round_steps
        total_selected += len(selected)

        # Cumulative model traffic through this round. Same accounting as the
        # final experiment metric: download + upload + 12% protocol overhead.
        cumulative_comm_raw_b += int(len(selected)) * 2 * int(param_b)
        cumulative_comm_mb = float(
            (cumulative_comm_raw_b * 1.12) / (1024 ** 2)
        )

        # Measure training runtime BEFORE this round's diagnostic evaluation.
        # Previous diagnostic-evaluation time is subtracted so progress printing
        # does not inflate the experiment's runtime metric.
        cumulative_runtime_s = float(
            max(0.0, (time.perf_counter() - start) - monitor_eval_s)
        )

        if train_monitor is not None:
            Xmon, ymon = train_monitor
            t_mon = time.perf_counter()
            monitor_metrics = evaluate_model(global_model, Xmon, ymon, n_classes)
            monitor_eval_s += float(time.perf_counter() - t_mon)
        else:
            monitor_metrics = {
                "accuracy": float("nan"),
                "precision_macro": float("nan"),
                "recall_macro": float("nan"),
                "f1_macro": float("nan"),
                "roc_auc_ovr_macro": float("nan"),
            }

        live_peak = float(monitor.peak_mb)
        round_row = {
            "round": r,
            "selected_clients": len(selected),
            "selected_ids": ";".join(selected),
            "optimizer_steps": int(round_steps),
            "cumulative_optimizer_steps": int(total_steps),
            "train_monitor_accuracy": float(monitor_metrics["accuracy"]),
            "train_monitor_precision_macro": float(monitor_metrics["precision_macro"]),
            "train_monitor_recall_macro": float(monitor_metrics["recall_macro"]),
            "train_monitor_f1_macro": float(monitor_metrics["f1_macro"]),
            "train_monitor_auc_ovr_macro": float(monitor_metrics["roc_auc_ovr_macro"]),
            "cumulative_runtime_s": float(cumulative_runtime_s),
            "cumulative_energy_wh_est": float(
                ROUND_PROGRESS_POWER_W * cumulative_runtime_s / 3600.0
            ),
            "cumulative_communication_mb": float(cumulative_comm_mb),
            "ram_peak_mb_live": float(live_peak),
            "ram_delta_mb_live": float(max(0.0, live_peak - monitor.start_mb)),
        }
        round_rows.append(round_row)

        print_round_progress(
            progress_context=progress_context,
            round_idx=r,
            round_total=len(selected_per_round),
            selected_ids=list(selected),
            round_steps=int(round_steps),
            cumulative_steps=int(total_steps),
            monitor_metrics=monitor_metrics,
            cumulative_runtime_s=cumulative_runtime_s,
            cumulative_communication_mb=cumulative_comm_mb,
            ram_start_mb=float(monitor.start_mb),
            ram_peak_mb=live_peak,
        )

    runtime = float(max(0.0, (time.perf_counter() - start) - monitor_eval_s))
    metrics = evaluate_model(global_model, X_test, y_test, n_classes)
    ram = monitor.stop()

    communication_b = int(cumulative_comm_raw_b * 1.12)

    return {
        "model": global_model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(total_steps),
        "communication_mb": float(communication_b / (1024 ** 2)),
        "round_audit": round_rows,
        "participants_mean_per_round": float(
            total_selected / max(1, len(round_rows))
        ),
        **ram,
    }



def select_dq_only_clients(
    dq_scores: Dict[str, Dict[str, float]],
    k: int,
) -> Tuple[List[str], pd.DataFrame]:
    """Select top-K clients by the machine-measured TADP Data Quality dimension only."""
    rows = []
    for cid, scores in dq_scores.items():
        vals = [float(scores[f]) for f in FACTOR_NAMES["dim2"]]
        rows.append({
            "client": str(cid),
            "dq_only_score": float(np.mean(vals)),
            "dq_factor_count": int(len(vals)),
        })
    audit = pd.DataFrame(rows).sort_values(
        ["dq_only_score", "client"], ascending=[False, True]
    ).reset_index(drop=True)
    audit["dq_only_rank"] = np.arange(1, len(audit) + 1)
    selected = audit.head(int(k))["client"].astype(str).tolist()
    audit["dq_only_selected"] = audit["client"].isin(selected)
    return selected, audit


def local_training_loss(model: keras.Model, X: np.ndarray, y: np.ndarray, batch_size: int = 512) -> float:
    """Mean sparse cross-entropy on client TRAIN data only; used by Power-of-Choice."""
    p = model.predict(X, batch_size=batch_size, verbose=0)
    y = np.asarray(y, dtype=np.int32)
    idx = np.arange(len(y))
    probs = np.clip(p[idx, y], 1e-12, 1.0)
    return float(-np.mean(np.log(probs)))


def federated_train_power_of_choice(
    build_model_fn,
    initial_weights: List[np.ndarray],
    candidate_clients: List[str],
    select_k: int,
    target_steps_per_round: int,
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    local_epochs: int,
    class_weights: Dict[int, float],
    run_seed: int,
    candidate_multiplier: int = 2,
    train_monitor: Optional[Tuple[np.ndarray, np.ndarray]] = None,
    progress_context: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Power-of-Choice baseline: each round samples d candidates and selects the K
    clients with largest current local TRAIN loss. K, rounds, initialisation, and
    total optimizer steps per round are matched to TADP-VR. No TEST information is used.
    """
    monitor = RAMMonitor().start()
    start = time.perf_counter()
    global_model = build_model_fn()
    global_model.set_weights([np.array(w, copy=True) for w in initial_weights])
    rng = np.random.default_rng(int(run_seed) + 880000)
    round_rows = []
    total_steps = 0
    total_selected = 0
    communication_b = 0
    monitor_eval_s = 0.0

    all_candidates = list(candidate_clients)
    if int(select_k) < 1 or int(select_k) > len(all_candidates):
        raise RuntimeError("Invalid Power-of-Choice K.")

    for r in range(1, NUM_ROUNDS_FL + 1):
        d = min(len(all_candidates), max(int(select_k), int(candidate_multiplier) * int(select_k)))
        if d == len(all_candidates):
            candidate_pool = list(all_candidates)
        else:
            # Canonical pow-d samples candidate clients without replacement
            # according to p_k, the client's fraction of total TRAIN data.
            sizes = np.asarray(
                [len(client_arrays[c][1]) for c in all_candidates], dtype=float
            )
            probs = sizes / sizes.sum()
            candidate_pool = rng.choice(
                all_candidates, size=d, replace=False, p=probs
            ).tolist()

        losses = []
        for cid in candidate_pool:
            Xc, yc = client_arrays[cid]
            losses.append((str(cid), local_training_loss(global_model, Xc, yc)))
        losses.sort(key=lambda x: (-x[1], x[0]))
        selected = [cid for cid, _ in losses[:int(select_k)]]
        step_map = allocate_exact_step_budget(
            selected, client_arrays, int(target_steps_per_round), batch_size, local_epochs
        )

        global_weights = [np.array(w, copy=True) for w in global_model.get_weights()]
        local_weights, local_sizes = [], []
        for j, cid in enumerate(selected):
            Xc, yc = client_arrays[cid]
            local_model = build_model_fn()
            local_model.set_weights(global_weights)
            train_exact_steps(
                local_model, Xc, yc,
                steps=int(step_map[cid]), batch_size=batch_size,
                seed=int(run_seed + 1000 * r + 17 * j),
                class_weights=class_weights,
            )
            local_weights.append([np.array(w, copy=True) for w in local_model.get_weights()])
            local_sizes.append(len(yc))
            del local_model
            tf.keras.backend.clear_session()

        agg = aggregate_weights(local_weights, local_sizes, equal_weight=False)
        global_model = build_model_fn()
        global_model.set_weights(agg)
        round_steps = int(sum(step_map.values()))
        total_steps += round_steps
        total_selected += len(selected)
        param_b = model_parameter_bytes(global_model)
        # Candidate clients receive the current model to evaluate local loss;
        # selected clients return one model update. Scalar loss uploads are negligible.
        communication_b += (len(candidate_pool) + len(selected)) * param_b
        cumulative_comm_mb = float(
            (communication_b * 1.12) / (1024 ** 2)
        )
        cumulative_runtime_s = float(
            max(0.0, (time.perf_counter() - start) - monitor_eval_s)
        )

        if train_monitor is not None:
            Xmon, ymon = train_monitor
            t_mon = time.perf_counter()
            monitor_metrics = evaluate_model(global_model, Xmon, ymon, n_classes)
            monitor_eval_s += float(time.perf_counter() - t_mon)
        else:
            monitor_metrics = {
                "accuracy": float("nan"),
                "precision_macro": float("nan"),
                "recall_macro": float("nan"),
                "f1_macro": float("nan"),
                "roc_auc_ovr_macro": float("nan"),
            }

        live_peak = float(monitor.peak_mb)
        round_rows.append({
            "round": r,
            "candidate_count": len(candidate_pool),
            "candidate_ids": ";".join(candidate_pool),
            "selected_clients": len(selected),
            "selected_ids": ";".join(selected),
            "optimizer_steps": round_steps,
            "cumulative_optimizer_steps": int(total_steps),
            "local_losses": json.dumps({cid: loss for cid, loss in losses}, sort_keys=True),
            "train_monitor_accuracy": float(monitor_metrics["accuracy"]),
            "train_monitor_precision_macro": float(monitor_metrics["precision_macro"]),
            "train_monitor_recall_macro": float(monitor_metrics["recall_macro"]),
            "train_monitor_f1_macro": float(monitor_metrics["f1_macro"]),
            "train_monitor_auc_ovr_macro": float(monitor_metrics["roc_auc_ovr_macro"]),
            "cumulative_runtime_s": float(cumulative_runtime_s),
            "cumulative_energy_wh_est": float(
                ROUND_PROGRESS_POWER_W * cumulative_runtime_s / 3600.0
            ),
            "cumulative_communication_mb": float(cumulative_comm_mb),
            "ram_peak_mb_live": float(live_peak),
            "ram_delta_mb_live": float(max(0.0, live_peak - monitor.start_mb)),
        })

        loss_preview = ", ".join(
            f"{cid}:{loss:.3f}" for cid, loss in losses[:min(5, len(losses))]
        )
        print_round_progress(
            progress_context=progress_context,
            round_idx=r,
            round_total=NUM_ROUNDS_FL,
            selected_ids=list(selected),
            round_steps=int(round_steps),
            cumulative_steps=int(total_steps),
            monitor_metrics=monitor_metrics,
            cumulative_runtime_s=cumulative_runtime_s,
            cumulative_communication_mb=cumulative_comm_mb,
            ram_start_mb=float(monitor.start_mb),
            ram_peak_mb=live_peak,
            extra=f"PoC candidates={len(candidate_pool)} | highest TRAIN losses: {loss_preview}",
        )

    runtime = float(max(0.0, (time.perf_counter() - start) - monitor_eval_s))
    metrics = evaluate_model(global_model, X_test, y_test, n_classes)
    ram = monitor.stop()
    communication_b = int(communication_b * 1.12)
    return {
        "model": global_model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(total_steps),
        "communication_mb": float(communication_b / (1024 ** 2)),
        "round_audit": round_rows,
        "participants_mean_per_round": float(total_selected / max(1, NUM_ROUNDS_FL)),
        **ram,
    }


def centralized_train(
    build_model_fn,
    initial_weights: List[np.ndarray],
    selected_clients: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    exact_steps: int,
    class_weights: Dict[int, float],
    seed: int,
) -> Dict[str, Any]:
    if not selected_clients:
        raise RuntimeError("Centralized scenario has no TRAIN clients.")

    monitor = RAMMonitor().start()
    start = time.perf_counter()

    # Pool ACCEPTED TRAIN partitions only. TEST is not present here.
    X = np.concatenate([client_arrays[c][0] for c in selected_clients], axis=0)
    y = np.concatenate([client_arrays[c][1] for c in selected_clients], axis=0)

    model = build_model_fn()
    model.set_weights([np.array(w, copy=True) for w in initial_weights])
    train_exact_steps(
        model, X, y,
        steps=int(exact_steps),
        batch_size=batch_size,
        seed=int(seed),
        class_weights=class_weights,
    )

    runtime = time.perf_counter() - start
    metrics = evaluate_model(model, X_test, y_test, n_classes)
    ram = monitor.stop()

    return {
        "model": model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(exact_steps),
        "communication_mb": 0.0,
        "participants_mean_per_round": float(len(selected_clients)),
        **ram,
    }


def result_row(
    run: int,
    seed: int,
    scenario: str,
    result: Dict[str, Any],
    initial_hash: str,
    power_w: float = 12.0,
) -> Dict[str, Any]:
    row = {
        "run": int(run),
        "seed": int(seed),
        "scenario": str(scenario),
        **result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "participants": float(result.get("participants_mean_per_round", 0.0)),
        "ram_start_mb": float(result.get("ram_start_mb", 0.0)),
        "ram_end_mb": float(result.get("ram_end_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
        "ram_delta_mb": float(result.get("ram_delta_mb", 0.0)),
        "ram_mb": float(result.get("ram_mb", result.get("ram_peak_mb", 0.0))),
        "initial_weights_sha256": initial_hash,
    }
    row["energy_wh"] = float(power_w * row["runtime_s"] / 3600.0)
    row["energy_kwh"] = float(row["energy_wh"] / 1000.0)
    row["co2_kg"] = float(row["energy_kwh"] * 0.430)
    row["energy_cost_usd"] = float(row["energy_kwh"] * 0.20)
    row["communication_cost_usd"] = float(row["communication_mb"] * 0.005)
    row["total_estimated_cost_usd"] = float(
        row["energy_cost_usd"] + row["communication_cost_usd"]
    )
    return row


# ======================================================================================
# LEAKAGE AUDIT
# ======================================================================================

def write_leakage_audit(
    out_dir: Path,
    train_ids,
    test_ids,
    client_train_ids: Dict[str, np.ndarray],
    extra: Optional[Dict[str, Any]] = None,
):
    train_set = set(map(str, train_ids))
    test_set = set(map(str, test_ids))
    overlap = train_set & test_set

    client_union = set()
    duplicates_across_clients = 0
    for cid, ids in client_train_ids.items():
        s = set(map(str, ids))
        duplicates_across_clients += len(client_union & s)
        client_union |= s

    test_in_clients = len(test_set & client_union)
    missing_train = len(train_set - client_union)
    extra_client_rows = len(client_union - train_set)

    row = {
        "train_test_overlap": len(overlap),
        "test_rows_in_any_client": test_in_clients,
        "train_rows_missing_from_clients": missing_train,
        "client_rows_not_in_global_train": extra_client_rows,
        "duplicate_train_rows_across_clients": duplicates_across_clients,
        "pass": (
            len(overlap) == 0
            and test_in_clients == 0
            and missing_train == 0
            and extra_client_rows == 0
            and duplicates_across_clients == 0
        ),
    }
    if extra:
        row.update(extra)

    pd.DataFrame([row]).to_csv(
        Path(out_dir) / "leakage_audit.csv", index=False
    )

    if not bool(row["pass"]):
        raise RuntimeError(f"FAIL-CLOSED leakage audit failed: {row}")
    return row


# ======================================================================================
# DIABETES 130-US — LEAKAGE-SAFE DATA PREPARATION
# ======================================================================================

TARGET = "readmitted"
ID_COLUMNS = ["encounter_id", "patient_nbr"]


def locate_diabetes_csv() -> str:
    candidates = [
        os.environ.get("DIABETES_CSV", ""),
        "/content/diabetes_130US.csv",
        "./diabetes_130US.csv",
        "/content/drive/MyDrive/diabetes_130US.csv",
    ]
    for p in candidates:
        if p and os.path.exists(p):
            return p

    try:
        from google.colab import files as colab_files
        print("Please upload the Diabetes 130-US CSV file.")
        uploaded = colab_files.upload()
        csvs = [name for name in uploaded if str(name).lower().endswith(".csv")]
        if not csvs:
            raise RuntimeError("No CSV file was uploaded.")
        return str(csvs[0])
    except ImportError:
        pass

    raise FileNotFoundError(
        "Diabetes CSV not found. Set DIABETES_CSV or upload the CSV in Colab."
    )


def load_diabetes(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df = df.replace("?", np.nan)

    if TARGET not in df.columns:
        raise RuntimeError(f"Missing target column: {TARGET}")

    valid_target = {"NO": 0, ">30": 1, "<30": 2}
    df = df[df[TARGET].isin(valid_target)].copy()
    df["_target"] = df[TARGET].map(valid_target).astype(np.int32)
    df["_row_id"] = np.arange(len(df), dtype=np.int64)

    return df


def global_patient_grouped_split(
    df: pd.DataFrame, seed: int, test_fraction: float = 0.20
):
    """
    Patient-grouped and stratified whenever patient_nbr is available.
    The split happens before any client partitioning or data-dependent preprocessing.
    """
    if "patient_nbr" in df.columns:
        splitter = StratifiedGroupKFold(
            n_splits=5, shuffle=True, random_state=int(seed)
        )
        train_idx, test_idx = next(
            splitter.split(
                np.zeros(len(df)),
                y=df["_target"].to_numpy(),
                groups=df["patient_nbr"].astype(str).to_numpy(),
            )
        )
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        patient_overlap = len(
            set(train_df["patient_nbr"].astype(str))
            & set(test_df["patient_nbr"].astype(str))
        )
        if patient_overlap != 0:
            raise RuntimeError("Patient leakage detected across TRAIN/TEST.")
    else:
        train_df, test_df = train_test_split(
            df, test_size=float(test_fraction),
            stratify=df["_target"], random_state=int(seed)
        )
        patient_overlap = 0

    return train_df.reset_index(drop=True), test_df.reset_index(drop=True), patient_overlap


def dirichlet_partition_dataframe(
    train_df: pd.DataFrame,
    n_clients: int,
    alpha: float,
    seed: int,
    min_client_records: int = 100,
) -> Dict[str, pd.DataFrame]:
    y = train_df["_target"].to_numpy()
    client_ids = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")[:n_clients]

    for attempt in range(100):
        rng = np.random.default_rng(int(seed + attempt))
        buckets = [[] for _ in range(n_clients)]

        for cls in sorted(np.unique(y)):
            idx = np.where(y == cls)[0]
            rng.shuffle(idx)
            props = rng.dirichlet(np.full(n_clients, float(alpha)))
            cuts = (np.cumsum(props)[:-1] * len(idx)).astype(int)
            splits = np.split(idx, cuts)
            for k, s in enumerate(splits):
                buckets[k].extend(s.tolist())

        sizes = [len(b) for b in buckets]
        if min(sizes) >= int(min_client_records):
            out = {}
            for cid, idxs in zip(client_ids, buckets):
                out[cid] = train_df.iloc[np.array(idxs, dtype=int)].copy()
            return out

    raise RuntimeError("Could not obtain a valid Dirichlet client partition.")


@dataclass
class TabularPreprocessor:
    feature_cols: List[str]
    numeric_cols: List[str]
    categorical_cols: List[str]
    mean: Dict[str, float]
    std: Dict[str, float]
    categories: Dict[str, List[str]]
    encoder: Any

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        pieces = []

        if self.numeric_cols:
            x_num = []
            for c in self.numeric_cols:
                s = pd.to_numeric(df[c], errors="coerce").astype(float)
                a = s.fillna(self.mean[c]).to_numpy(dtype=np.float32)
                a = (a - self.mean[c]) / self.std[c]
                x_num.append(a[:, None])
            pieces.append(np.concatenate(x_num, axis=1).astype(np.float32))

        if self.categorical_cols:
            cat = pd.DataFrame({
                c: df[c].astype("string").fillna("__MISSING__").astype(str)
                for c in self.categorical_cols
            })
            x_cat = self.encoder.transform(cat)
            pieces.append(np.asarray(x_cat, dtype=np.float32))

        if not pieces:
            raise RuntimeError("No predictor columns remained.")
        return np.concatenate(pieces, axis=1).astype(np.float32)


def fit_federated_train_only_preprocessor(
    client_frames: Dict[str, pd.DataFrame]
) -> TabularPreprocessor:
    """
    No raw TRAIN pooling is used to ESTIMATE numeric parameters.
    Numeric mean/std comes from aggregated local count/sum/sum-of-squares.
    Categorical vocabulary comes from union of local TRAIN category sets.
    """
    any_df = next(iter(client_frames.values()))
    feature_cols = [
        c for c in any_df.columns
        if c not in {TARGET, "_target", "_row_id", *ID_COLUMNS}
    ]

    # Infer expected type from TRAIN only.
    numeric_cols = []
    categorical_cols = []
    for c in feature_cols:
        total_nonmissing = 0
        numeric_valid = 0
        for df in client_frames.values():
            raw = df[c]
            nm = raw.notna()
            total_nonmissing += int(nm.sum())
            if nm.any():
                numeric_valid += int(
                    pd.to_numeric(raw[nm], errors="coerce").notna().sum()
                )
        ratio = numeric_valid / max(1, total_nonmissing)
        if ratio >= 0.95:
            numeric_cols.append(c)
        else:
            categorical_cols.append(c)

    mean = {}
    std = {}
    for c in numeric_cols:
        count = 0
        sum_ = 0.0
        sumsq = 0.0
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            count += len(a)
            sum_ += float(a.sum())
            sumsq += float(np.square(a).sum())
        mu = sum_ / max(1, count)
        var = max(1e-12, sumsq / max(1, count) - mu * mu)
        mean[c] = float(mu)
        std[c] = float(math.sqrt(var))

    categories = {}
    for c in categorical_cols:
        values = set()
        for df in client_frames.values():
            s = df[c].astype("string").fillna("__MISSING__").astype(str)
            values.update(s.unique().tolist())
        categories[c] = sorted(values)

    encoder = OneHotEncoder(
        categories=[categories[c] for c in categorical_cols],
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float32,
    )

    # Fit only metadata-shaped dummy rows; the vocabulary is already frozen from TRAIN.
    if categorical_cols:
        max_len = max(len(categories[c]) for c in categorical_cols)
        dummy = {}
        for c in categorical_cols:
            vals = categories[c]
            dummy[c] = [vals[i % len(vals)] for i in range(max_len)]
        encoder.fit(pd.DataFrame(dummy))

    return TabularPreprocessor(
        feature_cols=feature_cols,
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        mean=mean,
        std=std,
        categories=categories,
        encoder=encoder,
    )


def build_tabular_reference(
    client_frames: Dict[str, pd.DataFrame],
    preprocessor: TabularPreprocessor,
) -> Dict[str, Any]:
    """
    TRAIN-only DQ reference built from client-local summaries only.

    Numerical reference histograms are constructed by combining local
    min/max summaries and then summing client-local histogram counts.
    Categorical reference support is the union of local TRAIN category sets.
    Raw client records are not concatenated to construct the reference.
    """
    ref = {"num_hist": {}, "cat_values": {}}

    chosen_num = preprocessor.numeric_cols[:12]
    for c in chosen_num:
        local_min, local_max = [], []
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                local_min.append(float(np.min(a)))
                local_max.append(float(np.max(a)))
        if not local_min:
            continue
        gmin, gmax = float(min(local_min)), float(max(local_max))
        if gmax <= gmin:
            edges = np.array([gmin - 1e-6, gmax + 1e-6], dtype=float)
        else:
            edges = np.linspace(gmin, gmax, 11, dtype=float)
        global_hist = np.zeros(len(edges)-1, dtype=np.float64)
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                h, _ = np.histogram(a, bins=edges)
                global_hist += h.astype(np.float64)
        ref["num_hist"][c] = {
            "edges": edges.tolist(),
            "hist": global_hist.tolist(),
            "construction": "aggregated_client_local_histograms_only",
        }

    for c in preprocessor.categorical_cols:
        values = set()
        for df in client_frames.values():
            values.update(
                df[c].astype("string").fillna("__MISSING__")
                .astype(str).unique().tolist()
            )
        ref["cat_values"][c] = sorted(values)
    return ref

def tabular_dq_scores(
    df: pd.DataFrame,
    preprocessor: TabularPreprocessor,
    reference: Dict[str, Any],
) -> Tuple[Dict[str, float], Dict[str, float]]:
    """
    Eight machine-measured TRAIN-only DQ factors for the healthcare experiment.
    """
    feature_df = df[preprocessor.feature_cols]

    # 1) Completeness.
    missing_fraction = float(feature_df.isna().mean().mean())
    completeness = score_lower_is_better(
        missing_fraction,
        [0.01, 0.05, 0.10, 0.20, 0.50],
    )

    # 2) Duplication rate.
    duplicate_fraction = float(feature_df.duplicated().mean())
    duplication = score_lower_is_better(
        duplicate_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 3) Value validity / error rate.
    bad = 0
    observed = 0

    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())

        if nm.any():
            conv = pd.to_numeric(
                raw[nm], errors="coerce"
            ).to_numpy(dtype=float)
            bad += int(np.sum(~np.isfinite(conv)))

    for c in preprocessor.categorical_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())

        if nm.any():
            bad += int(
                np.sum(raw[nm].astype(str).str.strip().eq(""))
            )

    error_fraction = float(bad / max(1, observed))
    value_validity_error_rate = score_lower_is_better(
        error_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.15],
    )

    # 4) Type consistency.
    type_bad = 0
    type_obs = 0

    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        type_obs += int(nm.sum())

        if nm.any():
            type_bad += int(
                pd.to_numeric(
                    raw[nm], errors="coerce"
                ).isna().sum()
            )

    type_inconsistency = float(type_bad / max(1, type_obs))
    type_consistency = score_lower_is_better(
        type_inconsistency,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 5) Label integrity.
    invalid_label_fraction = float(
        (~df["_target"].isin([0, 1, 2])).mean()
    )
    label_integrity = score_lower_is_better(
        invalid_label_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    # 6) Feature-distribution consistency — TRAIN-only reference.
    jsds = []

    for c, spec in reference["num_hist"].items():
        a = pd.to_numeric(
            df[c], errors="coerce"
        ).to_numpy(dtype=float)
        a = a[np.isfinite(a)]

        if len(a):
            hist, _ = np.histogram(
                a,
                bins=np.array(spec["edges"], dtype=float),
            )
            jsds.append(
                js_divergence(
                    hist,
                    np.array(spec["hist"], dtype=float),
                )
            )

    max_jsd = float(max(jsds)) if jsds else 0.0
    distribution_consistency = score_lower_is_better(
        max_jsd,
        [0.01, 0.025, 0.05, 0.10, 0.20],
    )

    # 7) Feature/category coverage.
    coverage_vals = []

    for c, ref_vals in reference["cat_values"].items():
        ref_set = set(ref_vals)

        if ref_set:
            client_set = set(
                df[c]
                .astype("string")
                .fillna("__MISSING__")
                .astype(str)
                .unique()
            )
            coverage_vals.append(
                len(client_set & ref_set) / len(ref_set)
            )

    mean_coverage = (
        float(np.mean(coverage_vals))
        if coverage_vals else 1.0
    )
    feature_coverage = score_higher_is_better(
        mean_coverage,
        [0.90, 0.825, 0.75, 0.65, 0.50],
    )

    # 8) Structural / constraint integrity.
    # Uses only TRAIN records. It checks:
    #   - key presence / encounter uniqueness;
    #   - non-negative count-like clinical fields;
    #   - finite numeric values where a numeric value is expected.
    n = len(df)
    record_violation = np.zeros(n, dtype=bool)

    if "encounter_id" not in df.columns:
        record_violation[:] = True
    else:
        encounter = df["encounter_id"]
        record_violation |= encounter.isna().to_numpy()
        record_violation |= encounter.duplicated(keep=False).to_numpy()

    nonnegative_fields = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]

    for c in nonnegative_fields:
        if c in df.columns:
            a = pd.to_numeric(
                df[c], errors="coerce"
            ).to_numpy(dtype=float)
            bad_c = (~np.isfinite(a)) | (a < 0)
            record_violation |= bad_c

    structural_violation_fraction = float(
        np.mean(record_violation)
    ) if n else 1.0

    structural_integrity = score_lower_is_better(
        structural_violation_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    scores = {
        "completeness": completeness,
        "duplication_rate": duplication,
        "value_validity_error_rate": value_validity_error_rate,
        "type_consistency": type_consistency,
        "label_integrity": label_integrity,
        "feature_distribution_consistency": distribution_consistency,
        "feature_category_coverage": feature_coverage,
        "structural_constraint_integrity": structural_integrity,
    }

    raw = {
        "missing_fraction": missing_fraction,
        "duplicate_fraction": duplicate_fraction,
        "error_fraction": error_fraction,
        "type_inconsistency_fraction": type_inconsistency,
        "invalid_label_fraction": invalid_label_fraction,
        "max_jsd": max_jsd,
        "mean_category_coverage": mean_coverage,
        "structural_violation_fraction": structural_violation_fraction,
    }

    return scores, raw

def prepare_diabetes_no_leakage(
    csv_path: str,
    split_seed: int,
    partition_seed: int,
    n_clients: int = 10,
    alpha: float = 1.0,
):
    raw = load_diabetes(csv_path)
    train_df, test_df, patient_overlap = global_patient_grouped_split(
        raw, split_seed
    )

    clients = dirichlet_partition_dataframe(
        train_df, n_clients=n_clients, alpha=alpha, seed=partition_seed
    )
    client_ids = list(clients.keys())

    # Hard row-level leakage audit.
    client_train_ids = {
        cid: df["_row_id"].astype(str).to_numpy()
        for cid, df in clients.items()
    }

    pre = fit_federated_train_only_preprocessor(clients)
    reference = build_tabular_reference(clients, pre)

    dq_scores = {}
    dq_raw_rows = []
    client_arrays = {}

    for cid, df in clients.items():
        scores, raw_metrics = tabular_dq_scores(df, pre, reference)
        dq_scores[cid] = scores

        row = {"client": cid, **scores, **raw_metrics}
        dq_raw_rows.append(row)

        X = pre.transform(df)
        y = df["_target"].to_numpy(dtype=np.int32)
        client_arrays[cid] = (X, y)

    X_test = pre.transform(test_df)
    y_test = test_df["_target"].to_numpy(dtype=np.int32)

    y_train_all = np.concatenate(
        [client_arrays[c][1] for c in client_ids]
    )
    cw = class_weight_dict(y_train_all)

    meta = {
        "raw_rows": len(raw),
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "patient_overlap": int(patient_overlap),
        "input_dim": int(X_test.shape[1]),
        "n_clients": int(n_clients),
        "numeric_features": len(pre.numeric_cols),
        "categorical_features": len(pre.categorical_cols),
    }

    return {
        "raw": raw,
        "train_df": train_df,
        "test_df": test_df,
        "clients_raw": clients,
        "client_arrays": client_arrays,
        "client_ids": client_ids,
        "X_test": X_test,
        "y_test": y_test,
        "dq_scores": dq_scores,
        "dq_audit": pd.DataFrame(dq_raw_rows),
        "class_weights": cw,
        "meta": meta,
        "client_train_ids": client_train_ids,
        "global_train_ids": train_df["_row_id"].astype(str).to_numpy(),
        "global_test_ids": test_df["_row_id"].astype(str).to_numpy(),
        "preprocessor": pre,
        "dq_reference": reference,
    }


def build_diabetes_model(input_dim: int, lr: float = 1e-3) -> keras.Model:
    inp = keras.Input(shape=(int(input_dim),), dtype=tf.float32)
    x = layers.Dense(128, activation="relu")(inp)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(3, activation="softmax", dtype=tf.float32)(x)
    model = keras.Model(inp, out)
    model.optimizer = keras.optimizers.Adam(learning_rate=float(lr))
    return model



# ======================================================================================
# FULL EXPERIMENT-A REPORTING, CHECKPOINTING, LEDGER, AND STATISTICS
# ======================================================================================
from datetime import datetime, timezone
import re

# =============================================================================
# MAIN REPEATED TRAINING SET
#
# These eight configurations are genuinely distinct and are repeated over all
# five training seeds for mean ± SD / CI reporting.
#
# NOT repeated here:
#   - Great Expectations Centralized
#   - TADP-AA Centralized
#   - Great Expectations Federated
#   - TADP-AA Federated
# Those all-client equivalences are verified independently by the companion
# one-seed equivalence-audit script.
#
# TADP-SDA is retained as a boundary/stress condition but is run once only,
# outside the five-seed headline statistical comparison.
# =============================================================================
MANUSCRIPT_SCENARIOS = [
    "Naïve Centralized",
    "TADP-VR Centralized",
    "Vanilla FedAvg",
    "FedProx",
    "Random-K",
    "DQ-only Federated",
    "Power-of-Choice",
    "TADP-VR Federated",
]
assert len(MANUSCRIPT_SCENARIOS) == 8

BOUNDARY_SCENARIOS = [
    "TADP-SDA Centralized",
    "TADP-SDA Federated",
]
BOUNDARY_SEED = 42



def print_banner(title: str, width: int = 108):
    print("\n" + "=" * width)
    print(title)
    print("=" * width)


def client_partition_table(clients_raw):
    rows = []
    for cid, df in clients_raw.items():
        counts = df["_target"].value_counts().to_dict()
        rows.append({
            "client": cid,
            "records": int(len(df)),
            "class_0_NO": int(counts.get(0, 0)),
            "class_1_GT30": int(counts.get(1, 0)),
            "class_2_LT30": int(counts.get(2, 0)),
        })
    return pd.DataFrame(rows)


def evidence_assignment_summary(
    evidence_df: pd.DataFrame
) -> pd.DataFrame:
    """
    Summarize the frozen controlled documentary-evidence assignment.

    v16.7 reports the pre-specified governance archetype for each evidence
    bundle. These archetypes are branch-coverage scenarios, not observed
    real-world prevalence classes.
    """
    required = {
        "client",
        "bundle_id",
        "evidence_profile",
        "scenario_role",
        "profile_variant",
        "factor",
        "rubric_score_0_5",
        "meets_factor_adequacy",
        "evidence_seed",
    }

    missing = sorted(
        required - set(evidence_df.columns)
    )

    if missing:
        raise RuntimeError(
            "Controlled documentary-evidence table is missing required "
            f"column(s): {missing}. Available columns: "
            f"{sorted(evidence_df.columns.tolist())}"
        )

    work = evidence_df.copy()
    work["meets_factor_adequacy"] = (
        work["meets_factor_adequacy"]
        .astype(bool)
    )

    summary = (
        work
        .groupby(
            ["client", "bundle_id"],
            as_index=False,
        )
        .agg(
            evidence_profile=(
                "evidence_profile",
                "first",
            ),
            scenario_role=(
                "scenario_role",
                "first",
            ),
            profile_variant=(
                "profile_variant",
                "first",
            ),
            documentary_factor_count=(
                "factor",
                "count",
            ),
            documentary_adequate_factor_count=(
                "meets_factor_adequacy",
                "sum",
            ),
            documentary_mean_score=(
                "rubric_score_0_5",
                "mean",
            ),
            documentary_min_score=(
                "rubric_score_0_5",
                "min",
            ),
            documentary_max_score=(
                "rubric_score_0_5",
                "max",
            ),
            evidence_seed=(
                "evidence_seed",
                "first",
            ),
        )
        .sort_values("client")
        .reset_index(drop=True)
    )

    summary["documentary_adequacy_fraction"] = (
        summary["documentary_adequate_factor_count"]
        / summary["documentary_factor_count"].clip(lower=1)
    )

    expected_documentary_factors = sum(
        len(FACTOR_NAMES[d])
        for d in DOCUMENTARY_DIMS
    )

    count_ok = summary[
        "documentary_factor_count"
    ].eq(
        expected_documentary_factors
    )

    if not count_ok.all():
        bad = summary.loc[
            ~count_ok,
            [
                "client",
                "bundle_id",
                "documentary_factor_count",
            ],
        ]

        raise RuntimeError(
            "Unexpected controlled-evidence factor count. "
            f"Expected {expected_documentary_factors} documentary factors "
            "per client. Offending rows:\n"
            + bad.to_string(index=False)
        )

    return summary

def print_governance_details(
    gov: pd.DataFrame,
    ge: pd.DataFrame,
    dq: pd.DataFrame,
    vr_clients: List[str],
    sda_clients: List[str],
    ge_clients: List[str],
    domain: str,
):
    domain = str(domain).lower()

    print_banner(
        "DOMAIN ADEQUACY POLICY — FULL WAC + CRITICAL WAC"
    )

    policy_rows = []
    for dim in FACTOR_NAMES:
        policy_rows.append({
            "dimension":
                dim,
            "dimension_name":
                DIMENSION_NAMES[dim],
            "n_factors":
                len(FACTOR_NAMES[dim]),
            "minimum_adequacy_ranks":
                ",".join(
                    str(
                        DOMAIN_FACTOR_MINIMA[
                            domain
                        ][dim][f]
                    )
                    for f in FACTOR_NAMES[
                        dim
                    ]
                ),
            "dimension_policy_wac":
                DOMAIN_DIMENSION_WAC[
                    domain
                ][dim],
        })

    print(
        pd.DataFrame(
            policy_rows
        ).to_string(
            index=False
        )
    )

    global_critical_wac = derive_global_critical_wac(
        domain
    )

    print(
        f"\nGLOBAL {domain.upper()} WAC "
        f"(descriptive full-policy summary) = "
        f"{DOMAIN_GLOBAL_WAC[domain]:.6f}"
    )
    print(
        f"GLOBAL {domain.upper()} CRITICAL WAC "
        f"(automated Review threshold) = "
        f"{global_critical_wac:.6f}"
    )

    print_banner(
        "CRITICAL FACTORS — INDIVIDUAL ADEQUACY REQUIREMENTS"
    )

    critical_rows = []

    for dim, factor_list in (
        CRITICAL_FACTORS_BY_DOMAIN[
            domain
        ].items()
    ):
        for factor in factor_list:
            minimum = float(
                DOMAIN_FACTOR_MINIMA[
                    domain
                ][dim][factor]
            )
            critical_rows.append({
                "dimension":
                    dim,
                "factor":
                    factor,
                "individual_adequacy_min_0_5":
                    minimum,
                "normalized_reference":
                    minimum / MAX_FACTOR_SCORE,
                "direct_auto_accept_rule":
                    f"score >= {minimum:.1f}",
            })

    print(
        pd.DataFrame(
            critical_rows
        ).to_string(
            index=False
        )
    )

    print(
        f"\nMinimum dimension floor: EVERY averaged dimension "
        f"must be >= {DIMENSION_MIN_FLOOR:.1f}/5."
    )

    print(
        "Human reviewer role: verify uploaded questionnaire evidence only. "
        "The server makes the admission decision automatically."
    )

    print_banner(
        "TRAIN-ONLY DATA-QUALITY FACTORS — 8 FACTORS"
    )

    dq_cols = [
        "client",
        "completeness",
        "duplication_rate",
        "value_validity_error_rate",
        "type_consistency",
        "label_integrity",
        "feature_distribution_consistency",
        "feature_category_coverage",
        "structural_constraint_integrity",
    ]

    print(
        dq[dq_cols].to_string(
            index=False
        )
    )

    print_banner(
        "FROZEN TADP GOVERNANCE — HPS + CRITICAL WAC"
    )

    display_cols = [
        "client",
        "hps",
        "critical_wac_i",
        "global_critical_wac",
        "critical_wac_margin",
        "all_critical_meet_adequacy",
        "critical_below_adequacy",
        "dimension_floor_failures",
        "dim1_score_0_5",
        "dim2_score_0_5",
        "dim3_score_0_5",
        "dim4_score_0_5",
        "dim5_score_0_5",
        "dim6_score_0_5",
        "decision_path",
        "initial_action",
        "final_action",
        "status",
        "reason",
    ]

    print(
        gov[
            display_cols
        ].to_string(
            index=False
        )
    )

    print(
        "\nTADP v16.7 final decision order:"
    )
    print(
        f"  1) ANY averaged dimension < "
        f"{DIMENSION_MIN_FLOOR:.1f}/5 -> AUTO-REJECT"
    )
    print(
        f"  2) HPS < {GOOD_CUT:.1f} -> AUTO-REJECT"
    )
    print(
        f"  3) HPS >= {HIGH_CUT:.1f}:"
    )
    print(
        "       all critical factors meet their own adequacy minima "
        "-> DIRECT AUTO-ACCEPT"
    )
    print(
        "       otherwise -> AUTOMATED REVIEW fallback"
    )
    print(
        f"  4) {GOOD_CUT:.1f} <= HPS < "
        f"{HIGH_CUT:.1f} -> AUTOMATED REVIEW"
    )
    print(
        "  5) AUTOMATED REVIEW:"
    )
    print(
        f"       Critical WAC_i >= Global Critical WAC "
        f"({global_critical_wac:.6f}) -> ACCEPT AFTER REVIEW"
    )
    print(
        "       otherwise -> AUTO-REJECT"
    )

    print(
        f"\nFrozen TADP-VR cohort: "
        f"{len(vr_clients)}/10 -> "
        f"{vr_clients}"
    )
    print(
        f"Frozen TADP-SDA cohort: "
        f"{len(sda_clients)}/10 -> "
        f"{sda_clients}"
    )

    print_banner("GX CORE NATIVE TRAIN-ONLY VALIDATOR BASELINE")
    gx_base_cols = [
        "client", "gx_native_suite_success", "ge_expectations_passed",
        "ge_expectations_total", "ge_pass_rate", "ge_native_validation_class",
        "ge_final_action",
    ]
    gx_category_cols = [
        c for c in ge.columns
        if c.startswith("ge_") and (c.endswith("_passed") or c.endswith("_total"))
        and c not in {"ge_expectations_passed", "ge_expectations_total"}
    ]
    print(ge[gx_base_cols + sorted(gx_category_cols)].to_string(index=False))
    print(f"\nGX operationally valid clients (zero critical failures): {len(ge_clients)}/{len(ge)} -> {ge_clients}")
    print(
        "GX is used only as a native rule-based data validator. PASS/FAIL is the "
        "suite-level GX result; no ranking, no forced-K selection, and no TADP signal is used."
    )





def build_hash_chained_governance_ledger(
    gov,
    ge,
    output_path,
):
    rows = []
    prev_hash = "GENESIS"
    seq = 0

    for _, r in (
        gov.sort_values(
            "client"
        ).iterrows()
    ):
        seq += 1

        payload = {
            "sequence":
                seq,
            "governance_system":
                "TADP",
            "client":
                str(r["client"]),
            "hps":
                float(r["hps"]),
            "global_domain_wac":
                float(r["global_domain_wac"]),
            "global_critical_wac":
                float(r["global_critical_wac"]),
            "critical_wac_i": (
                float(r["critical_wac_i"])
                if np.isfinite(r["critical_wac_i"])
                else None
            ),
            "critical_wac_margin": (
                float(r["critical_wac_margin"])
                if np.isfinite(r["critical_wac_margin"])
                else None
            ),
            "all_critical_meet_adequacy":
                bool(r["all_critical_meet_adequacy"]),
            "critical_below_adequacy":
                str(r["critical_below_adequacy"]),
            "dimension_floor_failures":
                str(r["dimension_floor_failures"]),
            "decision_path":
                str(r["decision_path"]),
            "initial_action":
                str(r["initial_action"]),
            "final_action":
                str(r["final_action"]),
            "status":
                str(r["status"]),
            "reason":
                str(r["reason"]),
            "previous_hash":
                prev_hash,
        }

        canonical = json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
        )

        entry_hash = hashlib.sha256(
            canonical.encode("utf-8")
        ).hexdigest()

        payload["entry_hash"] = entry_hash

        rows.append(payload)
        prev_hash = entry_hash

    for _, r in (
        ge.sort_values(
            "client"
        ).iterrows()
    ):
        seq += 1

        payload = {
            "sequence":
                seq,
            "governance_system":
                "Great Expectations GX Core native validator",
            "client":
                str(r["client"]),
            "hps":
                None,
            "global_domain_wac":
                None,
            "global_critical_wac":
                None,
            "critical_wac_i":
                None,
            "critical_wac_margin":
                None,
            "all_critical_meet_adequacy":
                None,
            "critical_below_adequacy":
                None,
            "dimension_floor_failures":
                None,
            "decision_path":
                "RULE_BASED_VALIDATION",
            "initial_action":
                "RULE_BASED_VALIDATION",
            "final_action":
                str(r["ge_final_action"]),
            "status":
                "GE_CONTROLLED_TRAIN_ONLY",
            "reason": (
                f"GX native suite {'PASSED' if bool(r['gx_native_suite_success']) else 'FAILED'}; "
                f"{int(r['ge_expectations_passed'])}/{int(r['ge_expectations_total'])} "
                "configured technical Expectations passed"
            ),
            "previous_hash":
                prev_hash,
        }

        canonical = json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
        )

        entry_hash = hashlib.sha256(
            canonical.encode("utf-8")
        ).hexdigest()

        payload["entry_hash"] = entry_hash

        rows.append(payload)
        prev_hash = entry_hash

    out = pd.DataFrame(rows)

    out.to_csv(
        output_path,
        index=False,
    )

    return out

def choose_experiment_root(experiment_name, use_drive=True):
    if use_drive:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            root = Path("/content/drive/MyDrive/TADP_CHECKPOINTS") / experiment_name
            root.mkdir(parents=True, exist_ok=True)
            print(f"Persistent checkpoint root: {root}")
            return root
        except Exception as exc:
            print(f"Google Drive checkpoint mount unavailable: {exc}")
    root = Path("/content") / experiment_name
    root.mkdir(parents=True, exist_ok=True)
    print(f"Local checkpoint root: {root}")
    return root


def atomic_write_json(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")
    tmp.replace(path)


def load_checkpoint_state(path):
    path = Path(path)
    if not path.exists():
        return {"completed": [], "last_completed": None, "updated_utc": None}
    return json.loads(path.read_text(encoding="utf-8"))


def mark_checkpoint_complete(state_path, key, extra=None):
    state = load_checkpoint_state(state_path)
    completed = list(state.get("completed", []))
    if key not in completed:
        completed.append(key)
    state["completed"] = completed
    state["last_completed"] = key
    state["updated_utc"] = datetime.now(timezone.utc).isoformat()
    if extra:
        state.update(extra)
    atomic_write_json(state, state_path)


def upsert_csv(row, path, key_cols):
    path = Path(path)
    new = pd.DataFrame([row])
    if path.exists():
        old = pd.read_csv(path)
        if not old.empty:
            mask = pd.Series(True, index=old.index)
            for c in key_cols:
                mask &= old[c].astype(str).eq(str(row[c]))
            old = old.loc[~mask].copy()
            new = pd.concat([old, new], ignore_index=True)
    new.to_csv(path, index=False)


def write_scenario_checkpoint(root, run_idx, scenario, result):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", scenario).strip("_")
    cdir = ensure_dir(root / "scenario_checkpoints" / f"run_{run_idx:02d}")
    payload = {
        "run": int(run_idx),
        "scenario": scenario,
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "metrics": result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
    }
    atomic_write_json(payload, cdir / f"{safe}.json")
    if "round_audit" in result:
        pd.DataFrame(result["round_audit"]).to_csv(
            cdir / f"{safe}_rounds.csv", index=False
        )


def ci95_mean(values):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan
    if len(x) == 1:
        return float(x[0]), float(x[0])
    mean = float(np.mean(x))
    sd = float(np.std(x, ddof=1))
    try:
        from scipy.stats import t
        crit = float(t.ppf(0.975, df=len(x)-1))
    except Exception:
        crit = 1.96
    half = crit * sd / math.sqrt(len(x))
    return mean-half, mean+half


def summarize_runs_with_ci(perf):
    metrics = [
        "accuracy", "precision_macro", "recall_macro", "f1_macro",
        "roc_auc_ovr_macro", "runtime_s", "energy_wh", "energy_kwh",
        "co2_kg", "communication_mb", "ram_peak_mb", "ram_delta_mb",
        "optimizer_steps", "participants", "energy_cost_usd",
        "communication_cost_usd", "total_estimated_cost_usd",
    ]
    rows = []
    for scenario, d in perf.groupby("scenario", sort=False):
        row = {"scenario": scenario, "n_runs": int(len(d))}
        for metric in metrics:
            if metric not in d.columns:
                continue
            vals = pd.to_numeric(d[metric], errors="coerce")
            vals = vals[np.isfinite(vals)]
            row[f"{metric}_mean"] = float(vals.mean()) if len(vals) else np.nan
            row[f"{metric}_sd"] = float(vals.std(ddof=1)) if len(vals)>1 else 0.0
            lo, hi = ci95_mean(vals)
            row[f"{metric}_ci95_low"] = lo
            row[f"{metric}_ci95_high"] = hi
        rows.append(row)
    return pd.DataFrame(rows)


def paired_vr_randomk_statistics(perf):
    vr = perf[perf["scenario"].eq("TADP-VR Federated")].copy()
    rk = perf[perf["scenario"].eq("Random-K")].copy()
    merged = vr.merge(rk, on=["run", "seed"], suffixes=("_vr", "_randomk"), validate="one_to_one")
    rows = []
    for metric in ["accuracy", "f1_macro", "roc_auc_ovr_macro"]:
        a = merged[f"{metric}_vr"].to_numpy(dtype=float)
        b = merged[f"{metric}_randomk"].to_numpy(dtype=float)
        diff = a-b
        mean = float(np.mean(diff))
        sd = float(np.std(diff, ddof=1)) if len(diff)>1 else 0.0
        lo, hi = ci95_mean(diff)
        t_stat=t_p=wil_stat=wil_p=np.nan
        try:
            from scipy.stats import ttest_rel, wilcoxon
            tr = ttest_rel(a,b,nan_policy="omit")
            t_stat, t_p = float(tr.statistic), float(tr.pvalue)
            if np.any(np.abs(diff)>0):
                wr = wilcoxon(a,b)
                wil_stat, wil_p = float(wr.statistic), float(wr.pvalue)
        except Exception:
            pass
        rows.append({
            "metric": metric,
            "n_pairs": len(diff),
            "mean_difference_vr_minus_randomk": mean,
            "sd_difference": sd,
            "ci95_low": lo,
            "ci95_high": hi,
            "cohens_dz": float(mean/sd) if sd>0 else np.nan,
            "paired_t_stat": t_stat,
            "paired_t_p": t_p,
            "wilcoxon_stat": wil_stat,
            "wilcoxon_p": wil_p,
            "vr_wins": int(np.sum(diff>0)),
            "ties": int(np.sum(np.isclose(diff,0))),
            "vr_losses": int(np.sum(diff<0)),
        })
    return pd.DataFrame(rows)


def scenario_method_table():
    return pd.DataFrame([
        ["Naïve Centralized","centralized","baseline","all clients","5-seed headline"],
        ["Great Expectations Centralized","centralized","GX native validator","GX clients with zero critical failures","equivalence audit only"],
        ["TADP-AA Centralized","centralized","TADP-AA","all clients","equivalence audit only"],
        ["TADP-VR Centralized","centralized","TADP-VR","frozen TADP-VR cohort","5-seed headline"],
        ["TADP-SDA Centralized","centralized","TADP-SDA","frozen best eligible client","one-seed boundary"],
        ["Vanilla FedAvg","federated","FedAvg","all clients","5-seed headline"],
        ["FedProx","federated","FedProx","all clients","5-seed headline"],
        ["Random-K","federated","matched random control","same K/rounds/steps as TADP-VR","5-seed headline"],
        ["Great Expectations Federated","federated","GX native validator","GX clients with zero critical failures","equivalence audit only"],
        ["DQ-only Federated","federated","DQ-only","top-K by machine-measured DQ; same K/rounds/steps as TADP-VR","5-seed headline"],
        ["Power-of-Choice","federated","Power-of-Choice","dynamic loss-based selection; same K/rounds/steps as TADP-VR","5-seed headline"],
        ["TADP-AA Federated","federated","TADP-AA","all clients","equivalence audit only"],
        ["TADP-VR Federated","federated","TADP-VR","frozen TADP-VR cohort","5-seed headline"],
        ["TADP-SDA Federated","federated","TADP-SDA","frozen best eligible client","one-seed boundary"],
    ], columns=["scenario","paradigm","method","participation","execution_role"])



def governance_only_monte_carlo(
    client_ids,
    dq_scores,
    base_seed,
    n_realizations=1000,
    domain="healthcare",
):
    domain = str(domain).lower()
    rows=[]
    for j in range(int(n_realizations)):
        seed=int(base_seed+j)
        evidence,_=generate_controlled_documentary_evidence(
            client_ids,
            seed,
            DOMAIN_FACTOR_MINIMA[domain],
            domain=domain,
        )
        gov=build_tadp_governance(
            client_ids,
            evidence,
            dq_scores,
            run=0,
            evidence_seed=seed,
            domain=domain,
        )
        accepted=accepted_tadp_vr(gov)
        rows.append({
            "realization":j+1,
            "evidence_seed":seed,
            "accepted_count":len(accepted),
            "accepted_clients":";".join(accepted),
            "mean_hps":float(gov["hps"].mean()),
            "mean_critical_wac_i":float(gov["critical_wac_i"].mean()),
            "dimension_floor_rejects":int(gov["status"].eq("AUTO_REJECTED_DIMENSION_FLOOR").sum()),
            "low_hps_rejects":int(gov["status"].eq("AUTO_REJECTED_LOW_HPS").sum()),
            "review_accepts":int(
                gov["status"].eq("ACCEPTED_AFTER_AUTOMATED_REVIEW").sum()
            ),
            "direct_auto_accepts":int(
                gov["status"].eq("DIRECT_AUTO_ACCEPTED").sum()
            ),
        })
    return pd.DataFrame(rows)


# ======================================================================================
# EXPERIMENT B3 — PREDICTIVE UTILITY UNDER CLIENT SCALING
# ======================================================================================
#
# PURPOSE
# -------
# Complement Experiment B1/B2 (governance-only scalability) with downstream
# predictive-utility scalability.
#
# Tested client counts:
#     K = {20, 50, 100}
#
# K=10 is intentionally not rerun here because it is already evaluated in the
# final Experiment-A protocol. For cross-K manuscript plots, use only the
# corresponding 4-round Experiment-A K=10 rows for seeds {42,142,242}.
#
# Three scenarios are intentionally retained:
#
#   1) Vanilla FedAvg
#      - all submitted clients participate
#      - full-participation predictive-utility reference
#
#   2) Random-K
#      - same number of participating clients as TADP-VR
#      - exact same total optimizer-step budget per round as TADP-VR
#      - frozen random cohort within each K, across all rounds and seeds
#
#   3) TADP-VR Federated
#      - clients admitted by the frozen TADP governance policy
#      - natural one-local-epoch step budget
#
# FAIRNESS / INTERPRETATION
# -------------------------
# TADP-VR vs Random-K is the matched-compute selection comparison:
#   same K_selected, same FL rounds, same total optimizer steps per round,
#   same model architecture, same initial weights within seed, same batch size,
#   same optimizer, same global TEST set.
#
# Vanilla FedAvg is NOT step-matched to TADP-VR because it is the intended
# full-participation utility reference. Its larger compute/communication budget
# is reported rather than hidden.
#
# The experiment tests whether TADP-VR retains useful predictive performance
# as the submitted contributor population grows, while using fewer clients
# than full FedAvg and while being compared fairly against a matched random
# subset.
#
# The held-out TEST set is used only after training for final evaluation.
# It is never used for governance, preprocessing, client selection, step-budget
# construction, or checkpoint decisions.
# ======================================================================================

import platform
import shutil
from datetime import datetime, timezone

EXPERIMENT_VERSION = "TADP-B3-v16.9-PREDICTIVE-SCALABILITY-K20-50-100-3SEED-4ROUND"

CLIENT_COUNTS = [20, 50, 100]
TRAINING_RUN_SEEDS = [42, 142, 242]

NUM_ROUNDS_FL = 4
LOCAL_EPOCHS = 1
BATCH_SIZE = 64
LEARNING_RATE = 1e-3

DIRICHLET_ALPHA = 1.0
MIN_CLIENT_RECORDS = 30

GLOBAL_SPLIT_SEED = 7001
PARTITION_BASE_SEED = 9101
EVIDENCE_BASE_SEED = 12042
RANDOMK_BASE_SEED = 22042

DOMAIN = "healthcare"
USE_GOOGLE_DRIVE_CHECKPOINTS = True

SCENARIOS = [
    "Vanilla FedAvg",
    "Random-K",
    "TADP-VR Federated",
]

EXPERIMENT_ROOT = choose_experiment_root(
    "TADP_EXPERIMENT_B3_" + EXPERIMENT_VERSION,
    use_drive=USE_GOOGLE_DRIVE_CHECKPOINTS,
)
CHECKPOINT_STATE = EXPERIMENT_ROOT / "checkpoint_state.json"
PERF_CHECKPOINT = EXPERIMENT_ROOT / "performance_metrics_checkpoint.csv"


# ======================================================================================
# SCALABLE CLIENT PARTITIONING / CONTROLLED EVIDENCE
# ======================================================================================

def scalable_client_ids(n_clients: int) -> List[str]:
    return [f"C{i:03d}" for i in range(1, int(n_clients) + 1)]


def dirichlet_partition_dataframe_scalable(
    train_df: pd.DataFrame,
    n_clients: int,
    alpha: float,
    seed: int,
    min_client_records: int = 30,
    max_attempts: int = 500,
) -> Dict[str, pd.DataFrame]:
    """
    Label-wise Dirichlet partition that supports K > 26 while preserving every
    global TRAIN row exactly once.
    """
    y = train_df["_target"].to_numpy()
    client_ids = scalable_client_ids(n_clients)

    for attempt in range(int(max_attempts)):
        rng = np.random.default_rng(int(seed + attempt))
        buckets = [[] for _ in range(int(n_clients))]

        for cls in sorted(np.unique(y)):
            idx = np.where(y == cls)[0]
            rng.shuffle(idx)
            props = rng.dirichlet(np.full(int(n_clients), float(alpha)))
            cuts = (np.cumsum(props)[:-1] * len(idx)).astype(int)
            splits = np.split(idx, cuts)
            for j, split in enumerate(splits):
                buckets[j].extend(split.tolist())

        sizes = [len(b) for b in buckets]
        if min(sizes) >= int(min_client_records):
            return {
                cid: train_df.iloc[np.asarray(idxs, dtype=int)].copy()
                for cid, idxs in zip(client_ids, buckets)
            }

    raise RuntimeError(
        f"Could not obtain valid K={n_clients} partition after {max_attempts} attempts "
        f"with min_client_records={min_client_records}."
    )


def partition_integrity_audit(
    train_df: pd.DataFrame,
    clients: Dict[str, pd.DataFrame],
) -> Dict[str, Any]:
    global_ids = set(train_df["_row_id"].astype(int).tolist())
    all_ids = []
    for df in clients.values():
        all_ids.extend(df["_row_id"].astype(int).tolist())
    client_set = set(all_ids)

    result = {
        "global_train_rows": int(len(train_df)),
        "client_rows_total": int(len(all_ids)),
        "unique_client_rows": int(len(client_set)),
        "rows_missing_from_clients": int(len(global_ids - client_set)),
        "rows_not_in_global_train": int(len(client_set - global_ids)),
        "duplicate_rows_across_clients": int(len(all_ids) - len(client_set)),
    }
    result["pass"] = bool(
        len(all_ids) == len(train_df)
        and client_set == global_ids
        and len(all_ids) == len(client_set)
    )
    return result


def generate_scalable_controlled_evidence(
    client_ids: List[str],
    evidence_seed: int,
    domain: str = "healthcare",
):
    """
    Preserve the audited Experiment-A governance archetype composition.

    Every block of ten contributors receives the same 4/2/2/1/1 archetype mix:
      - 4 DIRECT_STRONG
      - 2 REVIEW_RECOVERABLE
      - 2 REVIEW_LIMITED
      - 1 LOW_HPS_WEAK
      - 1 DIMENSION_FLOOR_WEAK

    Assignment to identities is seeded before training and never uses TEST or
    downstream predictive performance.
    """
    if len(client_ids) % 10 != 0:
        raise ValueError(
            "B3 uses K values divisible by 10 so the controlled evidence-profile "
            "composition remains exactly proportional."
        )

    combined = {}
    frames = []
    minima = DOMAIN_FACTOR_MINIMA[str(domain).lower()]

    for block_idx, start in enumerate(range(0, len(client_ids), 10), start=1):
        block = client_ids[start:start + 10]
        block_seed = int(evidence_seed + 1009 * (block_idx - 1))
        evidence, df = generate_controlled_documentary_evidence(
            block,
            block_seed,
            minima,
            domain=domain,
        )
        combined.update(evidence)
        df = df.copy()
        df["scalability_block"] = int(block_idx)
        df["block_evidence_seed"] = int(block_seed)
        df["bundle_id"] = df["bundle_id"].map(
            lambda x: f"B{block_idx:02d}-{x}"
        )
        frames.append(df)

    return combined, pd.concat(frames, ignore_index=True)


def validate_expected_profile_mix(evidence_df: pd.DataFrame, k: int):
    one = evidence_df[["client", "scenario_role"]].drop_duplicates("client")
    got = one["scenario_role"].value_counts().to_dict()
    scale = int(k // 10)
    expected = {
        "DIRECT_STRONG": 4 * scale,
        "REVIEW_RECOVERABLE": 2 * scale,
        "REVIEW_LIMITED": 2 * scale,
        "LOW_HPS_WEAK": 1 * scale,
        "DIMENSION_FLOOR_WEAK": 1 * scale,
    }
    if got != expected:
        raise RuntimeError(
            f"Controlled evidence composition mismatch for K={k}: "
            f"got={got}, expected={expected}"
        )
    return expected


def prepare_k_data(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    k: int,
    partition_seed: int,
):
    clients = dirichlet_partition_dataframe_scalable(
        train_df=train_df,
        n_clients=k,
        alpha=DIRICHLET_ALPHA,
        seed=partition_seed,
        min_client_records=MIN_CLIENT_RECORDS,
    )

    integrity = partition_integrity_audit(train_df, clients)
    if not integrity["pass"]:
        raise RuntimeError(f"Partition integrity failed at K={k}: {integrity}")

    pre = fit_federated_train_only_preprocessor(clients)
    reference = build_tabular_reference(clients, pre)

    dq_scores = {}
    dq_rows = []
    client_arrays = {}

    for cid, df in clients.items():
        scores, raw_metrics = tabular_dq_scores(df, pre, reference)
        dq_scores[cid] = scores
        dq_rows.append({"client": cid, **scores, **raw_metrics})

        Xc = pre.transform(df)
        yc = df["_target"].to_numpy(dtype=np.int32)
        client_arrays[cid] = (Xc, yc)

    X_test = pre.transform(test_df)
    y_test = test_df["_target"].to_numpy(dtype=np.int32)

    y_train_all = np.concatenate(
        [client_arrays[cid][1] for cid in sorted(client_arrays)]
    )
    class_weights = class_weight_dict(y_train_all)

    return {
        "clients_raw": clients,
        "client_ids": list(clients.keys()),
        "client_arrays": client_arrays,
        "preprocessor": pre,
        "dq_reference": reference,
        "dq_scores": dq_scores,
        "dq_audit": pd.DataFrame(dq_rows),
        "X_test": X_test,
        "y_test": y_test,
        "class_weights": class_weights,
        "input_dim": int(X_test.shape[1]),
        "integrity": integrity,
    }


# ======================================================================================
# SUMMARIES / CHECKPOINTS
# ======================================================================================

def summarize_b3(perf: pd.DataFrame) -> pd.DataFrame:
    metrics = [
        "accuracy",
        "precision_macro",
        "recall_macro",
        "f1_macro",
        "roc_auc_ovr_macro",
        "runtime_s",
        "communication_mb",
        "optimizer_steps",
        "participants",
        "energy_wh",
        "ram_peak_mb",
        "ram_delta_mb",
    ]
    rows = []
    for (k, scenario), d in perf.groupby(
        ["k_submissions", "scenario"], sort=True
    ):
        row = {
            "k_submissions": int(k),
            "scenario": str(scenario),
            "n_runs": int(len(d)),
        }
        for metric in metrics:
            if metric not in d.columns:
                continue
            vals = pd.to_numeric(d[metric], errors="coerce").to_numpy(dtype=float)
            vals = vals[np.isfinite(vals)]
            row[f"{metric}_mean"] = (
                float(np.mean(vals)) if len(vals) else np.nan
            )
            row[f"{metric}_sd"] = (
                float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            )
            lo, hi = ci95_mean(vals)
            row[f"{metric}_ci95_low"] = lo
            row[f"{metric}_ci95_high"] = hi
        rows.append(row)
    return pd.DataFrame(rows)


def paired_b3(perf: pd.DataFrame) -> pd.DataFrame:
    """
    Paired seed-wise differences.

    TADP-VR vs Random-K:
        matched K and matched optimizer-step budget.

    TADP-VR vs Vanilla FedAvg:
        utility comparison against full participation; NOT compute-matched.
    """
    rows = []
    comparisons = [
        ("Random-K", "MATCHED_SELECTION_CONTROL"),
        ("Vanilla FedAvg", "FULL_PARTICIPATION_REFERENCE"),
    ]

    for k in sorted(perf["k_submissions"].unique()):
        tadp = perf[
            (perf["k_submissions"].eq(k))
            & (perf["scenario"].eq("TADP-VR Federated"))
        ].copy()

        for baseline, role in comparisons:
            base = perf[
                (perf["k_submissions"].eq(k))
                & (perf["scenario"].eq(baseline))
            ].copy()

            merged = tadp.merge(
                base,
                on=["k_submissions", "seed"],
                suffixes=("_tadp", "_baseline"),
                validate="one_to_one",
            )

            for metric in [
                "accuracy",
                "precision_macro",
                "recall_macro",
                "f1_macro",
                "roc_auc_ovr_macro",
            ]:
                a = merged[f"{metric}_tadp"].to_numpy(dtype=float)
                b = merged[f"{metric}_baseline"].to_numpy(dtype=float)
                diff = a - b
                lo, hi = ci95_mean(diff)

                rows.append({
                    "k_submissions": int(k),
                    "baseline": baseline,
                    "baseline_role": role,
                    "metric": metric,
                    "n_pairs": int(len(diff)),
                    "mean_difference_tadp_minus_baseline":
                        float(np.mean(diff)),
                    "sd_difference":
                        float(np.std(diff, ddof=1)) if len(diff) > 1 else 0.0,
                    "ci95_low": lo,
                    "ci95_high": hi,
                    "tadp_wins": int(np.sum(diff > 0)),
                    "ties": int(np.sum(np.isclose(diff, 0.0))),
                    "tadp_losses": int(np.sum(diff < 0)),
                })

    return pd.DataFrame(rows)


def utility_retention_table(summary: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for k in sorted(summary["k_submissions"].unique()):
        d = summary[summary["k_submissions"].eq(k)].set_index("scenario")
        if "TADP-VR Federated" not in d.index or "Vanilla FedAvg" not in d.index:
            continue

        row = {"k_submissions": int(k)}
        for metric in ["accuracy", "f1_macro", "roc_auc_ovr_macro"]:
            tadp = float(d.loc["TADP-VR Federated", f"{metric}_mean"])
            full = float(d.loc["Vanilla FedAvg", f"{metric}_mean"])
            rand = float(d.loc["Random-K", f"{metric}_mean"])

            row[f"tadp_{metric}_mean"] = tadp
            row[f"fedavg_{metric}_mean"] = full
            row[f"randomk_{metric}_mean"] = rand
            row[f"tadp_minus_fedavg_{metric}"] = tadp - full
            row[f"tadp_minus_randomk_{metric}"] = tadp - rand
            row[f"tadp_retention_pct_of_fedavg_{metric}"] = (
                100.0 * tadp / full if np.isfinite(full) and full != 0 else np.nan
            )

        row["tadp_participants_mean"] = float(
            d.loc["TADP-VR Federated", "participants_mean"]
        )
        row["fedavg_participants_mean"] = float(
            d.loc["Vanilla FedAvg", "participants_mean"]
        )
        row["randomk_participants_mean"] = float(
            d.loc["Random-K", "participants_mean"]
        )
        row["tadp_communication_mb_mean"] = float(
            d.loc["TADP-VR Federated", "communication_mb_mean"]
        )
        row["fedavg_communication_mb_mean"] = float(
            d.loc["Vanilla FedAvg", "communication_mb_mean"]
        )
        row["communication_reduction_pct_vs_fedavg"] = (
            100.0
            * (
                float(d.loc["Vanilla FedAvg", "communication_mb_mean"])
                - float(d.loc["TADP-VR Federated", "communication_mb_mean"])
            )
            / max(
                float(d.loc["Vanilla FedAvg", "communication_mb_mean"]),
                1e-12,
            )
        )
        rows.append(row)

    return pd.DataFrame(rows)


def write_b3_scenario_checkpoint(
    root: Path,
    k: int,
    run_idx: int,
    scenario: str,
    result: Dict[str, Any],
):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", scenario).strip("_")
    cdir = ensure_dir(
        root
        / "scenario_checkpoints"
        / f"K_{int(k):03d}"
        / f"run_{int(run_idx):02d}"
    )
    payload = {
        "k_submissions": int(k),
        "run": int(run_idx),
        "scenario": str(scenario),
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "metrics": result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
    }
    atomic_write_json(payload, cdir / f"{safe}.json")
    if "round_audit" in result:
        pd.DataFrame(result["round_audit"]).to_csv(
            cdir / f"{safe}_rounds.csv",
            index=False,
        )


def save_environment_metadata():
    meta = {
        "experiment_version": EXPERIMENT_VERSION,
        "python": sys.version,
        "platform": platform.platform(),
        "tensorflow": tf.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "client_counts": CLIENT_COUNTS,
        "training_seeds": TRAINING_RUN_SEEDS,
        "fl_rounds": NUM_ROUNDS_FL,
        "local_epochs": LOCAL_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "dirichlet_alpha": DIRICHLET_ALPHA,
        "global_split_seed": GLOBAL_SPLIT_SEED,
        "partition_seed_formula": "PARTITION_BASE_SEED + 101*K",
        "evidence_seed_formula": "EVIDENCE_BASE_SEED + 103*K",
        "randomk_seed_formula": "RANDOMK_BASE_SEED + 107*K",
        "test_semantics":
            "final evaluation only; never used for governance, selection, "
            "preprocessing, step budgets, or checkpoint decisions",
        "scenario_roles": {
            "Vanilla FedAvg":
                "full-participation predictive-utility reference; not compute matched",
            "Random-K":
                "same selected-client count and exact total optimizer-step budget per round as TADP-VR",
            "TADP-VR Federated":
                "governance-selected cohort",
        },
        "k10_note":
            "K=10 intentionally omitted here; use final Experiment-A 4-round rows "
            "for seeds 42,142,242 when constructing the manuscript K=10/20/50/100 plot.",
    }
    atomic_write_json(
        meta,
        EXPERIMENT_ROOT / "b3_experiment_design.json",
    )


# ======================================================================================
# RQ6 — THRESHOLD + HPS-WEIGHT SENSITIVITY (K=10, 5 seeds, 4 rounds)
# ======================================================================================
#
# Research question:
#   How sensitive are TADP-VR admission decisions and downstream utility to
#   controlled changes in HPS decision thresholds and HPS dimension weights?
#
# Frozen reference setting:
#   K=10, evidence seed=1042, partition seed=7101, split seed=7001,
#   training seeds=[42,142,242,342,442], 4 FL rounds, 1 local epoch, batch=64.
#
# Threshold perturbation families (each tested separately):
#   1) LOWER_ONLY  : lower threshold x (1 ± 10/20/30%), upper fixed at 3.5
#   2) UPPER_ONLY  : upper threshold x (1 ± 10/20/30%), lower fixed at 3.0
#   3) BOTH        : both thresholds x the same factor
# Invalid policies with lower >= upper are recorded and not executed.
#
# Weight sensitivity:
#   Each HPS dimension is perturbed ONE AT A TIME by ±10%, ±20%, ±30% around its
#   reference weight. The other five weights are rescaled proportionally so that
#   the full vector still sums to 1. This is necessary because multiplying every
#   weight by the same factor and renormalizing would reproduce the original vector
#   and therefore would not be a meaningful sensitivity test.
#
# For every valid configuration the code reports:
#   - admitted client count and admitted percentage;
#   - percentage-point and relative gain/loss versus the frozen reference;
#   - client decision flips and admitted-set Jaccard similarity;
#   - predictive utility (macro ROC-AUC, macro-F1, accuracy) over 5 training seeds;
#   - paired seed-level differences versus the frozen reference.
#
# This is sensitivity analysis, NOT policy tuning. No configuration is selected
# using held-out TEST performance.
# ======================================================================================

import matplotlib.pyplot as plt

EXPERIMENT_VERSION = "TADP-RQ6-v17.1-K10-THRESHOLD-WEIGHT-SENSITIVITY-5SEED-4ROUND"

K_SUBMISSIONS = 10
TRAINING_RUN_SEEDS = [42, 142, 242, 342, 442]
NUM_ROUNDS_FL = 4
LOCAL_EPOCHS = 1
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
DIRICHLET_ALPHA = 1.0

GLOBAL_SPLIT_SEED = 7001
CLIENT_PARTITION_SEED = 7101
FROZEN_EVIDENCE_ASSIGNMENT_SEED = 1042
DOMAIN = "healthcare"

REFERENCE_LOWER_CUT = 3.0
REFERENCE_UPPER_CUT = 3.5
REFERENCE_DIMENSION_FLOOR = 2.5
PERTURBATION_PCTS = [-30, -20, -10, 10, 20, 30]
DIMS = [f"dim{i}" for i in range(1, 7)]

USE_GOOGLE_DRIVE_CHECKPOINTS = True
EXPERIMENT_ROOT = choose_experiment_root(
    "TADP_EXPERIMENT_RQ6_" + EXPERIMENT_VERSION,
    use_drive=USE_GOOGLE_DRIVE_CHECKPOINTS,
)
CHECKPOINT_STATE = EXPERIMENT_ROOT / "checkpoint_state.json"
UNIQUE_COHORT_PERF = EXPERIMENT_ROOT / "unique_cohort_performance_checkpoint.csv"


def _bool_series_rq6(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s
    return s.astype(str).str.lower().isin(["true", "1", "yes"])


def _jaccard_rq6(a, b) -> float:
    a = set(map(str, a)); b = set(map(str, b)); u = a | b
    return 1.0 if not u else float(len(a & b) / len(u))


def build_policy_tables_rq6(full_factor_df: pd.DataFrame):
    f = full_factor_df.copy()
    f["client"] = f["client"].astype(str)
    f["dimension"] = f["dimension"].astype(str)
    f["rubric_score_0_5"] = pd.to_numeric(f["rubric_score_0_5"], errors="raise")
    dim_scores = (
        f.groupby(["client", "dimension"])["rubric_score_0_5"]
        .mean().unstack("dimension").sort_index().reindex(columns=DIMS)
    )
    crit = f.loc[_bool_series_rq6(f["is_critical_factor"])].copy()
    crit["critical_adequacy_min_rank"] = pd.to_numeric(
        crit["critical_adequacy_min_rank"], errors="coerce"
    )
    crit = crit.loc[crit["critical_adequacy_min_rank"].notna()].copy()
    return f, dim_scores, crit


def perturb_one_weight(target_dim: str, pct: int) -> dict:
    """
    Change one HPS dimension weight by pct% and proportionally rescale the other
    five weights so the final vector sums exactly to 1 while preserving their
    relative proportions.
    """
    if target_dim not in DIMS:
        raise ValueError(target_dim)
    base = {d: float(WEIGHTS_PSCORE_DEFAULT[d]) for d in DIMS}
    factor = 1.0 + float(pct) / 100.0
    target_new = base[target_dim] * factor
    if not (0.0 < target_new < 1.0):
        raise ValueError(f"Invalid perturbed weight for {target_dim}: {target_new}")
    remaining_old = 1.0 - base[target_dim]
    remaining_new = 1.0 - target_new
    scale = remaining_new / remaining_old
    out = {}
    for d in DIMS:
        out[d] = target_new if d == target_dim else base[d] * scale
    # Numerical cleanup.
    total = float(sum(out.values()))
    out = {d: float(w / total) for d, w in out.items()}
    if abs(sum(out.values()) - 1.0) > 1e-12:
        raise RuntimeError("Weight normalization failed")
    if any(w <= 0.0 for w in out.values()):
        raise RuntimeError("Weight perturbation produced non-positive weight")
    return out


def evaluate_rq6_policy(
    dim_scores: pd.DataFrame,
    crit: pd.DataFrame,
    lower_cut: float,
    upper_cut: float,
    weights: dict,
) -> pd.DataFrame:
    lower_cut = float(lower_cut); upper_cut = float(upper_cut)
    if not (0.0 <= lower_cut <= 5.0 and 0.0 <= upper_cut <= 5.0 and lower_cut < upper_cut):
        raise ValueError(f"Invalid thresholds: lower={lower_cut}, upper={upper_cut}")
    if set(weights) != set(DIMS):
        raise ValueError("Weight vector must contain dim1..dim6")
    if abs(sum(float(weights[d]) for d in DIMS) - 1.0) > 1e-10:
        raise ValueError("Weight vector must sum to 1")

    active_crit = crit.copy()
    if active_crit.empty:
        global_critical_wac = 0.0
    else:
        global_critical_wac = float(np.mean(
            active_crit["critical_adequacy_min_rank"].astype(float).to_numpy()
            / MAX_FACTOR_SCORE
        ))

    rows = []
    for client in dim_scores.index.astype(str):
        ds = dim_scores.loc[client]
        hps = float(sum(float(ds[d]) * float(weights[d]) for d in DIMS))
        floor_failures = [
            d for d in DIMS
            if (not np.isfinite(float(ds[d]))) or float(ds[d]) < REFERENCE_DIMENSION_FLOOR
        ]

        cdf = active_crit.loc[active_crit["client"].astype(str).eq(client)].copy()
        if cdf.empty:
            critical_wac_i = 1.0
            all_critical_meet = True
            below = []
        else:
            scores = cdf["rubric_score_0_5"].astype(float).to_numpy()
            minima = cdf["critical_adequacy_min_rank"].astype(float).to_numpy()
            critical_wac_i = float(np.mean(scores / MAX_FACTOR_SCORE))
            all_critical_meet = bool(np.all(scores >= minima))
            below = [
                f"{r.dimension}.{r.factor}:{float(r.rubric_score_0_5):.1f}"
                f"<{float(r.critical_adequacy_min_rank):.1f}"
                for r in cdf.itertuples()
                if float(r.rubric_score_0_5) < float(r.critical_adequacy_min_rank)
            ]

        if floor_failures:
            action = "REJECT"; path = "DIMENSION_FLOOR"
        elif hps < lower_cut:
            action = "REJECT"; path = "LOW_HPS"
        elif hps >= upper_cut and all_critical_meet:
            action = "ACCEPT"; path = "DIRECT_AUTO_ACCEPT"
        elif critical_wac_i >= global_critical_wac:
            action = "ACCEPT"; path = "ACCEPTED_AFTER_AUTOMATED_REVIEW"
        else:
            action = "REJECT"; path = "AUTO_REJECTED_REVIEW_CRITICAL_WAC"

        rows.append({
            "client": client, "hps": hps,
            "final_action": action, "decision_path": path,
            "dimension_floor_failures": ";".join(floor_failures),
            "critical_wac_i": critical_wac_i,
            "global_critical_wac": global_critical_wac,
            "all_critical_meet_adequacy": bool(all_critical_meet),
            "critical_below_adequacy": ";".join(below),
            "lower_cut": lower_cut, "upper_cut": upper_cut,
        })
    return pd.DataFrame(rows)


def build_rq6_configurations() -> pd.DataFrame:
    rows = [{
        "configuration": "REFERENCE",
        "family": "REFERENCE",
        "perturbation_pct": 0,
        "target_dimension": "",
        "lower_cut": REFERENCE_LOWER_CUT,
        "upper_cut": REFERENCE_UPPER_CUT,
        "valid_policy": True,
        "invalid_reason": "",
        "weights_json": json.dumps(WEIGHTS_PSCORE_DEFAULT, sort_keys=True),
    }]

    # Threshold families.
    for family in ["LOWER_ONLY", "UPPER_ONLY", "BOTH"]:
        for pct in PERTURBATION_PCTS:
            factor = 1.0 + float(pct)/100.0
            lower = REFERENCE_LOWER_CUT
            upper = REFERENCE_UPPER_CUT
            if family == "LOWER_ONLY":
                lower = REFERENCE_LOWER_CUT * factor
            elif family == "UPPER_ONLY":
                upper = REFERENCE_UPPER_CUT * factor
            else:
                lower = REFERENCE_LOWER_CUT * factor
                upper = REFERENCE_UPPER_CUT * factor
            valid = bool(0.0 <= lower <= 5.0 and 0.0 <= upper <= 5.0 and lower < upper)
            reason = "" if valid else (
                "Invalid threshold ordering/range: lower must remain strictly below upper within [0,5]."
            )
            rows.append({
                "configuration": f"{family}_{pct:+d}pct",
                "family": family,
                "perturbation_pct": int(pct),
                "target_dimension": "",
                "lower_cut": float(lower), "upper_cut": float(upper),
                "valid_policy": valid, "invalid_reason": reason,
                "weights_json": json.dumps(WEIGHTS_PSCORE_DEFAULT, sort_keys=True),
            })

    # HPS weight sensitivity: one dimension at a time; thresholds stay frozen.
    for dim in DIMS:
        for pct in PERTURBATION_PCTS:
            weights = perturb_one_weight(dim, pct)
            rows.append({
                "configuration": f"WEIGHT_{dim}_{pct:+d}pct",
                "family": "WEIGHT_ONE_DIM",
                "perturbation_pct": int(pct),
                "target_dimension": dim,
                "lower_cut": REFERENCE_LOWER_CUT,
                "upper_cut": REFERENCE_UPPER_CUT,
                "valid_policy": True,
                "invalid_reason": "",
                "weights_json": json.dumps(weights, sort_keys=True),
            })
    return pd.DataFrame(rows)


def summarize_rq6_governance(configs, dim_scores, crit):
    ref_weights = {d: float(WEIGHTS_PSCORE_DEFAULT[d]) for d in DIMS}
    ref = evaluate_rq6_policy(
        dim_scores, crit, REFERENCE_LOWER_CUT, REFERENCE_UPPER_CUT, ref_weights
    ).sort_values("client").reset_index(drop=True)
    ref_accept = ref.loc[ref["final_action"].eq("ACCEPT"), "client"].astype(str).tolist()
    ref_rate = len(ref_accept) / len(ref)

    rows = []; gov_by_config = {}
    for cfg in configs.itertuples(index=False):
        base = {
            "configuration": cfg.configuration,
            "family": cfg.family,
            "perturbation_pct": int(cfg.perturbation_pct),
            "target_dimension": str(cfg.target_dimension),
            "lower_cut": float(cfg.lower_cut), "upper_cut": float(cfg.upper_cut),
            "valid_policy": bool(cfg.valid_policy), "invalid_reason": str(cfg.invalid_reason),
            "weights_json": str(cfg.weights_json),
        }
        if not cfg.valid_policy:
            rows.append({
                **base,
                "accepted_count": np.nan, "accepted_rate": np.nan, "accepted_rate_pct": np.nan,
                "accepted_count_change_vs_reference": np.nan,
                "accepted_rate_change_pp_vs_reference": np.nan,
                "accepted_rate_relative_change_pct_vs_reference": np.nan,
                "decision_flip_count": np.nan, "decision_flip_rate": np.nan,
                "accepted_set_jaccard_vs_reference": np.nan,
                "mean_abs_hps_change": np.nan, "admitted_clients": "", "cohort_key": "",
            })
            continue

        weights = {k: float(v) for k, v in json.loads(cfg.weights_json).items()}
        cur = evaluate_rq6_policy(dim_scores, crit, cfg.lower_cut, cfg.upper_cut, weights)
        cur = cur.sort_values("client").reset_index(drop=True)
        gov_by_config[cfg.configuration] = cur
        admitted = cur.loc[cur["final_action"].eq("ACCEPT"), "client"].astype(str).tolist()
        rate = len(admitted)/len(cur)
        comp = ref[["client","final_action","hps"]].merge(
            cur[["client","final_action","hps"]], on="client",
            suffixes=("_reference","_current"), validate="one_to_one"
        )
        flips = comp["final_action_reference"].ne(comp["final_action_current"])
        rows.append({
            **base,
            "accepted_count": int(len(admitted)),
            "accepted_rate": float(rate), "accepted_rate_pct": float(100.0*rate),
            "accepted_count_change_vs_reference": int(len(admitted)-len(ref_accept)),
            "accepted_rate_change_pp_vs_reference": float(100.0*(rate-ref_rate)),
            "accepted_rate_relative_change_pct_vs_reference": (
                float(100.0*(rate-ref_rate)/ref_rate) if ref_rate>0 else np.nan
            ),
            "decision_flip_count": int(flips.sum()), "decision_flip_rate": float(flips.mean()),
            "accepted_set_jaccard_vs_reference": _jaccard_rq6(ref_accept, admitted),
            "mean_abs_hps_change": float(np.mean(np.abs(
                comp["hps_current"].to_numpy(float)-comp["hps_reference"].to_numpy(float)
            ))),
            "admitted_clients": ";".join(admitted), "cohort_key": ";".join(sorted(admitted)),
        })
    return pd.DataFrame(rows), gov_by_config, ref


def assign_rq6_cohort_ids(gov_summary: pd.DataFrame):
    valid = gov_summary[gov_summary["valid_policy"].eq(True) & gov_summary["cohort_key"].astype(str).ne("")]
    keys = sorted(valid["cohort_key"].unique().tolist())
    mapping = {key: f"COHORT_{i+1:02d}" for i, key in enumerate(keys)}
    out = gov_summary.copy(); out["cohort_id"] = out["cohort_key"].map(mapping).fillna("")
    return out, mapping


def train_rq6_unique_cohorts(cohort_mapping, gov_summary, data):
    cohort_clients = {
        cohort_id: [x for x in str(key).split(";") if x]
        for key, cohort_id in cohort_mapping.items()
    }
    train_monitor = build_train_monitor_subset(
        data["client_arrays"], max_samples=ROUND_PROGRESS_MONITOR_MAX_SAMPLES, seed=99137
    )
    build_model_fn = lambda: build_diabetes_model(data["meta"]["input_dim"], lr=LEARNING_RATE)
    completed = set(load_checkpoint_state(CHECKPOINT_STATE).get("completed", []))
    total_jobs = len(TRAINING_RUN_SEEDS)*len(cohort_clients); job_idx = 0

    for training_seed in TRAINING_RUN_SEEDS:
        seed_everything(training_seed)
        base_model = build_model_fn()
        initial_weights = [np.array(w, copy=True) for w in base_model.get_weights()]
        initial_hash = sha256_weights(initial_weights)
        del base_model; tf.keras.backend.clear_session(); gc.collect()

        for cohort_id, selected in cohort_clients.items():
            job_idx += 1; key = f"seed{training_seed}|{cohort_id}"
            if key in completed:
                print(f"CHECKPOINT FOUND -> SKIP: {key}"); continue
            if not selected:
                raise RuntimeError(f"Empty cohort {cohort_id}")
            print_banner(f"RQ6 UNIQUE COHORT {job_idx}/{total_jobs} | seed={training_seed} | {cohort_id}")
            print(f"Selected clients ({len(selected)}): {selected}")

            natural_map = {
                cid: natural_steps(len(data["client_arrays"][cid][1]), BATCH_SIZE, LOCAL_EPOCHS)
                for cid in selected
            }
            result = federated_train(
                build_model_fn=build_model_fn, initial_weights=initial_weights,
                selected_per_round=[list(selected) for _ in range(NUM_ROUNDS_FL)],
                client_arrays=data["client_arrays"], X_test=data["X_test"], y_test=data["y_test"],
                n_classes=3, batch_size=BATCH_SIZE, local_epochs=LOCAL_EPOCHS,
                class_weights=data["class_weights"], run_seed=training_seed,
                exact_step_maps=[dict(natural_map) for _ in range(NUM_ROUNDS_FL)],
                equal_weight=False, fedprox_mu=0.0, train_monitor=train_monitor,
                progress_context={
                    "scenario": f"RQ6 | {cohort_id}",
                    "run_idx": TRAINING_RUN_SEEDS.index(training_seed)+1,
                    "run_total": len(TRAINING_RUN_SEEDS),
                    "scenario_idx": job_idx, "scenario_total": total_jobs,
                    "overall_idx": job_idx, "overall_total": total_jobs,
                },
            )
            row = result_row(
                run=TRAINING_RUN_SEEDS.index(training_seed)+1, seed=training_seed,
                scenario=cohort_id, result=result, initial_hash=initial_hash,
            )
            row.update({
                "training_seed": int(training_seed), "cohort_id": cohort_id,
                "selected_k": int(len(selected)), "selected_clients": ";".join(selected),
                "steps_per_round": int(sum(natural_map.values())), "fl_rounds": NUM_ROUNDS_FL,
            })
            upsert_csv(row, UNIQUE_COHORT_PERF, ["training_seed","cohort_id"])
            mark_checkpoint_complete(CHECKPOINT_STATE, key, extra={"last_job": key})
            completed.add(key)
            del result; tf.keras.backend.clear_session(); gc.collect()

    perf = pd.read_csv(UNIQUE_COHORT_PERF)
    expected = len(TRAINING_RUN_SEEDS)*len(cohort_clients)
    if len(perf) != expected:
        raise RuntimeError(f"Expected {expected} unique-cohort rows; found {len(perf)}")

    audit_rows=[]
    for seed in TRAINING_RUN_SEEDS:
        d=perf[perf["training_seed"].eq(seed)]
        hashes=d["initial_weights_sha256"].astype(str).unique(); passed=len(hashes)==1
        audit_rows.append({
            "training_seed":seed,"n_unique_cohorts":len(d),"unique_W0_hashes":len(hashes),
            "same_W0_across_cohorts":passed,"initial_weights_sha256":hashes[0] if len(hashes) else "",
        })
        if not passed: raise RuntimeError(f"W0 parity failed for seed {seed}")
    pd.DataFrame(audit_rows).to_csv(EXPERIMENT_ROOT/"rq6_W0_parity_audit.csv",index=False)
    return perf


def summarize_rq6_performance(gov_summary, cohort_perf):
    valid = gov_summary[gov_summary["valid_policy"].eq(True) & gov_summary["cohort_id"].astype(str).ne("")]
    expanded = valid.merge(cohort_perf, on="cohort_id", how="left", validate="many_to_many")
    expanded.to_csv(EXPERIMENT_ROOT/"rq6_performance_all_seeds.csv",index=False)
    metrics=[
        "roc_auc_ovr_macro","f1_macro","accuracy","precision_macro","recall_macro",
        "runtime_s","communication_mb","optimizer_steps","participants","energy_wh",
    ]
    rows=[]
    for cfg,d in expanded.groupby("configuration",sort=False):
        g=valid[valid["configuration"].eq(cfg)].iloc[0]
        row={
            "configuration":cfg,"family":g["family"],"perturbation_pct":int(g["perturbation_pct"]),
            "target_dimension":g["target_dimension"],"lower_cut":float(g["lower_cut"]),
            "upper_cut":float(g["upper_cut"]),"weights_json":g["weights_json"],
            "cohort_id":g["cohort_id"],"accepted_count":int(g["accepted_count"]),
            "accepted_rate_pct":float(g["accepted_rate_pct"]),
            "accepted_count_change_vs_reference":int(g["accepted_count_change_vs_reference"]),
            "accepted_rate_change_pp_vs_reference":float(g["accepted_rate_change_pp_vs_reference"]),
            "accepted_rate_relative_change_pct_vs_reference":float(g["accepted_rate_relative_change_pct_vs_reference"]),
            "decision_flip_count":int(g["decision_flip_count"]),
            "accepted_set_jaccard_vs_reference":float(g["accepted_set_jaccard_vs_reference"]),
            "admitted_clients":g["admitted_clients"],"n_training_seeds":len(d),
        }
        for metric in metrics:
            if metric not in d.columns: continue
            x=pd.to_numeric(d[metric],errors="coerce").dropna().to_numpy(float)
            row[f"{metric}_mean"]=float(np.mean(x)) if len(x) else np.nan
            row[f"{metric}_sd"]=float(np.std(x,ddof=1)) if len(x)>1 else 0.0
            lo,hi=ci95_mean(x); row[f"{metric}_ci95_low"]=lo; row[f"{metric}_ci95_high"]=hi
        rows.append(row)
    summary=pd.DataFrame(rows)
    ref=summary[summary["configuration"].eq("REFERENCE")].iloc[0]
    for metric in ["roc_auc_ovr_macro","f1_macro","accuracy"]:
        base=float(ref[f"{metric}_mean"])
        summary[f"{metric}_delta_vs_reference"]=summary[f"{metric}_mean"]-base
        summary[f"{metric}_change_pct_vs_reference"]=100.0*summary[f"{metric}_delta_vs_reference"]/base
    summary.to_csv(EXPERIMENT_ROOT/"rq6_sensitivity_performance_summary.csv",index=False)

    # Paired seed-level differences.
    ref_seed=expanded[expanded["configuration"].eq("REFERENCE")][
        ["training_seed","roc_auc_ovr_macro","f1_macro","accuracy"]
    ].copy()
    paired=[]
    for cfg,d in expanded.groupby("configuration",sort=False):
        if cfg=="REFERENCE": continue
        m=d.merge(ref_seed,on="training_seed",suffixes=("_config","_reference"),validate="one_to_one")
        for metric in ["roc_auc_ovr_macro","f1_macro","accuracy"]:
            diff=m[f"{metric}_config"].to_numpy(float)-m[f"{metric}_reference"].to_numpy(float)
            lo,hi=ci95_mean(diff)
            paired.append({
                "configuration":cfg,"family":d["family"].iloc[0],"target_dimension":d["target_dimension"].iloc[0],
                "perturbation_pct":int(d["perturbation_pct"].iloc[0]),"metric":metric,"n_pairs":len(diff),
                "mean_difference":float(np.mean(diff)),"sd_difference":float(np.std(diff,ddof=1)) if len(diff)>1 else 0.0,
                "ci95_low":lo,"ci95_high":hi,"wins":int(np.sum(diff>0)),"ties":int(np.sum(np.isclose(diff,0))),
                "losses":int(np.sum(diff<0)),
            })
    pd.DataFrame(paired).to_csv(EXPERIMENT_ROOT/"rq6_paired_differences_vs_reference.csv",index=False)
    return expanded,summary


def save_rq6_split_outputs(gov_summary, perf_summary):
    threshold_families=["LOWER_ONLY","UPPER_ONLY","BOTH"]
    gov_summary[gov_summary["family"].isin(threshold_families)].to_csv(
        EXPERIMENT_ROOT/"rq6_threshold_governance_summary.csv",index=False
    )
    perf_summary[perf_summary["family"].isin(threshold_families)].to_csv(
        EXPERIMENT_ROOT/"rq6_threshold_performance_summary.csv",index=False
    )
    gov_summary[gov_summary["family"].eq("WEIGHT_ONE_DIM")].to_csv(
        EXPERIMENT_ROOT/"rq6_weight_governance_summary.csv",index=False
    )
    perf_summary[perf_summary["family"].eq("WEIGHT_ONE_DIM")].to_csv(
        EXPERIMENT_ROOT/"rq6_weight_performance_summary.csv",index=False
    )
    invalid=gov_summary[gov_summary["valid_policy"].eq(False)]
    invalid.to_csv(EXPERIMENT_ROOT/"rq6_invalid_threshold_configurations.csv",index=False)


def create_rq6_threshold_figure(perf_summary):
    d=perf_summary[perf_summary["family"].isin(["LOWER_ONLY","UPPER_ONLY","BOTH"])].copy()
    families=[("LOWER_ONLY","Lower only"),("UPPER_ONLY","Upper only"),("BOTH","Both")]
    pcts=PERTURBATION_PCTS; x=np.arange(len(pcts),dtype=float); width=0.24
    fig=plt.figure(figsize=(14,10)); gs=fig.add_gridspec(2,2,hspace=0.38,wspace=0.28)
    panels=[
        (gs[0,0],"accepted_rate_change_pp_vs_reference","A. Change in admitted-client percentage","Percentage-point change"),
        (gs[0,1],"roc_auc_ovr_macro_delta_vs_reference","B. Change in Macro ROC-AUC","Δ Macro ROC-AUC"),
        (gs[1,0],"f1_macro_delta_vs_reference","C. Change in Macro-F1","Δ Macro-F1"),
        (gs[1,1],"accuracy_delta_vs_reference","D. Change in Accuracy","Δ Accuracy"),
    ]
    for cell,col,title,ylabel in panels:
        ax=fig.add_subplot(cell)
        for i,(fam,label) in enumerate(families):
            vals=[]
            for pct in pcts:
                z=d[d["family"].eq(fam)&d["perturbation_pct"].eq(pct)]
                vals.append(float(z[col].iloc[0]) if len(z) else np.nan)
            ax.bar(x+(i-1)*width,vals,width,label=label)
        ax.axhline(0,linewidth=0.8); ax.set_xticks(x); ax.set_xticklabels([f"{p:+d}%" for p in pcts])
        ax.set_xlabel("Threshold perturbation"); ax.set_ylabel(ylabel); ax.set_title(title,fontweight="bold")
        if cell==gs[0,0]: ax.legend(fontsize=8)
    fig.suptitle("RQ6 — HPS Threshold Sensitivity under Frozen K=10 TADP-VR",fontsize=14,fontweight="bold")
    out=EXPERIMENT_ROOT/"RQ6_threshold_sensitivity_figure.png"; fig.savefig(out,dpi=300,bbox_inches="tight"); plt.close(fig)
    return out


def create_rq6_weight_figure(perf_summary):
    d=perf_summary[perf_summary["family"].eq("WEIGHT_ONE_DIM")].copy()
    dims=DIMS; pcts=PERTURBATION_PCTS
    panels=[
        ("accepted_rate_change_pp_vs_reference","A. Admitted-client percentage change","Percentage points"),
        ("roc_auc_ovr_macro_delta_vs_reference","B. Macro ROC-AUC change","Δ Macro ROC-AUC"),
        ("f1_macro_delta_vs_reference","C. Macro-F1 change","Δ Macro-F1"),
        ("accuracy_delta_vs_reference","D. Accuracy change","Δ Accuracy"),
    ]
    fig=plt.figure(figsize=(15,10)); gs=fig.add_gridspec(2,2,hspace=0.32,wspace=0.28)
    for idx,(col,title,cbar_label) in enumerate(panels):
        ax=fig.add_subplot(gs[idx//2,idx%2]); matrix=np.full((len(dims),len(pcts)),np.nan)
        for i,dim in enumerate(dims):
            for j,pct in enumerate(pcts):
                z=d[d["target_dimension"].eq(dim)&d["perturbation_pct"].eq(pct)]
                if len(z): matrix[i,j]=float(z[col].iloc[0])
        im=ax.imshow(matrix,aspect="auto")
        ax.set_xticks(np.arange(len(pcts))); ax.set_xticklabels([f"{p:+d}%" for p in pcts])
        ax.set_yticks(np.arange(len(dims))); ax.set_yticklabels([DIMENSION_NAMES[x] for x in dims])
        ax.set_xlabel("Single-dimension weight perturbation"); ax.set_title(title,fontweight="bold")
        for i in range(matrix.shape[0]):
            for j in range(matrix.shape[1]):
                if np.isfinite(matrix[i,j]):
                    ax.text(j,i,f"{matrix[i,j]:.3f}",ha="center",va="center",fontsize=7)
        cb=fig.colorbar(im,ax=ax,fraction=0.046,pad=0.04); cb.set_label(cbar_label)
    fig.suptitle("RQ6 — HPS Weight Sensitivity under Frozen K=10 TADP-VR",fontsize=14,fontweight="bold")
    out=EXPERIMENT_ROOT/"RQ6_weight_sensitivity_figure.png"; fig.savefig(out,dpi=300,bbox_inches="tight"); plt.close(fig)
    return out


def main():
    print_banner(EXPERIMENT_VERSION)
    print("Threshold families: lower-only, upper-only, both")
    print("Weight sensitivity: one HPS dimension at a time; other weights rescaled proportionally")
    print(f"Perturbations: {PERTURBATION_PCTS}")
    print(f"K={K_SUBMISSIONS} | seeds={TRAINING_RUN_SEEDS} | rounds={NUM_ROUNDS_FL}")

    configs=build_rq6_configurations()
    configs.to_csv(EXPERIMENT_ROOT/"rq6_requested_configurations.csv",index=False)
    weight_vectors=configs[configs["family"].eq("WEIGHT_ONE_DIM")][
        ["configuration","target_dimension","perturbation_pct","weights_json"]
    ].copy()
    weight_vectors.to_csv(EXPERIMENT_ROOT/"rq6_weight_vectors.csv",index=False)

    atomic_write_json({
        "experiment_version":EXPERIMENT_VERSION,
        "research_question":"How sensitive are TADP-VR admission decisions and downstream utility to HPS thresholds and dimension weights?",
        "k_submissions":K_SUBMISSIONS,"training_seeds":TRAINING_RUN_SEEDS,"fl_rounds":NUM_ROUNDS_FL,
        "local_epochs":LOCAL_EPOCHS,"batch_size":BATCH_SIZE,"learning_rate":LEARNING_RATE,
        "dirichlet_alpha":DIRICHLET_ALPHA,"global_split_seed":GLOBAL_SPLIT_SEED,
        "client_partition_seed":CLIENT_PARTITION_SEED,"frozen_evidence_assignment_seed":FROZEN_EVIDENCE_ASSIGNMENT_SEED,
        "reference_lower_cut":REFERENCE_LOWER_CUT,"reference_upper_cut":REFERENCE_UPPER_CUT,
        "reference_dimension_floor":REFERENCE_DIMENSION_FLOOR,"reference_weights":WEIGHTS_PSCORE_DEFAULT,
        "perturbation_pcts":PERTURBATION_PCTS,
        "threshold_families":["LOWER_ONLY","UPPER_ONLY","BOTH"],
        "weight_rule":"Perturb one target dimension by ±10/20/30%; rescale the other five proportionally so weights sum to 1.",
        "why_not_all_weights_together":"Common scaling of all weights followed by normalization leaves the original normalized vector unchanged.",
        "interpretation":"Sensitivity analysis only; no configuration is selected using held-out TEST performance.",
    },EXPERIMENT_ROOT/"rq6_experiment_design.json")

    csv_path=locate_diabetes_csv()
    data=prepare_diabetes_no_leakage(
        csv_path=csv_path,split_seed=GLOBAL_SPLIT_SEED,partition_seed=CLIENT_PARTITION_SEED,
        n_clients=K_SUBMISSIONS,alpha=DIRICHLET_ALPHA,
    )
    leakage=write_leakage_audit(
        EXPERIMENT_ROOT,data["global_train_ids"],data["global_test_ids"],data["client_train_ids"],
        extra={
            "patient_overlap":data["meta"]["patient_overlap"],"global_holdout_before_client_partition":True,
            "preprocessing_train_only":True,"dq_tadp_train_only":True,"class_weights_train_only":True,
            "test_used_for_final_evaluation_only":True,"rq6_frozen_main_setting":True,
        },
    )
    if not bool(leakage.get("pass",False)): raise RuntimeError(f"No-leakage audit failed: {leakage}")

    client_ids=list(data["client_ids"])
    if client_ids!=list("ABCDEFGHIJ"): raise RuntimeError(f"Expected A-J clients; got {client_ids}")

    documentary,evidence_df=generate_scalable_controlled_evidence(client_ids,FROZEN_EVIDENCE_ASSIGNMENT_SEED,domain=DOMAIN)
    validate_expected_profile_mix(evidence_df,K_SUBMISSIONS)
    evidence_df.to_csv(EXPERIMENT_ROOT/"rq6_frozen_controlled_evidence.csv",index=False)
    data["dq_audit"].to_csv(EXPERIMENT_ROOT/"rq6_frozen_DQ_audit.csv",index=False)

    original_gov=build_tadp_governance(
        client_ids,documentary,data["dq_scores"],run=0,evidence_seed=FROZEN_EVIDENCE_ASSIGNMENT_SEED,domain=DOMAIN
    ).sort_values("client").reset_index(drop=True)
    original_gov.to_csv(EXPERIMENT_ROOT/"rq6_original_frozen_governance.csv",index=False)

    full_factor_df=build_full_factor_evidence_table(
        client_ids,documentary,data["dq_scores"],data["dq_audit"],FROZEN_EVIDENCE_ASSIGNMENT_SEED,DOMAIN
    )
    full_factor_df.to_csv(EXPERIMENT_ROOT/"rq6_frozen_all_28_factor_evidence.csv",index=False)
    _,dim_scores,crit=build_policy_tables_rq6(full_factor_df)

    # Fail closed: reproduce exact reference policy first.
    ref_recomputed=evaluate_rq6_policy(
        dim_scores,crit,REFERENCE_LOWER_CUT,REFERENCE_UPPER_CUT,
        {d:float(WEIGHTS_PSCORE_DEFAULT[d]) for d in DIMS}
    ).sort_values("client").reset_index(drop=True)
    audit=ref_recomputed[["client","hps","final_action"]].merge(
        original_gov[["client","hps","final_action"]],on="client",suffixes=("_recomputed","_original"),validate="one_to_one"
    )
    audit["decision_match"]=audit["final_action_recomputed"].eq(audit["final_action_original"])
    audit["hps_abs_diff"]=np.abs(audit["hps_recomputed"]-audit["hps_original"])
    audit.to_csv(EXPERIMENT_ROOT/"rq6_reference_policy_reproduction_audit.csv",index=False)
    if not audit["decision_match"].all() or float(audit["hps_abs_diff"].max())>1e-10:
        raise RuntimeError("RQ6 reference-policy reproduction failed")

    gov_summary,gov_by_config,_=summarize_rq6_governance(configs,dim_scores,crit)
    gov_summary,cohort_mapping=assign_rq6_cohort_ids(gov_summary)
    gov_summary.to_csv(EXPERIMENT_ROOT/"rq6_governance_sensitivity_summary.csv",index=False)

    frames=[]
    for cfg,df in gov_by_config.items():
        x=df.copy();x.insert(0,"configuration",cfg);frames.append(x)
    pd.concat(frames,ignore_index=True).to_csv(EXPERIMENT_ROOT/"rq6_governance_client_level.csv",index=False)

    print_banner("RQ6 GOVERNANCE SENSITIVITY SUMMARY")
    show=[
        "configuration","family","target_dimension","perturbation_pct","lower_cut","upper_cut","valid_policy",
        "accepted_count","accepted_rate_pct","accepted_rate_change_pp_vs_reference","decision_flip_count",
        "accepted_set_jaccard_vs_reference","admitted_clients"
    ]
    print(gov_summary[[c for c in show if c in gov_summary.columns]].to_string(index=False))

    cohort_perf=train_rq6_unique_cohorts(cohort_mapping,gov_summary,data)
    _,perf_summary=summarize_rq6_performance(gov_summary,cohort_perf)
    save_rq6_split_outputs(gov_summary,perf_summary)
    fig_threshold=create_rq6_threshold_figure(perf_summary)
    fig_weight=create_rq6_weight_figure(perf_summary)

    print_banner("RQ6 PERFORMANCE SUMMARY")
    print(perf_summary[[
        "configuration","family","target_dimension","perturbation_pct","accepted_count","accepted_rate_pct",
        "accepted_rate_change_pp_vs_reference","roc_auc_ovr_macro_mean","roc_auc_ovr_macro_delta_vs_reference",
        "f1_macro_mean","f1_macro_delta_vs_reference","accuracy_mean","accuracy_delta_vs_reference"
    ]].to_string(index=False))

    n_threshold=int(configs["family"].isin(["LOWER_ONLY","UPPER_ONLY","BOTH"]).sum())
    n_weight=int(configs["family"].eq("WEIGHT_ONE_DIM").sum())
    n_invalid=int((configs["valid_policy"].eq(False)).sum())
    atomic_write_json({
        "reference_policy_reproduction":"PASS","no_leakage":"PASS","same_W0_within_seed":"PASS",
        "requested_threshold_configs":n_threshold,"requested_weight_configs":n_weight,
        "invalid_threshold_configs":n_invalid,"valid_configs_including_reference":int(gov_summary["valid_policy"].eq(True).sum()),
        "n_unique_admitted_cohorts":int(len(cohort_mapping)),
        "actual_model_training_runs":int(len(cohort_mapping)*len(TRAINING_RUN_SEEDS)),
        "threshold_figure":str(fig_threshold),"weight_figure":str(fig_weight),
    },EXPERIMENT_ROOT/"rq6_final_audit_summary.json")

    return EXPERIMENT_ROOT


if __name__=="__main__":
    finished_root=main()
    package_and_download_results(finished_root,EXPERIMENT_VERSION)


Google Drive checkpoint mount unavailable: Error: credential propagation was unsuccessful
Local checkpoint root: /content/TADP_EXPERIMENT_B3_TADP-B3-v16.9-PREDICTIVE-SCALABILITY-K20-50-100-3SEED-4ROUND
Google Drive checkpoint mount unavailable: Error: credential propagation was unsuccessful
Local checkpoint root: /content/TADP_EXPERIMENT_RQ6_TADP-RQ6-v17.1-K10-THRESHOLD-WEIGHT-SENSITIVITY-5SEED-4ROUND

TADP-RQ6-v17.1-K10-THRESHOLD-WEIGHT-SENSITIVITY-5SEED-4ROUND
Threshold families: lower-only, upper-only, both
Weight sensitivity: one HPS dimension at a time; other weights rescaled proportionally
Perturbations: [-30, -20, -10, 10, 20, 30]
K=10 | seeds=[42, 142, 242, 342, 442] | rounds=4

RQ6 GOVERNANCE SENSITIVITY SUMMARY
     configuration         family target_dimension  perturbation_pct  lower_cut  upper_cut  valid_policy  accepted_count  accepted_rate_pct  accepted_rate_change_pp_vs_reference  decision_flip_count  accepted_set_jaccard_vs_reference admitted_clients
         REFERENCE

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# ======================================================================================
# TADP v16.7 — REVIEWER-ALIGNED TRUSTWORTHY DATA PREPARATION EXPERIMENT CORE
# ======================================================================================
# Design guarantees:
#   1) GLOBAL holdout is created BEFORE client partitioning.
#   2) Only TRAIN is distributed to clients.
#   3) Preprocessing parameters/vocabularies use TRAIN only.
#   4) DQ, GE and TADP governance use TRAIN only.
#   5) Held-out TEST never affects preprocessing, governance, class weights,
#      client selection, model initialization, training, or matched-control budgets.
#   6) TEST is used only after training for final evaluation.
#
# v16.7 governance:
#   - 28 factors across 6 dimensions:
#       dim1=4, dim2(DQ)=8, dim3=4, dim4=3, dim5=5, dim6=4.
#   - Added Data Collection / Acquisition Lineage (dim1).
#   - Added Structural / Constraint Integrity (dim2).
#   - Documentary evidence is generated at the individual factor level.
#   - DQ evidence is machine-measured from client TRAIN partitions only.
#   - HPS remains client-specific and uses the six policy dimension weights.
#   - WAC is NOT client-specific.
#   - Each factor has a declared minimum adequate rubric rank.
#   - A domain WAC is derived once:
#         WAC_d = mean(minimum adequate ranks in dimension d) / 5
#         WAC_domain = equal mean of the six WAC_d values.
#   - Every averaged dimension must be >= 2.5/5 or the client is auto-rejected.
#   - HPS < 3.0 -> AUTO_REJECT.
#   - 3.0 <= HPS < 3.5 -> AUTOMATED REVIEW using Critical WAC_i.
#   - Review accepts iff Critical WAC_i >= the domain Global Critical WAC.
#   - HPS >= 3.5 -> DIRECT AUTO_ACCEPT when every critical factor meets its own adequacy minimum.
#   - Human reviewers verify evidence only; admission is server-automated.
#
# GX Core comparator:
#   - Separate from HPS/TADP.
#   - Uses REAL Great Expectations GX Core validation on TRAIN-only client data.
#   - Uses common technical checks: schema, datatype consistency, required ranges,
#     missingness, duplicate/ID integrity, label/domain validity, and structure.
#   - Uses GX native severity-aware validation: zero critical failures required; no ranking and no forced-K.
#
# Runtime/reporting:
#   - Every FL round prints configuration/run/scenario/round progress plus TRAIN-only diagnostic utility and operational metrics.
#   - Every completed scenario prints all predictive and operational metrics.
#   - Scenario checkpoints support restart/resume.
#   - Final result ZIP downloads automatically in Google Colab.
# ======================================================================================

import os
import sys
import gc
import math
import time
import json
import random
import hashlib
import threading
import zipfile
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# ======================================================================================
# POLICY CONSTANTS — v16.7
# ======================================================================================

GOOD_CUT = 3.0
HIGH_CUT = 3.5
MAX_FACTOR_SCORE = 5.0
GE_ACCEPT_COUNT = 6

# ----------------------------------------------------------------------
# v16.7 FINAL FULLY AUTOMATED ADMISSION POLICY
# ----------------------------------------------------------------------
# Human reviewers verify supporting evidence uploaded through the questionnaire.
# They do NOT make the admission decision. Once verified factor scores are
# available, the server applies this policy automatically.
#
# 1) EVERY averaged HPS dimension must be >= 2.5/5.
#    Any dimension < 2.5 -> AUTO-REJECT.
#
# 2) HPS must be >= 3.0.
#    HPS < 3.0 -> AUTO-REJECT.
#
# 3) DIRECT AUTO-ACCEPT:
#    HPS >= 3.5 AND every critical factor independently meets its own
#    factor-specific minimum adequacy requirement.
#
# 4) AUTOMATED REVIEW:
#    - all clients with 3.0 <= HPS < 3.5; and
#    - high-HPS clients that fail one or more individual critical-factor
#      adequacy requirements.
#
# 5) REVIEW RESOLUTION:
#    Critical WAC_i = mean(actual critical-factor scores / 5)
#    Global Critical WAC = mean(policy adequacy minima / 5 for same factors)
#
#    Critical WAC_i >= Global Critical WAC -> ACCEPT AFTER REVIEW
#    Critical WAC_i <  Global Critical WAC -> AUTO-REJECT
#
# Thus Direct Auto-Accept is strict at the individual critical-factor level,
# whereas Review intentionally allows controlled compensation across the
# critical subset.
DIMENSION_MIN_FLOOR = 2.5

CRITICAL_FACTORS_BY_DOMAIN = {
    "healthcare": {
        "dim1": [
            "data_controller",
            "data_collection_lineage",
        ],
        "dim5": [
            "regulation_coverage",
            "consent_ethics",
            "sensitivity_classification",
        ],
        "dim6": [
            "user_agreements",
        ],
    },
    "cifar10": {
        "dim1": [
            "data_controller",
            "data_collection_lineage",
        ],
        "dim6": [
            "license_terms",
            "user_agreements",
        ],
    },
}

WEIGHTS_PSCORE_DEFAULT = {
    "dim1": 0.25,  # Source Reliability
    "dim2": 0.15,  # Data Quality and Health
    "dim3": 0.10,  # Documentation Practices
    "dim4": 0.10,  # Timeliness and Refresh Rate
    "dim5": 0.30,  # Regulatory / Compliance Alignment
    "dim6": 0.10,  # Context / Usage Constraints
}

DIMENSION_NAMES = {
    "dim1": "Source Reliability",
    "dim2": "Data Quality and Health",
    "dim3": "Documentation Practices",
    "dim4": "Timeliness and Refresh Rate",
    "dim5": "Regulatory and Compliance Alignment",
    "dim6": "Context and Usage Constraints",
}

# v16.7: two reviewer-driven additions:
#   dim1: data_collection_lineage
#   dim2: structural_constraint_integrity
#
# Total = 4 + 8 + 4 + 3 + 5 + 4 = 28 factors.
FACTOR_NAMES = {
    "dim1": [
        "source_reputation",
        "data_controller",
        "data_objective",
        "data_collection_lineage",
    ],
    "dim2": [
        "completeness",
        "duplication_rate",
        "value_validity_error_rate",
        "type_consistency",
        "label_integrity",
        "feature_distribution_consistency",
        "feature_category_coverage",
        "structural_constraint_integrity",
    ],
    "dim3": [
        "data_dictionary",
        "version_logs",
        "collection_protocol",
        "definition_updates",
    ],
    "dim4": [
        "data_freshness",
        "scheduled_refresh",
        "retention_clarity",
    ],
    "dim5": [
        "regulation_coverage",
        "consent_ethics",
        "geo_restrictions",
        "sensitivity_classification",
        "audits",
    ],
    "dim6": [
        "license_terms",
        "ethical_reviews",
        "redistribution",
        "user_agreements",
    ],
}

DOCUMENTARY_DIMS = ("dim1", "dim3", "dim4", "dim5", "dim6")

# ----------------------------------------------------------------------
# COMPLETE 0--5 RUBRIC DESCRIPTORS
# ----------------------------------------------------------------------
# These descriptors reproduce the Appendix-A semantics and add the two
# v16.7 factors explicitly. Controlled evidence is sampled at the factor
# level; HPS is never generated directly.
RUBRIC_DESCRIPTORS = {
    "dim1": {
        "source_reputation": {
            0: "No info",
            1: "Poor",
            2: "Limited evidence",
            3: "Average, partially trusted",
            4: "Well-documented, reliable",
            5: "Highly reputable, verified",
        },
        "data_controller": {
            0: "No documented controller",
            1: "Unclear",
            2: "Partially clear",
            3: "Moderately clear",
            4: "Mostly clear",
            5: "Fully documented",
        },
        "data_objective": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General but unclear",
            4: "Mostly explicit",
            5: "Fully explicit, justified",
        },
        "data_collection_lineage": {
            0: "Collection origin unknown",
            1: "Informal or unverifiable origin",
            2: "Partially documented acquisition path",
            3: "Documented acquisition with limited traceability",
            4: "Well-documented and traceable acquisition path",
            5: "Fully source-linked, versioned, and auditable lineage",
        },
    },
    "dim2": {
        "completeness": {
            0: ">50% missing",
            1: "20-50% missing",
            2: "10-20% missing",
            3: "5-10% missing",
            4: "1-5% missing",
            5: "<1% missing",
        },
        "duplication_rate": {
            0: ">20% duplicates",
            1: "10-20% duplicates",
            2: "5-10% duplicates",
            3: "2-5% duplicates",
            4: "1-2% duplicates",
            5: "<1% duplicates",
        },
        "value_validity_error_rate": {
            0: ">15% invalid/error values",
            1: "10-15% invalid/error values",
            2: "5-10% invalid/error values",
            3: "2-5% invalid/error values",
            4: "1-2% invalid/error values",
            5: "<1% invalid/error values",
        },
        "type_consistency": {
            0: "Highly inconsistent",
            1: "Frequent type inconsistency",
            2: "Moderate type inconsistency",
            3: "Minor type inconsistency",
            4: "Rare type inconsistency",
            5: "Fully consistent",
        },
        "label_integrity": {
            0: ">10% missing/invalid/known erroneous labels",
            1: "5-10% missing/invalid/known erroneous labels",
            2: "2-5% missing/invalid/known erroneous labels",
            3: "1-2% missing/invalid/known erroneous labels",
            4: "0.1-1% missing/invalid/known erroneous labels",
            5: "<=0.1% missing/invalid/known erroneous labels",
        },
        "feature_distribution_consistency": {
            0: "JSD >0.20",
            1: "JSD 0.10-0.20",
            2: "JSD 0.05-0.10",
            3: "JSD 0.025-0.05",
            4: "JSD 0.01-0.025",
            5: "JSD <=0.01",
        },
        "feature_category_coverage": {
            0: "<50% reference support represented",
            1: "50-65% reference support represented",
            2: "65-75% reference support represented",
            3: "75-82.5% reference support represented",
            4: "82.5-90% reference support represented",
            5: ">=90% reference support represented",
        },
        "structural_constraint_integrity": {
            0: ">10% records/structures violate required constraints",
            1: "5-10% violate required constraints",
            2: "2-5% violate required constraints",
            3: "1-2% violate required constraints",
            4: "0.1-1% violate required constraints",
            5: "<=0.1% violate required constraints",
        },
    },
    "dim3": {
        "data_dictionary": {
            0: "None",
            1: "Minimal outline",
            2: "Partial coverage",
            3: "Moderate coverage",
            4: "Near-complete",
            5: "Fully detailed",
        },
        "version_logs": {
            0: "None",
            1: "Minimal logs",
            2: "Occasional logs",
            3: "Regular logs",
            4: "Near-complete",
            5: "Full version history",
        },
        "collection_protocol": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General methods",
            4: "Well-defined",
            5: "Fully transparent",
        },
        "definition_updates": {
            0: "None",
            1: "Rarely updated",
            2: "Occasional updates",
            3: "Regular but basic",
            4: "Frequent",
            5: "Real-time, documented",
        },
    },
    "dim4": {
        "data_freshness": {
            0: ">5 years old",
            1: "2-5 years old",
            2: "1-2 years old",
            3: "6-12 months old",
            4: "1-6 months old",
            5: "Real-time/current",
        },
        "scheduled_refresh": {
            0: "Never",
            1: "Irregular",
            2: "Annual",
            3: "Quarterly",
            4: "Monthly",
            5: "Daily/real-time",
        },
        "retention_clarity": {
            0: "None",
            1: "Minimal",
            2: "Basic guidelines",
            3: "Moderate clarity",
            4: "High clarity",
            5: "Fully documented",
        },
    },
    "dim5": {
        "regulation_coverage": {
            0: "None",
            1: "Minimal",
            2: "Partial",
            3: "Moderate",
            4: "Comprehensive but dated",
            5: "Fully documented/current",
        },
        "consent_ethics": {
            0: "None",
            1: "Minimal record",
            2: "Partial consent/ethics evidence",
            3: "Moderate logs",
            4: "Substantial",
            5: "Fully documented",
        },
        "geo_restrictions": {
            0: "None",
            1: "Basic mention",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully documented",
        },
        "sensitivity_classification": {
            0: "None",
            1: "Basic flagging",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully classified",
        },
        "audits": {
            0: "None",
            1: "Internal only",
            2: "Basic certification",
            3: "Occasional audit",
            4: "Recent audit",
            5: "Regular external audits",
        },
    },
    "dim6": {
        "license_terms": {
            0: "None",
            1: "Vague",
            2: "Basic",
            3: "Clear",
            4: "Detailed",
            5: "Industry-compliant",
        },
        "ethical_reviews": {
            0: "None",
            1: "Informal approval",
            2: "Partial",
            3: "Moderate",
            4: "Well-documented",
            5: "Certified",
        },
        "redistribution": {
            0: "No policy",
            1: "Unclear",
            2: "Partial",
            3: "Clear",
            4: "Detailed",
            5: "Fully compliant",
        },
        "user_agreements": {
            0: "Non-compliant",
            1: "Minimal adherence",
            2: "Partial",
            3: "Mostly compliant",
            4: "Fully compliant",
            5: "Audited compliance",
        },
    },
}

# ----------------------------------------------------------------------
# FACTOR-SPECIFIC ADEQUACY POLICY
# ----------------------------------------------------------------------
# The minimum adequate rank is derived factor-by-factor from the wording of
# the Appendix-A rubric. It is NOT learned from model/test outcomes.
#
# Healthcare is the primary policy. CIFAR-10 uses the same reference ranks
# for cross-domain comparability; its DQ factors are measured with image-
# specific checks, while documentary dimensions remain controlled evidence.
FACTOR_ADEQUACY_MIN_HEALTHCARE = {
    "dim1": {
        "source_reputation": 4,
        "data_controller": 4,
        "data_objective": 4,
        "data_collection_lineage": 4,
    },
    "dim2": {
        "completeness": 3,
        "duplication_rate": 3,
        "value_validity_error_rate": 3,
        "type_consistency": 3,
        "label_integrity": 3,
        "feature_distribution_consistency": 3,
        "feature_category_coverage": 3,
        "structural_constraint_integrity": 3,
    },
    "dim3": {
        "data_dictionary": 3,
        "version_logs": 3,
        "collection_protocol": 4,
        "definition_updates": 3,
    },
    "dim4": {
        "data_freshness": 3,
        "scheduled_refresh": 3,
        "retention_clarity": 4,
    },
    "dim5": {
        "regulation_coverage": 3,
        "consent_ethics": 3,
        "geo_restrictions": 3,
        "sensitivity_classification": 3,
        "audits": 4,
    },
    "dim6": {
        "license_terms": 3,
        "ethical_reviews": 4,
        "redistribution": 3,
        "user_agreements": 4,
    },
}

FACTOR_ADEQUACY_MIN_CIFAR10 = {
    dim: dict(values)
    for dim, values in FACTOR_ADEQUACY_MIN_HEALTHCARE.items()
}

def derive_domain_wac(
    factor_minima: Dict[str, Dict[str, float]]
) -> Tuple[Dict[str, float], float]:
    """
    Derive the domain policy WAC from factor-specific minimum adequate ranks.

    WAC_d = mean_k(adequate_rank_dk / 5)
    WAC_domain = equal mean across the six dimension WAC_d values.

    IMPORTANT:
      WAC is a DOMAIN POLICY value, not a client-specific score.
    """
    dimension_wac = {}
    for dim in FACTOR_NAMES:
        vals = [
            float(factor_minima[dim][factor])
            for factor in FACTOR_NAMES[dim]
        ]
        dimension_wac[dim] = float(np.mean(vals) / MAX_FACTOR_SCORE)
    global_wac = float(np.mean(list(dimension_wac.values())))
    return dimension_wac, global_wac


HEALTHCARE_DIMENSION_WAC, WAC_HEALTHCARE = derive_domain_wac(
    FACTOR_ADEQUACY_MIN_HEALTHCARE
)
CIFAR10_DIMENSION_WAC, WAC_CIFAR10 = derive_domain_wac(
    FACTOR_ADEQUACY_MIN_CIFAR10
)

DOMAIN_FACTOR_MINIMA = {
    "healthcare": FACTOR_ADEQUACY_MIN_HEALTHCARE,
    "cifar10": FACTOR_ADEQUACY_MIN_CIFAR10,
}
DOMAIN_DIMENSION_WAC = {
    "healthcare": HEALTHCARE_DIMENSION_WAC,
    "cifar10": CIFAR10_DIMENSION_WAC,
}
DOMAIN_GLOBAL_WAC = {
    "healthcare": WAC_HEALTHCARE,
    "cifar10": WAC_CIFAR10,
}

DQ_RAW_METRIC_BY_FACTOR = {
    "completeness": "missing_fraction",
    "duplication_rate": "duplicate_fraction",
    "value_validity_error_rate": "error_fraction",
    "type_consistency": "type_inconsistency_fraction",
    "label_integrity": "invalid_label_fraction",
    "feature_distribution_consistency": "max_jsd",
    "feature_category_coverage": "mean_category_coverage",
    "structural_constraint_integrity": "structural_violation_fraction",
}

assert abs(sum(WEIGHTS_PSCORE_DEFAULT.values()) - 1.0) < 1e-12
assert sum(len(v) for v in FACTOR_NAMES.values()) == 28
assert len(FACTOR_NAMES["dim1"]) == 4
assert len(FACTOR_NAMES["dim2"]) == 8
assert abs(WAC_HEALTHCARE - 0.6761111111111111) < 1e-12

# ======================================================================================
# GENERAL UTILITIES
# ======================================================================================

def seed_everything(seed: int):
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        tf.keras.utils.set_random_seed(seed)
    except Exception:
        tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def sha256_bytes(x: bytes) -> str:
    return hashlib.sha256(x).hexdigest()


def sha256_array(x: np.ndarray) -> str:
    x = np.asarray(x)
    return sha256_bytes(np.ascontiguousarray(x).view(np.uint8).tobytes())


def sha256_weights(weights: List[np.ndarray]) -> str:
    h = hashlib.sha256()
    for w in weights:
        a = np.ascontiguousarray(np.asarray(w))
        h.update(str(a.shape).encode())
        h.update(a.view(np.uint8).tobytes())
    return h.hexdigest()


def _process_rss_mb() -> float:
    try:
        import psutil
        return float(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2))
    except Exception:
        pass
    try:
        with open("/proc/self/statm", "r", encoding="utf-8") as f:
            pages = int(f.read().split()[1])
        return float(pages * int(os.sysconf("SC_PAGE_SIZE")) / (1024 ** 2))
    except Exception:
        pass
    try:
        import resource
        x = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
        return x / (1024 ** 2) if sys.platform == "darwin" else x / 1024.0
    except Exception:
        return 0.0


class RAMMonitor:
    def __init__(self, interval_s: float = 0.05):
        self.interval_s = float(interval_s)
        self.start_mb = 0.0
        self.end_mb = 0.0
        self.peak_mb = 0.0
        self._stop = threading.Event()
        self._thread = None

    def _loop(self):
        while not self._stop.wait(self.interval_s):
            self.peak_mb = max(self.peak_mb, _process_rss_mb())

    def start(self):
        self.start_mb = _process_rss_mb()
        self.peak_mb = self.start_mb
        self._stop.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def stop(self) -> Dict[str, float]:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=1.0)
        self.end_mb = _process_rss_mb()
        self.peak_mb = max(self.peak_mb, self.start_mb, self.end_mb)
        return {
            "ram_start_mb": float(self.start_mb),
            "ram_end_mb": float(self.end_mb),
            "ram_peak_mb": float(self.peak_mb),
            "ram_delta_mb": float(max(0.0, self.peak_mb - self.start_mb)),
            "ram_mb": float(self.peak_mb),
        }


def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
    return Path(path)


def append_csv(row: Dict[str, Any], path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([row]).to_csv(
        path, mode="a", header=not path.exists(), index=False
    )


def summarize_runs(perf: pd.DataFrame) -> pd.DataFrame:
    if perf.empty:
        return pd.DataFrame()
    metrics = [
        c for c in [
            "accuracy", "precision_macro", "recall_macro", "f1_macro",
            "roc_auc_ovr_macro", "runtime_s", "energy_wh", "communication_mb",
            "optimizer_steps", "participants", "ram_peak_mb", "ram_delta_mb"
        ] if c in perf.columns
    ]
    rows = []
    for scenario, d in perf.groupby("scenario", sort=False):
        row = {"scenario": scenario, "n_runs": len(d)}
        for m in metrics:
            vals = pd.to_numeric(d[m], errors="coerce")
            row[f"{m}_mean"] = float(vals.mean())
            row[f"{m}_sd"] = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
        rows.append(row)
    return pd.DataFrame(rows)


def package_and_download_results(output_dir: Path, label: str) -> Path:
    output_dir = Path(output_dir)
    manifest = []
    for p in sorted(output_dir.rglob("*")):
        if p.is_file():
            manifest.append({
                "relative_path": str(p.relative_to(output_dir)),
                "bytes": int(p.stat().st_size),
                "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
            })
    pd.DataFrame(manifest).to_csv(
        output_dir / "RESULTS_FILE_MANIFEST.csv", index=False
    )

    zip_path = output_dir.parent / f"{output_dir.name}_RESULTS.zip"
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
    ) as zf:
        for p in sorted(output_dir.rglob("*")):
            if p.is_file():
                zf.write(p, arcname=str(p.relative_to(output_dir)))

    print("\n" + "=" * 100)
    print(f"{label} COMPLETE")
    print(f"Results ZIP: {zip_path}")
    print(f"ZIP size: {zip_path.stat().st_size / (1024**2):.2f} MB")
    print("=" * 100)

    try:
        from google.colab import files as colab_files
        print("Starting automatic download to your laptop...")
        colab_files.download(str(zip_path))
    except Exception as exc:
        print("Automatic Colab download unavailable.")
        print(f"ZIP remains at: {zip_path}")
        print(f"Reason: {exc}")

    return zip_path


# ======================================================================================
# CONTROLLED DOCUMENTARY EVIDENCE — v16.7
# ======================================================================================

def rubric_descriptor(dim: str, factor: str, score: float) -> str:
    s = int(np.clip(np.rint(float(score)), 0, 5))
    return str(RUBRIC_DESCRIPTORS[dim][factor][s])


def generate_controlled_documentary_evidence(
    client_ids: List[str],
    evidence_seed: int,
    factor_minima: Dict[str, Dict[str, float]],
    domain: str,
) -> Tuple[Dict[str, Dict[str, Dict[str, float]]], pd.DataFrame]:
    """
    Generate controlled documentary evidence at the INDIVIDUAL FACTOR level.

    v16.7 uses PRE-SPECIFIED governance archetypes so the controlled experiment
    contains all three decision outcomes needed to validate the policy:

      - DIRECT_STRONG:
          designed to satisfy the strict Direct Auto-Accept route;
      - REVIEW_RECOVERABLE:
          borderline overall evidence, but sufficiently strong average evidence
          across the critical subset for Automated Review acceptance;
      - REVIEW_LIMITED:
          adequate enough to enter Automated Review, but insufficient average
          critical evidence for Review acceptance;
      - LOW_HPS_WEAK:
          every documentary dimension remains at/above the 2.5 dimension floor,
          but the overall HPS is expected to remain below 3.0;
      - DIMENSION_FLOOR_WEAK:
          contains a deliberately weak documentary dimension (<2.5).

    IMPORTANT SCIENTIFIC INTERPRETATION
    -----------------------------------
    These are controlled governance scenarios, not observed hospital-site
    provenance records and not estimates of real-world admission prevalence.
    The archetype composition is specified BEFORE model training and does not
    use predictive performance, test labels, or downstream model outcomes.

    DQ (dim2) is NEVER synthesized here; it remains measured from TRAIN data.
    The frozen evidence seed randomly assigns the pre-generated archetypes to
    client identities.
    """
    domain = str(domain).lower()
    if domain not in CRITICAL_FACTORS_BY_DOMAIN:
        raise ValueError(f"Unsupported domain: {domain!r}")

    n = len(client_ids)
    rng = np.random.default_rng(int(evidence_seed))

    critical_policy = CRITICAL_FACTORS_BY_DOMAIN[domain]
    critical_keys = {
        (dim, factor)
        for dim, factor_list in critical_policy.items()
        for factor in factor_list
    }

    def make_documentary_profile(role: str, variant: int):
        factors = {
            dim: {}
            for dim in DOCUMENTARY_DIMS
        }

        if role == "DIRECT_STRONG":
            # Strong but not uniformly perfect. Every factor is at least 4,
            # while some values reach 5 according to a deterministic variant.
            for dim in DOCUMENTARY_DIMS:
                for j, factor in enumerate(FACTOR_NAMES[dim]):
                    minimum = float(factor_minima[dim][factor])
                    base = max(4.0, minimum)
                    bonus = 1.0 if ((j + variant + len(dim)) % 3 == 0) else 0.0
                    factors[dim][factor] = float(min(5.0, base + bonus))

        elif role == "REVIEW_RECOVERABLE":
            # Start from moderate evidence: all documentary dimensions remain
            # safely above the 2.5 floor without making HPS automatically high.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = 3.0

            critical_sequence = [
                (dim, factor)
                for dim, factor_list in critical_policy.items()
                for factor in factor_list
            ]

            # Put critical factors at their policy adequacy references.
            for dim, factor in critical_sequence:
                factors[dim][factor] = float(
                    factor_minima[dim][factor]
                )

            # Intentionally allow ONE critical factor to fall one point below
            # its individual minimum, while another critical factor with room
            # is strengthened. This is exactly the compensatory situation that
            # Automated Review is intended to resolve.
            high_min = [
                (dim, factor)
                for dim, factor in critical_sequence
                if float(factor_minima[dim][factor]) >= 4.0
            ]
            lower_min = [
                (dim, factor)
                for dim, factor in critical_sequence
                if float(factor_minima[dim][factor]) <= 3.0
            ]

            if high_min and lower_min:
                weak_key = high_min[variant % len(high_min)]
                strong_key = lower_min[variant % len(lower_min)]

                weak_min = float(
                    factor_minima[weak_key[0]][weak_key[1]]
                )
                strong_min = float(
                    factor_minima[strong_key[0]][strong_key[1]]
                )

                factors[weak_key[0]][weak_key[1]] = float(
                    max(0.0, weak_min - 1.0)
                )
                factors[strong_key[0]][strong_key[1]] = float(
                    min(5.0, strong_min + 2.0)
                )

            # Small documentary variation between the two recoverable bundles.
            if variant % 2 == 1:
                factors["dim3"][FACTOR_NAMES["dim3"][0]] = 4.0

        elif role == "REVIEW_LIMITED":
            # Preserve dimensions at/above 2.5 and keep HPS in/near the review
            # region, but make the average critical evidence too weak.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    if dim in critical_policy and (dim, factor) not in critical_keys:
                        factors[dim][factor] = 4.0
                    else:
                        factors[dim][factor] = 3.0

            for dim, factor_list in critical_policy.items():
                for factor in factor_list:
                    minimum = float(factor_minima[dim][factor])
                    factors[dim][factor] = float(
                        max(2.0, minimum - 1.0)
                    )

            if variant % 2 == 1:
                factors["dim3"][FACTOR_NAMES["dim3"][0]] = 4.0

        elif role == "LOW_HPS_WEAK":
            # Each documentary dimension averages >=2.5, but remains weak.
            # This isolates the HPS<3.0 rejection path from the dimension floor.
            for dim in DOCUMENTARY_DIMS:
                names = list(FACTOR_NAMES[dim])
                values = [2.0] * len(names)
                n_three = (len(names) + 1) // 2
                for j in range(n_three):
                    values[j] = 3.0

                for factor, value in zip(names, values):
                    factors[dim][factor] = float(value)

        elif role == "DIMENSION_FLOOR_WEAK":
            # General evidence is moderate, but Timeliness is deliberately below
            # the 2.5 dimension trustworthiness floor.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = 3.0

            for factor in FACTOR_NAMES["dim4"]:
                factors["dim4"][factor] = 2.0

        else:
            raise ValueError(f"Unknown evidence role: {role!r}")

        return factors

    # For the fixed K=10 experiments, pre-specify a balanced governance
    # branch-coverage set:
    #   4 direct-strong
    #   2 review-recoverable
    #   2 review-limited
    #   1 low-HPS weak
    #   1 dimension-floor weak
    #
    # This is not a post-hoc selection by client identity. The ten bundles are
    # generated first and then randomly assigned to clients using evidence_seed.
    if n == 10:
        bundle_specs = [
            ("DIRECT_STRONG", 0),
            ("DIRECT_STRONG", 1),
            ("DIRECT_STRONG", 2),
            ("DIRECT_STRONG", 3),
            ("REVIEW_RECOVERABLE", 0),
            ("REVIEW_RECOVERABLE", 1),
            ("REVIEW_LIMITED", 0),
            ("REVIEW_LIMITED", 1),
            ("LOW_HPS_WEAK", 0),
            ("DIMENSION_FLOOR_WEAK", 0),
        ]
    else:
        # Generic fallback for non-10-client studies: cycle through the same
        # archetypes without conditioning on client data or model performance.
        archetypes = [
            "DIRECT_STRONG",
            "REVIEW_RECOVERABLE",
            "REVIEW_LIMITED",
            "LOW_HPS_WEAK",
            "DIMENSION_FLOOR_WEAK",
        ]
        bundle_specs = [
            (archetypes[j % len(archetypes)], j // len(archetypes))
            for j in range(n)
        ]

    bundles = []
    for b, (role, variant) in enumerate(bundle_specs):
        factors = make_documentary_profile(role, int(variant))

        bundles.append({
            "bundle_id": f"E{b+1:02d}",
            "profile": role,
            "scenario_role": role,
            "profile_variant": int(variant),
            "factors": factors,
        })

    # Randomly assign already-generated governance bundles to client identities.
    assignment = rng.permutation(n)

    evidence_by_client = {}
    rows = []

    for client_pos, cid in enumerate(client_ids):
        bundle = bundles[int(assignment[client_pos])]

        evidence_by_client[cid] = {
            dim: dict(bundle["factors"][dim])
            for dim in DOCUMENTARY_DIMS
        }

        for dim in DOCUMENTARY_DIMS:
            for factor, score in bundle["factors"][dim].items():
                min_rank = float(
                    factor_minima[dim][factor]
                )

                rows.append({
                    "client":
                        str(cid),
                    "bundle_id":
                        bundle["bundle_id"],
                    "evidence_profile":
                        bundle["profile"],
                    "scenario_role":
                        bundle["scenario_role"],
                    "profile_variant":
                        int(bundle["profile_variant"]),
                    "dimension":
                        dim,
                    "dimension_name":
                        DIMENSION_NAMES[dim],
                    "factor":
                        factor,
                    "evidence_source_type":
                        "CONTROLLED_DOCUMENTARY_EVIDENCE",
                    "verified_rubric_level_0_5":
                        float(score),
                    "rubric_score_0_5":
                        float(score),
                    "server_mapped_score_0_5":
                        float(score),
                    "rubric_descriptor":
                        rubric_descriptor(
                            dim,
                            factor,
                            score,
                        ),
                    "human_role":
                        "VERIFY_EVIDENCE_ONLY",
                    "score_assignment":
                        "DETERMINISTIC_SERVER_MAPPING_FROM_VERIFIED_RUBRIC_LEVEL",
                    "admission_decision_by":
                        "SERVER_POLICY",
                    "adequacy_min_rank":
                        min_rank,
                    "adequacy_min_normalized":
                        min_rank / MAX_FACTOR_SCORE,
                    "meets_factor_adequacy":
                        bool(float(score) >= min_rank),
                    "evidence_artifact_id": (
                        f"{cid}-{bundle['bundle_id']}-{dim}-{factor}"
                    ),
                    "validation_status":
                        "CONTROLLED_SCENARIO_EVIDENCE",
                    "evidence_seed":
                        int(evidence_seed),
                    "domain":
                        domain,
                })

    return evidence_by_client, pd.DataFrame(rows)

def dimension_scores_from_factors(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, float]:
    """
    Client-specific HPS dimension scores (0..5):
    mean of the observed factor rubric scores within each dimension.
    """
    out = {}
    for dim in FACTOR_NAMES:
        vals = [
            float(v)
            for v in factors.get(dim, {}).values()
            if np.isfinite(float(v))
        ]
        out[dim] = float(np.mean(vals)) if vals else 0.0
    return out


def compute_hps(
    dimensions: Dict[str, float],
    weights: Dict[str, float] = WEIGHTS_PSCORE_DEFAULT,
) -> float:
    return float(
        sum(float(weights[d]) * float(dimensions[d]) for d in weights)
    )


def client_dimension_adequacy(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, float]:
    """
    Client ACHIEVED adequacy, not WAC.

    A_i,d = mean(client factor ranks in dimension d) / 5.
    """
    scores = {}
    for dim in FACTOR_NAMES:
        vals = [
            float(v)
            for v in factors.get(dim, {}).values()
            if np.isfinite(float(v))
        ]
        scores[dim] = (
            float(np.mean(vals) / MAX_FACTOR_SCORE)
            if vals else 0.0
        )
    return scores


def client_adequacy_score(
    factors: Dict[str, Dict[str, float]]
) -> Tuple[Dict[str, float], float]:
    """
    Legacy audit helper retained for backward comparability only.

    v16.7 does NOT use this value for admission. Review decisions use the
    client Critical WAC_i versus the domain Global Critical WAC.
    """
    by_dim = client_dimension_adequacy(factors)
    cas = float(np.mean(list(by_dim.values())))
    return by_dim, cas


def all_zero_score_factors(
    factors: Dict[str, Dict[str, float]]
) -> List[str]:
    """Audit-only zero list. Zero is not a separate admission rule in v16.7."""
    zeros = []
    for dim in FACTOR_NAMES:
        for factor, value in factors.get(dim, {}).items():
            if np.isfinite(float(value)) and float(value) <= 0.0:
                zeros.append(f"{dim}.{factor}")
    return zeros


def derive_global_critical_wac(domain: str) -> float:
    """
    Domain Global Critical WAC = mean(minimum critical adequacy rank / 5).

    This is a policy reference used only to resolve the HPS Review band.
    """
    domain = str(domain).lower()
    vals = []
    for dim, factors_ in CRITICAL_FACTORS_BY_DOMAIN[domain].items():
        for factor in factors_:
            vals.append(
                float(DOMAIN_FACTOR_MINIMA[domain][dim][factor])
                / MAX_FACTOR_SCORE
            )
    if not vals:
        raise ValueError(f"No critical factors for domain={domain!r}")
    return float(np.mean(vals))


def critical_factor_audit(
    factors: Dict[str, Dict[str, float]],
    domain: str,
) -> Dict[str, Any]:
    """
    Compute the client Critical WAC_i and the strict individual critical gate.

    Direct Auto-Accept:
        every critical factor >= its own factor-specific adequacy minimum.

    Automated Review:
        uses the overall average Critical WAC_i, allowing compensation across
        critical factors while preserving the dimension and HPS floors.
    """
    domain = str(domain).lower()

    scores = {}
    minima = {}
    missing = []
    below_adequacy = []

    for dim, factor_list in CRITICAL_FACTORS_BY_DOMAIN[domain].items():
        for factor in factor_list:
            key = f"{dim}.{factor}"
            minimum = float(
                DOMAIN_FACTOR_MINIMA[domain][dim][factor]
            )
            minima[key] = minimum

            if factor not in factors.get(dim, {}):
                missing.append(key)
                continue

            value = float(factors[dim][factor])

            if not np.isfinite(value):
                missing.append(key)
                continue

            scores[key] = value

            if value < minimum:
                below_adequacy.append(
                    f"{key}:{value:.1f}<{minimum:.1f}"
                )

    expected_n = sum(
        len(v)
        for v in CRITICAL_FACTORS_BY_DOMAIN[domain].values()
    )

    if missing or len(scores) != expected_n:
        return {
            "scores": scores,
            "minima": minima,
            "missing": missing,
            "critical_count": expected_n,
            "critical_wac_i": np.nan,
            "critical_mean_score_0_5": np.nan,
            "global_critical_wac":
                float(derive_global_critical_wac(domain)),
            "all_critical_meet_adequacy": False,
            "critical_below_adequacy": below_adequacy,
        }

    values = np.array(
        list(scores.values()),
        dtype=float,
    )

    return {
        "scores": scores,
        "minima": minima,
        "missing": [],
        "critical_count": expected_n,
        "critical_wac_i":
            float(np.mean(values / MAX_FACTOR_SCORE)),
        "critical_mean_score_0_5":
            float(np.mean(values)),
        "global_critical_wac":
            float(derive_global_critical_wac(domain)),
        "all_critical_meet_adequacy":
            bool(len(below_adequacy) == 0),
        "critical_below_adequacy":
            below_adequacy,
    }

def dimension_floor_failures(
    dimensions: Dict[str, float],
    floor: float = DIMENSION_MIN_FLOOR,
) -> List[str]:
    """Return averaged HPS dimensions below the minimum floor."""
    return [
        dim for dim, value in dimensions.items()
        if (not np.isfinite(float(value))) or float(value) < float(floor)
    ]

def factor_adequacy_attainment(
    factors: Dict[str, Dict[str, float]],
    factor_minima: Dict[str, Dict[str, float]],
) -> Dict[str, Any]:
    total = 0
    passed = 0
    per_dim = {}

    for dim in FACTOR_NAMES:
        dim_total = 0
        dim_passed = 0

        for factor in FACTOR_NAMES[dim]:
            total += 1
            dim_total += 1

            score = float(factors[dim][factor])
            threshold = float(factor_minima[dim][factor])

            if score >= threshold:
                passed += 1
                dim_passed += 1

        per_dim[dim] = {
            "passed": dim_passed,
            "total": dim_total,
            "fraction": float(dim_passed / max(1, dim_total)),
        }

    return {
        "passed": passed,
        "total": total,
        "fraction": float(passed / max(1, total)),
        "per_dim": per_dim,
    }


def tadp_decision(
    factors: Dict[str, Dict[str, float]],
    domain: str,
    good_cut: float = GOOD_CUT,
    high_cut: float = HIGH_CUT,
) -> Dict[str, Any]:
    """
    v16.7 final fully automated admission policy.

    Flow:
      EVERY dimension >= 2.5?
          NO -> AUTO-REJECT

      HPS >= 3.0?
          NO -> AUTO-REJECT

      HPS >= 3.5?
          YES:
              all critical factors >= own adequacy minima?
                  YES -> DIRECT AUTO-ACCEPT
                  NO  -> AUTOMATED REVIEW
          NO (3.0 <= HPS < 3.5):
              -> AUTOMATED REVIEW

      AUTOMATED REVIEW:
          Critical WAC_i >= Global Critical WAC
              -> ACCEPT AFTER REVIEW
          otherwise
              -> AUTO-REJECT
    """
    domain = str(domain).lower()

    factor_minima = DOMAIN_FACTOR_MINIMA[domain]
    dimension_wac = DOMAIN_DIMENSION_WAC[domain]
    global_wac = float(DOMAIN_GLOBAL_WAC[domain])
    global_critical_wac = float(
        derive_global_critical_wac(domain)
    )

    dims = dimension_scores_from_factors(factors)
    hps = compute_hps(dims)

    attainment = factor_adequacy_attainment(
        factors,
        factor_minima,
    )

    zeros = all_zero_score_factors(factors)

    dim_floor_failed = dimension_floor_failures(
        dims,
        floor=DIMENSION_MIN_FLOOR,
    )

    critical = critical_factor_audit(
        factors,
        domain,
    )

    critical_score_string = ";".join(
        (
            f"{key}={value:.1f}"
            f"(min={critical['minima'][key]:.1f})"
        )
        for key, value in critical["scores"].items()
    )

    base = {
        "hps": float(hps),
        "policy_dimension_wac": dimension_wac,
        "global_domain_wac": global_wac,
        "global_critical_wac": global_critical_wac,
        "critical_wac_i": critical["critical_wac_i"],
        "critical_wac_margin": (
            float(critical["critical_wac_i"])
            - global_critical_wac
            if np.isfinite(critical["critical_wac_i"])
            else np.nan
        ),
        "critical_mean_score_0_5":
            critical["critical_mean_score_0_5"],
        "critical_factor_count":
            int(critical["critical_count"]),
        "critical_scores":
            critical_score_string,
        "critical_missing":
            ";".join(critical["missing"]),
        "all_critical_meet_adequacy":
            bool(critical["all_critical_meet_adequacy"]),
        "critical_below_adequacy":
            ";".join(critical["critical_below_adequacy"]),
        "factor_adequacy_passed":
            int(attainment["passed"]),
        "factor_adequacy_total":
            int(attainment["total"]),
        "factor_adequacy_fraction":
            float(attainment["fraction"]),
        "dimensions":
            dims,
        "dimension_min_floor":
            float(DIMENSION_MIN_FLOOR),
        "dimension_floor_failures":
            ";".join(dim_floor_failed),
        # Audit only; zero is not a separate decision rule.
        "all_zero_factors":
            ";".join(zeros),
    }

    # Gate 1: every complete trustworthiness dimension must clear 2.5/5.
    if dim_floor_failed:
        return {
            **base,
            "decision_path":
                "DIMENSION_FLOOR",
            "initial_action":
                "AUTO_REJECT",
            "final_action":
                "REJECT",
            "status":
                "AUTO_REJECTED_DIMENSION_FLOOR",
            "reason": (
                f"At least one averaged dimension is below "
                f"{DIMENSION_MIN_FLOOR:.1f}/5: "
                + ";".join(
                    f"{dim}={dims[dim]:.3f}"
                    for dim in dim_floor_failed
                )
            ),
        }

    # Missing/non-finite critical evidence is fail-closed in this experiment.
    if critical["missing"]:
        return {
            **base,
            "decision_path":
                "MISSING_CRITICAL_EVIDENCE",
            "initial_action":
                "AUTO_REJECT",
            "final_action":
                "REJECT",
            "status":
                "AUTO_REJECTED_MISSING_CRITICAL_EVIDENCE",
            "reason": (
                "Missing/non-finite verified critical evidence: "
                + ";".join(critical["missing"])
            ),
        }

    # Gate 2: overall HPS lower bound.
    if hps < float(good_cut):
        return {
            **base,
            "decision_path":
                "LOW_HPS",
            "initial_action":
                "AUTO_REJECT",
            "final_action":
                "REJECT",
            "status":
                "AUTO_REJECTED_LOW_HPS",
            "reason":
                f"HPS {hps:.3f} < {good_cut:.3f}",
        }

    # High-HPS strict direct route.
    if hps >= float(high_cut):
        if critical["all_critical_meet_adequacy"]:
            return {
                **base,
                "decision_path":
                    "DIRECT_AUTO_ACCEPT",
                "initial_action":
                    "AUTO_ACCEPT",
                "final_action":
                    "ACCEPT",
                "status":
                    "DIRECT_AUTO_ACCEPTED",
                "reason": (
                    f"HPS {hps:.3f} >= {high_cut:.3f}; all critical "
                    "factors meet their individual adequacy requirements"
                ),
            }

        # High HPS but failed strict critical gate:
        # fall back to the same automated review rather than immediate rejection.
        review_origin = (
            "HIGH_HPS_CRITICAL_FALLBACK"
        )

    else:
        # 3.0 <= HPS < 3.5
        review_origin = (
            "HPS_REVIEW_BAND"
        )

    # Automated Review for both origins.
    review_pass = bool(
        float(critical["critical_wac_i"])
        >= global_critical_wac
    )

    if review_pass:
        return {
            **base,
            "decision_path":
                review_origin,
            "initial_action":
                "AUTOMATED_REVIEW",
            "final_action":
                "ACCEPT",
            "status":
                "ACCEPTED_AFTER_AUTOMATED_REVIEW",
            "reason": (
                f"{review_origin}: Critical WAC_i "
                f"{critical['critical_wac_i']:.3f} >= "
                f"Global Critical WAC {global_critical_wac:.3f}"
            ),
        }

    return {
        **base,
        "decision_path":
            review_origin,
        "initial_action":
            "AUTOMATED_REVIEW",
        "final_action":
            "REJECT",
        "status":
            "AUTO_REJECTED_REVIEW_CRITICAL_WAC",
        "reason": (
            f"{review_origin}: Critical WAC_i "
            f"{critical['critical_wac_i']:.3f} < "
            f"Global Critical WAC {global_critical_wac:.3f}"
        ),
    }

def build_tadp_governance(
    client_ids: List[str],
    documentary_evidence: Dict[str, Dict[str, Dict[str, float]]],
    dq_scores: Dict[str, Dict[str, float]],
    run: int,
    evidence_seed: int,
    domain: str,
) -> pd.DataFrame:
    domain = str(domain).lower()
    rows = []

    for cid in client_ids:
        factors = {
            dim: dict(documentary_evidence[cid][dim])
            for dim in DOCUMENTARY_DIMS
        }

        factors["dim2"] = {
            name: float(dq_scores[cid][name])
            for name in FACTOR_NAMES["dim2"]
        }

        d = tadp_decision(
            factors,
            domain=domain,
        )

        row = {
            "run": int(run),
            "domain": domain,
            "evidence_seed": int(evidence_seed),
            "client": str(cid),
            "hps": float(d["hps"]),
            "global_domain_wac":
                float(d["global_domain_wac"]),
            "global_critical_wac":
                float(d["global_critical_wac"]),
            "critical_wac_i": (
                float(d["critical_wac_i"])
                if np.isfinite(d["critical_wac_i"])
                else np.nan
            ),
            "critical_wac_margin": (
                float(d["critical_wac_margin"])
                if np.isfinite(d["critical_wac_margin"])
                else np.nan
            ),
            "critical_mean_score_0_5": (
                float(d["critical_mean_score_0_5"])
                if np.isfinite(d["critical_mean_score_0_5"])
                else np.nan
            ),
            "critical_factor_count":
                int(d["critical_factor_count"]),
            "critical_scores":
                d["critical_scores"],
            "critical_missing":
                d["critical_missing"],
            "all_critical_meet_adequacy":
                bool(d["all_critical_meet_adequacy"]),
            "critical_below_adequacy":
                d["critical_below_adequacy"],
            "dimension_min_floor":
                float(d["dimension_min_floor"]),
            "dimension_floor_failures":
                d["dimension_floor_failures"],
            "factor_adequacy_passed":
                int(d["factor_adequacy_passed"]),
            "factor_adequacy_total":
                int(d["factor_adequacy_total"]),
            "factor_adequacy_fraction":
                float(d["factor_adequacy_fraction"]),
            "decision_path":
                d["decision_path"],
            "initial_action":
                d["initial_action"],
            "final_action":
                d["final_action"],
            "status":
                d["status"],
            "reason":
                d["reason"],
            "all_zero_factors":
                d["all_zero_factors"],
        }

        for dim, value in d["dimensions"].items():
            row[
                f"{dim}_score_0_5"
            ] = float(value)

        for dim, value in d["policy_dimension_wac"].items():
            row[
                f"{dim}_policy_wac"
            ] = float(value)

        rows.append(row)

    return pd.DataFrame(rows)

def accepted_tadp_vr(governance_df: pd.DataFrame) -> List[str]:
    return governance_df.loc[
        governance_df["final_action"].eq("ACCEPT"), "client"
    ].astype(str).tolist()


def accepted_tadp_sda(
    governance_df: pd.DataFrame
) -> List[str]:
    """
    Select one best TADP-eligible client for SDA from already accepted clients.
    """
    eligible = governance_df[
        governance_df["final_action"].eq("ACCEPT")
    ].copy()

    if eligible.empty:
        raise RuntimeError(
            "TADP-SDA cannot select a client: no TADP-eligible client."
        )

    eligible["direct_accept_priority"] = (
        eligible["status"]
        .eq("DIRECT_AUTO_ACCEPTED")
        .astype(int)
    )

    eligible = eligible.sort_values(
        [
            "hps",
            "direct_accept_priority",
            "critical_wac_i",
            "client",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    )

    return [
        str(
            eligible.iloc[0]["client"]
        )
    ]

def build_full_factor_evidence_table(
    client_ids: List[str],
    documentary_evidence: Dict[str, Dict[str, Dict[str, float]]],
    dq_scores: Dict[str, Dict[str, float]],
    dq_audit: pd.DataFrame,
    evidence_seed: int,
    domain: str,
) -> pd.DataFrame:
    """
    Full 28-factor evidence audit.

    Critical factors are flagged with their own policy adequacy minima.
    Direct Auto-Accept uses these individual minima; Automated Review uses the
    aggregate Critical WAC_i.
    """
    domain = str(domain).lower()
    factor_minima = DOMAIN_FACTOR_MINIMA[domain]
    critical_policy = CRITICAL_FACTORS_BY_DOMAIN[domain]

    dq_index = dq_audit.copy()
    dq_index["client"] = dq_index["client"].astype(str)
    dq_index = dq_index.set_index(
        "client",
        drop=False,
    )

    def policy_fields(dim, factor):
        is_critical = bool(
            factor in critical_policy.get(
                dim,
                [],
            )
        )

        return {
            "is_critical_factor":
                is_critical,
            "critical_adequacy_min_rank": (
                float(factor_minima[dim][factor])
                if is_critical
                else np.nan
            ),
            "critical_adequacy_min_normalized": (
                float(factor_minima[dim][factor])
                / MAX_FACTOR_SCORE
                if is_critical
                else np.nan
            ),
        }

    rows = []

    for cid in client_ids:
        cid = str(cid)

        for dim in DOCUMENTARY_DIMS:
            for factor in FACTOR_NAMES[dim]:
                score = float(
                    documentary_evidence[cid][dim][factor]
                )
                min_rank = float(
                    factor_minima[dim][factor]
                )

                rows.append({
                    "client": cid,
                    "domain": domain,
                    "dimension": dim,
                    "dimension_name":
                        DIMENSION_NAMES[dim],
                    "factor": factor,
                    "factor_source":
                        "CONTROLLED_DOCUMENTARY_EVIDENCE",
                    "raw_measured_value":
                        np.nan,
                    "rubric_score_0_5":
                        score,
                    "rubric_descriptor":
                        rubric_descriptor(
                            dim,
                            factor,
                            score,
                        ),
                    "adequacy_min_rank":
                        min_rank,
                    "adequacy_min_normalized":
                        min_rank / MAX_FACTOR_SCORE,
                    "meets_factor_adequacy":
                        bool(score >= min_rank),
                    **policy_fields(
                        dim,
                        factor,
                    ),
                    "evidence_seed":
                        int(evidence_seed),
                })

        for factor in FACTOR_NAMES["dim2"]:
            score = float(
                dq_scores[cid][factor]
            )
            min_rank = float(
                factor_minima["dim2"][factor]
            )
            raw_col = DQ_RAW_METRIC_BY_FACTOR[
                factor
            ]

            raw_value = (
                float(
                    dq_index.loc[
                        cid,
                        raw_col,
                    ]
                )
                if raw_col in dq_index.columns
                else np.nan
            )

            rows.append({
                "client": cid,
                "domain": domain,
                "dimension": "dim2",
                "dimension_name":
                    DIMENSION_NAMES["dim2"],
                "factor": factor,
                "factor_source":
                    "MACHINE_MEASURED_TRAIN_ONLY",
                "raw_measured_value":
                    raw_value,
                "rubric_score_0_5":
                    score,
                "rubric_descriptor":
                    rubric_descriptor(
                        "dim2",
                        factor,
                        score,
                    ),
                "adequacy_min_rank":
                    min_rank,
                "adequacy_min_normalized":
                    min_rank / MAX_FACTOR_SCORE,
                "meets_factor_adequacy":
                    bool(score >= min_rank),
                **policy_fields(
                    "dim2",
                    factor,
                ),
                "evidence_seed":
                    int(evidence_seed),
            })

    out = pd.DataFrame(
        rows
    )

    expected_rows = len(client_ids) * 28

    if len(out) != expected_rows:
        raise RuntimeError(
            f"Full evidence matrix should contain "
            f"{expected_rows} rows; found {len(out)}."
        )

    return out

# ======================================================================================
# GREAT EXPECTATIONS (GX CORE) — NATIVE VALIDATOR BASELINE
# ======================================================================================
# GX is used here in its native role: validate each client's TRAIN-only data against
# a predefined Expectation Suite and use the suite-level success flag as PASS/FAIL.
# There is NO ranking, NO forced-K selection, and NO HPS/WAC information in this
# baseline. Great Expectations reports suite success=True only when all configured
# Expectations pass. The suite deliberately uses common technical checks only:
#   1) schema, 2) datatype consistency, 3) required-value ranges,
#   4) missingness, 5) duplicates / unique IDs, 6) label/domain validity,
#   7) structural integrity.
# Distribution checks are intentionally omitted because non-IID client distributions
# are expected in federated learning and should not by themselves constitute failure.
GX_CORE_VERSION = "1.23.0"
GX_VALUE_MOSTLY = 0.99
GX_NONNULL_REFERENCE_MIN = 0.80
GX_NONNULL_TOLERANCE = 0.15


def ensure_great_expectations():
    """Import pinned GX Core; install once in Colab if unavailable."""
    try:
        import great_expectations as gx
        return gx
    except ImportError:
        import subprocess
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            f"great_expectations=={GX_CORE_VERSION}",
        ])
        import great_expectations as gx
        return gx


def _gx_meta(category: str, name: str) -> Dict[str, Any]:
    return {
        "dq_category": str(category),
        "check_name": str(name),
        "baseline": "GX_NATIVE_VALIDATOR",
    }


def _gx_add(suite, specs: List[Dict[str, Any]], expectation, category: str, name: str):
    suite.add_expectation(expectation)
    specs.append({"category": str(category), "name": str(name)})


def _aggregate_nonnull_reference(
    client_frames: Dict[str, pd.DataFrame],
    columns: List[str],
) -> Dict[str, float]:
    total_rows = float(sum(len(df) for df in client_frames.values()))
    out = {}
    for c in columns:
        nonnull = sum(int(df[c].notna().sum()) for df in client_frames.values() if c in df.columns)
        out[c] = float(nonnull / max(1.0, total_rows))
    return out


def build_gx_healthcare_reference(client_frames: Dict[str, pd.DataFrame], preprocessor: Any) -> Dict[str, Any]:
    first = next(iter(client_frames.values()))

    # GX datatype validation is intentionally independent of TADP's model
    # preprocessing type inference.  The preprocessor labels a feature numeric
    # when >=95% of pooled TRAIN non-missing values are parseable; reusing that
    # inferred list inside GX and then requiring 99% parseability per client
    # creates an artificial contradiction.  GX therefore validates datatype
    # consistency only for fields whose numeric meaning is explicit in the
    # Diabetes data schema.
    known_numeric_fields = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]
    gx_numeric_cols = [c for c in known_numeric_fields if c in first.columns]

    return {
        "original_columns": list(first.columns),
        "numeric_cols": gx_numeric_cols,
        "feature_cols": list(preprocessor.feature_cols),
        "nonnull_reference": _aggregate_nonnull_reference(
            client_frames, list(preprocessor.feature_cols)
        ),
    }


def build_gx_healthcare_validation_frame(df: pd.DataFrame, preprocessor: Any) -> pd.DataFrame:
    """GX validation view; raw TRAIN records remain the source of all checks."""
    out = df.copy()
    for c in preprocessor.numeric_cols:
        raw = df[c]
        parsed = pd.to_numeric(raw, errors="coerce")
        type_ok = raw.isna() | parsed.notna()
        out[c] = parsed.astype(float)
        out[f"__gx_type_ok__{c}"] = type_ok.astype(np.int8)
    return out


def build_gx_healthcare_suite(gx, reference: Dict[str, Any]):
    """Simple native GX technical-validation suite for the Diabetes TRAIN shards."""
    gxe = gx.expectations
    suite = gx.ExpectationSuite(name="tadp_healthcare_gx_native_suite")
    specs = []
    cols = reference["original_columns"]

    # 1) Schema.
    _gx_add(
        suite, specs,
        gxe.ExpectTableColumnsToMatchSet(
            column_set=cols,
            exact_match=False,
            severity="critical",
            meta=_gx_meta("Schema", "required_column_set"),
        ),
        "Schema", "required_column_set",
    )

    # 2) Datatype consistency for columns inferred as numeric from TRAIN only.
    for c in reference["numeric_cols"]:
        diag = f"__gx_type_ok__{c}"
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=diag,
                value_set=[1],
                mostly=GX_VALUE_MOSTLY,
                severity="critical",
                meta=_gx_meta("Datatype Consistency", f"{c}_numeric_parseability"),
            ),
            "Datatype Consistency", f"{c}_numeric_parseability",
        )

    # 3) Required-value ranges for known count/duration fields.
    nonnegative_fields = [
        "num_lab_procedures", "num_procedures", "num_medications",
        "number_outpatient", "number_emergency", "number_inpatient",
        "number_diagnoses",
    ]
    for c in nonnegative_fields:
        if c in cols:
            _gx_add(
                suite, specs,
                gxe.ExpectColumnValuesToBeBetween(
                    column=c, min_value=0.0, mostly=GX_VALUE_MOSTLY,
                    severity="critical",
                    meta=_gx_meta("Required Value Ranges", f"{c}_nonnegative"),
                ),
                "Required Value Ranges", f"{c}_nonnegative",
            )
    if "time_in_hospital" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeBetween(
                column="time_in_hospital", min_value=1.0, max_value=14.0,
                mostly=GX_VALUE_MOSTLY,
                severity="critical",
                meta=_gx_meta("Required Value Ranges", "time_in_hospital_range"),
            ),
            "Required Value Ranges", "time_in_hospital_range",
        )

    # 4) Missingness. Only columns that are substantially populated in the frozen
    # TRAIN reference are treated as required-enough for a missingness expectation.
    for c, global_nonnull in reference["nonnull_reference"].items():
        if c not in cols or float(global_nonnull) < GX_NONNULL_REFERENCE_MIN:
            continue
        minimum = max(0.70, float(global_nonnull) - GX_NONNULL_TOLERANCE)
        _gx_add(
            suite, specs,
            gxe.ExpectColumnProportionOfNonNullValuesToBeBetween(
                column=c, min_value=float(minimum), max_value=1.0,
                severity="warning",
                meta=_gx_meta("Missingness", f"{c}_nonnull"),
            ),
            "Missingness", f"{c}_nonnull",
        )

    # 5) Duplicates / unique identifiers.
    if "encounter_id" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeUnique(
                column="encounter_id",
                severity="critical",
                meta=_gx_meta("Duplicates / Unique IDs", "encounter_id_unique"),
            ),
            "Duplicates / Unique IDs", "encounter_id_unique",
        )

    # 6) Labels / domain validity.
    if "_target" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column="_target", value_set=[0, 1, 2],
                severity="critical",
                meta=_gx_meta("Labels / Domain Validity", "target_domain"),
            ),
            "Labels / Domain Validity", "target_domain",
        )

    # 7) Structural integrity.
    if "_row_id" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeUnique(
                column="_row_id",
                severity="critical",
                meta=_gx_meta("Structural Integrity", "row_id_unique"),
            ),
            "Structural Integrity", "row_id_unique",
        )
    _gx_add(
        suite, specs,
        gxe.ExpectTableRowCountToBeBetween(
            min_value=100,
            severity="warning",
            meta=_gx_meta("Structural Integrity", "minimum_client_rows"),
        ),
        "Structural Integrity", "minimum_client_rows",
    )
    return suite, specs


def build_gx_cifar_metadata(X: np.ndarray, y: np.ndarray) -> pd.DataFrame:
    X = np.asarray(X)
    y = np.asarray(y).reshape(-1)
    if X.ndim != 4:
        raise RuntimeError(f"Expected CIFAR tensor [N,H,W,C], got shape={X.shape}")
    finite = np.isfinite(X.astype(np.float32)).reshape(len(X), -1).all(axis=1)
    flat = X.reshape(len(X), -1)
    return pd.DataFrame({
        "sample_id": np.arange(len(X), dtype=np.int64),
        "label": y.astype(np.int32),
        "height": np.full(len(X), X.shape[1], dtype=np.int32),
        "width": np.full(len(X), X.shape[2], dtype=np.int32),
        "channels": np.full(len(X), X.shape[3], dtype=np.int32),
        "dtype_ok": np.full(len(X), int(X.dtype == np.uint8), dtype=np.int8),
        "finite": finite.astype(np.int8),
        "pixel_min": flat.min(axis=1).astype(float),
        "pixel_max": flat.max(axis=1).astype(float),
    })


def build_gx_cifar_reference(client_raw: Dict[str, Tuple[np.ndarray, np.ndarray]]) -> Dict[str, Any]:
    # Native validator uses fixed technical constraints; no distribution reference needed.
    return {}


def build_gx_cifar_suite(gx, reference: Dict[str, Any]):
    """Simple native GX technical-validation suite for CIFAR-10 client metadata."""
    gxe = gx.expectations
    suite = gx.ExpectationSuite(name="tadp_cifar10_gx_native_suite")
    specs = []
    columns = [
        "sample_id", "label", "height", "width", "channels",
        "dtype_ok", "finite", "pixel_min", "pixel_max",
    ]

    _gx_add(
        suite, specs,
        gxe.ExpectTableColumnsToMatchSet(
            column_set=columns, exact_match=True,
            meta=_gx_meta("Schema", "image_metadata_schema"),
        ),
        "Schema", "image_metadata_schema",
    )
    for c in columns:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToNotBeNull(
                column=c,
                severity="warning",
                meta=_gx_meta("Missingness", f"{c}_not_null"),
            ),
            "Missingness", f"{c}_not_null",
        )
    for c, value in [("height", 32), ("width", 32), ("channels", 3), ("dtype_ok", 1), ("finite", 1)]:
        category = "Datatype Consistency" if c in {"dtype_ok", "finite"} else "Structural Integrity"
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=c, value_set=[value],
                meta=_gx_meta(category, f"{c}_constraint"),
            ),
            category, f"{c}_constraint",
        )
    for c in ["pixel_min", "pixel_max"]:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeBetween(
                column=c, min_value=0.0, max_value=255.0,
                meta=_gx_meta("Required Value Ranges", f"{c}_valid_range"),
            ),
            "Required Value Ranges", f"{c}_valid_range",
        )
    _gx_add(
        suite, specs,
        gxe.ExpectColumnValuesToBeInSet(
            column="label", value_set=list(range(10)),
            meta=_gx_meta("Labels / Domain Validity", "label_domain"),
        ),
        "Labels / Domain Validity", "label_domain",
    )
    _gx_add(
        suite, specs,
        gxe.ExpectColumnValuesToBeUnique(
            column="sample_id",
            meta=_gx_meta("Duplicates / Unique IDs", "sample_id_unique"),
        ),
        "Duplicates / Unique IDs", "sample_id_unique",
    )
    _gx_add(
        suite, specs,
        gxe.ExpectTableRowCountToBeBetween(
            min_value=100,
            severity="warning",
            meta=_gx_meta("Structural Integrity", "minimum_client_images"),
        ),
        "Structural Integrity", "minimum_client_images",
    )
    return suite, specs


def ge_governance(
    client_data: Dict[str, Any],
    client_ids: List[str],
    accept_count: Optional[int] = None,  # ignored; retained only for call compatibility
    domain: str = "healthcare",
    preprocessor: Optional[Any] = None,
    dq_scores: Optional[Dict[str, Dict[str, float]]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run native GX suite validation and PASS only clients whose entire suite succeeds."""
    domain = str(domain).lower()
    gx = ensure_great_expectations()
    context = gx.get_context(mode="ephemeral")
    datasource = context.data_sources.add_pandas(name=f"tadp_gx_native_{domain}_datasource")
    asset = datasource.add_dataframe_asset(name=f"tadp_gx_native_{domain}_asset")
    batch_definition = asset.add_batch_definition_whole_dataframe(name="whole_client_train_partition")

    if domain == "healthcare":
        if preprocessor is None:
            raise ValueError("Healthcare GX native baseline requires TRAIN-only preprocessor metadata.")
        reference = build_gx_healthcare_reference(client_data, preprocessor)
        suite, specs = build_gx_healthcare_suite(gx, reference)
        make_frame = lambda cid: build_gx_healthcare_validation_frame(client_data[cid], preprocessor)
    elif domain == "cifar10":
        reference = build_gx_cifar_reference(client_data)
        suite, specs = build_gx_cifar_suite(gx, reference)
        def make_frame(cid):
            X, y = client_data[cid]
            return build_gx_cifar_metadata(X, y)
    else:
        raise ValueError(f"Unsupported GX domain: {domain!r}")

    summary_rows, detail_rows = [], []
    for cid in client_ids:
        frame = make_frame(cid)
        batch = batch_definition.get_batch(batch_parameters={"dataframe": frame})
        validation = batch.validate(suite)
        results = list(validation.results)
        if len(results) != len(specs):
            raise RuntimeError(
                f"GX result/spec mismatch for client {cid}: {len(results)} vs {len(specs)}"
            )

        passed = 0
        critical_failures = 0
        warning_failures = 0
        info_failures = 0
        category_totals, category_passed = {}, {}
        observed_names = []

        for result in results:
            success = bool(result.success)
            passed += int(success)
            cfg = result.expectation_config
            meta = getattr(cfg, "meta", None) or {}
            name = str(meta.get("check_name", getattr(cfg, "type", "GX_EXPECTATION")))
            cat = str(meta.get("dq_category", "Technical Validation"))

            sev_obj = getattr(cfg, "severity", None)
            sev = getattr(sev_obj, "value", sev_obj)
            sev = str(sev if sev is not None else "critical").lower()
            if "." in sev:
                sev = sev.split(".")[-1]
            if sev not in {"critical", "warning", "info"}:
                sev = "critical"

            if not success:
                if sev == "critical":
                    critical_failures += 1
                elif sev == "warning":
                    warning_failures += 1
                else:
                    info_failures += 1

            observed_names.append(name)
            category_totals[cat] = category_totals.get(cat, 0) + 1
            category_passed[cat] = category_passed.get(cat, 0) + int(success)

            detail_rows.append({
                "client": str(cid),
                "domain": domain,
                "gx_version": GX_CORE_VERSION,
                "expectation_name": name,
                "dq_category": cat,
                "severity": sev,
                "success": success,
            })

        expected_names = sorted(str(x["name"]) for x in specs)
        if sorted(observed_names) != expected_names:
            raise RuntimeError(
                f"GX expectation identity mismatch for client {cid}: "
                f"expected={expected_names}, observed={sorted(observed_names)}"
            )

        total = len(specs)
        stats = getattr(validation, "statistics", {}) or {}
        suite_success = bool(validation.success)

        # GX-native severity-aware operational gate.
        # GX itself exposes the maximum failed severity for a Validation Result.
        # We use that native result rather than reconstructing the gate from a
        # custom ranking.  A failed Expectation execution is also treated by GX
        # as CRITICAL.
        max_failed_severity_obj = validation.get_max_severity_failure()
        if max_failed_severity_obj is None:
            max_failed_severity = "none"
        else:
            max_failed_severity = str(
                getattr(max_failed_severity_obj, "value", max_failed_severity_obj)
            ).lower()
            if "." in max_failed_severity:
                max_failed_severity = max_failed_severity.split(".")[-1]

        operational_valid = (max_failed_severity != "critical")

        row = {
            "client": str(cid),
            "gx_version": GX_CORE_VERSION,
            "gx_native_suite_success": suite_success,
            "gx_operational_valid": bool(operational_valid),
            "gx_critical_failures": int(critical_failures),
            "gx_warning_failures": int(warning_failures),
            "gx_info_failures": int(info_failures),
            "gx_max_failed_severity": max_failed_severity,
            "ge_expectations_passed": int(stats.get("successful_expectations", passed)),
            "ge_expectations_total": int(stats.get("evaluated_expectations", total)),
            "ge_pass_rate": float(
                stats.get("success_percent", 100.0 * passed / max(1, total))
            ) / 100.0,
            "ge_native_validation_class": (
                "GX_SUITE_PASS"
                if suite_success
                else (
                    "GX_WARNING_ONLY"
                    if operational_valid
                    else "GX_CRITICAL_FAILURE"
                )
            ),
            "ge_final_action": "ACCEPT" if operational_valid else "REJECT",
            "ge_policy": (
                "GX Core severity-aware validation; ACCEPT requires zero critical "
                "Expectation failures; warning/info failures are reported but do not "
                "exclude; no ranking and no forced-K selection"
            ),
        }

        for cat in sorted(category_totals):
            safe = re.sub(r"[^a-z0-9]+", "_", cat.lower()).strip("_")
            row[f"ge_{safe}_passed"] = int(category_passed.get(cat, 0))
            row[f"ge_{safe}_total"] = int(category_totals[cat])

        summary_rows.append(row)

    return pd.DataFrame(summary_rows), pd.DataFrame(detail_rows)


def score_lower_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [best_upper, score4_upper, score3_upper, score2_upper, score1_upper]
    value <= cuts[0] => 5; ... value <= cuts[4] => 1; else 0.
    """
    v = float(value)
    for score, upper in zip([5, 4, 3, 2, 1], cuts):
        if v <= float(upper):
            return float(score)
    return 0.0


def score_higher_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [score5_lower, score4_lower, score3_lower, score2_lower, score1_lower]
    """
    v = float(value)
    for score, lower in zip([5, 4, 3, 2, 1], cuts):
        if v >= float(lower):
            return float(score)
    return 0.0


def js_divergence(p, q, eps=1e-12) -> float:
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / max(eps, p.sum())
    q = q / max(eps, q.sum())
    m = 0.5 * (p + q)
    kl_pm = np.sum(np.where(p > 0, p * np.log((p + eps) / (m + eps)), 0.0))
    kl_qm = np.sum(np.where(q > 0, q * np.log((q + eps) / (m + eps)), 0.0))
    return float(0.5 * (kl_pm + kl_qm))


# ======================================================================================
# MODEL / METRICS / FL TRAINING
# ======================================================================================

def class_weight_dict(y: np.ndarray) -> Dict[int, float]:
    y = np.asarray(y, dtype=np.int32)
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(w) for c, w in zip(classes, weights)}


def model_parameter_bytes(model: keras.Model) -> int:
    return int(sum(np.asarray(w).nbytes for w in model.get_weights()))


def evaluate_model(
    model: keras.Model, X: np.ndarray, y: np.ndarray, n_classes: int
) -> Dict[str, float]:
    p = model.predict(X, batch_size=512, verbose=0)
    pred = np.argmax(p, axis=1)
    out = {
        "accuracy": float(accuracy_score(y, pred)),
        "precision_macro": float(
            precision_score(y, pred, average="macro", zero_division=0)
        ),
        "recall_macro": float(
            recall_score(y, pred, average="macro", zero_division=0)
        ),
        "f1_macro": float(
            f1_score(y, pred, average="macro", zero_division=0)
        ),
    }
    try:
        out["roc_auc_ovr_macro"] = float(
            roc_auc_score(y, p, multi_class="ovr", average="macro")
        )
    except Exception:
        out["roc_auc_ovr_macro"] = float("nan")
    return out



ROUND_PROGRESS_MONITOR_MAX_SAMPLES = 4096
ROUND_PROGRESS_POWER_W = 12.0


def build_train_monitor_subset(
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    max_samples: int = ROUND_PROGRESS_MONITOR_MAX_SAMPLES,
    seed: int = 99117,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Build a deterministic lightweight monitoring subset from TRAIN client arrays only.

    This subset is used solely for console progress after FL rounds. It never affects
    preprocessing, governance, client selection, optimizer budgets, checkpoint
    decisions, or final reporting. The held-out TEST set remains final-evaluation-only.
    """
    rng = np.random.default_rng(int(seed))
    client_ids = sorted(client_arrays.keys())
    sizes = {cid: int(len(client_arrays[cid][1])) for cid in client_ids}
    total = int(sum(sizes.values()))
    target = int(min(max_samples, total))

    # Proportional allocation across clients, then distribute rounding remainder.
    raw = {cid: target * sizes[cid] / max(1, total) for cid in client_ids}
    take = {cid: min(sizes[cid], int(np.floor(raw[cid]))) for cid in client_ids}

    while sum(take.values()) < target:
        candidates = [c for c in client_ids if take[c] < sizes[c]]
        if not candidates:
            break
        cid = max(candidates, key=lambda c: (raw[c] - take[c], sizes[c], c))
        take[cid] += 1

    xs, ys = [], []
    for cid in client_ids:
        n_take = int(take[cid])
        if n_take <= 0:
            continue
        Xc, yc = client_arrays[cid]
        if n_take >= len(yc):
            idx = np.arange(len(yc))
        else:
            idx = rng.choice(len(yc), size=n_take, replace=False)
        xs.append(np.asarray(Xc[idx]))
        ys.append(np.asarray(yc[idx], dtype=np.int32))

    Xmon = np.concatenate(xs, axis=0)
    ymon = np.concatenate(ys, axis=0)

    # Deterministic shuffle so monitoring batches do not follow client order.
    order = rng.permutation(len(ymon))
    return Xmon[order], ymon[order]


def _fmt_metric(x: float) -> str:
    return "nan" if not np.isfinite(float(x)) else f"{float(x):.4f}"


def print_round_progress(
    progress_context: Optional[Dict[str, Any]],
    round_idx: int,
    round_total: int,
    selected_ids: List[str],
    round_steps: int,
    cumulative_steps: int,
    monitor_metrics: Dict[str, float],
    cumulative_runtime_s: float,
    cumulative_communication_mb: float,
    ram_start_mb: float,
    ram_peak_mb: float,
    extra: str = "",
) -> None:
    """Compact, human-readable progress block after each federated round."""
    ctx = progress_context or {}
    scenario = str(ctx.get("scenario", "Federated scenario"))
    run_idx = int(ctx.get("run_idx", 0))
    run_total = int(ctx.get("run_total", 0))
    scenario_idx = int(ctx.get("scenario_idx", 0))
    scenario_total = int(ctx.get("scenario_total", 0))
    overall_idx = int(ctx.get("overall_idx", 0))
    overall_total = int(ctx.get("overall_total", 0))

    energy_wh = float(
        ROUND_PROGRESS_POWER_W * float(cumulative_runtime_s) / 3600.0
    )
    ram_delta = max(0.0, float(ram_peak_mb) - float(ram_start_mb))

    print("\n" + "-" * 112)
    print(
        f"PROGRESS | overall configuration {overall_idx}/{overall_total} | "
        f"run {run_idx}/{run_total} | scenario {scenario_idx}/{scenario_total}"
    )
    print(f"SCENARIO | {scenario}")
    print(
        f"ROUND    | {round_idx}/{round_total} | "
        f"selected={len(selected_ids)} [{','.join(map(str, selected_ids))}] | "
        f"steps={round_steps} | cumulative_steps={cumulative_steps}"
    )
    if extra:
        print(f"DETAIL   | {extra}")
    print(
        "TRAIN-MONITOR (diagnostic only; TEST untouched) | "
        f"Accuracy={_fmt_metric(monitor_metrics.get('accuracy', np.nan))} | "
        f"F1={_fmt_metric(monitor_metrics.get('f1_macro', np.nan))} | "
        f"AUC={_fmt_metric(monitor_metrics.get('roc_auc_ovr_macro', np.nan))} | "
        f"Precision={_fmt_metric(monitor_metrics.get('precision_macro', np.nan))} | "
        f"Recall={_fmt_metric(monitor_metrics.get('recall_macro', np.nan))}"
    )
    print(
        f"CUMULATIVE OPERATIONAL | runtime={cumulative_runtime_s:.2f}s | "
        f"energy≈{energy_wh:.5f}Wh | communication={cumulative_communication_mb:.3f}MB | "
        f"RAM peak={ram_peak_mb:.1f}MB | RAM Δ={ram_delta:.1f}MB"
    )
    print("-" * 112)


def train_exact_steps(
    model: keras.Model,
    X: np.ndarray,
    y: np.ndarray,
    steps: int,
    batch_size: int,
    seed: int,
    class_weights: Optional[Dict[int, float]] = None,
    prox_reference: Optional[List[np.ndarray]] = None,
    prox_mu: float = 0.0,
):
    """
    Exact mini-batch update count. Used for step-parity audits.
    """
    steps = int(max(1, steps))
    rng = np.random.default_rng(int(seed))
    n = len(y)
    if n == 0:
        raise RuntimeError("Cannot train on an empty dataset.")

    loss_fn = keras.losses.SparseCategoricalCrossentropy(
        reduction=keras.losses.Reduction.NONE
    )

    order = rng.permutation(n)
    cursor = 0

    prox_tensors = None
    if prox_reference is not None and prox_mu > 0:
        prox_tensors = [tf.convert_to_tensor(w) for w in prox_reference]

    for _ in range(steps):
        if cursor + batch_size > n:
            order = rng.permutation(n)
            cursor = 0

        idx = order[cursor:cursor + batch_size]
        cursor += batch_size

        xb = tf.convert_to_tensor(np.asarray(X[idx]), dtype=tf.float32)
        yb_np = np.asarray(y[idx], dtype=np.int32)
        yb = tf.convert_to_tensor(yb_np, dtype=tf.int32)

        with tf.GradientTape() as tape:
            probs = model(xb, training=True)
            per_loss = loss_fn(yb, probs)

            if class_weights:
                sw = np.array(
                    [class_weights.get(int(v), 1.0) for v in yb_np],
                    dtype=np.float32,
                )
                sw_t = tf.convert_to_tensor(sw)
                data_loss = tf.reduce_sum(per_loss * sw_t) / tf.reduce_sum(sw_t)
            else:
                data_loss = tf.reduce_mean(per_loss)

            loss = data_loss

            if prox_tensors is not None:
                prox = tf.constant(0.0, dtype=tf.float32)
                for var, ref in zip(model.trainable_variables, prox_tensors):
                    prox += tf.reduce_sum(tf.square(var - tf.cast(ref, var.dtype)))
                loss = loss + 0.5 * float(prox_mu) * prox

        grads = tape.gradient(loss, model.trainable_variables)
        model.optimizer.apply_gradients(zip(grads, model.trainable_variables))


def natural_steps(n_records: int, batch_size: int, local_epochs: int = 1) -> int:
    return int(max(1, math.ceil(int(n_records) / int(batch_size)) * int(local_epochs)))


def allocate_exact_step_budget(
    selected: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    target_total: int,
    batch_size: int,
    local_epochs: int = 1,
) -> Dict[str, int]:
    natural = {
        cid: natural_steps(len(client_arrays[cid][1]), batch_size, local_epochs)
        for cid in selected
    }
    total_nat = max(1, sum(natural.values()))
    raw = {cid: target_total * natural[cid] / total_nat for cid in selected}
    alloc = {cid: max(1, int(math.floor(raw[cid]))) for cid in selected}

    # Adjust to exact target.
    while sum(alloc.values()) < target_total:
        cid = max(selected, key=lambda c: raw[c] - alloc[c])
        alloc[cid] += 1
    while sum(alloc.values()) > target_total:
        candidates = [c for c in selected if alloc[c] > 1]
        if not candidates:
            break
        cid = min(candidates, key=lambda c: raw[c] - alloc[c])
        alloc[cid] -= 1

    if sum(alloc.values()) != int(target_total):
        raise RuntimeError("Exact step-budget allocation failed.")
    return alloc


def aggregate_weights(
    local_weights: List[List[np.ndarray]],
    sample_sizes: List[int],
    equal_weight: bool = False,
) -> List[np.ndarray]:
    if not local_weights:
        raise RuntimeError("No local weights to aggregate.")
    if equal_weight:
        alpha = np.ones(len(local_weights), dtype=float) / len(local_weights)
    else:
        sizes = np.asarray(sample_sizes, dtype=float)
        alpha = sizes / sizes.sum()

    out = []
    for layer_idx in range(len(local_weights[0])):
        x = sum(alpha[j] * np.asarray(local_weights[j][layer_idx])
                for j in range(len(local_weights)))
        out.append(np.asarray(x))
    return out


def federated_train(
    build_model_fn,
    initial_weights: List[np.ndarray],
    selected_per_round: List[List[str]],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    local_epochs: int,
    class_weights: Dict[int, float],
    run_seed: int,
    exact_step_maps: Optional[List[Dict[str, int]]] = None,
    equal_weight: bool = False,
    fedprox_mu: float = 0.0,
    train_monitor: Optional[Tuple[np.ndarray, np.ndarray]] = None,
    progress_context: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    monitor = RAMMonitor().start()
    start = time.perf_counter()

    global_model = build_model_fn()
    global_model.set_weights([np.array(w, copy=True) for w in initial_weights])

    round_rows = []
    total_steps = 0
    total_selected = 0
    monitor_eval_s = 0.0
    cumulative_comm_raw_b = 0
    param_b = model_parameter_bytes(global_model)

    for r, selected in enumerate(selected_per_round, start=1):
        global_weights = [np.array(w, copy=True) for w in global_model.get_weights()]
        local_weights = []
        local_sizes = []
        round_steps = 0

        if exact_step_maps is None:
            step_map = {
                cid: natural_steps(
                    len(client_arrays[cid][1]), batch_size, local_epochs
                )
                for cid in selected
            }
        else:
            step_map = exact_step_maps[r - 1]

        for j, cid in enumerate(selected):
            Xc, yc = client_arrays[cid]
            local_model = build_model_fn()
            local_model.set_weights(global_weights)
            train_exact_steps(
                local_model, Xc, yc,
                steps=int(step_map[cid]),
                batch_size=batch_size,
                seed=int(run_seed + 1000 * r + 17 * j),
                class_weights=class_weights,
                prox_reference=global_weights if fedprox_mu > 0 else None,
                prox_mu=float(fedprox_mu),
            )
            local_weights.append(
                [np.array(w, copy=True) for w in local_model.get_weights()]
            )
            local_sizes.append(len(yc))
            round_steps += int(step_map[cid])

            del local_model
            tf.keras.backend.clear_session()

        agg = aggregate_weights(
            local_weights, local_sizes, equal_weight=equal_weight
        )
        global_model = build_model_fn()
        global_model.set_weights(agg)

        total_steps += round_steps
        total_selected += len(selected)

        # Cumulative model traffic through this round. Same accounting as the
        # final experiment metric: download + upload + 12% protocol overhead.
        cumulative_comm_raw_b += int(len(selected)) * 2 * int(param_b)
        cumulative_comm_mb = float(
            (cumulative_comm_raw_b * 1.12) / (1024 ** 2)
        )

        # Measure training runtime BEFORE this round's diagnostic evaluation.
        # Previous diagnostic-evaluation time is subtracted so progress printing
        # does not inflate the experiment's runtime metric.
        cumulative_runtime_s = float(
            max(0.0, (time.perf_counter() - start) - monitor_eval_s)
        )

        if train_monitor is not None:
            Xmon, ymon = train_monitor
            t_mon = time.perf_counter()
            monitor_metrics = evaluate_model(global_model, Xmon, ymon, n_classes)
            monitor_eval_s += float(time.perf_counter() - t_mon)
        else:
            monitor_metrics = {
                "accuracy": float("nan"),
                "precision_macro": float("nan"),
                "recall_macro": float("nan"),
                "f1_macro": float("nan"),
                "roc_auc_ovr_macro": float("nan"),
            }

        live_peak = float(monitor.peak_mb)
        round_row = {
            "round": r,
            "selected_clients": len(selected),
            "selected_ids": ";".join(selected),
            "optimizer_steps": int(round_steps),
            "cumulative_optimizer_steps": int(total_steps),
            "train_monitor_accuracy": float(monitor_metrics["accuracy"]),
            "train_monitor_precision_macro": float(monitor_metrics["precision_macro"]),
            "train_monitor_recall_macro": float(monitor_metrics["recall_macro"]),
            "train_monitor_f1_macro": float(monitor_metrics["f1_macro"]),
            "train_monitor_auc_ovr_macro": float(monitor_metrics["roc_auc_ovr_macro"]),
            "cumulative_runtime_s": float(cumulative_runtime_s),
            "cumulative_energy_wh_est": float(
                ROUND_PROGRESS_POWER_W * cumulative_runtime_s / 3600.0
            ),
            "cumulative_communication_mb": float(cumulative_comm_mb),
            "ram_peak_mb_live": float(live_peak),
            "ram_delta_mb_live": float(max(0.0, live_peak - monitor.start_mb)),
        }
        round_rows.append(round_row)

        print_round_progress(
            progress_context=progress_context,
            round_idx=r,
            round_total=len(selected_per_round),
            selected_ids=list(selected),
            round_steps=int(round_steps),
            cumulative_steps=int(total_steps),
            monitor_metrics=monitor_metrics,
            cumulative_runtime_s=cumulative_runtime_s,
            cumulative_communication_mb=cumulative_comm_mb,
            ram_start_mb=float(monitor.start_mb),
            ram_peak_mb=live_peak,
        )

    runtime = float(max(0.0, (time.perf_counter() - start) - monitor_eval_s))
    metrics = evaluate_model(global_model, X_test, y_test, n_classes)
    ram = monitor.stop()

    communication_b = int(cumulative_comm_raw_b * 1.12)

    return {
        "model": global_model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(total_steps),
        "communication_mb": float(communication_b / (1024 ** 2)),
        "round_audit": round_rows,
        "participants_mean_per_round": float(
            total_selected / max(1, len(round_rows))
        ),
        **ram,
    }



def select_dq_only_clients(
    dq_scores: Dict[str, Dict[str, float]],
    k: int,
) -> Tuple[List[str], pd.DataFrame]:
    """Select top-K clients by the machine-measured TADP Data Quality dimension only."""
    rows = []
    for cid, scores in dq_scores.items():
        vals = [float(scores[f]) for f in FACTOR_NAMES["dim2"]]
        rows.append({
            "client": str(cid),
            "dq_only_score": float(np.mean(vals)),
            "dq_factor_count": int(len(vals)),
        })
    audit = pd.DataFrame(rows).sort_values(
        ["dq_only_score", "client"], ascending=[False, True]
    ).reset_index(drop=True)
    audit["dq_only_rank"] = np.arange(1, len(audit) + 1)
    selected = audit.head(int(k))["client"].astype(str).tolist()
    audit["dq_only_selected"] = audit["client"].isin(selected)
    return selected, audit


def local_training_loss(model: keras.Model, X: np.ndarray, y: np.ndarray, batch_size: int = 512) -> float:
    """Mean sparse cross-entropy on client TRAIN data only; used by Power-of-Choice."""
    p = model.predict(X, batch_size=batch_size, verbose=0)
    y = np.asarray(y, dtype=np.int32)
    idx = np.arange(len(y))
    probs = np.clip(p[idx, y], 1e-12, 1.0)
    return float(-np.mean(np.log(probs)))


def federated_train_power_of_choice(
    build_model_fn,
    initial_weights: List[np.ndarray],
    candidate_clients: List[str],
    select_k: int,
    target_steps_per_round: int,
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    local_epochs: int,
    class_weights: Dict[int, float],
    run_seed: int,
    candidate_multiplier: int = 2,
    train_monitor: Optional[Tuple[np.ndarray, np.ndarray]] = None,
    progress_context: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Power-of-Choice baseline: each round samples d candidates and selects the K
    clients with largest current local TRAIN loss. K, rounds, initialisation, and
    total optimizer steps per round are matched to TADP-VR. No TEST information is used.
    """
    monitor = RAMMonitor().start()
    start = time.perf_counter()
    global_model = build_model_fn()
    global_model.set_weights([np.array(w, copy=True) for w in initial_weights])
    rng = np.random.default_rng(int(run_seed) + 880000)
    round_rows = []
    total_steps = 0
    total_selected = 0
    communication_b = 0
    monitor_eval_s = 0.0

    all_candidates = list(candidate_clients)
    if int(select_k) < 1 or int(select_k) > len(all_candidates):
        raise RuntimeError("Invalid Power-of-Choice K.")

    for r in range(1, NUM_ROUNDS_FL + 1):
        d = min(len(all_candidates), max(int(select_k), int(candidate_multiplier) * int(select_k)))
        if d == len(all_candidates):
            candidate_pool = list(all_candidates)
        else:
            # Canonical pow-d samples candidate clients without replacement
            # according to p_k, the client's fraction of total TRAIN data.
            sizes = np.asarray(
                [len(client_arrays[c][1]) for c in all_candidates], dtype=float
            )
            probs = sizes / sizes.sum()
            candidate_pool = rng.choice(
                all_candidates, size=d, replace=False, p=probs
            ).tolist()

        losses = []
        for cid in candidate_pool:
            Xc, yc = client_arrays[cid]
            losses.append((str(cid), local_training_loss(global_model, Xc, yc)))
        losses.sort(key=lambda x: (-x[1], x[0]))
        selected = [cid for cid, _ in losses[:int(select_k)]]
        step_map = allocate_exact_step_budget(
            selected, client_arrays, int(target_steps_per_round), batch_size, local_epochs
        )

        global_weights = [np.array(w, copy=True) for w in global_model.get_weights()]
        local_weights, local_sizes = [], []
        for j, cid in enumerate(selected):
            Xc, yc = client_arrays[cid]
            local_model = build_model_fn()
            local_model.set_weights(global_weights)
            train_exact_steps(
                local_model, Xc, yc,
                steps=int(step_map[cid]), batch_size=batch_size,
                seed=int(run_seed + 1000 * r + 17 * j),
                class_weights=class_weights,
            )
            local_weights.append([np.array(w, copy=True) for w in local_model.get_weights()])
            local_sizes.append(len(yc))
            del local_model
            tf.keras.backend.clear_session()

        agg = aggregate_weights(local_weights, local_sizes, equal_weight=False)
        global_model = build_model_fn()
        global_model.set_weights(agg)
        round_steps = int(sum(step_map.values()))
        total_steps += round_steps
        total_selected += len(selected)
        param_b = model_parameter_bytes(global_model)
        # Candidate clients receive the current model to evaluate local loss;
        # selected clients return one model update. Scalar loss uploads are negligible.
        communication_b += (len(candidate_pool) + len(selected)) * param_b
        cumulative_comm_mb = float(
            (communication_b * 1.12) / (1024 ** 2)
        )
        cumulative_runtime_s = float(
            max(0.0, (time.perf_counter() - start) - monitor_eval_s)
        )

        if train_monitor is not None:
            Xmon, ymon = train_monitor
            t_mon = time.perf_counter()
            monitor_metrics = evaluate_model(global_model, Xmon, ymon, n_classes)
            monitor_eval_s += float(time.perf_counter() - t_mon)
        else:
            monitor_metrics = {
                "accuracy": float("nan"),
                "precision_macro": float("nan"),
                "recall_macro": float("nan"),
                "f1_macro": float("nan"),
                "roc_auc_ovr_macro": float("nan"),
            }

        live_peak = float(monitor.peak_mb)
        round_rows.append({
            "round": r,
            "candidate_count": len(candidate_pool),
            "candidate_ids": ";".join(candidate_pool),
            "selected_clients": len(selected),
            "selected_ids": ";".join(selected),
            "optimizer_steps": round_steps,
            "cumulative_optimizer_steps": int(total_steps),
            "local_losses": json.dumps({cid: loss for cid, loss in losses}, sort_keys=True),
            "train_monitor_accuracy": float(monitor_metrics["accuracy"]),
            "train_monitor_precision_macro": float(monitor_metrics["precision_macro"]),
            "train_monitor_recall_macro": float(monitor_metrics["recall_macro"]),
            "train_monitor_f1_macro": float(monitor_metrics["f1_macro"]),
            "train_monitor_auc_ovr_macro": float(monitor_metrics["roc_auc_ovr_macro"]),
            "cumulative_runtime_s": float(cumulative_runtime_s),
            "cumulative_energy_wh_est": float(
                ROUND_PROGRESS_POWER_W * cumulative_runtime_s / 3600.0
            ),
            "cumulative_communication_mb": float(cumulative_comm_mb),
            "ram_peak_mb_live": float(live_peak),
            "ram_delta_mb_live": float(max(0.0, live_peak - monitor.start_mb)),
        })

        loss_preview = ", ".join(
            f"{cid}:{loss:.3f}" for cid, loss in losses[:min(5, len(losses))]
        )
        print_round_progress(
            progress_context=progress_context,
            round_idx=r,
            round_total=NUM_ROUNDS_FL,
            selected_ids=list(selected),
            round_steps=int(round_steps),
            cumulative_steps=int(total_steps),
            monitor_metrics=monitor_metrics,
            cumulative_runtime_s=cumulative_runtime_s,
            cumulative_communication_mb=cumulative_comm_mb,
            ram_start_mb=float(monitor.start_mb),
            ram_peak_mb=live_peak,
            extra=f"PoC candidates={len(candidate_pool)} | highest TRAIN losses: {loss_preview}",
        )

    runtime = float(max(0.0, (time.perf_counter() - start) - monitor_eval_s))
    metrics = evaluate_model(global_model, X_test, y_test, n_classes)
    ram = monitor.stop()
    communication_b = int(communication_b * 1.12)
    return {
        "model": global_model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(total_steps),
        "communication_mb": float(communication_b / (1024 ** 2)),
        "round_audit": round_rows,
        "participants_mean_per_round": float(total_selected / max(1, NUM_ROUNDS_FL)),
        **ram,
    }


def centralized_train(
    build_model_fn,
    initial_weights: List[np.ndarray],
    selected_clients: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    exact_steps: int,
    class_weights: Dict[int, float],
    seed: int,
) -> Dict[str, Any]:
    if not selected_clients:
        raise RuntimeError("Centralized scenario has no TRAIN clients.")

    monitor = RAMMonitor().start()
    start = time.perf_counter()

    # Pool ACCEPTED TRAIN partitions only. TEST is not present here.
    X = np.concatenate([client_arrays[c][0] for c in selected_clients], axis=0)
    y = np.concatenate([client_arrays[c][1] for c in selected_clients], axis=0)

    model = build_model_fn()
    model.set_weights([np.array(w, copy=True) for w in initial_weights])
    train_exact_steps(
        model, X, y,
        steps=int(exact_steps),
        batch_size=batch_size,
        seed=int(seed),
        class_weights=class_weights,
    )

    runtime = time.perf_counter() - start
    metrics = evaluate_model(model, X_test, y_test, n_classes)
    ram = monitor.stop()

    return {
        "model": model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(exact_steps),
        "communication_mb": 0.0,
        "participants_mean_per_round": float(len(selected_clients)),
        **ram,
    }


def result_row(
    run: int,
    seed: int,
    scenario: str,
    result: Dict[str, Any],
    initial_hash: str,
    power_w: float = 12.0,
) -> Dict[str, Any]:
    row = {
        "run": int(run),
        "seed": int(seed),
        "scenario": str(scenario),
        **result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "participants": float(result.get("participants_mean_per_round", 0.0)),
        "ram_start_mb": float(result.get("ram_start_mb", 0.0)),
        "ram_end_mb": float(result.get("ram_end_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
        "ram_delta_mb": float(result.get("ram_delta_mb", 0.0)),
        "ram_mb": float(result.get("ram_mb", result.get("ram_peak_mb", 0.0))),
        "initial_weights_sha256": initial_hash,
    }
    row["energy_wh"] = float(power_w * row["runtime_s"] / 3600.0)
    row["energy_kwh"] = float(row["energy_wh"] / 1000.0)
    row["co2_kg"] = float(row["energy_kwh"] * 0.430)
    row["energy_cost_usd"] = float(row["energy_kwh"] * 0.20)
    row["communication_cost_usd"] = float(row["communication_mb"] * 0.005)
    row["total_estimated_cost_usd"] = float(
        row["energy_cost_usd"] + row["communication_cost_usd"]
    )
    return row


# ======================================================================================
# LEAKAGE AUDIT
# ======================================================================================

def write_leakage_audit(
    out_dir: Path,
    train_ids,
    test_ids,
    client_train_ids: Dict[str, np.ndarray],
    extra: Optional[Dict[str, Any]] = None,
):
    train_set = set(map(str, train_ids))
    test_set = set(map(str, test_ids))
    overlap = train_set & test_set

    client_union = set()
    duplicates_across_clients = 0
    for cid, ids in client_train_ids.items():
        s = set(map(str, ids))
        duplicates_across_clients += len(client_union & s)
        client_union |= s

    test_in_clients = len(test_set & client_union)
    missing_train = len(train_set - client_union)
    extra_client_rows = len(client_union - train_set)

    row = {
        "train_test_overlap": len(overlap),
        "test_rows_in_any_client": test_in_clients,
        "train_rows_missing_from_clients": missing_train,
        "client_rows_not_in_global_train": extra_client_rows,
        "duplicate_train_rows_across_clients": duplicates_across_clients,
        "pass": (
            len(overlap) == 0
            and test_in_clients == 0
            and missing_train == 0
            and extra_client_rows == 0
            and duplicates_across_clients == 0
        ),
    }
    if extra:
        row.update(extra)

    pd.DataFrame([row]).to_csv(
        Path(out_dir) / "leakage_audit.csv", index=False
    )

    if not bool(row["pass"]):
        raise RuntimeError(f"FAIL-CLOSED leakage audit failed: {row}")
    return row


# ======================================================================================
# DIABETES 130-US — LEAKAGE-SAFE DATA PREPARATION
# ======================================================================================

TARGET = "readmitted"
ID_COLUMNS = ["encounter_id", "patient_nbr"]


def locate_diabetes_csv() -> str:
    candidates = [
        os.environ.get("DIABETES_CSV", ""),
        "/content/diabetes_130US.csv",
        "./diabetes_130US.csv",
        "/content/drive/MyDrive/diabetes_130US.csv",
    ]
    for p in candidates:
        if p and os.path.exists(p):
            return p

    try:
        from google.colab import files as colab_files
        print("Please upload the Diabetes 130-US CSV file.")
        uploaded = colab_files.upload()
        csvs = [name for name in uploaded if str(name).lower().endswith(".csv")]
        if not csvs:
            raise RuntimeError("No CSV file was uploaded.")
        return str(csvs[0])
    except ImportError:
        pass

    raise FileNotFoundError(
        "Diabetes CSV not found. Set DIABETES_CSV or upload the CSV in Colab."
    )


def load_diabetes(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df = df.replace("?", np.nan)

    if TARGET not in df.columns:
        raise RuntimeError(f"Missing target column: {TARGET}")

    valid_target = {"NO": 0, ">30": 1, "<30": 2}
    df = df[df[TARGET].isin(valid_target)].copy()
    df["_target"] = df[TARGET].map(valid_target).astype(np.int32)
    df["_row_id"] = np.arange(len(df), dtype=np.int64)

    return df


def global_patient_grouped_split(
    df: pd.DataFrame, seed: int, test_fraction: float = 0.20
):
    """
    Patient-grouped and stratified whenever patient_nbr is available.
    The split happens before any client partitioning or data-dependent preprocessing.
    """
    if "patient_nbr" in df.columns:
        splitter = StratifiedGroupKFold(
            n_splits=5, shuffle=True, random_state=int(seed)
        )
        train_idx, test_idx = next(
            splitter.split(
                np.zeros(len(df)),
                y=df["_target"].to_numpy(),
                groups=df["patient_nbr"].astype(str).to_numpy(),
            )
        )
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        patient_overlap = len(
            set(train_df["patient_nbr"].astype(str))
            & set(test_df["patient_nbr"].astype(str))
        )
        if patient_overlap != 0:
            raise RuntimeError("Patient leakage detected across TRAIN/TEST.")
    else:
        train_df, test_df = train_test_split(
            df, test_size=float(test_fraction),
            stratify=df["_target"], random_state=int(seed)
        )
        patient_overlap = 0

    return train_df.reset_index(drop=True), test_df.reset_index(drop=True), patient_overlap


def dirichlet_partition_dataframe(
    train_df: pd.DataFrame,
    n_clients: int,
    alpha: float,
    seed: int,
    min_client_records: int = 100,
) -> Dict[str, pd.DataFrame]:
    y = train_df["_target"].to_numpy()
    client_ids = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")[:n_clients]

    for attempt in range(100):
        rng = np.random.default_rng(int(seed + attempt))
        buckets = [[] for _ in range(n_clients)]

        for cls in sorted(np.unique(y)):
            idx = np.where(y == cls)[0]
            rng.shuffle(idx)
            props = rng.dirichlet(np.full(n_clients, float(alpha)))
            cuts = (np.cumsum(props)[:-1] * len(idx)).astype(int)
            splits = np.split(idx, cuts)
            for k, s in enumerate(splits):
                buckets[k].extend(s.tolist())

        sizes = [len(b) for b in buckets]
        if min(sizes) >= int(min_client_records):
            out = {}
            for cid, idxs in zip(client_ids, buckets):
                out[cid] = train_df.iloc[np.array(idxs, dtype=int)].copy()
            return out

    raise RuntimeError("Could not obtain a valid Dirichlet client partition.")


@dataclass
class TabularPreprocessor:
    feature_cols: List[str]
    numeric_cols: List[str]
    categorical_cols: List[str]
    mean: Dict[str, float]
    std: Dict[str, float]
    categories: Dict[str, List[str]]
    encoder: Any

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        pieces = []

        if self.numeric_cols:
            x_num = []
            for c in self.numeric_cols:
                s = pd.to_numeric(df[c], errors="coerce").astype(float)
                a = s.fillna(self.mean[c]).to_numpy(dtype=np.float32)
                a = (a - self.mean[c]) / self.std[c]
                x_num.append(a[:, None])
            pieces.append(np.concatenate(x_num, axis=1).astype(np.float32))

        if self.categorical_cols:
            cat = pd.DataFrame({
                c: df[c].astype("string").fillna("__MISSING__").astype(str)
                for c in self.categorical_cols
            })
            x_cat = self.encoder.transform(cat)
            pieces.append(np.asarray(x_cat, dtype=np.float32))

        if not pieces:
            raise RuntimeError("No predictor columns remained.")
        return np.concatenate(pieces, axis=1).astype(np.float32)


def fit_federated_train_only_preprocessor(
    client_frames: Dict[str, pd.DataFrame]
) -> TabularPreprocessor:
    """
    No raw TRAIN pooling is used to ESTIMATE numeric parameters.
    Numeric mean/std comes from aggregated local count/sum/sum-of-squares.
    Categorical vocabulary comes from union of local TRAIN category sets.
    """
    any_df = next(iter(client_frames.values()))
    feature_cols = [
        c for c in any_df.columns
        if c not in {TARGET, "_target", "_row_id", *ID_COLUMNS}
    ]

    # Infer expected type from TRAIN only.
    numeric_cols = []
    categorical_cols = []
    for c in feature_cols:
        total_nonmissing = 0
        numeric_valid = 0
        for df in client_frames.values():
            raw = df[c]
            nm = raw.notna()
            total_nonmissing += int(nm.sum())
            if nm.any():
                numeric_valid += int(
                    pd.to_numeric(raw[nm], errors="coerce").notna().sum()
                )
        ratio = numeric_valid / max(1, total_nonmissing)
        if ratio >= 0.95:
            numeric_cols.append(c)
        else:
            categorical_cols.append(c)

    mean = {}
    std = {}
    for c in numeric_cols:
        count = 0
        sum_ = 0.0
        sumsq = 0.0
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            count += len(a)
            sum_ += float(a.sum())
            sumsq += float(np.square(a).sum())
        mu = sum_ / max(1, count)
        var = max(1e-12, sumsq / max(1, count) - mu * mu)
        mean[c] = float(mu)
        std[c] = float(math.sqrt(var))

    categories = {}
    for c in categorical_cols:
        values = set()
        for df in client_frames.values():
            s = df[c].astype("string").fillna("__MISSING__").astype(str)
            values.update(s.unique().tolist())
        categories[c] = sorted(values)

    encoder = OneHotEncoder(
        categories=[categories[c] for c in categorical_cols],
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float32,
    )

    # Fit only metadata-shaped dummy rows; the vocabulary is already frozen from TRAIN.
    if categorical_cols:
        max_len = max(len(categories[c]) for c in categorical_cols)
        dummy = {}
        for c in categorical_cols:
            vals = categories[c]
            dummy[c] = [vals[i % len(vals)] for i in range(max_len)]
        encoder.fit(pd.DataFrame(dummy))

    return TabularPreprocessor(
        feature_cols=feature_cols,
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        mean=mean,
        std=std,
        categories=categories,
        encoder=encoder,
    )


def build_tabular_reference(
    client_frames: Dict[str, pd.DataFrame],
    preprocessor: TabularPreprocessor,
) -> Dict[str, Any]:
    """
    TRAIN-only DQ reference built from client-local summaries only.

    Numerical reference histograms are constructed by combining local
    min/max summaries and then summing client-local histogram counts.
    Categorical reference support is the union of local TRAIN category sets.
    Raw client records are not concatenated to construct the reference.
    """
    ref = {"num_hist": {}, "cat_values": {}}

    chosen_num = preprocessor.numeric_cols[:12]
    for c in chosen_num:
        local_min, local_max = [], []
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                local_min.append(float(np.min(a)))
                local_max.append(float(np.max(a)))
        if not local_min:
            continue
        gmin, gmax = float(min(local_min)), float(max(local_max))
        if gmax <= gmin:
            edges = np.array([gmin - 1e-6, gmax + 1e-6], dtype=float)
        else:
            edges = np.linspace(gmin, gmax, 11, dtype=float)
        global_hist = np.zeros(len(edges)-1, dtype=np.float64)
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                h, _ = np.histogram(a, bins=edges)
                global_hist += h.astype(np.float64)
        ref["num_hist"][c] = {
            "edges": edges.tolist(),
            "hist": global_hist.tolist(),
            "construction": "aggregated_client_local_histograms_only",
        }

    for c in preprocessor.categorical_cols:
        values = set()
        for df in client_frames.values():
            values.update(
                df[c].astype("string").fillna("__MISSING__")
                .astype(str).unique().tolist()
            )
        ref["cat_values"][c] = sorted(values)
    return ref

def tabular_dq_scores(
    df: pd.DataFrame,
    preprocessor: TabularPreprocessor,
    reference: Dict[str, Any],
) -> Tuple[Dict[str, float], Dict[str, float]]:
    """
    Eight machine-measured TRAIN-only DQ factors for the healthcare experiment.
    """
    feature_df = df[preprocessor.feature_cols]

    # 1) Completeness.
    missing_fraction = float(feature_df.isna().mean().mean())
    completeness = score_lower_is_better(
        missing_fraction,
        [0.01, 0.05, 0.10, 0.20, 0.50],
    )

    # 2) Duplication rate.
    duplicate_fraction = float(feature_df.duplicated().mean())
    duplication = score_lower_is_better(
        duplicate_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 3) Value validity / error rate.
    bad = 0
    observed = 0

    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())

        if nm.any():
            conv = pd.to_numeric(
                raw[nm], errors="coerce"
            ).to_numpy(dtype=float)
            bad += int(np.sum(~np.isfinite(conv)))

    for c in preprocessor.categorical_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())

        if nm.any():
            bad += int(
                np.sum(raw[nm].astype(str).str.strip().eq(""))
            )

    error_fraction = float(bad / max(1, observed))
    value_validity_error_rate = score_lower_is_better(
        error_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.15],
    )

    # 4) Type consistency.
    type_bad = 0
    type_obs = 0

    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        type_obs += int(nm.sum())

        if nm.any():
            type_bad += int(
                pd.to_numeric(
                    raw[nm], errors="coerce"
                ).isna().sum()
            )

    type_inconsistency = float(type_bad / max(1, type_obs))
    type_consistency = score_lower_is_better(
        type_inconsistency,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 5) Label integrity.
    invalid_label_fraction = float(
        (~df["_target"].isin([0, 1, 2])).mean()
    )
    label_integrity = score_lower_is_better(
        invalid_label_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    # 6) Feature-distribution consistency — TRAIN-only reference.
    jsds = []

    for c, spec in reference["num_hist"].items():
        a = pd.to_numeric(
            df[c], errors="coerce"
        ).to_numpy(dtype=float)
        a = a[np.isfinite(a)]

        if len(a):
            hist, _ = np.histogram(
                a,
                bins=np.array(spec["edges"], dtype=float),
            )
            jsds.append(
                js_divergence(
                    hist,
                    np.array(spec["hist"], dtype=float),
                )
            )

    max_jsd = float(max(jsds)) if jsds else 0.0
    distribution_consistency = score_lower_is_better(
        max_jsd,
        [0.01, 0.025, 0.05, 0.10, 0.20],
    )

    # 7) Feature/category coverage.
    coverage_vals = []

    for c, ref_vals in reference["cat_values"].items():
        ref_set = set(ref_vals)

        if ref_set:
            client_set = set(
                df[c]
                .astype("string")
                .fillna("__MISSING__")
                .astype(str)
                .unique()
            )
            coverage_vals.append(
                len(client_set & ref_set) / len(ref_set)
            )

    mean_coverage = (
        float(np.mean(coverage_vals))
        if coverage_vals else 1.0
    )
    feature_coverage = score_higher_is_better(
        mean_coverage,
        [0.90, 0.825, 0.75, 0.65, 0.50],
    )

    # 8) Structural / constraint integrity.
    # Uses only TRAIN records. It checks:
    #   - key presence / encounter uniqueness;
    #   - non-negative count-like clinical fields;
    #   - finite numeric values where a numeric value is expected.
    n = len(df)
    record_violation = np.zeros(n, dtype=bool)

    if "encounter_id" not in df.columns:
        record_violation[:] = True
    else:
        encounter = df["encounter_id"]
        record_violation |= encounter.isna().to_numpy()
        record_violation |= encounter.duplicated(keep=False).to_numpy()

    nonnegative_fields = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]

    for c in nonnegative_fields:
        if c in df.columns:
            a = pd.to_numeric(
                df[c], errors="coerce"
            ).to_numpy(dtype=float)
            bad_c = (~np.isfinite(a)) | (a < 0)
            record_violation |= bad_c

    structural_violation_fraction = float(
        np.mean(record_violation)
    ) if n else 1.0

    structural_integrity = score_lower_is_better(
        structural_violation_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    scores = {
        "completeness": completeness,
        "duplication_rate": duplication,
        "value_validity_error_rate": value_validity_error_rate,
        "type_consistency": type_consistency,
        "label_integrity": label_integrity,
        "feature_distribution_consistency": distribution_consistency,
        "feature_category_coverage": feature_coverage,
        "structural_constraint_integrity": structural_integrity,
    }

    raw = {
        "missing_fraction": missing_fraction,
        "duplicate_fraction": duplicate_fraction,
        "error_fraction": error_fraction,
        "type_inconsistency_fraction": type_inconsistency,
        "invalid_label_fraction": invalid_label_fraction,
        "max_jsd": max_jsd,
        "mean_category_coverage": mean_coverage,
        "structural_violation_fraction": structural_violation_fraction,
    }

    return scores, raw

def prepare_diabetes_no_leakage(
    csv_path: str,
    split_seed: int,
    partition_seed: int,
    n_clients: int = 10,
    alpha: float = 1.0,
):
    raw = load_diabetes(csv_path)
    train_df, test_df, patient_overlap = global_patient_grouped_split(
        raw, split_seed
    )

    clients = dirichlet_partition_dataframe(
        train_df, n_clients=n_clients, alpha=alpha, seed=partition_seed
    )
    client_ids = list(clients.keys())

    # Hard row-level leakage audit.
    client_train_ids = {
        cid: df["_row_id"].astype(str).to_numpy()
        for cid, df in clients.items()
    }

    pre = fit_federated_train_only_preprocessor(clients)
    reference = build_tabular_reference(clients, pre)

    dq_scores = {}
    dq_raw_rows = []
    client_arrays = {}

    for cid, df in clients.items():
        scores, raw_metrics = tabular_dq_scores(df, pre, reference)
        dq_scores[cid] = scores

        row = {"client": cid, **scores, **raw_metrics}
        dq_raw_rows.append(row)

        X = pre.transform(df)
        y = df["_target"].to_numpy(dtype=np.int32)
        client_arrays[cid] = (X, y)

    X_test = pre.transform(test_df)
    y_test = test_df["_target"].to_numpy(dtype=np.int32)

    y_train_all = np.concatenate(
        [client_arrays[c][1] for c in client_ids]
    )
    cw = class_weight_dict(y_train_all)

    meta = {
        "raw_rows": len(raw),
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "patient_overlap": int(patient_overlap),
        "input_dim": int(X_test.shape[1]),
        "n_clients": int(n_clients),
        "numeric_features": len(pre.numeric_cols),
        "categorical_features": len(pre.categorical_cols),
    }

    return {
        "raw": raw,
        "train_df": train_df,
        "test_df": test_df,
        "clients_raw": clients,
        "client_arrays": client_arrays,
        "client_ids": client_ids,
        "X_test": X_test,
        "y_test": y_test,
        "dq_scores": dq_scores,
        "dq_audit": pd.DataFrame(dq_raw_rows),
        "class_weights": cw,
        "meta": meta,
        "client_train_ids": client_train_ids,
        "global_train_ids": train_df["_row_id"].astype(str).to_numpy(),
        "global_test_ids": test_df["_row_id"].astype(str).to_numpy(),
        "preprocessor": pre,
        "dq_reference": reference,
    }


def build_diabetes_model(input_dim: int, lr: float = 1e-3) -> keras.Model:
    inp = keras.Input(shape=(int(input_dim),), dtype=tf.float32)
    x = layers.Dense(128, activation="relu")(inp)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(3, activation="softmax", dtype=tf.float32)(x)
    model = keras.Model(inp, out)
    model.optimizer = keras.optimizers.Adam(learning_rate=float(lr))
    return model



# ======================================================================================
# FULL EXPERIMENT-A REPORTING, CHECKPOINTING, LEDGER, AND STATISTICS
# ======================================================================================
from datetime import datetime, timezone
import re

# =============================================================================
# MAIN REPEATED TRAINING SET
#
# These eight configurations are genuinely distinct and are repeated over all
# five training seeds for mean ± SD / CI reporting.
#
# NOT repeated here:
#   - Great Expectations Centralized
#   - TADP-AA Centralized
#   - Great Expectations Federated
#   - TADP-AA Federated
# Those all-client equivalences are verified independently by the companion
# one-seed equivalence-audit script.
#
# TADP-SDA is retained as a boundary/stress condition but is run once only,
# outside the five-seed headline statistical comparison.
# =============================================================================
MANUSCRIPT_SCENARIOS = [
    "Naïve Centralized",
    "TADP-VR Centralized",
    "Vanilla FedAvg",
    "FedProx",
    "Random-K",
    "DQ-only Federated",
    "Power-of-Choice",
    "TADP-VR Federated",
]
assert len(MANUSCRIPT_SCENARIOS) == 8

BOUNDARY_SCENARIOS = [
    "TADP-SDA Centralized",
    "TADP-SDA Federated",
]
BOUNDARY_SEED = 42



def print_banner(title: str, width: int = 108):
    print("\n" + "=" * width)
    print(title)
    print("=" * width)


def client_partition_table(clients_raw):
    rows = []
    for cid, df in clients_raw.items():
        counts = df["_target"].value_counts().to_dict()
        rows.append({
            "client": cid,
            "records": int(len(df)),
            "class_0_NO": int(counts.get(0, 0)),
            "class_1_GT30": int(counts.get(1, 0)),
            "class_2_LT30": int(counts.get(2, 0)),
        })
    return pd.DataFrame(rows)


def evidence_assignment_summary(
    evidence_df: pd.DataFrame
) -> pd.DataFrame:
    """
    Summarize the frozen controlled documentary-evidence assignment.

    v16.7 reports the pre-specified governance archetype for each evidence
    bundle. These archetypes are branch-coverage scenarios, not observed
    real-world prevalence classes.
    """
    required = {
        "client",
        "bundle_id",
        "evidence_profile",
        "scenario_role",
        "profile_variant",
        "factor",
        "rubric_score_0_5",
        "meets_factor_adequacy",
        "evidence_seed",
    }

    missing = sorted(
        required - set(evidence_df.columns)
    )

    if missing:
        raise RuntimeError(
            "Controlled documentary-evidence table is missing required "
            f"column(s): {missing}. Available columns: "
            f"{sorted(evidence_df.columns.tolist())}"
        )

    work = evidence_df.copy()
    work["meets_factor_adequacy"] = (
        work["meets_factor_adequacy"]
        .astype(bool)
    )

    summary = (
        work
        .groupby(
            ["client", "bundle_id"],
            as_index=False,
        )
        .agg(
            evidence_profile=(
                "evidence_profile",
                "first",
            ),
            scenario_role=(
                "scenario_role",
                "first",
            ),
            profile_variant=(
                "profile_variant",
                "first",
            ),
            documentary_factor_count=(
                "factor",
                "count",
            ),
            documentary_adequate_factor_count=(
                "meets_factor_adequacy",
                "sum",
            ),
            documentary_mean_score=(
                "rubric_score_0_5",
                "mean",
            ),
            documentary_min_score=(
                "rubric_score_0_5",
                "min",
            ),
            documentary_max_score=(
                "rubric_score_0_5",
                "max",
            ),
            evidence_seed=(
                "evidence_seed",
                "first",
            ),
        )
        .sort_values("client")
        .reset_index(drop=True)
    )

    summary["documentary_adequacy_fraction"] = (
        summary["documentary_adequate_factor_count"]
        / summary["documentary_factor_count"].clip(lower=1)
    )

    expected_documentary_factors = sum(
        len(FACTOR_NAMES[d])
        for d in DOCUMENTARY_DIMS
    )

    count_ok = summary[
        "documentary_factor_count"
    ].eq(
        expected_documentary_factors
    )

    if not count_ok.all():
        bad = summary.loc[
            ~count_ok,
            [
                "client",
                "bundle_id",
                "documentary_factor_count",
            ],
        ]

        raise RuntimeError(
            "Unexpected controlled-evidence factor count. "
            f"Expected {expected_documentary_factors} documentary factors "
            "per client. Offending rows:\n"
            + bad.to_string(index=False)
        )

    return summary

def print_governance_details(
    gov: pd.DataFrame,
    ge: pd.DataFrame,
    dq: pd.DataFrame,
    vr_clients: List[str],
    sda_clients: List[str],
    ge_clients: List[str],
    domain: str,
):
    domain = str(domain).lower()

    print_banner(
        "DOMAIN ADEQUACY POLICY — FULL WAC + CRITICAL WAC"
    )

    policy_rows = []
    for dim in FACTOR_NAMES:
        policy_rows.append({
            "dimension":
                dim,
            "dimension_name":
                DIMENSION_NAMES[dim],
            "n_factors":
                len(FACTOR_NAMES[dim]),
            "minimum_adequacy_ranks":
                ",".join(
                    str(
                        DOMAIN_FACTOR_MINIMA[
                            domain
                        ][dim][f]
                    )
                    for f in FACTOR_NAMES[
                        dim
                    ]
                ),
            "dimension_policy_wac":
                DOMAIN_DIMENSION_WAC[
                    domain
                ][dim],
        })

    print(
        pd.DataFrame(
            policy_rows
        ).to_string(
            index=False
        )
    )

    global_critical_wac = derive_global_critical_wac(
        domain
    )

    print(
        f"\nGLOBAL {domain.upper()} WAC "
        f"(descriptive full-policy summary) = "
        f"{DOMAIN_GLOBAL_WAC[domain]:.6f}"
    )
    print(
        f"GLOBAL {domain.upper()} CRITICAL WAC "
        f"(automated Review threshold) = "
        f"{global_critical_wac:.6f}"
    )

    print_banner(
        "CRITICAL FACTORS — INDIVIDUAL ADEQUACY REQUIREMENTS"
    )

    critical_rows = []

    for dim, factor_list in (
        CRITICAL_FACTORS_BY_DOMAIN[
            domain
        ].items()
    ):
        for factor in factor_list:
            minimum = float(
                DOMAIN_FACTOR_MINIMA[
                    domain
                ][dim][factor]
            )
            critical_rows.append({
                "dimension":
                    dim,
                "factor":
                    factor,
                "individual_adequacy_min_0_5":
                    minimum,
                "normalized_reference":
                    minimum / MAX_FACTOR_SCORE,
                "direct_auto_accept_rule":
                    f"score >= {minimum:.1f}",
            })

    print(
        pd.DataFrame(
            critical_rows
        ).to_string(
            index=False
        )
    )

    print(
        f"\nMinimum dimension floor: EVERY averaged dimension "
        f"must be >= {DIMENSION_MIN_FLOOR:.1f}/5."
    )

    print(
        "Human reviewer role: verify uploaded questionnaire evidence only. "
        "The server makes the admission decision automatically."
    )

    print_banner(
        "TRAIN-ONLY DATA-QUALITY FACTORS — 8 FACTORS"
    )

    dq_cols = [
        "client",
        "completeness",
        "duplication_rate",
        "value_validity_error_rate",
        "type_consistency",
        "label_integrity",
        "feature_distribution_consistency",
        "feature_category_coverage",
        "structural_constraint_integrity",
    ]

    print(
        dq[dq_cols].to_string(
            index=False
        )
    )

    print_banner(
        "FROZEN TADP GOVERNANCE — HPS + CRITICAL WAC"
    )

    display_cols = [
        "client",
        "hps",
        "critical_wac_i",
        "global_critical_wac",
        "critical_wac_margin",
        "all_critical_meet_adequacy",
        "critical_below_adequacy",
        "dimension_floor_failures",
        "dim1_score_0_5",
        "dim2_score_0_5",
        "dim3_score_0_5",
        "dim4_score_0_5",
        "dim5_score_0_5",
        "dim6_score_0_5",
        "decision_path",
        "initial_action",
        "final_action",
        "status",
        "reason",
    ]

    print(
        gov[
            display_cols
        ].to_string(
            index=False
        )
    )

    print(
        "\nTADP v16.7 final decision order:"
    )
    print(
        f"  1) ANY averaged dimension < "
        f"{DIMENSION_MIN_FLOOR:.1f}/5 -> AUTO-REJECT"
    )
    print(
        f"  2) HPS < {GOOD_CUT:.1f} -> AUTO-REJECT"
    )
    print(
        f"  3) HPS >= {HIGH_CUT:.1f}:"
    )
    print(
        "       all critical factors meet their own adequacy minima "
        "-> DIRECT AUTO-ACCEPT"
    )
    print(
        "       otherwise -> AUTOMATED REVIEW fallback"
    )
    print(
        f"  4) {GOOD_CUT:.1f} <= HPS < "
        f"{HIGH_CUT:.1f} -> AUTOMATED REVIEW"
    )
    print(
        "  5) AUTOMATED REVIEW:"
    )
    print(
        f"       Critical WAC_i >= Global Critical WAC "
        f"({global_critical_wac:.6f}) -> ACCEPT AFTER REVIEW"
    )
    print(
        "       otherwise -> AUTO-REJECT"
    )

    print(
        f"\nFrozen TADP-VR cohort: "
        f"{len(vr_clients)}/10 -> "
        f"{vr_clients}"
    )
    print(
        f"Frozen TADP-SDA cohort: "
        f"{len(sda_clients)}/10 -> "
        f"{sda_clients}"
    )

    print_banner("GX CORE NATIVE TRAIN-ONLY VALIDATOR BASELINE")
    gx_base_cols = [
        "client", "gx_native_suite_success", "ge_expectations_passed",
        "ge_expectations_total", "ge_pass_rate", "ge_native_validation_class",
        "ge_final_action",
    ]
    gx_category_cols = [
        c for c in ge.columns
        if c.startswith("ge_") and (c.endswith("_passed") or c.endswith("_total"))
        and c not in {"ge_expectations_passed", "ge_expectations_total"}
    ]
    print(ge[gx_base_cols + sorted(gx_category_cols)].to_string(index=False))
    print(f"\nGX operationally valid clients (zero critical failures): {len(ge_clients)}/{len(ge)} -> {ge_clients}")
    print(
        "GX is used only as a native rule-based data validator. PASS/FAIL is the "
        "suite-level GX result; no ranking, no forced-K selection, and no TADP signal is used."
    )





def build_hash_chained_governance_ledger(
    gov,
    ge,
    output_path,
):
    rows = []
    prev_hash = "GENESIS"
    seq = 0

    for _, r in (
        gov.sort_values(
            "client"
        ).iterrows()
    ):
        seq += 1

        payload = {
            "sequence":
                seq,
            "governance_system":
                "TADP",
            "client":
                str(r["client"]),
            "hps":
                float(r["hps"]),
            "global_domain_wac":
                float(r["global_domain_wac"]),
            "global_critical_wac":
                float(r["global_critical_wac"]),
            "critical_wac_i": (
                float(r["critical_wac_i"])
                if np.isfinite(r["critical_wac_i"])
                else None
            ),
            "critical_wac_margin": (
                float(r["critical_wac_margin"])
                if np.isfinite(r["critical_wac_margin"])
                else None
            ),
            "all_critical_meet_adequacy":
                bool(r["all_critical_meet_adequacy"]),
            "critical_below_adequacy":
                str(r["critical_below_adequacy"]),
            "dimension_floor_failures":
                str(r["dimension_floor_failures"]),
            "decision_path":
                str(r["decision_path"]),
            "initial_action":
                str(r["initial_action"]),
            "final_action":
                str(r["final_action"]),
            "status":
                str(r["status"]),
            "reason":
                str(r["reason"]),
            "previous_hash":
                prev_hash,
        }

        canonical = json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
        )

        entry_hash = hashlib.sha256(
            canonical.encode("utf-8")
        ).hexdigest()

        payload["entry_hash"] = entry_hash

        rows.append(payload)
        prev_hash = entry_hash

    for _, r in (
        ge.sort_values(
            "client"
        ).iterrows()
    ):
        seq += 1

        payload = {
            "sequence":
                seq,
            "governance_system":
                "Great Expectations GX Core native validator",
            "client":
                str(r["client"]),
            "hps":
                None,
            "global_domain_wac":
                None,
            "global_critical_wac":
                None,
            "critical_wac_i":
                None,
            "critical_wac_margin":
                None,
            "all_critical_meet_adequacy":
                None,
            "critical_below_adequacy":
                None,
            "dimension_floor_failures":
                None,
            "decision_path":
                "RULE_BASED_VALIDATION",
            "initial_action":
                "RULE_BASED_VALIDATION",
            "final_action":
                str(r["ge_final_action"]),
            "status":
                "GE_CONTROLLED_TRAIN_ONLY",
            "reason": (
                f"GX native suite {'PASSED' if bool(r['gx_native_suite_success']) else 'FAILED'}; "
                f"{int(r['ge_expectations_passed'])}/{int(r['ge_expectations_total'])} "
                "configured technical Expectations passed"
            ),
            "previous_hash":
                prev_hash,
        }

        canonical = json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
        )

        entry_hash = hashlib.sha256(
            canonical.encode("utf-8")
        ).hexdigest()

        payload["entry_hash"] = entry_hash

        rows.append(payload)
        prev_hash = entry_hash

    out = pd.DataFrame(rows)

    out.to_csv(
        output_path,
        index=False,
    )

    return out

def choose_experiment_root(experiment_name, use_drive=True):
    if use_drive:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            root = Path("/content/drive/MyDrive/TADP_CHECKPOINTS") / experiment_name
            root.mkdir(parents=True, exist_ok=True)
            print(f"Persistent checkpoint root: {root}")
            return root
        except Exception as exc:
            print(f"Google Drive checkpoint mount unavailable: {exc}")
    root = Path("/content") / experiment_name
    root.mkdir(parents=True, exist_ok=True)
    print(f"Local checkpoint root: {root}")
    return root


def atomic_write_json(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")
    tmp.replace(path)


def load_checkpoint_state(path):
    path = Path(path)
    if not path.exists():
        return {"completed": [], "last_completed": None, "updated_utc": None}
    return json.loads(path.read_text(encoding="utf-8"))


def mark_checkpoint_complete(state_path, key, extra=None):
    state = load_checkpoint_state(state_path)
    completed = list(state.get("completed", []))
    if key not in completed:
        completed.append(key)
    state["completed"] = completed
    state["last_completed"] = key
    state["updated_utc"] = datetime.now(timezone.utc).isoformat()
    if extra:
        state.update(extra)
    atomic_write_json(state, state_path)


def upsert_csv(row, path, key_cols):
    path = Path(path)
    new = pd.DataFrame([row])
    if path.exists():
        old = pd.read_csv(path)
        if not old.empty:
            mask = pd.Series(True, index=old.index)
            for c in key_cols:
                mask &= old[c].astype(str).eq(str(row[c]))
            old = old.loc[~mask].copy()
            new = pd.concat([old, new], ignore_index=True)
    new.to_csv(path, index=False)


def write_scenario_checkpoint(root, run_idx, scenario, result):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", scenario).strip("_")
    cdir = ensure_dir(root / "scenario_checkpoints" / f"run_{run_idx:02d}")
    payload = {
        "run": int(run_idx),
        "scenario": scenario,
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "metrics": result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
    }
    atomic_write_json(payload, cdir / f"{safe}.json")
    if "round_audit" in result:
        pd.DataFrame(result["round_audit"]).to_csv(
            cdir / f"{safe}_rounds.csv", index=False
        )


def ci95_mean(values):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan
    if len(x) == 1:
        return float(x[0]), float(x[0])
    mean = float(np.mean(x))
    sd = float(np.std(x, ddof=1))
    try:
        from scipy.stats import t
        crit = float(t.ppf(0.975, df=len(x)-1))
    except Exception:
        crit = 1.96
    half = crit * sd / math.sqrt(len(x))
    return mean-half, mean+half


def summarize_runs_with_ci(perf):
    metrics = [
        "accuracy", "precision_macro", "recall_macro", "f1_macro",
        "roc_auc_ovr_macro", "runtime_s", "energy_wh", "energy_kwh",
        "co2_kg", "communication_mb", "ram_peak_mb", "ram_delta_mb",
        "optimizer_steps", "participants", "energy_cost_usd",
        "communication_cost_usd", "total_estimated_cost_usd",
    ]
    rows = []
    for scenario, d in perf.groupby("scenario", sort=False):
        row = {"scenario": scenario, "n_runs": int(len(d))}
        for metric in metrics:
            if metric not in d.columns:
                continue
            vals = pd.to_numeric(d[metric], errors="coerce")
            vals = vals[np.isfinite(vals)]
            row[f"{metric}_mean"] = float(vals.mean()) if len(vals) else np.nan
            row[f"{metric}_sd"] = float(vals.std(ddof=1)) if len(vals)>1 else 0.0
            lo, hi = ci95_mean(vals)
            row[f"{metric}_ci95_low"] = lo
            row[f"{metric}_ci95_high"] = hi
        rows.append(row)
    return pd.DataFrame(rows)


def paired_vr_randomk_statistics(perf):
    vr = perf[perf["scenario"].eq("TADP-VR Federated")].copy()
    rk = perf[perf["scenario"].eq("Random-K")].copy()
    merged = vr.merge(rk, on=["run", "seed"], suffixes=("_vr", "_randomk"), validate="one_to_one")
    rows = []
    for metric in ["accuracy", "f1_macro", "roc_auc_ovr_macro"]:
        a = merged[f"{metric}_vr"].to_numpy(dtype=float)
        b = merged[f"{metric}_randomk"].to_numpy(dtype=float)
        diff = a-b
        mean = float(np.mean(diff))
        sd = float(np.std(diff, ddof=1)) if len(diff)>1 else 0.0
        lo, hi = ci95_mean(diff)
        t_stat=t_p=wil_stat=wil_p=np.nan
        try:
            from scipy.stats import ttest_rel, wilcoxon
            tr = ttest_rel(a,b,nan_policy="omit")
            t_stat, t_p = float(tr.statistic), float(tr.pvalue)
            if np.any(np.abs(diff)>0):
                wr = wilcoxon(a,b)
                wil_stat, wil_p = float(wr.statistic), float(wr.pvalue)
        except Exception:
            pass
        rows.append({
            "metric": metric,
            "n_pairs": len(diff),
            "mean_difference_vr_minus_randomk": mean,
            "sd_difference": sd,
            "ci95_low": lo,
            "ci95_high": hi,
            "cohens_dz": float(mean/sd) if sd>0 else np.nan,
            "paired_t_stat": t_stat,
            "paired_t_p": t_p,
            "wilcoxon_stat": wil_stat,
            "wilcoxon_p": wil_p,
            "vr_wins": int(np.sum(diff>0)),
            "ties": int(np.sum(np.isclose(diff,0))),
            "vr_losses": int(np.sum(diff<0)),
        })
    return pd.DataFrame(rows)


def scenario_method_table():
    return pd.DataFrame([
        ["Naïve Centralized","centralized","baseline","all clients","5-seed headline"],
        ["Great Expectations Centralized","centralized","GX native validator","GX clients with zero critical failures","equivalence audit only"],
        ["TADP-AA Centralized","centralized","TADP-AA","all clients","equivalence audit only"],
        ["TADP-VR Centralized","centralized","TADP-VR","frozen TADP-VR cohort","5-seed headline"],
        ["TADP-SDA Centralized","centralized","TADP-SDA","frozen best eligible client","one-seed boundary"],
        ["Vanilla FedAvg","federated","FedAvg","all clients","5-seed headline"],
        ["FedProx","federated","FedProx","all clients","5-seed headline"],
        ["Random-K","federated","matched random control","same K/rounds/steps as TADP-VR","5-seed headline"],
        ["Great Expectations Federated","federated","GX native validator","GX clients with zero critical failures","equivalence audit only"],
        ["DQ-only Federated","federated","DQ-only","top-K by machine-measured DQ; same K/rounds/steps as TADP-VR","5-seed headline"],
        ["Power-of-Choice","federated","Power-of-Choice","dynamic loss-based selection; same K/rounds/steps as TADP-VR","5-seed headline"],
        ["TADP-AA Federated","federated","TADP-AA","all clients","equivalence audit only"],
        ["TADP-VR Federated","federated","TADP-VR","frozen TADP-VR cohort","5-seed headline"],
        ["TADP-SDA Federated","federated","TADP-SDA","frozen best eligible client","one-seed boundary"],
    ], columns=["scenario","paradigm","method","participation","execution_role"])



def governance_only_monte_carlo(
    client_ids,
    dq_scores,
    base_seed,
    n_realizations=1000,
    domain="healthcare",
):
    domain = str(domain).lower()
    rows=[]
    for j in range(int(n_realizations)):
        seed=int(base_seed+j)
        evidence,_=generate_controlled_documentary_evidence(
            client_ids,
            seed,
            DOMAIN_FACTOR_MINIMA[domain],
            domain=domain,
        )
        gov=build_tadp_governance(
            client_ids,
            evidence,
            dq_scores,
            run=0,
            evidence_seed=seed,
            domain=domain,
        )
        accepted=accepted_tadp_vr(gov)
        rows.append({
            "realization":j+1,
            "evidence_seed":seed,
            "accepted_count":len(accepted),
            "accepted_clients":";".join(accepted),
            "mean_hps":float(gov["hps"].mean()),
            "mean_critical_wac_i":float(gov["critical_wac_i"].mean()),
            "dimension_floor_rejects":int(gov["status"].eq("AUTO_REJECTED_DIMENSION_FLOOR").sum()),
            "low_hps_rejects":int(gov["status"].eq("AUTO_REJECTED_LOW_HPS").sum()),
            "review_accepts":int(
                gov["status"].eq("ACCEPTED_AFTER_AUTOMATED_REVIEW").sum()
            ),
            "direct_auto_accepts":int(
                gov["status"].eq("DIRECT_AUTO_ACCEPTED").sum()
            ),
        })
    return pd.DataFrame(rows)


# ======================================================================================
# EXPERIMENT B3 — PREDICTIVE UTILITY UNDER CLIENT SCALING
# ======================================================================================
#
# PURPOSE
# -------
# Complement Experiment B1/B2 (governance-only scalability) with downstream
# predictive-utility scalability.
#
# Tested client counts:
#     K = {20, 50, 100}
#
# K=10 is intentionally not rerun here because it is already evaluated in the
# final Experiment-A protocol. For cross-K manuscript plots, use only the
# corresponding 4-round Experiment-A K=10 rows for seeds {42,142,242}.
#
# Three scenarios are intentionally retained:
#
#   1) Vanilla FedAvg
#      - all submitted clients participate
#      - full-participation predictive-utility reference
#
#   2) Random-K
#      - same number of participating clients as TADP-VR
#      - exact same total optimizer-step budget per round as TADP-VR
#      - frozen random cohort within each K, across all rounds and seeds
#
#   3) TADP-VR Federated
#      - clients admitted by the frozen TADP governance policy
#      - natural one-local-epoch step budget
#
# FAIRNESS / INTERPRETATION
# -------------------------
# TADP-VR vs Random-K is the matched-compute selection comparison:
#   same K_selected, same FL rounds, same total optimizer steps per round,
#   same model architecture, same initial weights within seed, same batch size,
#   same optimizer, same global TEST set.
#
# Vanilla FedAvg is NOT step-matched to TADP-VR because it is the intended
# full-participation utility reference. Its larger compute/communication budget
# is reported rather than hidden.
#
# The experiment tests whether TADP-VR retains useful predictive performance
# as the submitted contributor population grows, while using fewer clients
# than full FedAvg and while being compared fairly against a matched random
# subset.
#
# The held-out TEST set is used only after training for final evaluation.
# It is never used for governance, preprocessing, client selection, step-budget
# construction, or checkpoint decisions.
# ======================================================================================

import platform
import shutil
from datetime import datetime, timezone

EXPERIMENT_VERSION = "TADP-B3-v16.9-PREDICTIVE-SCALABILITY-K20-50-100-3SEED-4ROUND"

CLIENT_COUNTS = [20, 50, 100]
TRAINING_RUN_SEEDS = [42, 142, 242]

NUM_ROUNDS_FL = 4
LOCAL_EPOCHS = 1
BATCH_SIZE = 64
LEARNING_RATE = 1e-3

DIRICHLET_ALPHA = 1.0
MIN_CLIENT_RECORDS = 30

GLOBAL_SPLIT_SEED = 7001
PARTITION_BASE_SEED = 9101
EVIDENCE_BASE_SEED = 12042
RANDOMK_BASE_SEED = 22042

DOMAIN = "healthcare"
USE_GOOGLE_DRIVE_CHECKPOINTS = True

SCENARIOS = [
    "Vanilla FedAvg",
    "Random-K",
    "TADP-VR Federated",
]

EXPERIMENT_ROOT = choose_experiment_root(
    "TADP_EXPERIMENT_B3_" + EXPERIMENT_VERSION,
    use_drive=USE_GOOGLE_DRIVE_CHECKPOINTS,
)
CHECKPOINT_STATE = EXPERIMENT_ROOT / "checkpoint_state.json"
PERF_CHECKPOINT = EXPERIMENT_ROOT / "performance_metrics_checkpoint.csv"


# ======================================================================================
# SCALABLE CLIENT PARTITIONING / CONTROLLED EVIDENCE
# ======================================================================================

def scalable_client_ids(n_clients: int) -> List[str]:
    return [f"C{i:03d}" for i in range(1, int(n_clients) + 1)]


def dirichlet_partition_dataframe_scalable(
    train_df: pd.DataFrame,
    n_clients: int,
    alpha: float,
    seed: int,
    min_client_records: int = 30,
    max_attempts: int = 500,
) -> Dict[str, pd.DataFrame]:
    """
    Label-wise Dirichlet partition that supports K > 26 while preserving every
    global TRAIN row exactly once.
    """
    y = train_df["_target"].to_numpy()
    client_ids = scalable_client_ids(n_clients)

    for attempt in range(int(max_attempts)):
        rng = np.random.default_rng(int(seed + attempt))
        buckets = [[] for _ in range(int(n_clients))]

        for cls in sorted(np.unique(y)):
            idx = np.where(y == cls)[0]
            rng.shuffle(idx)
            props = rng.dirichlet(np.full(int(n_clients), float(alpha)))
            cuts = (np.cumsum(props)[:-1] * len(idx)).astype(int)
            splits = np.split(idx, cuts)
            for j, split in enumerate(splits):
                buckets[j].extend(split.tolist())

        sizes = [len(b) for b in buckets]
        if min(sizes) >= int(min_client_records):
            return {
                cid: train_df.iloc[np.asarray(idxs, dtype=int)].copy()
                for cid, idxs in zip(client_ids, buckets)
            }

    raise RuntimeError(
        f"Could not obtain valid K={n_clients} partition after {max_attempts} attempts "
        f"with min_client_records={min_client_records}."
    )


def partition_integrity_audit(
    train_df: pd.DataFrame,
    clients: Dict[str, pd.DataFrame],
) -> Dict[str, Any]:
    global_ids = set(train_df["_row_id"].astype(int).tolist())
    all_ids = []
    for df in clients.values():
        all_ids.extend(df["_row_id"].astype(int).tolist())
    client_set = set(all_ids)

    result = {
        "global_train_rows": int(len(train_df)),
        "client_rows_total": int(len(all_ids)),
        "unique_client_rows": int(len(client_set)),
        "rows_missing_from_clients": int(len(global_ids - client_set)),
        "rows_not_in_global_train": int(len(client_set - global_ids)),
        "duplicate_rows_across_clients": int(len(all_ids) - len(client_set)),
    }
    result["pass"] = bool(
        len(all_ids) == len(train_df)
        and client_set == global_ids
        and len(all_ids) == len(client_set)
    )
    return result


def _evidence_factor_signature(factors: Dict[str, Dict[str, float]]) -> Tuple[float, ...]:
    """Stable factor-level signature used to guarantee 20 distinct evidence bundles."""
    return tuple(
        float(factors[dim][factor])
        for dim in DOCUMENTARY_DIMS
        for factor in FACTOR_NAMES[dim]
    )


def build_20_profile_documentary_evidence_library(
    library_seed: int,
    domain: str = "healthcare",
) -> Tuple[List[Dict[str, Any]], pd.DataFrame]:
    """
    Build exactly 20 DISTINCT documentary-evidence bundles before any client is
    assigned an evidence profile.

    The library deliberately covers a broad but policy-valid range of evidence
    strengths. It contains 12 profiles expected to be governance-eligible under
    typical TRAIN-measured DQ values and 8 weaker profiles. This gives an expected
    admission tendency near 60% without forcing the realized K=20 admission rate.

    IMPORTANT:
      * The 20 bundles are generated independently of client identity, TEST data,
        model loss, and downstream predictive performance.
      * DQ (dim2) is never synthesized here; it remains machine-measured from each
        client's TRAIN shard.
      * The library is frozen by ``library_seed`` and then reused unchanged by
        both RQ5 and RQ6.
    """
    domain = str(domain).lower()
    if domain not in DOMAIN_FACTOR_MINIMA:
        raise ValueError(f"Unsupported domain: {domain!r}")

    rng = np.random.default_rng(int(library_seed))
    factor_minima = DOMAIN_FACTOR_MINIMA[domain]
    critical_policy = CRITICAL_FACTORS_BY_DOMAIN[domain]
    critical_sequence = [
        (dim, factor)
        for dim, factor_list in critical_policy.items()
        for factor in factor_list
    ]
    critical_keys = set(critical_sequence)

    # 20 distinct profiles. The role mix is specified before client assignment.
    # 7 + 5 = 12 profiles are designed around the accepted/review-recoverable
    # region; the remaining 8 exercise rejection paths. Because clients sample
    # WITH REPLACEMENT, the realized admitted percentage is not forced to 60%.
    profile_specs = (
        [("DIRECT_STRONG", v) for v in range(7)]
        + [("REVIEW_RECOVERABLE", v) for v in range(5)]
        + [("REVIEW_LIMITED", v) for v in range(3)]
        + [("LOW_HPS_WEAK", v) for v in range(3)]
        + [("DIMENSION_FLOOR_WEAK", v) for v in range(2)]
    )
    if len(profile_specs) != 20:
        raise AssertionError("The evidence library must contain exactly 20 profiles")

    def make_profile(role: str, variant: int) -> Dict[str, Dict[str, float]]:
        factors = {dim: {} for dim in DOCUMENTARY_DIMS}

        if role == "DIRECT_STRONG":
            # Strong evidence with factor-level variation. Critical factors always
            # remain at or above their declared adequacy minima.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    minimum = float(factor_minima[dim][factor])
                    low = int(max(4.0, minimum))
                    score = low if low >= 5 else int(rng.integers(low, 6))
                    factors[dim][factor] = float(score)

        elif role == "REVIEW_RECOVERABLE":
            # Moderate evidence. At least one critical factor is set below its
            # individual minimum, while another is strengthened so that the
            # compensatory CWAC review path is meaningfully exercised.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = float(rng.choice([3, 3, 3, 4]))

            for dim, factor in critical_sequence:
                factors[dim][factor] = float(factor_minima[dim][factor])

            high_min = [
                key for key in critical_sequence
                if float(factor_minima[key[0]][key[1]]) >= 4.0
            ]
            compensators = [
                key for key in critical_sequence
                if float(factor_minima[key[0]][key[1]]) <= 3.0
            ]
            weak_key = high_min[variant % len(high_min)]
            comp_key = compensators[(variant + 1) % len(compensators)]
            weak_min = float(factor_minima[weak_key[0]][weak_key[1]])
            comp_min = float(factor_minima[comp_key[0]][comp_key[1]])
            factors[weak_key[0]][weak_key[1]] = float(max(0.0, weak_min - 1.0))
            factors[comp_key[0]][comp_key[1]] = float(min(5.0, comp_min + 2.0))

        elif role == "REVIEW_LIMITED":
            # Dimensions remain around the review region, but critical evidence is
            # deliberately weaker than the domain adequacy reference.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    base = 4 if (dim, factor) not in critical_keys and rng.random() < 0.45 else 3
                    factors[dim][factor] = float(base)
            for dim, factor in critical_sequence:
                minimum = float(factor_minima[dim][factor])
                drop = 1.0 + (1.0 if ((variant + len(factor)) % 3 == 0) else 0.0)
                factors[dim][factor] = float(max(2.0, minimum - drop))

        elif role == "LOW_HPS_WEAK":
            # Every documentary dimension stays at or slightly above the 2.5
            # dimension floor, but the overall documentary evidence remains weak.
            for dim in DOCUMENTARY_DIMS:
                names = list(FACTOR_NAMES[dim])
                n_three = (len(names) + 1) // 2
                values = [3.0] * n_three + [2.0] * (len(names) - n_three)
                rng.shuffle(values)
                # Variant-specific small change that preserves the weak region.
                if variant == 1 and len(values) >= 4:
                    values[0], values[-1] = values[-1], values[0]
                elif variant == 2 and len(values) >= 3:
                    values = values[1:] + values[:1]
                for factor, value in zip(names, values):
                    factors[dim][factor] = float(value)

        elif role == "DIMENSION_FLOOR_WEAK":
            # Keep most evidence moderate but deliberately push one documentary
            # dimension below the 2.5 floor. Two variants use different dimensions.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = float(rng.choice([3, 3, 4]))
            weak_dim = ["dim4", "dim3"][variant % 2]
            weak_names = list(FACTOR_NAMES[weak_dim])
            for j, factor in enumerate(weak_names):
                factors[weak_dim][factor] = float(1.0 if j == (variant % len(weak_names)) else 2.0)

        else:
            raise ValueError(f"Unknown evidence role: {role!r}")

        return factors

    library: List[Dict[str, Any]] = []
    seen = set()

    for idx, (role, variant) in enumerate(profile_specs, start=1):
        # Very large discrete space; collisions are unlikely, but fail closed and
        # regenerate if one occurs so the library really contains 20 profiles.
        factors = None
        signature = None
        for _attempt in range(200):
            candidate = make_profile(role, int(variant))
            sig = _evidence_factor_signature(candidate)
            if sig not in seen:
                factors, signature = candidate, sig
                break
        if factors is None:
            raise RuntimeError("Could not generate 20 unique documentary evidence profiles")
        seen.add(signature)
        library.append({
            "library_profile_id": f"P{idx:02d}",
            "profile_role": role,
            "profile_variant": int(variant),
            "factors": factors,
        })

    if len(library) != 20 or len(seen) != 20:
        raise RuntimeError("Evidence-library uniqueness audit failed")

    rows = []
    for entry in library:
        for dim in DOCUMENTARY_DIMS:
            for factor, score in entry["factors"][dim].items():
                min_rank = float(factor_minima[dim][factor])
                rows.append({
                    "library_profile_id": entry["library_profile_id"],
                    "evidence_profile": entry["profile_role"],
                    "scenario_role": entry["profile_role"],
                    "profile_variant": int(entry["profile_variant"]),
                    "dimension": dim,
                    "dimension_name": DIMENSION_NAMES[dim],
                    "factor": factor,
                    "rubric_score_0_5": float(score),
                    "rubric_descriptor": rubric_descriptor(dim, factor, score),
                    "adequacy_min_rank": min_rank,
                    "meets_factor_adequacy": bool(float(score) >= min_rank),
                    "evidence_library_seed": int(library_seed),
                    "domain": domain,
                })

    library_df = pd.DataFrame(rows)
    return library, library_df


def generate_random_library_controlled_evidence(
    client_ids: List[str],
    evidence_seed: int,
    library_seed: int,
    domain: str = "healthcare",
) -> Tuple[Dict[str, Dict[str, Dict[str, float]]], pd.DataFrame, pd.DataFrame]:
    """
    Assign one of 20 distinct documentary evidence profiles independently to
    every client WITH REPLACEMENT.

    Therefore:
      * two or more clients may receive the same documentary evidence profile;
      * some of the 20 profiles may not be sampled in a particular realization;
      * the admitted-client percentage is an observed result, not a fixed design
        target or a repeated 10-client block.

    The sampled assignment is frozen before training and reused by RQ5 and RQ6.
    """
    library, library_df = build_20_profile_documentary_evidence_library(
        library_seed=library_seed,
        domain=domain,
    )
    if len(library) != 20:
        raise RuntimeError("Expected exactly 20 evidence profiles")

    rng = np.random.default_rng(int(evidence_seed))
    draw_idx = rng.integers(0, len(library), size=len(client_ids))

    profile_ids = [library[int(i)]["library_profile_id"] for i in draw_idx]
    usage_counts = pd.Series(profile_ids).value_counts().to_dict()

    evidence_by_client: Dict[str, Dict[str, Dict[str, float]]] = {}
    rows = []

    for draw_position, (cid, lib_idx) in enumerate(zip(client_ids, draw_idx), start=1):
        entry = library[int(lib_idx)]
        evidence_by_client[str(cid)] = {
            dim: dict(entry["factors"][dim])
            for dim in DOCUMENTARY_DIMS
        }

        for dim in DOCUMENTARY_DIMS:
            for factor, score in entry["factors"][dim].items():
                min_rank = float(DOMAIN_FACTOR_MINIMA[domain][dim][factor])
                rows.append({
                    "client": str(cid),
                    "assignment_draw_position": int(draw_position),
                    "library_profile_id": entry["library_profile_id"],
                    "bundle_id": entry["library_profile_id"],
                    "evidence_profile": entry["profile_role"],
                    "scenario_role": entry["profile_role"],
                    "profile_variant": int(entry["profile_variant"]),
                    "times_profile_sampled_in_realization": int(usage_counts[entry["library_profile_id"]]),
                    "assignment_sampling": "UNIFORM_WITH_REPLACEMENT",
                    "dimension": dim,
                    "dimension_name": DIMENSION_NAMES[dim],
                    "factor": factor,
                    "evidence_source_type": "CONTROLLED_DOCUMENTARY_EVIDENCE_20_PROFILE_LIBRARY",
                    "verified_rubric_level_0_5": float(score),
                    "rubric_score_0_5": float(score),
                    "server_mapped_score_0_5": float(score),
                    "rubric_descriptor": rubric_descriptor(dim, factor, score),
                    "human_role": "VERIFY_EVIDENCE_ONLY",
                    "score_assignment": "DETERMINISTIC_SERVER_MAPPING_FROM_VERIFIED_RUBRIC_LEVEL",
                    "admission_decision_by": "SERVER_POLICY",
                    "adequacy_min_rank": min_rank,
                    "adequacy_min_normalized": min_rank / MAX_FACTOR_SCORE,
                    "meets_factor_adequacy": bool(float(score) >= min_rank),
                    "evidence_artifact_id": f"{cid}-{entry['library_profile_id']}-{dim}-{factor}",
                    "validation_status": "CONTROLLED_RANDOM_LIBRARY_EVIDENCE",
                    "evidence_seed": int(evidence_seed),
                    "evidence_library_seed": int(library_seed),
                    "domain": domain,
                })

    assignment_df = pd.DataFrame(rows)

    # Fail-closed assignment audit: exactly one sampled profile per client.
    one = assignment_df[["client", "library_profile_id"]].drop_duplicates()
    per_client = one.groupby("client")["library_profile_id"].nunique()
    if len(per_client) != len(client_ids) or not (per_client == 1).all():
        raise RuntimeError("Evidence assignment audit failed: each client must receive exactly one profile")

    return evidence_by_client, assignment_df, library_df


def summarize_random_library_assignment(evidence_df: pd.DataFrame) -> pd.DataFrame:
    """One-row-per-client audit of the frozen with-replacement evidence draw."""
    cols = [
        "client", "library_profile_id", "evidence_profile", "profile_variant",
        "times_profile_sampled_in_realization", "assignment_sampling",
        "evidence_seed", "evidence_library_seed",
    ]
    out = evidence_df[cols].drop_duplicates("client").sort_values("client").reset_index(drop=True)
    return out


def prepare_k_data(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    k: int,
    partition_seed: int,
):
    clients = dirichlet_partition_dataframe_scalable(
        train_df=train_df,
        n_clients=k,
        alpha=DIRICHLET_ALPHA,
        seed=partition_seed,
        min_client_records=MIN_CLIENT_RECORDS,
    )

    integrity = partition_integrity_audit(train_df, clients)
    if not integrity["pass"]:
        raise RuntimeError(f"Partition integrity failed at K={k}: {integrity}")

    pre = fit_federated_train_only_preprocessor(clients)
    reference = build_tabular_reference(clients, pre)

    dq_scores = {}
    dq_rows = []
    client_arrays = {}

    for cid, df in clients.items():
        scores, raw_metrics = tabular_dq_scores(df, pre, reference)
        dq_scores[cid] = scores
        dq_rows.append({"client": cid, **scores, **raw_metrics})

        Xc = pre.transform(df)
        yc = df["_target"].to_numpy(dtype=np.int32)
        client_arrays[cid] = (Xc, yc)

    X_test = pre.transform(test_df)
    y_test = test_df["_target"].to_numpy(dtype=np.int32)

    y_train_all = np.concatenate(
        [client_arrays[cid][1] for cid in sorted(client_arrays)]
    )
    class_weights = class_weight_dict(y_train_all)

    return {
        "clients_raw": clients,
        "client_ids": list(clients.keys()),
        "client_arrays": client_arrays,
        "preprocessor": pre,
        "dq_reference": reference,
        "dq_scores": dq_scores,
        "dq_audit": pd.DataFrame(dq_rows),
        "X_test": X_test,
        "y_test": y_test,
        "class_weights": class_weights,
        "input_dim": int(X_test.shape[1]),
        "integrity": integrity,
    }


# ======================================================================================
# SUMMARIES / CHECKPOINTS
# ======================================================================================

def summarize_b3(perf: pd.DataFrame) -> pd.DataFrame:
    metrics = [
        "accuracy",
        "precision_macro",
        "recall_macro",
        "f1_macro",
        "roc_auc_ovr_macro",
        "runtime_s",
        "communication_mb",
        "optimizer_steps",
        "participants",
        "energy_wh",
        "ram_peak_mb",
        "ram_delta_mb",
    ]
    rows = []
    for (k, scenario), d in perf.groupby(
        ["k_submissions", "scenario"], sort=True
    ):
        row = {
            "k_submissions": int(k),
            "scenario": str(scenario),
            "n_runs": int(len(d)),
        }
        for metric in metrics:
            if metric not in d.columns:
                continue
            vals = pd.to_numeric(d[metric], errors="coerce").to_numpy(dtype=float)
            vals = vals[np.isfinite(vals)]
            row[f"{metric}_mean"] = (
                float(np.mean(vals)) if len(vals) else np.nan
            )
            row[f"{metric}_sd"] = (
                float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            )
            lo, hi = ci95_mean(vals)
            row[f"{metric}_ci95_low"] = lo
            row[f"{metric}_ci95_high"] = hi
        rows.append(row)
    return pd.DataFrame(rows)


def paired_b3(perf: pd.DataFrame) -> pd.DataFrame:
    """
    Paired seed-wise differences.

    TADP-VR vs Random-K:
        matched K and matched optimizer-step budget.

    TADP-VR vs Vanilla FedAvg:
        utility comparison against full participation; NOT compute-matched.
    """
    rows = []
    comparisons = [
        ("Random-K", "MATCHED_SELECTION_CONTROL"),
        ("Vanilla FedAvg", "FULL_PARTICIPATION_REFERENCE"),
    ]

    for k in sorted(perf["k_submissions"].unique()):
        tadp = perf[
            (perf["k_submissions"].eq(k))
            & (perf["scenario"].eq("TADP-VR Federated"))
        ].copy()

        for baseline, role in comparisons:
            base = perf[
                (perf["k_submissions"].eq(k))
                & (perf["scenario"].eq(baseline))
            ].copy()

            merged = tadp.merge(
                base,
                on=["k_submissions", "seed"],
                suffixes=("_tadp", "_baseline"),
                validate="one_to_one",
            )

            for metric in [
                "accuracy",
                "precision_macro",
                "recall_macro",
                "f1_macro",
                "roc_auc_ovr_macro",
            ]:
                a = merged[f"{metric}_tadp"].to_numpy(dtype=float)
                b = merged[f"{metric}_baseline"].to_numpy(dtype=float)
                diff = a - b
                lo, hi = ci95_mean(diff)

                rows.append({
                    "k_submissions": int(k),
                    "baseline": baseline,
                    "baseline_role": role,
                    "metric": metric,
                    "n_pairs": int(len(diff)),
                    "mean_difference_tadp_minus_baseline":
                        float(np.mean(diff)),
                    "sd_difference":
                        float(np.std(diff, ddof=1)) if len(diff) > 1 else 0.0,
                    "ci95_low": lo,
                    "ci95_high": hi,
                    "tadp_wins": int(np.sum(diff > 0)),
                    "ties": int(np.sum(np.isclose(diff, 0.0))),
                    "tadp_losses": int(np.sum(diff < 0)),
                })

    return pd.DataFrame(rows)


def utility_retention_table(summary: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for k in sorted(summary["k_submissions"].unique()):
        d = summary[summary["k_submissions"].eq(k)].set_index("scenario")
        if "TADP-VR Federated" not in d.index or "Vanilla FedAvg" not in d.index:
            continue

        row = {"k_submissions": int(k)}
        for metric in ["accuracy", "f1_macro", "roc_auc_ovr_macro"]:
            tadp = float(d.loc["TADP-VR Federated", f"{metric}_mean"])
            full = float(d.loc["Vanilla FedAvg", f"{metric}_mean"])
            rand = float(d.loc["Random-K", f"{metric}_mean"])

            row[f"tadp_{metric}_mean"] = tadp
            row[f"fedavg_{metric}_mean"] = full
            row[f"randomk_{metric}_mean"] = rand
            row[f"tadp_minus_fedavg_{metric}"] = tadp - full
            row[f"tadp_minus_randomk_{metric}"] = tadp - rand
            row[f"tadp_retention_pct_of_fedavg_{metric}"] = (
                100.0 * tadp / full if np.isfinite(full) and full != 0 else np.nan
            )

        row["tadp_participants_mean"] = float(
            d.loc["TADP-VR Federated", "participants_mean"]
        )
        row["fedavg_participants_mean"] = float(
            d.loc["Vanilla FedAvg", "participants_mean"]
        )
        row["randomk_participants_mean"] = float(
            d.loc["Random-K", "participants_mean"]
        )
        row["tadp_communication_mb_mean"] = float(
            d.loc["TADP-VR Federated", "communication_mb_mean"]
        )
        row["fedavg_communication_mb_mean"] = float(
            d.loc["Vanilla FedAvg", "communication_mb_mean"]
        )
        row["communication_reduction_pct_vs_fedavg"] = (
            100.0
            * (
                float(d.loc["Vanilla FedAvg", "communication_mb_mean"])
                - float(d.loc["TADP-VR Federated", "communication_mb_mean"])
            )
            / max(
                float(d.loc["Vanilla FedAvg", "communication_mb_mean"]),
                1e-12,
            )
        )
        rows.append(row)

    return pd.DataFrame(rows)


def write_b3_scenario_checkpoint(
    root: Path,
    k: int,
    run_idx: int,
    scenario: str,
    result: Dict[str, Any],
):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", scenario).strip("_")
    cdir = ensure_dir(
        root
        / "scenario_checkpoints"
        / f"K_{int(k):03d}"
        / f"run_{int(run_idx):02d}"
    )
    payload = {
        "k_submissions": int(k),
        "run": int(run_idx),
        "scenario": str(scenario),
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "metrics": result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
    }
    atomic_write_json(payload, cdir / f"{safe}.json")
    if "round_audit" in result:
        pd.DataFrame(result["round_audit"]).to_csv(
            cdir / f"{safe}_rounds.csv",
            index=False,
        )


def save_environment_metadata():
    meta = {
        "experiment_version": EXPERIMENT_VERSION,
        "python": sys.version,
        "platform": platform.platform(),
        "tensorflow": tf.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "client_counts": CLIENT_COUNTS,
        "training_seeds": TRAINING_RUN_SEEDS,
        "fl_rounds": NUM_ROUNDS_FL,
        "local_epochs": LOCAL_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "dirichlet_alpha": DIRICHLET_ALPHA,
        "global_split_seed": GLOBAL_SPLIT_SEED,
        "partition_seed_formula": "PARTITION_BASE_SEED + 101*K",
        "evidence_seed_formula": "EVIDENCE_BASE_SEED + 103*K",
        "randomk_seed_formula": "RANDOMK_BASE_SEED + 107*K",
        "test_semantics":
            "final evaluation only; never used for governance, selection, "
            "preprocessing, step budgets, or checkpoint decisions",
        "scenario_roles": {
            "Vanilla FedAvg":
                "full-participation predictive-utility reference; not compute matched",
            "Random-K":
                "same selected-client count and exact total optimizer-step budget per round as TADP-VR",
            "TADP-VR Federated":
                "governance-selected cohort",
        },
        "k10_note":
            "K=10 intentionally omitted here; use final Experiment-A 4-round rows "
            "for seeds 42,142,242 when constructing the manuscript K=10/20/50/100 plot.",
    }
    atomic_write_json(
        meta,
        EXPERIMENT_ROOT / "b3_experiment_design.json",
    )


# ======================================================================================
# RQ5 — TRUST-DIMENSION ABLATION (K=20, 20-profile evidence library, with replacement)
# ======================================================================================
#
# Research question:
#   How does removing individual trust dimensions affect governance decisions and
#   downstream predictive utility?
#
# IMPORTANT DESIGN NOTE
# ---------------------
# This rerun does NOT change the experiment because of a preferred outcome. It reports
# the full-policy leave-one-dimension-out (LODO) result even if the effect is small or
# concentrated in one dimension. A second, clearly labelled HPS-only ablation is added
# to explain whether a dimension matters through the weighted HPS itself or through the
# wider policy gates (dimension floor / critical-factor checks).
#
# Primary analysis:
#   FULL_POLICY_LODO
#     - remove one dimension from HPS;
#     - renormalize remaining HPS weights;
#     - remove that dimension from the dimension-floor gate;
#     - remove critical factors belonging to that dimension.
#
# Secondary diagnostic:
#   HPS_ONLY_ZERO_WEIGHT
#     - set one dimension's HPS weight to zero;
#     - renormalize remaining HPS weights;
#     - KEEP all dimension-floor and critical-factor requirements unchanged.
#
# Frozen K=20 sensitivity setting:
#   20 distinct documentary evidence profiles are generated first; each client
#   samples one profile independently WITH REPLACEMENT using a frozen seed.
#   K=20, assignment seed=14102, library seed=17220, partition seed=11121, split seed=7001,
#   training seeds=[42,142,242,342,442], 4 FL rounds, 1 local epoch, batch=64.
# ======================================================================================

import matplotlib.pyplot as plt

EXPERIMENT_VERSION = "TADP-RQ5-v17.3-K20-20EVIDENCE-WR-DIMENSION-ABLATION-5SEED-4ROUND"

K_SUBMISSIONS = 20
TRAINING_RUN_SEEDS = [42, 142, 242, 342, 442]
NUM_ROUNDS_FL = 4
LOCAL_EPOCHS = 1
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
DIRICHLET_ALPHA = 1.0

GLOBAL_SPLIT_SEED = 7001
CLIENT_PARTITION_SEED = 11121
FROZEN_EVIDENCE_ASSIGNMENT_SEED = 14102
EVIDENCE_LIBRARY_SEED = 17220
EVIDENCE_LIBRARY_SIZE = 20
EVIDENCE_ASSIGNMENT_SAMPLING = "UNIFORM_WITH_REPLACEMENT"
DOMAIN = "healthcare"

REFERENCE_LOWER_CUT = 3.0
REFERENCE_UPPER_CUT = 3.5
REFERENCE_DIMENSION_FLOOR = 2.5

USE_GOOGLE_DRIVE_CHECKPOINTS = True
EXPERIMENT_ROOT = choose_experiment_root(
    "TADP_EXPERIMENT_RQ5_" + EXPERIMENT_VERSION,
    use_drive=USE_GOOGLE_DRIVE_CHECKPOINTS,
)
CHECKPOINT_STATE = EXPERIMENT_ROOT / "checkpoint_state.json"
UNIQUE_COHORT_PERF = EXPERIMENT_ROOT / "unique_cohort_performance_checkpoint.csv"

DIMS = [f"dim{i}" for i in range(1, 7)]


def _bool_series_rq5(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s
    return s.astype(str).str.lower().isin(["true", "1", "yes"])


def _jaccard_rq5(a, b) -> float:
    a = set(map(str, a))
    b = set(map(str, b))
    u = a | b
    return 1.0 if not u else float(len(a & b) / len(u))


def build_policy_tables_rq5(full_factor_df: pd.DataFrame):
    f = full_factor_df.copy()
    f["client"] = f["client"].astype(str)
    f["dimension"] = f["dimension"].astype(str)
    f["rubric_score_0_5"] = pd.to_numeric(f["rubric_score_0_5"], errors="raise")

    dim_scores = (
        f.groupby(["client", "dimension"])["rubric_score_0_5"]
        .mean()
        .unstack("dimension")
        .sort_index()
        .reindex(columns=DIMS)
    )

    crit = f.loc[_bool_series_rq5(f["is_critical_factor"])].copy()
    crit["critical_adequacy_min_rank"] = pd.to_numeric(
        crit["critical_adequacy_min_rank"], errors="coerce"
    )
    crit = crit.loc[crit["critical_adequacy_min_rank"].notna()].copy()
    return f, dim_scores, crit


def _renormalize_without_dim(removed_dim: str) -> dict:
    active = [d for d in DIMS if d != removed_dim]
    denom = float(sum(float(WEIGHTS_PSCORE_DEFAULT[d]) for d in active))
    return {
        d: (0.0 if d == removed_dim else float(WEIGHTS_PSCORE_DEFAULT[d]) / denom)
        for d in DIMS
    }


def evaluate_rq5_policy(
    dim_scores: pd.DataFrame,
    crit: pd.DataFrame,
    mode: str = "REFERENCE",
    removed_dim: str | None = None,
) -> tuple[pd.DataFrame, dict]:
    """Evaluate reference, HPS-only, or full-policy dimension ablation."""
    if mode not in {"REFERENCE", "HPS_ONLY_ZERO_WEIGHT", "FULL_POLICY_LODO"}:
        raise ValueError(f"Unknown RQ5 ablation mode: {mode}")
    if mode != "REFERENCE" and removed_dim not in DIMS:
        raise ValueError(f"Invalid removed_dim={removed_dim}")

    if mode == "REFERENCE":
        weights = {d: float(WEIGHTS_PSCORE_DEFAULT[d]) for d in DIMS}
        active_floor_dims = list(DIMS)
        active_critical_dims = list(DIMS)
    elif mode == "HPS_ONLY_ZERO_WEIGHT":
        weights = _renormalize_without_dim(removed_dim)
        # Diagnostic only: isolate the HPS contribution while retaining policy gates.
        active_floor_dims = list(DIMS)
        active_critical_dims = list(DIMS)
    else:  # FULL_POLICY_LODO
        weights = _renormalize_without_dim(removed_dim)
        active_floor_dims = [d for d in DIMS if d != removed_dim]
        active_critical_dims = [d for d in DIMS if d != removed_dim]

    active_crit = crit.loc[crit["dimension"].isin(active_critical_dims)].copy()
    if active_crit.empty:
        global_critical_wac = 0.0
    else:
        global_critical_wac = float(
            np.mean(
                active_crit["critical_adequacy_min_rank"].astype(float).to_numpy()
                / MAX_FACTOR_SCORE
            )
        )

    rows = []
    for client in dim_scores.index.astype(str):
        ds = dim_scores.loc[client]
        hps = float(sum(float(ds[d]) * float(weights[d]) for d in DIMS))

        floor_failures = [
            d for d in active_floor_dims
            if (not np.isfinite(float(ds[d])))
            or float(ds[d]) < REFERENCE_DIMENSION_FLOOR
        ]

        cdf = active_crit.loc[active_crit["client"].astype(str).eq(client)].copy()
        if cdf.empty:
            critical_wac_i = 1.0
            all_critical_meet = True
            below = []
        else:
            scores = cdf["rubric_score_0_5"].astype(float).to_numpy()
            minima = cdf["critical_adequacy_min_rank"].astype(float).to_numpy()
            critical_wac_i = float(np.mean(scores / MAX_FACTOR_SCORE))
            all_critical_meet = bool(np.all(scores >= minima))
            below = [
                f"{r.dimension}.{r.factor}:{float(r.rubric_score_0_5):.1f}"
                f"<{float(r.critical_adequacy_min_rank):.1f}"
                for r in cdf.itertuples()
                if float(r.rubric_score_0_5) < float(r.critical_adequacy_min_rank)
            ]

        if floor_failures:
            action = "REJECT"
            path = "DIMENSION_FLOOR"
        elif hps < REFERENCE_LOWER_CUT:
            action = "REJECT"
            path = "LOW_HPS"
        elif hps >= REFERENCE_UPPER_CUT and all_critical_meet:
            action = "ACCEPT"
            path = "DIRECT_AUTO_ACCEPT"
        elif critical_wac_i >= global_critical_wac:
            action = "ACCEPT"
            path = "ACCEPTED_AFTER_AUTOMATED_REVIEW"
        else:
            action = "REJECT"
            path = "AUTO_REJECTED_REVIEW_CRITICAL_WAC"

        rows.append({
            "client": client,
            "hps": hps,
            "final_action": action,
            "decision_path": path,
            "dimension_floor_failures": ";".join(floor_failures),
            "critical_wac_i": critical_wac_i,
            "global_critical_wac": global_critical_wac,
            "all_critical_meet_adequacy": bool(all_critical_meet),
            "critical_below_adequacy": ";".join(below),
            "ablation_mode": mode,
            "removed_dimension": removed_dim or "",
        })

    return pd.DataFrame(rows), {
        "weights": weights,
        "active_floor_dimensions": active_floor_dims,
        "active_critical_dimensions": active_critical_dims,
        "global_critical_wac": global_critical_wac,
    }


def build_rq5_configurations() -> pd.DataFrame:
    rows = [{
        "configuration": "REFERENCE",
        "ablation_mode": "REFERENCE",
        "removed_dimension": "",
    }]
    for dim in DIMS:
        rows.append({
            "configuration": f"HPS_ONLY_ZERO_{dim}",
            "ablation_mode": "HPS_ONLY_ZERO_WEIGHT",
            "removed_dimension": dim,
        })
    for dim in DIMS:
        rows.append({
            "configuration": f"FULL_POLICY_LODO_{dim}",
            "ablation_mode": "FULL_POLICY_LODO",
            "removed_dimension": dim,
        })
    return pd.DataFrame(rows)


def summarize_rq5_governance(configs, dim_scores, crit):
    ref, _ = evaluate_rq5_policy(dim_scores, crit, "REFERENCE", None)
    ref = ref.sort_values("client").reset_index(drop=True)
    ref_accept = ref.loc[ref["final_action"].eq("ACCEPT"), "client"].astype(str).tolist()
    ref_rate = len(ref_accept) / len(ref)

    gov_by_config = {}
    rows = []
    for cfg in configs.itertuples(index=False):
        removed = str(cfg.removed_dimension).strip() or None
        cur, meta = evaluate_rq5_policy(
            dim_scores, crit, cfg.ablation_mode, removed
        )
        cur = cur.sort_values("client").reset_index(drop=True)
        gov_by_config[cfg.configuration] = cur
        admitted = cur.loc[cur["final_action"].eq("ACCEPT"), "client"].astype(str).tolist()

        comp = ref[["client", "final_action", "hps"]].merge(
            cur[["client", "final_action", "hps"]],
            on="client", suffixes=("_reference", "_current"), validate="one_to_one"
        )
        flips = comp["final_action_reference"].ne(comp["final_action_current"])
        cur_rate = len(admitted) / len(cur)

        rows.append({
            "configuration": cfg.configuration,
            "ablation_mode": cfg.ablation_mode,
            "removed_dimension": removed or "",
            "dimension_name": DIMENSION_NAMES.get(removed, "Reference"),
            "accepted_count": int(len(admitted)),
            "accepted_rate": float(cur_rate),
            "accepted_rate_pct": float(100.0 * cur_rate),
            "accepted_count_change_vs_reference": int(len(admitted) - len(ref_accept)),
            "accepted_rate_change_pp_vs_reference": float(100.0 * (cur_rate - ref_rate)),
            "accepted_rate_relative_change_pct_vs_reference": (
                float(100.0 * (cur_rate - ref_rate) / ref_rate) if ref_rate > 0 else np.nan
            ),
            "decision_flip_count": int(flips.sum()),
            "decision_flip_rate": float(flips.mean()),
            "accepted_set_jaccard_vs_reference": _jaccard_rq5(ref_accept, admitted),
            "mean_abs_hps_change": float(np.mean(np.abs(
                comp["hps_current"].to_numpy(float) - comp["hps_reference"].to_numpy(float)
            ))),
            "admitted_clients": ";".join(admitted),
            "cohort_key": ";".join(sorted(admitted)),
            "weights_json": json.dumps(meta["weights"], sort_keys=True),
        })

    return pd.DataFrame(rows), gov_by_config, ref


def assign_rq5_cohort_ids(gov_summary: pd.DataFrame):
    keys = sorted(gov_summary["cohort_key"].astype(str).unique().tolist())
    mapping = {key: f"COHORT_{i+1:02d}" for i, key in enumerate(keys)}
    out = gov_summary.copy()
    out["cohort_id"] = out["cohort_key"].map(mapping)
    return out, mapping


def train_rq5_unique_cohorts(cohort_mapping, gov_summary, data):
    cohort_clients = {
        cohort_id: [x for x in str(key).split(";") if x]
        for key, cohort_id in cohort_mapping.items()
    }
    train_monitor = build_train_monitor_subset(
        data["client_arrays"],
        max_samples=ROUND_PROGRESS_MONITOR_MAX_SAMPLES,
        seed=99137,
    )
    build_model_fn = lambda: build_diabetes_model(data["meta"]["input_dim"], lr=LEARNING_RATE)
    completed = set(load_checkpoint_state(CHECKPOINT_STATE).get("completed", []))
    total_jobs = len(TRAINING_RUN_SEEDS) * len(cohort_clients)
    job_idx = 0

    for training_seed in TRAINING_RUN_SEEDS:
        seed_everything(training_seed)
        base_model = build_model_fn()
        initial_weights = [np.array(w, copy=True) for w in base_model.get_weights()]
        initial_hash = sha256_weights(initial_weights)
        del base_model
        tf.keras.backend.clear_session(); gc.collect()

        for cohort_id, selected in cohort_clients.items():
            job_idx += 1
            key = f"seed{training_seed}|{cohort_id}"
            if key in completed:
                print(f"CHECKPOINT FOUND -> SKIP: {key}")
                continue
            if not selected:
                raise RuntimeError(f"Empty cohort {cohort_id}")

            print_banner(
                f"RQ5 UNIQUE COHORT {job_idx}/{total_jobs} | seed={training_seed} | {cohort_id}"
            )
            print(f"Selected clients ({len(selected)}): {selected}")

            natural_map = {
                cid: natural_steps(len(data["client_arrays"][cid][1]), BATCH_SIZE, LOCAL_EPOCHS)
                for cid in selected
            }
            result = federated_train(
                build_model_fn=build_model_fn,
                initial_weights=initial_weights,
                selected_per_round=[list(selected) for _ in range(NUM_ROUNDS_FL)],
                client_arrays=data["client_arrays"],
                X_test=data["X_test"], y_test=data["y_test"], n_classes=3,
                batch_size=BATCH_SIZE, local_epochs=LOCAL_EPOCHS,
                class_weights=data["class_weights"], run_seed=training_seed,
                exact_step_maps=[dict(natural_map) for _ in range(NUM_ROUNDS_FL)],
                equal_weight=False, fedprox_mu=0.0, train_monitor=train_monitor,
                progress_context={
                    "scenario": f"RQ5 | {cohort_id}",
                    "run_idx": TRAINING_RUN_SEEDS.index(training_seed)+1,
                    "run_total": len(TRAINING_RUN_SEEDS),
                    "scenario_idx": job_idx, "scenario_total": total_jobs,
                    "overall_idx": job_idx, "overall_total": total_jobs,
                },
            )
            row = result_row(
                run=TRAINING_RUN_SEEDS.index(training_seed)+1,
                seed=training_seed, scenario=cohort_id, result=result,
                initial_hash=initial_hash,
            )
            row.update({
                "training_seed": int(training_seed),
                "cohort_id": cohort_id,
                "selected_k": int(len(selected)),
                "selected_clients": ";".join(selected),
                "steps_per_round": int(sum(natural_map.values())),
                "fl_rounds": int(NUM_ROUNDS_FL),
            })
            upsert_csv(row, UNIQUE_COHORT_PERF, ["training_seed", "cohort_id"])
            mark_checkpoint_complete(CHECKPOINT_STATE, key, extra={"last_job": key})
            completed.add(key)
            del result
            tf.keras.backend.clear_session(); gc.collect()

    perf = pd.read_csv(UNIQUE_COHORT_PERF)
    expected = len(TRAINING_RUN_SEEDS) * len(cohort_clients)
    if len(perf) != expected:
        raise RuntimeError(f"Expected {expected} performance rows; found {len(perf)}")

    audits = []
    for seed in TRAINING_RUN_SEEDS:
        d = perf[perf["training_seed"].eq(seed)]
        hashes = d["initial_weights_sha256"].astype(str).unique()
        passed = len(hashes) == 1
        audits.append({
            "training_seed": seed,
            "n_unique_cohorts": len(d),
            "unique_W0_hashes": len(hashes),
            "same_W0_across_cohorts": passed,
            "initial_weights_sha256": hashes[0] if len(hashes) else "",
        })
        if not passed:
            raise RuntimeError(f"W0 parity failed for seed {seed}")
    pd.DataFrame(audits).to_csv(EXPERIMENT_ROOT / "rq5_W0_parity_audit.csv", index=False)
    return perf


def summarize_rq5_performance(gov_summary, cohort_perf):
    expanded = gov_summary.merge(cohort_perf, on="cohort_id", how="left", validate="many_to_many")
    expanded.to_csv(EXPERIMENT_ROOT / "rq5_performance_all_seeds.csv", index=False)

    metrics = [
        "roc_auc_ovr_macro", "f1_macro", "accuracy", "precision_macro", "recall_macro",
        "runtime_s", "communication_mb", "optimizer_steps", "participants", "energy_wh",
    ]
    rows = []
    for cfg, d in expanded.groupby("configuration", sort=False):
        g = gov_summary[gov_summary["configuration"].eq(cfg)].iloc[0]
        row = {
            "configuration": cfg,
            "ablation_mode": g["ablation_mode"],
            "removed_dimension": g["removed_dimension"],
            "dimension_name": g["dimension_name"],
            "accepted_count": int(g["accepted_count"]),
            "accepted_rate_pct": float(g["accepted_rate_pct"]),
            "accepted_rate_change_pp_vs_reference": float(g["accepted_rate_change_pp_vs_reference"]),
            "decision_flip_count": int(g["decision_flip_count"]),
            "accepted_set_jaccard_vs_reference": float(g["accepted_set_jaccard_vs_reference"]),
            "admitted_clients": g["admitted_clients"],
            "cohort_id": g["cohort_id"],
            "n_training_seeds": int(len(d)),
        }
        for metric in metrics:
            if metric not in d.columns:
                continue
            x = pd.to_numeric(d[metric], errors="coerce").dropna().to_numpy(float)
            row[f"{metric}_mean"] = float(np.mean(x)) if len(x) else np.nan
            row[f"{metric}_sd"] = float(np.std(x, ddof=1)) if len(x)>1 else 0.0
            lo, hi = ci95_mean(x)
            row[f"{metric}_ci95_low"] = lo
            row[f"{metric}_ci95_high"] = hi
        rows.append(row)

    summary = pd.DataFrame(rows)
    ref = summary[summary["configuration"].eq("REFERENCE")].iloc[0]
    for metric in ["roc_auc_ovr_macro", "f1_macro", "accuracy"]:
        base = float(ref[f"{metric}_mean"])
        summary[f"{metric}_delta_vs_reference"] = summary[f"{metric}_mean"] - base
        summary[f"{metric}_change_pct_vs_reference"] = (
            100.0 * summary[f"{metric}_delta_vs_reference"] / base
        )
    summary.to_csv(EXPERIMENT_ROOT / "rq5_dimension_ablation_summary.csv", index=False)

    # Paired seed-level differences versus reference.
    ref_seed = expanded[expanded["configuration"].eq("REFERENCE")][
        ["training_seed", "roc_auc_ovr_macro", "f1_macro", "accuracy"]
    ].copy()
    paired_rows = []
    for cfg, d in expanded.groupby("configuration", sort=False):
        if cfg == "REFERENCE":
            continue
        m = d.merge(ref_seed, on="training_seed", suffixes=("_config", "_reference"), validate="one_to_one")
        for metric in ["roc_auc_ovr_macro", "f1_macro", "accuracy"]:
            diff = m[f"{metric}_config"].to_numpy(float) - m[f"{metric}_reference"].to_numpy(float)
            lo, hi = ci95_mean(diff)
            paired_rows.append({
                "configuration": cfg,
                "metric": metric,
                "n_pairs": len(diff),
                "mean_difference": float(np.mean(diff)),
                "sd_difference": float(np.std(diff, ddof=1)) if len(diff)>1 else 0.0,
                "ci95_low": lo, "ci95_high": hi,
                "wins": int(np.sum(diff>0)), "ties": int(np.sum(np.isclose(diff,0))),
                "losses": int(np.sum(diff<0)),
            })
    pd.DataFrame(paired_rows).to_csv(EXPERIMENT_ROOT / "rq5_paired_differences_vs_reference.csv", index=False)
    return expanded, summary


def create_rq5_figure(summary: pd.DataFrame):
    plot = summary[summary["ablation_mode"].ne("REFERENCE")].copy()
    fig = plt.figure(figsize=(14, 10))
    gs = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.28)

    modes = [
        ("HPS_ONLY_ZERO_WEIGHT", "HPS-only"),
        ("FULL_POLICY_LODO", "Full-policy LODO"),
    ]
    dims = DIMS
    names = [DIMENSION_NAMES[d] for d in dims]
    x = np.arange(len(dims), dtype=float)
    width = 0.36

    ax1 = fig.add_subplot(gs[0,0])
    for i, (mode, label) in enumerate(modes):
        vals = []
        for dim in dims:
            d = plot[(plot["ablation_mode"].eq(mode)) & (plot["removed_dimension"].eq(dim))]
            vals.append(float(d["accepted_rate_change_pp_vs_reference"].iloc[0]))
        ax1.bar(x + (i-0.5)*width, vals, width, label=label)
    ax1.axhline(0, linewidth=0.8)
    ax1.set_xticks(x); ax1.set_xticklabels(names, rotation=30, ha="right")
    ax1.set_ylabel("Change in admitted clients (percentage points)")
    ax1.set_title("A. Governance impact of dimension ablation", fontweight="bold")
    ax1.legend(fontsize=8)

    panels = [
        (gs[0,1], "roc_auc_ovr_macro_delta_vs_reference", "B. Change in Macro ROC-AUC", "Δ Macro ROC-AUC"),
        (gs[1,0], "f1_macro_delta_vs_reference", "C. Change in Macro-F1", "Δ Macro-F1"),
        (gs[1,1], "accuracy_delta_vs_reference", "D. Change in Accuracy", "Δ Accuracy"),
    ]
    for cell, metric, title, ylabel in panels:
        ax = fig.add_subplot(cell)
        for i, (mode, label) in enumerate(modes):
            vals = []
            for dim in dims:
                d = plot[(plot["ablation_mode"].eq(mode)) & (plot["removed_dimension"].eq(dim))]
                vals.append(float(d[metric].iloc[0]))
            ax.bar(x + (i-0.5)*width, vals, width, label=label)
        ax.axhline(0, linewidth=0.8)
        ax.set_xticks(x); ax.set_xticklabels(names, rotation=30, ha="right")
        ax.set_ylabel(ylabel); ax.set_title(title, fontweight="bold")

    fig.suptitle("RQ5 — Trust-Dimension Ablation under Frozen K=20 TADP-VR", fontsize=14, fontweight="bold")
    out = EXPERIMENT_ROOT / "RQ5_dimension_ablation_figure.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return out


def main():
    print_banner(EXPERIMENT_VERSION)
    print("RQ5 primary analysis: FULL_POLICY_LODO")
    print("RQ5 secondary diagnostic: HPS_ONLY_ZERO_WEIGHT")
    print(f"K={K_SUBMISSIONS} | seeds={TRAINING_RUN_SEEDS} | rounds={NUM_ROUNDS_FL}")

    atomic_write_json({
        "experiment_version": EXPERIMENT_VERSION,
        "research_question": "How does removing individual trust dimensions affect governance decisions and downstream predictive utility?",
        "primary_analysis": "FULL_POLICY_LODO",
        "secondary_diagnostic": "HPS_ONLY_ZERO_WEIGHT",
        "k_submissions": K_SUBMISSIONS,
        "training_seeds": TRAINING_RUN_SEEDS,
        "fl_rounds": NUM_ROUNDS_FL,
        "local_epochs": LOCAL_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "dirichlet_alpha": DIRICHLET_ALPHA,
        "global_split_seed": GLOBAL_SPLIT_SEED,
        "client_partition_seed": CLIENT_PARTITION_SEED,
        "frozen_evidence_assignment_seed": FROZEN_EVIDENCE_ASSIGNMENT_SEED,
        "evidence_library_seed": EVIDENCE_LIBRARY_SEED,
        "evidence_library_size": EVIDENCE_LIBRARY_SIZE,
        "evidence_assignment_sampling": EVIDENCE_ASSIGNMENT_SAMPLING,
        "evidence_assignment_note": "Twenty distinct documentary evidence profiles are created before client assignment; each of the 20 clients samples one uniformly with replacement. Duplicate client assignments are allowed and the realized admission rate is not forced.",
        "reference_weights": WEIGHTS_PSCORE_DEFAULT,
        "reference_lower_cut": REFERENCE_LOWER_CUT,
        "reference_upper_cut": REFERENCE_UPPER_CUT,
        "reference_dimension_floor": REFERENCE_DIMENSION_FLOOR,
        "interpretation": "Pre-specified ablation on one frozen K=20 evidence realization drawn with replacement from a 20-profile library. Report all dimensions and both ablation modes; do not select results based on predictive performance.",
    }, EXPERIMENT_ROOT / "rq5_experiment_design.json")

    csv_path = locate_diabetes_csv()
    data = prepare_diabetes_no_leakage(
        csv_path=csv_path, split_seed=GLOBAL_SPLIT_SEED,
        partition_seed=CLIENT_PARTITION_SEED,
        n_clients=K_SUBMISSIONS, alpha=DIRICHLET_ALPHA,
    )

    leakage = write_leakage_audit(
        EXPERIMENT_ROOT, data["global_train_ids"], data["global_test_ids"], data["client_train_ids"],
        extra={
            "patient_overlap": data["meta"]["patient_overlap"],
            "global_holdout_before_client_partition": True,
            "preprocessing_train_only": True,
            "dq_tadp_train_only": True,
            "class_weights_train_only": True,
            "test_used_for_final_evaluation_only": True,
            "rq5_frozen_k20_setting": True,
            "evidence_library_size": EVIDENCE_LIBRARY_SIZE,
            "evidence_assignment_sampling": EVIDENCE_ASSIGNMENT_SAMPLING,
        },
    )
    if not bool(leakage.get("pass", False)):
        raise RuntimeError(f"No-leakage audit failed: {leakage}")

    client_ids = list(data["client_ids"])
    expected_client_ids = list("ABCDEFGHIJKLMNOPQRST")
    if client_ids != expected_client_ids:
        raise RuntimeError(f"Expected A-T clients for K=20, got {client_ids}")

    documentary, evidence_df, evidence_library_df = generate_random_library_controlled_evidence(
        client_ids=client_ids,
        evidence_seed=FROZEN_EVIDENCE_ASSIGNMENT_SEED,
        library_seed=EVIDENCE_LIBRARY_SEED,
        domain=DOMAIN,
    )
    evidence_library_df.to_csv(EXPERIMENT_ROOT / "rq5_20_profile_evidence_library.csv", index=False)
    evidence_df.to_csv(EXPERIMENT_ROOT / "rq5_frozen_controlled_evidence.csv", index=False)
    assignment_summary = summarize_random_library_assignment(evidence_df)
    assignment_summary.to_csv(EXPERIMENT_ROOT / "rq5_frozen_evidence_assignment_map.csv", index=False)
    usage_summary = (assignment_summary.groupby(["library_profile_id", "evidence_profile"])
                     .size().reset_index(name="clients_receiving_profile")
                     .sort_values(["clients_receiving_profile", "library_profile_id"], ascending=[False, True]))
    usage_summary.to_csv(EXPERIMENT_ROOT / "rq5_evidence_profile_usage_summary.csv", index=False)
    print_banner("RQ5 FROZEN 20-PROFILE EVIDENCE ASSIGNMENT (WITH REPLACEMENT)")
    print(assignment_summary.to_string(index=False))
    data["dq_audit"].to_csv(EXPERIMENT_ROOT / "rq5_frozen_DQ_audit.csv", index=False)

    original_gov = build_tadp_governance(
        client_ids, documentary, data["dq_scores"], run=0,
        evidence_seed=FROZEN_EVIDENCE_ASSIGNMENT_SEED, domain=DOMAIN,
    ).sort_values("client").reset_index(drop=True)
    original_gov.to_csv(EXPERIMENT_ROOT / "rq5_original_frozen_governance.csv", index=False)

    full_factor_df = build_full_factor_evidence_table(
        client_ids, documentary, data["dq_scores"], data["dq_audit"],
        FROZEN_EVIDENCE_ASSIGNMENT_SEED, DOMAIN,
    )
    full_factor_df.to_csv(EXPERIMENT_ROOT / "rq5_frozen_all_28_factor_evidence.csv", index=False)
    _, dim_scores, crit = build_policy_tables_rq5(full_factor_df)

    # Fail closed: reference recomputation must exactly reproduce the frozen main governance.
    ref_recomputed, _ = evaluate_rq5_policy(dim_scores, crit, "REFERENCE", None)
    ref_recomputed = ref_recomputed.sort_values("client").reset_index(drop=True)
    audit = ref_recomputed[["client", "hps", "final_action"]].merge(
        original_gov[["client", "hps", "final_action"]], on="client",
        suffixes=("_recomputed", "_original"), validate="one_to_one"
    )
    audit["decision_match"] = audit["final_action_recomputed"].eq(audit["final_action_original"])
    audit["hps_abs_diff"] = np.abs(audit["hps_recomputed"] - audit["hps_original"])
    audit.to_csv(EXPERIMENT_ROOT / "rq5_reference_policy_reproduction_audit.csv", index=False)
    if not audit["decision_match"].all() or float(audit["hps_abs_diff"].max()) > 1e-10:
        raise RuntimeError("RQ5 reference-policy reproduction failed")

    configs = build_rq5_configurations()
    configs.to_csv(EXPERIMENT_ROOT / "rq5_requested_configurations.csv", index=False)
    gov_summary, gov_by_config, _ = summarize_rq5_governance(configs, dim_scores, crit)
    gov_summary, cohort_mapping = assign_rq5_cohort_ids(gov_summary)
    gov_summary.to_csv(EXPERIMENT_ROOT / "rq5_governance_summary.csv", index=False)

    frames = []
    for cfg, df in gov_by_config.items():
        x = df.copy(); x.insert(0, "configuration", cfg); frames.append(x)
    pd.concat(frames, ignore_index=True).to_csv(
        EXPERIMENT_ROOT / "rq5_governance_client_level.csv", index=False
    )

    print_banner("RQ5 GOVERNANCE SUMMARY")
    print(gov_summary[[
        "configuration", "ablation_mode", "removed_dimension", "dimension_name",
        "accepted_count", "accepted_rate_pct", "accepted_rate_change_pp_vs_reference",
        "decision_flip_count", "accepted_set_jaccard_vs_reference", "admitted_clients"
    ]].to_string(index=False))

    cohort_perf = train_rq5_unique_cohorts(cohort_mapping, gov_summary, data)
    _, perf_summary = summarize_rq5_performance(gov_summary, cohort_perf)
    fig = create_rq5_figure(perf_summary)

    print_banner("RQ5 PERFORMANCE SUMMARY")
    print(perf_summary[[
        "configuration", "ablation_mode", "removed_dimension", "accepted_count",
        "roc_auc_ovr_macro_mean", "roc_auc_ovr_macro_delta_vs_reference",
        "f1_macro_mean", "f1_macro_delta_vs_reference",
        "accuracy_mean", "accuracy_delta_vs_reference",
    ]].to_string(index=False))

    atomic_write_json({
        "reference_policy_reproduction": "PASS",
        "no_leakage": "PASS",
        "same_W0_within_seed": "PASS",
        "n_policy_configurations": int(len(configs)),
        "n_unique_admitted_cohorts": int(len(cohort_mapping)),
        "actual_model_training_runs": int(len(cohort_mapping) * len(TRAINING_RUN_SEEDS)),
        "figure": str(fig),
    }, EXPERIMENT_ROOT / "rq5_final_audit_summary.json")

    return EXPERIMENT_ROOT


if __name__ == "__main__":
    finished_root = main()
    package_and_download_results(finished_root, EXPERIMENT_VERSION)


Google Drive checkpoint mount unavailable: Error: credential propagation was unsuccessful
Local checkpoint root: /content/TADP_EXPERIMENT_B3_TADP-B3-v16.9-PREDICTIVE-SCALABILITY-K20-50-100-3SEED-4ROUND
Google Drive checkpoint mount unavailable: mount failed
Local checkpoint root: /content/TADP_EXPERIMENT_RQ5_TADP-RQ5-v17.3-K20-20EVIDENCE-WR-DIMENSION-ABLATION-5SEED-4ROUND

TADP-RQ5-v17.3-K20-20EVIDENCE-WR-DIMENSION-ABLATION-5SEED-4ROUND
RQ5 primary analysis: FULL_POLICY_LODO
RQ5 secondary diagnostic: HPS_ONLY_ZERO_WEIGHT
K=20 | seeds=[42, 142, 242, 342, 442] | rounds=4

RQ5 FROZEN 20-PROFILE EVIDENCE ASSIGNMENT (WITH REPLACEMENT)
client library_profile_id     evidence_profile  profile_variant  times_profile_sampled_in_realization      assignment_sampling  evidence_seed  evidence_library_seed
     A                P19 DIMENSION_FLOOR_WEAK                0                                     4 UNIFORM_WITH_REPLACEMENT          14102                  17220
     B                P19 DIMENS

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
# ======================================================================================
# TADP v16.7 — REVIEWER-ALIGNED TRUSTWORTHY DATA PREPARATION EXPERIMENT CORE
# ======================================================================================
# Design guarantees:
#   1) GLOBAL holdout is created BEFORE client partitioning.
#   2) Only TRAIN is distributed to clients.
#   3) Preprocessing parameters/vocabularies use TRAIN only.
#   4) DQ, GE and TADP governance use TRAIN only.
#   5) Held-out TEST never affects preprocessing, governance, class weights,
#      client selection, model initialization, training, or matched-control budgets.
#   6) TEST is used only after training for final evaluation.
#
# v16.7 governance:
#   - 28 factors across 6 dimensions:
#       dim1=4, dim2(DQ)=8, dim3=4, dim4=3, dim5=5, dim6=4.
#   - Added Data Collection / Acquisition Lineage (dim1).
#   - Added Structural / Constraint Integrity (dim2).
#   - Documentary evidence is generated at the individual factor level.
#   - DQ evidence is machine-measured from client TRAIN partitions only.
#   - HPS remains client-specific and uses the six policy dimension weights.
#   - WAC is NOT client-specific.
#   - Each factor has a declared minimum adequate rubric rank.
#   - A domain WAC is derived once:
#         WAC_d = mean(minimum adequate ranks in dimension d) / 5
#         WAC_domain = equal mean of the six WAC_d values.
#   - Every averaged dimension must be >= 2.5/5 or the client is auto-rejected.
#   - HPS < 3.0 -> AUTO_REJECT.
#   - 3.0 <= HPS < 3.5 -> AUTOMATED REVIEW using Critical WAC_i.
#   - Review accepts iff Critical WAC_i >= the domain Global Critical WAC.
#   - HPS >= 3.5 -> DIRECT AUTO_ACCEPT when every critical factor meets its own adequacy minimum.
#   - Human reviewers verify evidence only; admission is server-automated.
#
# GX Core comparator:
#   - Separate from HPS/TADP.
#   - Uses REAL Great Expectations GX Core validation on TRAIN-only client data.
#   - Uses common technical checks: schema, datatype consistency, required ranges,
#     missingness, duplicate/ID integrity, label/domain validity, and structure.
#   - Uses GX native severity-aware validation: zero critical failures required; no ranking and no forced-K.
#
# Runtime/reporting:
#   - Every FL round prints configuration/run/scenario/round progress plus TRAIN-only diagnostic utility and operational metrics.
#   - Every completed scenario prints all predictive and operational metrics.
#   - Scenario checkpoints support restart/resume.
#   - Final result ZIP downloads automatically in Google Colab.
# ======================================================================================

import os
import sys
import gc
import math
import time
import json
import random
import hashlib
import threading
import zipfile
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# ======================================================================================
# POLICY CONSTANTS — v16.7
# ======================================================================================

GOOD_CUT = 3.0
HIGH_CUT = 3.5
MAX_FACTOR_SCORE = 5.0
GE_ACCEPT_COUNT = 6

# ----------------------------------------------------------------------
# v16.7 FINAL FULLY AUTOMATED ADMISSION POLICY
# ----------------------------------------------------------------------
# Human reviewers verify supporting evidence uploaded through the questionnaire.
# They do NOT make the admission decision. Once verified factor scores are
# available, the server applies this policy automatically.
#
# 1) EVERY averaged HPS dimension must be >= 2.5/5.
#    Any dimension < 2.5 -> AUTO-REJECT.
#
# 2) HPS must be >= 3.0.
#    HPS < 3.0 -> AUTO-REJECT.
#
# 3) DIRECT AUTO-ACCEPT:
#    HPS >= 3.5 AND every critical factor independently meets its own
#    factor-specific minimum adequacy requirement.
#
# 4) AUTOMATED REVIEW:
#    - all clients with 3.0 <= HPS < 3.5; and
#    - high-HPS clients that fail one or more individual critical-factor
#      adequacy requirements.
#
# 5) REVIEW RESOLUTION:
#    Critical WAC_i = mean(actual critical-factor scores / 5)
#    Global Critical WAC = mean(policy adequacy minima / 5 for same factors)
#
#    Critical WAC_i >= Global Critical WAC -> ACCEPT AFTER REVIEW
#    Critical WAC_i <  Global Critical WAC -> AUTO-REJECT
#
# Thus Direct Auto-Accept is strict at the individual critical-factor level,
# whereas Review intentionally allows controlled compensation across the
# critical subset.
DIMENSION_MIN_FLOOR = 2.5

CRITICAL_FACTORS_BY_DOMAIN = {
    "healthcare": {
        "dim1": [
            "data_controller",
            "data_collection_lineage",
        ],
        "dim5": [
            "regulation_coverage",
            "consent_ethics",
            "sensitivity_classification",
        ],
        "dim6": [
            "user_agreements",
        ],
    },
    "cifar10": {
        "dim1": [
            "data_controller",
            "data_collection_lineage",
        ],
        "dim6": [
            "license_terms",
            "user_agreements",
        ],
    },
}

WEIGHTS_PSCORE_DEFAULT = {
    "dim1": 0.25,  # Source Reliability
    "dim2": 0.15,  # Data Quality and Health
    "dim3": 0.10,  # Documentation Practices
    "dim4": 0.10,  # Timeliness and Refresh Rate
    "dim5": 0.30,  # Regulatory / Compliance Alignment
    "dim6": 0.10,  # Context / Usage Constraints
}

DIMENSION_NAMES = {
    "dim1": "Source Reliability",
    "dim2": "Data Quality and Health",
    "dim3": "Documentation Practices",
    "dim4": "Timeliness and Refresh Rate",
    "dim5": "Regulatory and Compliance Alignment",
    "dim6": "Context and Usage Constraints",
}

# v16.7: two reviewer-driven additions:
#   dim1: data_collection_lineage
#   dim2: structural_constraint_integrity
#
# Total = 4 + 8 + 4 + 3 + 5 + 4 = 28 factors.
FACTOR_NAMES = {
    "dim1": [
        "source_reputation",
        "data_controller",
        "data_objective",
        "data_collection_lineage",
    ],
    "dim2": [
        "completeness",
        "duplication_rate",
        "value_validity_error_rate",
        "type_consistency",
        "label_integrity",
        "feature_distribution_consistency",
        "feature_category_coverage",
        "structural_constraint_integrity",
    ],
    "dim3": [
        "data_dictionary",
        "version_logs",
        "collection_protocol",
        "definition_updates",
    ],
    "dim4": [
        "data_freshness",
        "scheduled_refresh",
        "retention_clarity",
    ],
    "dim5": [
        "regulation_coverage",
        "consent_ethics",
        "geo_restrictions",
        "sensitivity_classification",
        "audits",
    ],
    "dim6": [
        "license_terms",
        "ethical_reviews",
        "redistribution",
        "user_agreements",
    ],
}

DOCUMENTARY_DIMS = ("dim1", "dim3", "dim4", "dim5", "dim6")

# ----------------------------------------------------------------------
# COMPLETE 0--5 RUBRIC DESCRIPTORS
# ----------------------------------------------------------------------
# These descriptors reproduce the Appendix-A semantics and add the two
# v16.7 factors explicitly. Controlled evidence is sampled at the factor
# level; HPS is never generated directly.
RUBRIC_DESCRIPTORS = {
    "dim1": {
        "source_reputation": {
            0: "No info",
            1: "Poor",
            2: "Limited evidence",
            3: "Average, partially trusted",
            4: "Well-documented, reliable",
            5: "Highly reputable, verified",
        },
        "data_controller": {
            0: "No documented controller",
            1: "Unclear",
            2: "Partially clear",
            3: "Moderately clear",
            4: "Mostly clear",
            5: "Fully documented",
        },
        "data_objective": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General but unclear",
            4: "Mostly explicit",
            5: "Fully explicit, justified",
        },
        "data_collection_lineage": {
            0: "Collection origin unknown",
            1: "Informal or unverifiable origin",
            2: "Partially documented acquisition path",
            3: "Documented acquisition with limited traceability",
            4: "Well-documented and traceable acquisition path",
            5: "Fully source-linked, versioned, and auditable lineage",
        },
    },
    "dim2": {
        "completeness": {
            0: ">50% missing",
            1: "20-50% missing",
            2: "10-20% missing",
            3: "5-10% missing",
            4: "1-5% missing",
            5: "<1% missing",
        },
        "duplication_rate": {
            0: ">20% duplicates",
            1: "10-20% duplicates",
            2: "5-10% duplicates",
            3: "2-5% duplicates",
            4: "1-2% duplicates",
            5: "<1% duplicates",
        },
        "value_validity_error_rate": {
            0: ">15% invalid/error values",
            1: "10-15% invalid/error values",
            2: "5-10% invalid/error values",
            3: "2-5% invalid/error values",
            4: "1-2% invalid/error values",
            5: "<1% invalid/error values",
        },
        "type_consistency": {
            0: "Highly inconsistent",
            1: "Frequent type inconsistency",
            2: "Moderate type inconsistency",
            3: "Minor type inconsistency",
            4: "Rare type inconsistency",
            5: "Fully consistent",
        },
        "label_integrity": {
            0: ">10% missing/invalid/known erroneous labels",
            1: "5-10% missing/invalid/known erroneous labels",
            2: "2-5% missing/invalid/known erroneous labels",
            3: "1-2% missing/invalid/known erroneous labels",
            4: "0.1-1% missing/invalid/known erroneous labels",
            5: "<=0.1% missing/invalid/known erroneous labels",
        },
        "feature_distribution_consistency": {
            0: "JSD >0.20",
            1: "JSD 0.10-0.20",
            2: "JSD 0.05-0.10",
            3: "JSD 0.025-0.05",
            4: "JSD 0.01-0.025",
            5: "JSD <=0.01",
        },
        "feature_category_coverage": {
            0: "<50% reference support represented",
            1: "50-65% reference support represented",
            2: "65-75% reference support represented",
            3: "75-82.5% reference support represented",
            4: "82.5-90% reference support represented",
            5: ">=90% reference support represented",
        },
        "structural_constraint_integrity": {
            0: ">10% records/structures violate required constraints",
            1: "5-10% violate required constraints",
            2: "2-5% violate required constraints",
            3: "1-2% violate required constraints",
            4: "0.1-1% violate required constraints",
            5: "<=0.1% violate required constraints",
        },
    },
    "dim3": {
        "data_dictionary": {
            0: "None",
            1: "Minimal outline",
            2: "Partial coverage",
            3: "Moderate coverage",
            4: "Near-complete",
            5: "Fully detailed",
        },
        "version_logs": {
            0: "None",
            1: "Minimal logs",
            2: "Occasional logs",
            3: "Regular logs",
            4: "Near-complete",
            5: "Full version history",
        },
        "collection_protocol": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General methods",
            4: "Well-defined",
            5: "Fully transparent",
        },
        "definition_updates": {
            0: "None",
            1: "Rarely updated",
            2: "Occasional updates",
            3: "Regular but basic",
            4: "Frequent",
            5: "Real-time, documented",
        },
    },
    "dim4": {
        "data_freshness": {
            0: ">5 years old",
            1: "2-5 years old",
            2: "1-2 years old",
            3: "6-12 months old",
            4: "1-6 months old",
            5: "Real-time/current",
        },
        "scheduled_refresh": {
            0: "Never",
            1: "Irregular",
            2: "Annual",
            3: "Quarterly",
            4: "Monthly",
            5: "Daily/real-time",
        },
        "retention_clarity": {
            0: "None",
            1: "Minimal",
            2: "Basic guidelines",
            3: "Moderate clarity",
            4: "High clarity",
            5: "Fully documented",
        },
    },
    "dim5": {
        "regulation_coverage": {
            0: "None",
            1: "Minimal",
            2: "Partial",
            3: "Moderate",
            4: "Comprehensive but dated",
            5: "Fully documented/current",
        },
        "consent_ethics": {
            0: "None",
            1: "Minimal record",
            2: "Partial consent/ethics evidence",
            3: "Moderate logs",
            4: "Substantial",
            5: "Fully documented",
        },
        "geo_restrictions": {
            0: "None",
            1: "Basic mention",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully documented",
        },
        "sensitivity_classification": {
            0: "None",
            1: "Basic flagging",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully classified",
        },
        "audits": {
            0: "None",
            1: "Internal only",
            2: "Basic certification",
            3: "Occasional audit",
            4: "Recent audit",
            5: "Regular external audits",
        },
    },
    "dim6": {
        "license_terms": {
            0: "None",
            1: "Vague",
            2: "Basic",
            3: "Clear",
            4: "Detailed",
            5: "Industry-compliant",
        },
        "ethical_reviews": {
            0: "None",
            1: "Informal approval",
            2: "Partial",
            3: "Moderate",
            4: "Well-documented",
            5: "Certified",
        },
        "redistribution": {
            0: "No policy",
            1: "Unclear",
            2: "Partial",
            3: "Clear",
            4: "Detailed",
            5: "Fully compliant",
        },
        "user_agreements": {
            0: "Non-compliant",
            1: "Minimal adherence",
            2: "Partial",
            3: "Mostly compliant",
            4: "Fully compliant",
            5: "Audited compliance",
        },
    },
}

# ----------------------------------------------------------------------
# FACTOR-SPECIFIC ADEQUACY POLICY
# ----------------------------------------------------------------------
# The minimum adequate rank is derived factor-by-factor from the wording of
# the Appendix-A rubric. It is NOT learned from model/test outcomes.
#
# Healthcare is the primary policy. CIFAR-10 uses the same reference ranks
# for cross-domain comparability; its DQ factors are measured with image-
# specific checks, while documentary dimensions remain controlled evidence.
FACTOR_ADEQUACY_MIN_HEALTHCARE = {
    "dim1": {
        "source_reputation": 4,
        "data_controller": 4,
        "data_objective": 4,
        "data_collection_lineage": 4,
    },
    "dim2": {
        "completeness": 3,
        "duplication_rate": 3,
        "value_validity_error_rate": 3,
        "type_consistency": 3,
        "label_integrity": 3,
        "feature_distribution_consistency": 3,
        "feature_category_coverage": 3,
        "structural_constraint_integrity": 3,
    },
    "dim3": {
        "data_dictionary": 3,
        "version_logs": 3,
        "collection_protocol": 4,
        "definition_updates": 3,
    },
    "dim4": {
        "data_freshness": 3,
        "scheduled_refresh": 3,
        "retention_clarity": 4,
    },
    "dim5": {
        "regulation_coverage": 3,
        "consent_ethics": 3,
        "geo_restrictions": 3,
        "sensitivity_classification": 3,
        "audits": 4,
    },
    "dim6": {
        "license_terms": 3,
        "ethical_reviews": 4,
        "redistribution": 3,
        "user_agreements": 4,
    },
}

FACTOR_ADEQUACY_MIN_CIFAR10 = {
    dim: dict(values)
    for dim, values in FACTOR_ADEQUACY_MIN_HEALTHCARE.items()
}

def derive_domain_wac(
    factor_minima: Dict[str, Dict[str, float]]
) -> Tuple[Dict[str, float], float]:
    """
    Derive the domain policy WAC from factor-specific minimum adequate ranks.

    WAC_d = mean_k(adequate_rank_dk / 5)
    WAC_domain = equal mean across the six dimension WAC_d values.

    IMPORTANT:
      WAC is a DOMAIN POLICY value, not a client-specific score.
    """
    dimension_wac = {}
    for dim in FACTOR_NAMES:
        vals = [
            float(factor_minima[dim][factor])
            for factor in FACTOR_NAMES[dim]
        ]
        dimension_wac[dim] = float(np.mean(vals) / MAX_FACTOR_SCORE)
    global_wac = float(np.mean(list(dimension_wac.values())))
    return dimension_wac, global_wac


HEALTHCARE_DIMENSION_WAC, WAC_HEALTHCARE = derive_domain_wac(
    FACTOR_ADEQUACY_MIN_HEALTHCARE
)
CIFAR10_DIMENSION_WAC, WAC_CIFAR10 = derive_domain_wac(
    FACTOR_ADEQUACY_MIN_CIFAR10
)

DOMAIN_FACTOR_MINIMA = {
    "healthcare": FACTOR_ADEQUACY_MIN_HEALTHCARE,
    "cifar10": FACTOR_ADEQUACY_MIN_CIFAR10,
}
DOMAIN_DIMENSION_WAC = {
    "healthcare": HEALTHCARE_DIMENSION_WAC,
    "cifar10": CIFAR10_DIMENSION_WAC,
}
DOMAIN_GLOBAL_WAC = {
    "healthcare": WAC_HEALTHCARE,
    "cifar10": WAC_CIFAR10,
}

DQ_RAW_METRIC_BY_FACTOR = {
    "completeness": "missing_fraction",
    "duplication_rate": "duplicate_fraction",
    "value_validity_error_rate": "error_fraction",
    "type_consistency": "type_inconsistency_fraction",
    "label_integrity": "invalid_label_fraction",
    "feature_distribution_consistency": "max_jsd",
    "feature_category_coverage": "mean_category_coverage",
    "structural_constraint_integrity": "structural_violation_fraction",
}

assert abs(sum(WEIGHTS_PSCORE_DEFAULT.values()) - 1.0) < 1e-12
assert sum(len(v) for v in FACTOR_NAMES.values()) == 28
assert len(FACTOR_NAMES["dim1"]) == 4
assert len(FACTOR_NAMES["dim2"]) == 8
assert abs(WAC_HEALTHCARE - 0.6761111111111111) < 1e-12

# ======================================================================================
# GENERAL UTILITIES
# ======================================================================================

def seed_everything(seed: int):
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        tf.keras.utils.set_random_seed(seed)
    except Exception:
        tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def sha256_bytes(x: bytes) -> str:
    return hashlib.sha256(x).hexdigest()


def sha256_array(x: np.ndarray) -> str:
    x = np.asarray(x)
    return sha256_bytes(np.ascontiguousarray(x).view(np.uint8).tobytes())


def sha256_weights(weights: List[np.ndarray]) -> str:
    h = hashlib.sha256()
    for w in weights:
        a = np.ascontiguousarray(np.asarray(w))
        h.update(str(a.shape).encode())
        h.update(a.view(np.uint8).tobytes())
    return h.hexdigest()


def _process_rss_mb() -> float:
    try:
        import psutil
        return float(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2))
    except Exception:
        pass
    try:
        with open("/proc/self/statm", "r", encoding="utf-8") as f:
            pages = int(f.read().split()[1])
        return float(pages * int(os.sysconf("SC_PAGE_SIZE")) / (1024 ** 2))
    except Exception:
        pass
    try:
        import resource
        x = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
        return x / (1024 ** 2) if sys.platform == "darwin" else x / 1024.0
    except Exception:
        return 0.0


class RAMMonitor:
    def __init__(self, interval_s: float = 0.05):
        self.interval_s = float(interval_s)
        self.start_mb = 0.0
        self.end_mb = 0.0
        self.peak_mb = 0.0
        self._stop = threading.Event()
        self._thread = None

    def _loop(self):
        while not self._stop.wait(self.interval_s):
            self.peak_mb = max(self.peak_mb, _process_rss_mb())

    def start(self):
        self.start_mb = _process_rss_mb()
        self.peak_mb = self.start_mb
        self._stop.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def stop(self) -> Dict[str, float]:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=1.0)
        self.end_mb = _process_rss_mb()
        self.peak_mb = max(self.peak_mb, self.start_mb, self.end_mb)
        return {
            "ram_start_mb": float(self.start_mb),
            "ram_end_mb": float(self.end_mb),
            "ram_peak_mb": float(self.peak_mb),
            "ram_delta_mb": float(max(0.0, self.peak_mb - self.start_mb)),
            "ram_mb": float(self.peak_mb),
        }


def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
    return Path(path)


def append_csv(row: Dict[str, Any], path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([row]).to_csv(
        path, mode="a", header=not path.exists(), index=False
    )


def summarize_runs(perf: pd.DataFrame) -> pd.DataFrame:
    if perf.empty:
        return pd.DataFrame()
    metrics = [
        c for c in [
            "accuracy", "precision_macro", "recall_macro", "f1_macro",
            "roc_auc_ovr_macro", "runtime_s", "energy_wh", "communication_mb",
            "optimizer_steps", "participants", "ram_peak_mb", "ram_delta_mb"
        ] if c in perf.columns
    ]
    rows = []
    for scenario, d in perf.groupby("scenario", sort=False):
        row = {"scenario": scenario, "n_runs": len(d)}
        for m in metrics:
            vals = pd.to_numeric(d[m], errors="coerce")
            row[f"{m}_mean"] = float(vals.mean())
            row[f"{m}_sd"] = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
        rows.append(row)
    return pd.DataFrame(rows)


def package_and_download_results(output_dir: Path, label: str) -> Path:
    output_dir = Path(output_dir)
    manifest = []
    for p in sorted(output_dir.rglob("*")):
        if p.is_file():
            manifest.append({
                "relative_path": str(p.relative_to(output_dir)),
                "bytes": int(p.stat().st_size),
                "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
            })
    pd.DataFrame(manifest).to_csv(
        output_dir / "RESULTS_FILE_MANIFEST.csv", index=False
    )

    zip_path = output_dir.parent / f"{output_dir.name}_RESULTS.zip"
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
    ) as zf:
        for p in sorted(output_dir.rglob("*")):
            if p.is_file():
                zf.write(p, arcname=str(p.relative_to(output_dir)))

    print("\n" + "=" * 100)
    print(f"{label} COMPLETE")
    print(f"Results ZIP: {zip_path}")
    print(f"ZIP size: {zip_path.stat().st_size / (1024**2):.2f} MB")
    print("=" * 100)

    try:
        from google.colab import files as colab_files
        print("Starting automatic download to your laptop...")
        colab_files.download(str(zip_path))
    except Exception as exc:
        print("Automatic Colab download unavailable.")
        print(f"ZIP remains at: {zip_path}")
        print(f"Reason: {exc}")

    return zip_path


# ======================================================================================
# CONTROLLED DOCUMENTARY EVIDENCE — v16.7
# ======================================================================================

def rubric_descriptor(dim: str, factor: str, score: float) -> str:
    s = int(np.clip(np.rint(float(score)), 0, 5))
    return str(RUBRIC_DESCRIPTORS[dim][factor][s])


def generate_controlled_documentary_evidence(
    client_ids: List[str],
    evidence_seed: int,
    factor_minima: Dict[str, Dict[str, float]],
    domain: str,
) -> Tuple[Dict[str, Dict[str, Dict[str, float]]], pd.DataFrame]:
    """
    Generate controlled documentary evidence at the INDIVIDUAL FACTOR level.

    v16.7 uses PRE-SPECIFIED governance archetypes so the controlled experiment
    contains all three decision outcomes needed to validate the policy:

      - DIRECT_STRONG:
          designed to satisfy the strict Direct Auto-Accept route;
      - REVIEW_RECOVERABLE:
          borderline overall evidence, but sufficiently strong average evidence
          across the critical subset for Automated Review acceptance;
      - REVIEW_LIMITED:
          adequate enough to enter Automated Review, but insufficient average
          critical evidence for Review acceptance;
      - LOW_HPS_WEAK:
          every documentary dimension remains at/above the 2.5 dimension floor,
          but the overall HPS is expected to remain below 3.0;
      - DIMENSION_FLOOR_WEAK:
          contains a deliberately weak documentary dimension (<2.5).

    IMPORTANT SCIENTIFIC INTERPRETATION
    -----------------------------------
    These are controlled governance scenarios, not observed hospital-site
    provenance records and not estimates of real-world admission prevalence.
    The archetype composition is specified BEFORE model training and does not
    use predictive performance, test labels, or downstream model outcomes.

    DQ (dim2) is NEVER synthesized here; it remains measured from TRAIN data.
    The frozen evidence seed randomly assigns the pre-generated archetypes to
    client identities.
    """
    domain = str(domain).lower()
    if domain not in CRITICAL_FACTORS_BY_DOMAIN:
        raise ValueError(f"Unsupported domain: {domain!r}")

    n = len(client_ids)
    rng = np.random.default_rng(int(evidence_seed))

    critical_policy = CRITICAL_FACTORS_BY_DOMAIN[domain]
    critical_keys = {
        (dim, factor)
        for dim, factor_list in critical_policy.items()
        for factor in factor_list
    }

    def make_documentary_profile(role: str, variant: int):
        factors = {
            dim: {}
            for dim in DOCUMENTARY_DIMS
        }

        if role == "DIRECT_STRONG":
            # Strong but not uniformly perfect. Every factor is at least 4,
            # while some values reach 5 according to a deterministic variant.
            for dim in DOCUMENTARY_DIMS:
                for j, factor in enumerate(FACTOR_NAMES[dim]):
                    minimum = float(factor_minima[dim][factor])
                    base = max(4.0, minimum)
                    bonus = 1.0 if ((j + variant + len(dim)) % 3 == 0) else 0.0
                    factors[dim][factor] = float(min(5.0, base + bonus))

        elif role == "REVIEW_RECOVERABLE":
            # Start from moderate evidence: all documentary dimensions remain
            # safely above the 2.5 floor without making HPS automatically high.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = 3.0

            critical_sequence = [
                (dim, factor)
                for dim, factor_list in critical_policy.items()
                for factor in factor_list
            ]

            # Put critical factors at their policy adequacy references.
            for dim, factor in critical_sequence:
                factors[dim][factor] = float(
                    factor_minima[dim][factor]
                )

            # Intentionally allow ONE critical factor to fall one point below
            # its individual minimum, while another critical factor with room
            # is strengthened. This is exactly the compensatory situation that
            # Automated Review is intended to resolve.
            high_min = [
                (dim, factor)
                for dim, factor in critical_sequence
                if float(factor_minima[dim][factor]) >= 4.0
            ]
            lower_min = [
                (dim, factor)
                for dim, factor in critical_sequence
                if float(factor_minima[dim][factor]) <= 3.0
            ]

            if high_min and lower_min:
                weak_key = high_min[variant % len(high_min)]
                strong_key = lower_min[variant % len(lower_min)]

                weak_min = float(
                    factor_minima[weak_key[0]][weak_key[1]]
                )
                strong_min = float(
                    factor_minima[strong_key[0]][strong_key[1]]
                )

                factors[weak_key[0]][weak_key[1]] = float(
                    max(0.0, weak_min - 1.0)
                )
                factors[strong_key[0]][strong_key[1]] = float(
                    min(5.0, strong_min + 2.0)
                )

            # Small documentary variation between the two recoverable bundles.
            if variant % 2 == 1:
                factors["dim3"][FACTOR_NAMES["dim3"][0]] = 4.0

        elif role == "REVIEW_LIMITED":
            # Preserve dimensions at/above 2.5 and keep HPS in/near the review
            # region, but make the average critical evidence too weak.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    if dim in critical_policy and (dim, factor) not in critical_keys:
                        factors[dim][factor] = 4.0
                    else:
                        factors[dim][factor] = 3.0

            for dim, factor_list in critical_policy.items():
                for factor in factor_list:
                    minimum = float(factor_minima[dim][factor])
                    factors[dim][factor] = float(
                        max(2.0, minimum - 1.0)
                    )

            if variant % 2 == 1:
                factors["dim3"][FACTOR_NAMES["dim3"][0]] = 4.0

        elif role == "LOW_HPS_WEAK":
            # Each documentary dimension averages >=2.5, but remains weak.
            # This isolates the HPS<3.0 rejection path from the dimension floor.
            for dim in DOCUMENTARY_DIMS:
                names = list(FACTOR_NAMES[dim])
                values = [2.0] * len(names)
                n_three = (len(names) + 1) // 2
                for j in range(n_three):
                    values[j] = 3.0

                for factor, value in zip(names, values):
                    factors[dim][factor] = float(value)

        elif role == "DIMENSION_FLOOR_WEAK":
            # General evidence is moderate, but Timeliness is deliberately below
            # the 2.5 dimension trustworthiness floor.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = 3.0

            for factor in FACTOR_NAMES["dim4"]:
                factors["dim4"][factor] = 2.0

        else:
            raise ValueError(f"Unknown evidence role: {role!r}")

        return factors

    # For the fixed K=10 experiments, pre-specify a balanced governance
    # branch-coverage set:
    #   4 direct-strong
    #   2 review-recoverable
    #   2 review-limited
    #   1 low-HPS weak
    #   1 dimension-floor weak
    #
    # This is not a post-hoc selection by client identity. The ten bundles are
    # generated first and then randomly assigned to clients using evidence_seed.
    if n == 10:
        bundle_specs = [
            ("DIRECT_STRONG", 0),
            ("DIRECT_STRONG", 1),
            ("DIRECT_STRONG", 2),
            ("DIRECT_STRONG", 3),
            ("REVIEW_RECOVERABLE", 0),
            ("REVIEW_RECOVERABLE", 1),
            ("REVIEW_LIMITED", 0),
            ("REVIEW_LIMITED", 1),
            ("LOW_HPS_WEAK", 0),
            ("DIMENSION_FLOOR_WEAK", 0),
        ]
    else:
        # Generic fallback for non-10-client studies: cycle through the same
        # archetypes without conditioning on client data or model performance.
        archetypes = [
            "DIRECT_STRONG",
            "REVIEW_RECOVERABLE",
            "REVIEW_LIMITED",
            "LOW_HPS_WEAK",
            "DIMENSION_FLOOR_WEAK",
        ]
        bundle_specs = [
            (archetypes[j % len(archetypes)], j // len(archetypes))
            for j in range(n)
        ]

    bundles = []
    for b, (role, variant) in enumerate(bundle_specs):
        factors = make_documentary_profile(role, int(variant))

        bundles.append({
            "bundle_id": f"E{b+1:02d}",
            "profile": role,
            "scenario_role": role,
            "profile_variant": int(variant),
            "factors": factors,
        })

    # Randomly assign already-generated governance bundles to client identities.
    assignment = rng.permutation(n)

    evidence_by_client = {}
    rows = []

    for client_pos, cid in enumerate(client_ids):
        bundle = bundles[int(assignment[client_pos])]

        evidence_by_client[cid] = {
            dim: dict(bundle["factors"][dim])
            for dim in DOCUMENTARY_DIMS
        }

        for dim in DOCUMENTARY_DIMS:
            for factor, score in bundle["factors"][dim].items():
                min_rank = float(
                    factor_minima[dim][factor]
                )

                rows.append({
                    "client":
                        str(cid),
                    "bundle_id":
                        bundle["bundle_id"],
                    "evidence_profile":
                        bundle["profile"],
                    "scenario_role":
                        bundle["scenario_role"],
                    "profile_variant":
                        int(bundle["profile_variant"]),
                    "dimension":
                        dim,
                    "dimension_name":
                        DIMENSION_NAMES[dim],
                    "factor":
                        factor,
                    "evidence_source_type":
                        "CONTROLLED_DOCUMENTARY_EVIDENCE",
                    "verified_rubric_level_0_5":
                        float(score),
                    "rubric_score_0_5":
                        float(score),
                    "server_mapped_score_0_5":
                        float(score),
                    "rubric_descriptor":
                        rubric_descriptor(
                            dim,
                            factor,
                            score,
                        ),
                    "human_role":
                        "VERIFY_EVIDENCE_ONLY",
                    "score_assignment":
                        "DETERMINISTIC_SERVER_MAPPING_FROM_VERIFIED_RUBRIC_LEVEL",
                    "admission_decision_by":
                        "SERVER_POLICY",
                    "adequacy_min_rank":
                        min_rank,
                    "adequacy_min_normalized":
                        min_rank / MAX_FACTOR_SCORE,
                    "meets_factor_adequacy":
                        bool(float(score) >= min_rank),
                    "evidence_artifact_id": (
                        f"{cid}-{bundle['bundle_id']}-{dim}-{factor}"
                    ),
                    "validation_status":
                        "CONTROLLED_SCENARIO_EVIDENCE",
                    "evidence_seed":
                        int(evidence_seed),
                    "domain":
                        domain,
                })

    return evidence_by_client, pd.DataFrame(rows)

def dimension_scores_from_factors(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, float]:
    """
    Client-specific HPS dimension scores (0..5):
    mean of the observed factor rubric scores within each dimension.
    """
    out = {}
    for dim in FACTOR_NAMES:
        vals = [
            float(v)
            for v in factors.get(dim, {}).values()
            if np.isfinite(float(v))
        ]
        out[dim] = float(np.mean(vals)) if vals else 0.0
    return out


def compute_hps(
    dimensions: Dict[str, float],
    weights: Dict[str, float] = WEIGHTS_PSCORE_DEFAULT,
) -> float:
    return float(
        sum(float(weights[d]) * float(dimensions[d]) for d in weights)
    )


def client_dimension_adequacy(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, float]:
    """
    Client ACHIEVED adequacy, not WAC.

    A_i,d = mean(client factor ranks in dimension d) / 5.
    """
    scores = {}
    for dim in FACTOR_NAMES:
        vals = [
            float(v)
            for v in factors.get(dim, {}).values()
            if np.isfinite(float(v))
        ]
        scores[dim] = (
            float(np.mean(vals) / MAX_FACTOR_SCORE)
            if vals else 0.0
        )
    return scores


def client_adequacy_score(
    factors: Dict[str, Dict[str, float]]
) -> Tuple[Dict[str, float], float]:
    """
    Legacy audit helper retained for backward comparability only.

    v16.7 does NOT use this value for admission. Review decisions use the
    client Critical WAC_i versus the domain Global Critical WAC.
    """
    by_dim = client_dimension_adequacy(factors)
    cas = float(np.mean(list(by_dim.values())))
    return by_dim, cas


def all_zero_score_factors(
    factors: Dict[str, Dict[str, float]]
) -> List[str]:
    """Audit-only zero list. Zero is not a separate admission rule in v16.7."""
    zeros = []
    for dim in FACTOR_NAMES:
        for factor, value in factors.get(dim, {}).items():
            if np.isfinite(float(value)) and float(value) <= 0.0:
                zeros.append(f"{dim}.{factor}")
    return zeros


def derive_global_critical_wac(domain: str) -> float:
    """
    Domain Global Critical WAC = mean(minimum critical adequacy rank / 5).

    This is a policy reference used only to resolve the HPS Review band.
    """
    domain = str(domain).lower()
    vals = []
    for dim, factors_ in CRITICAL_FACTORS_BY_DOMAIN[domain].items():
        for factor in factors_:
            vals.append(
                float(DOMAIN_FACTOR_MINIMA[domain][dim][factor])
                / MAX_FACTOR_SCORE
            )
    if not vals:
        raise ValueError(f"No critical factors for domain={domain!r}")
    return float(np.mean(vals))


def critical_factor_audit(
    factors: Dict[str, Dict[str, float]],
    domain: str,
) -> Dict[str, Any]:
    """
    Compute the client Critical WAC_i and the strict individual critical gate.

    Direct Auto-Accept:
        every critical factor >= its own factor-specific adequacy minimum.

    Automated Review:
        uses the overall average Critical WAC_i, allowing compensation across
        critical factors while preserving the dimension and HPS floors.
    """
    domain = str(domain).lower()

    scores = {}
    minima = {}
    missing = []
    below_adequacy = []

    for dim, factor_list in CRITICAL_FACTORS_BY_DOMAIN[domain].items():
        for factor in factor_list:
            key = f"{dim}.{factor}"
            minimum = float(
                DOMAIN_FACTOR_MINIMA[domain][dim][factor]
            )
            minima[key] = minimum

            if factor not in factors.get(dim, {}):
                missing.append(key)
                continue

            value = float(factors[dim][factor])

            if not np.isfinite(value):
                missing.append(key)
                continue

            scores[key] = value

            if value < minimum:
                below_adequacy.append(
                    f"{key}:{value:.1f}<{minimum:.1f}"
                )

    expected_n = sum(
        len(v)
        for v in CRITICAL_FACTORS_BY_DOMAIN[domain].values()
    )

    if missing or len(scores) != expected_n:
        return {
            "scores": scores,
            "minima": minima,
            "missing": missing,
            "critical_count": expected_n,
            "critical_wac_i": np.nan,
            "critical_mean_score_0_5": np.nan,
            "global_critical_wac":
                float(derive_global_critical_wac(domain)),
            "all_critical_meet_adequacy": False,
            "critical_below_adequacy": below_adequacy,
        }

    values = np.array(
        list(scores.values()),
        dtype=float,
    )

    return {
        "scores": scores,
        "minima": minima,
        "missing": [],
        "critical_count": expected_n,
        "critical_wac_i":
            float(np.mean(values / MAX_FACTOR_SCORE)),
        "critical_mean_score_0_5":
            float(np.mean(values)),
        "global_critical_wac":
            float(derive_global_critical_wac(domain)),
        "all_critical_meet_adequacy":
            bool(len(below_adequacy) == 0),
        "critical_below_adequacy":
            below_adequacy,
    }

def dimension_floor_failures(
    dimensions: Dict[str, float],
    floor: float = DIMENSION_MIN_FLOOR,
) -> List[str]:
    """Return averaged HPS dimensions below the minimum floor."""
    return [
        dim for dim, value in dimensions.items()
        if (not np.isfinite(float(value))) or float(value) < float(floor)
    ]

def factor_adequacy_attainment(
    factors: Dict[str, Dict[str, float]],
    factor_minima: Dict[str, Dict[str, float]],
) -> Dict[str, Any]:
    total = 0
    passed = 0
    per_dim = {}

    for dim in FACTOR_NAMES:
        dim_total = 0
        dim_passed = 0

        for factor in FACTOR_NAMES[dim]:
            total += 1
            dim_total += 1

            score = float(factors[dim][factor])
            threshold = float(factor_minima[dim][factor])

            if score >= threshold:
                passed += 1
                dim_passed += 1

        per_dim[dim] = {
            "passed": dim_passed,
            "total": dim_total,
            "fraction": float(dim_passed / max(1, dim_total)),
        }

    return {
        "passed": passed,
        "total": total,
        "fraction": float(passed / max(1, total)),
        "per_dim": per_dim,
    }


def tadp_decision(
    factors: Dict[str, Dict[str, float]],
    domain: str,
    good_cut: float = GOOD_CUT,
    high_cut: float = HIGH_CUT,
) -> Dict[str, Any]:
    """
    v16.7 final fully automated admission policy.

    Flow:
      EVERY dimension >= 2.5?
          NO -> AUTO-REJECT

      HPS >= 3.0?
          NO -> AUTO-REJECT

      HPS >= 3.5?
          YES:
              all critical factors >= own adequacy minima?
                  YES -> DIRECT AUTO-ACCEPT
                  NO  -> AUTOMATED REVIEW
          NO (3.0 <= HPS < 3.5):
              -> AUTOMATED REVIEW

      AUTOMATED REVIEW:
          Critical WAC_i >= Global Critical WAC
              -> ACCEPT AFTER REVIEW
          otherwise
              -> AUTO-REJECT
    """
    domain = str(domain).lower()

    factor_minima = DOMAIN_FACTOR_MINIMA[domain]
    dimension_wac = DOMAIN_DIMENSION_WAC[domain]
    global_wac = float(DOMAIN_GLOBAL_WAC[domain])
    global_critical_wac = float(
        derive_global_critical_wac(domain)
    )

    dims = dimension_scores_from_factors(factors)
    hps = compute_hps(dims)

    attainment = factor_adequacy_attainment(
        factors,
        factor_minima,
    )

    zeros = all_zero_score_factors(factors)

    dim_floor_failed = dimension_floor_failures(
        dims,
        floor=DIMENSION_MIN_FLOOR,
    )

    critical = critical_factor_audit(
        factors,
        domain,
    )

    critical_score_string = ";".join(
        (
            f"{key}={value:.1f}"
            f"(min={critical['minima'][key]:.1f})"
        )
        for key, value in critical["scores"].items()
    )

    base = {
        "hps": float(hps),
        "policy_dimension_wac": dimension_wac,
        "global_domain_wac": global_wac,
        "global_critical_wac": global_critical_wac,
        "critical_wac_i": critical["critical_wac_i"],
        "critical_wac_margin": (
            float(critical["critical_wac_i"])
            - global_critical_wac
            if np.isfinite(critical["critical_wac_i"])
            else np.nan
        ),
        "critical_mean_score_0_5":
            critical["critical_mean_score_0_5"],
        "critical_factor_count":
            int(critical["critical_count"]),
        "critical_scores":
            critical_score_string,
        "critical_missing":
            ";".join(critical["missing"]),
        "all_critical_meet_adequacy":
            bool(critical["all_critical_meet_adequacy"]),
        "critical_below_adequacy":
            ";".join(critical["critical_below_adequacy"]),
        "factor_adequacy_passed":
            int(attainment["passed"]),
        "factor_adequacy_total":
            int(attainment["total"]),
        "factor_adequacy_fraction":
            float(attainment["fraction"]),
        "dimensions":
            dims,
        "dimension_min_floor":
            float(DIMENSION_MIN_FLOOR),
        "dimension_floor_failures":
            ";".join(dim_floor_failed),
        # Audit only; zero is not a separate decision rule.
        "all_zero_factors":
            ";".join(zeros),
    }

    # Gate 1: every complete trustworthiness dimension must clear 2.5/5.
    if dim_floor_failed:
        return {
            **base,
            "decision_path":
                "DIMENSION_FLOOR",
            "initial_action":
                "AUTO_REJECT",
            "final_action":
                "REJECT",
            "status":
                "AUTO_REJECTED_DIMENSION_FLOOR",
            "reason": (
                f"At least one averaged dimension is below "
                f"{DIMENSION_MIN_FLOOR:.1f}/5: "
                + ";".join(
                    f"{dim}={dims[dim]:.3f}"
                    for dim in dim_floor_failed
                )
            ),
        }

    # Missing/non-finite critical evidence is fail-closed in this experiment.
    if critical["missing"]:
        return {
            **base,
            "decision_path":
                "MISSING_CRITICAL_EVIDENCE",
            "initial_action":
                "AUTO_REJECT",
            "final_action":
                "REJECT",
            "status":
                "AUTO_REJECTED_MISSING_CRITICAL_EVIDENCE",
            "reason": (
                "Missing/non-finite verified critical evidence: "
                + ";".join(critical["missing"])
            ),
        }

    # Gate 2: overall HPS lower bound.
    if hps < float(good_cut):
        return {
            **base,
            "decision_path":
                "LOW_HPS",
            "initial_action":
                "AUTO_REJECT",
            "final_action":
                "REJECT",
            "status":
                "AUTO_REJECTED_LOW_HPS",
            "reason":
                f"HPS {hps:.3f} < {good_cut:.3f}",
        }

    # High-HPS strict direct route.
    if hps >= float(high_cut):
        if critical["all_critical_meet_adequacy"]:
            return {
                **base,
                "decision_path":
                    "DIRECT_AUTO_ACCEPT",
                "initial_action":
                    "AUTO_ACCEPT",
                "final_action":
                    "ACCEPT",
                "status":
                    "DIRECT_AUTO_ACCEPTED",
                "reason": (
                    f"HPS {hps:.3f} >= {high_cut:.3f}; all critical "
                    "factors meet their individual adequacy requirements"
                ),
            }

        # High HPS but failed strict critical gate:
        # fall back to the same automated review rather than immediate rejection.
        review_origin = (
            "HIGH_HPS_CRITICAL_FALLBACK"
        )

    else:
        # 3.0 <= HPS < 3.5
        review_origin = (
            "HPS_REVIEW_BAND"
        )

    # Automated Review for both origins.
    review_pass = bool(
        float(critical["critical_wac_i"])
        >= global_critical_wac
    )

    if review_pass:
        return {
            **base,
            "decision_path":
                review_origin,
            "initial_action":
                "AUTOMATED_REVIEW",
            "final_action":
                "ACCEPT",
            "status":
                "ACCEPTED_AFTER_AUTOMATED_REVIEW",
            "reason": (
                f"{review_origin}: Critical WAC_i "
                f"{critical['critical_wac_i']:.3f} >= "
                f"Global Critical WAC {global_critical_wac:.3f}"
            ),
        }

    return {
        **base,
        "decision_path":
            review_origin,
        "initial_action":
            "AUTOMATED_REVIEW",
        "final_action":
            "REJECT",
        "status":
            "AUTO_REJECTED_REVIEW_CRITICAL_WAC",
        "reason": (
            f"{review_origin}: Critical WAC_i "
            f"{critical['critical_wac_i']:.3f} < "
            f"Global Critical WAC {global_critical_wac:.3f}"
        ),
    }

def build_tadp_governance(
    client_ids: List[str],
    documentary_evidence: Dict[str, Dict[str, Dict[str, float]]],
    dq_scores: Dict[str, Dict[str, float]],
    run: int,
    evidence_seed: int,
    domain: str,
) -> pd.DataFrame:
    domain = str(domain).lower()
    rows = []

    for cid in client_ids:
        factors = {
            dim: dict(documentary_evidence[cid][dim])
            for dim in DOCUMENTARY_DIMS
        }

        factors["dim2"] = {
            name: float(dq_scores[cid][name])
            for name in FACTOR_NAMES["dim2"]
        }

        d = tadp_decision(
            factors,
            domain=domain,
        )

        row = {
            "run": int(run),
            "domain": domain,
            "evidence_seed": int(evidence_seed),
            "client": str(cid),
            "hps": float(d["hps"]),
            "global_domain_wac":
                float(d["global_domain_wac"]),
            "global_critical_wac":
                float(d["global_critical_wac"]),
            "critical_wac_i": (
                float(d["critical_wac_i"])
                if np.isfinite(d["critical_wac_i"])
                else np.nan
            ),
            "critical_wac_margin": (
                float(d["critical_wac_margin"])
                if np.isfinite(d["critical_wac_margin"])
                else np.nan
            ),
            "critical_mean_score_0_5": (
                float(d["critical_mean_score_0_5"])
                if np.isfinite(d["critical_mean_score_0_5"])
                else np.nan
            ),
            "critical_factor_count":
                int(d["critical_factor_count"]),
            "critical_scores":
                d["critical_scores"],
            "critical_missing":
                d["critical_missing"],
            "all_critical_meet_adequacy":
                bool(d["all_critical_meet_adequacy"]),
            "critical_below_adequacy":
                d["critical_below_adequacy"],
            "dimension_min_floor":
                float(d["dimension_min_floor"]),
            "dimension_floor_failures":
                d["dimension_floor_failures"],
            "factor_adequacy_passed":
                int(d["factor_adequacy_passed"]),
            "factor_adequacy_total":
                int(d["factor_adequacy_total"]),
            "factor_adequacy_fraction":
                float(d["factor_adequacy_fraction"]),
            "decision_path":
                d["decision_path"],
            "initial_action":
                d["initial_action"],
            "final_action":
                d["final_action"],
            "status":
                d["status"],
            "reason":
                d["reason"],
            "all_zero_factors":
                d["all_zero_factors"],
        }

        for dim, value in d["dimensions"].items():
            row[
                f"{dim}_score_0_5"
            ] = float(value)

        for dim, value in d["policy_dimension_wac"].items():
            row[
                f"{dim}_policy_wac"
            ] = float(value)

        rows.append(row)

    return pd.DataFrame(rows)

def accepted_tadp_vr(governance_df: pd.DataFrame) -> List[str]:
    return governance_df.loc[
        governance_df["final_action"].eq("ACCEPT"), "client"
    ].astype(str).tolist()


def accepted_tadp_sda(
    governance_df: pd.DataFrame
) -> List[str]:
    """
    Select one best TADP-eligible client for SDA from already accepted clients.
    """
    eligible = governance_df[
        governance_df["final_action"].eq("ACCEPT")
    ].copy()

    if eligible.empty:
        raise RuntimeError(
            "TADP-SDA cannot select a client: no TADP-eligible client."
        )

    eligible["direct_accept_priority"] = (
        eligible["status"]
        .eq("DIRECT_AUTO_ACCEPTED")
        .astype(int)
    )

    eligible = eligible.sort_values(
        [
            "hps",
            "direct_accept_priority",
            "critical_wac_i",
            "client",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    )

    return [
        str(
            eligible.iloc[0]["client"]
        )
    ]

def build_full_factor_evidence_table(
    client_ids: List[str],
    documentary_evidence: Dict[str, Dict[str, Dict[str, float]]],
    dq_scores: Dict[str, Dict[str, float]],
    dq_audit: pd.DataFrame,
    evidence_seed: int,
    domain: str,
) -> pd.DataFrame:
    """
    Full 28-factor evidence audit.

    Critical factors are flagged with their own policy adequacy minima.
    Direct Auto-Accept uses these individual minima; Automated Review uses the
    aggregate Critical WAC_i.
    """
    domain = str(domain).lower()
    factor_minima = DOMAIN_FACTOR_MINIMA[domain]
    critical_policy = CRITICAL_FACTORS_BY_DOMAIN[domain]

    dq_index = dq_audit.copy()
    dq_index["client"] = dq_index["client"].astype(str)
    dq_index = dq_index.set_index(
        "client",
        drop=False,
    )

    def policy_fields(dim, factor):
        is_critical = bool(
            factor in critical_policy.get(
                dim,
                [],
            )
        )

        return {
            "is_critical_factor":
                is_critical,
            "critical_adequacy_min_rank": (
                float(factor_minima[dim][factor])
                if is_critical
                else np.nan
            ),
            "critical_adequacy_min_normalized": (
                float(factor_minima[dim][factor])
                / MAX_FACTOR_SCORE
                if is_critical
                else np.nan
            ),
        }

    rows = []

    for cid in client_ids:
        cid = str(cid)

        for dim in DOCUMENTARY_DIMS:
            for factor in FACTOR_NAMES[dim]:
                score = float(
                    documentary_evidence[cid][dim][factor]
                )
                min_rank = float(
                    factor_minima[dim][factor]
                )

                rows.append({
                    "client": cid,
                    "domain": domain,
                    "dimension": dim,
                    "dimension_name":
                        DIMENSION_NAMES[dim],
                    "factor": factor,
                    "factor_source":
                        "CONTROLLED_DOCUMENTARY_EVIDENCE",
                    "raw_measured_value":
                        np.nan,
                    "rubric_score_0_5":
                        score,
                    "rubric_descriptor":
                        rubric_descriptor(
                            dim,
                            factor,
                            score,
                        ),
                    "adequacy_min_rank":
                        min_rank,
                    "adequacy_min_normalized":
                        min_rank / MAX_FACTOR_SCORE,
                    "meets_factor_adequacy":
                        bool(score >= min_rank),
                    **policy_fields(
                        dim,
                        factor,
                    ),
                    "evidence_seed":
                        int(evidence_seed),
                })

        for factor in FACTOR_NAMES["dim2"]:
            score = float(
                dq_scores[cid][factor]
            )
            min_rank = float(
                factor_minima["dim2"][factor]
            )
            raw_col = DQ_RAW_METRIC_BY_FACTOR[
                factor
            ]

            raw_value = (
                float(
                    dq_index.loc[
                        cid,
                        raw_col,
                    ]
                )
                if raw_col in dq_index.columns
                else np.nan
            )

            rows.append({
                "client": cid,
                "domain": domain,
                "dimension": "dim2",
                "dimension_name":
                    DIMENSION_NAMES["dim2"],
                "factor": factor,
                "factor_source":
                    "MACHINE_MEASURED_TRAIN_ONLY",
                "raw_measured_value":
                    raw_value,
                "rubric_score_0_5":
                    score,
                "rubric_descriptor":
                    rubric_descriptor(
                        "dim2",
                        factor,
                        score,
                    ),
                "adequacy_min_rank":
                    min_rank,
                "adequacy_min_normalized":
                    min_rank / MAX_FACTOR_SCORE,
                "meets_factor_adequacy":
                    bool(score >= min_rank),
                **policy_fields(
                    "dim2",
                    factor,
                ),
                "evidence_seed":
                    int(evidence_seed),
            })

    out = pd.DataFrame(
        rows
    )

    expected_rows = len(client_ids) * 28

    if len(out) != expected_rows:
        raise RuntimeError(
            f"Full evidence matrix should contain "
            f"{expected_rows} rows; found {len(out)}."
        )

    return out

# ======================================================================================
# GREAT EXPECTATIONS (GX CORE) — NATIVE VALIDATOR BASELINE
# ======================================================================================
# GX is used here in its native role: validate each client's TRAIN-only data against
# a predefined Expectation Suite and use the suite-level success flag as PASS/FAIL.
# There is NO ranking, NO forced-K selection, and NO HPS/WAC information in this
# baseline. Great Expectations reports suite success=True only when all configured
# Expectations pass. The suite deliberately uses common technical checks only:
#   1) schema, 2) datatype consistency, 3) required-value ranges,
#   4) missingness, 5) duplicates / unique IDs, 6) label/domain validity,
#   7) structural integrity.
# Distribution checks are intentionally omitted because non-IID client distributions
# are expected in federated learning and should not by themselves constitute failure.
GX_CORE_VERSION = "1.23.0"
GX_VALUE_MOSTLY = 0.99
GX_NONNULL_REFERENCE_MIN = 0.80
GX_NONNULL_TOLERANCE = 0.15


def ensure_great_expectations():
    """Import pinned GX Core; install once in Colab if unavailable."""
    try:
        import great_expectations as gx
        return gx
    except ImportError:
        import subprocess
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            f"great_expectations=={GX_CORE_VERSION}",
        ])
        import great_expectations as gx
        return gx


def _gx_meta(category: str, name: str) -> Dict[str, Any]:
    return {
        "dq_category": str(category),
        "check_name": str(name),
        "baseline": "GX_NATIVE_VALIDATOR",
    }


def _gx_add(suite, specs: List[Dict[str, Any]], expectation, category: str, name: str):
    suite.add_expectation(expectation)
    specs.append({"category": str(category), "name": str(name)})


def _aggregate_nonnull_reference(
    client_frames: Dict[str, pd.DataFrame],
    columns: List[str],
) -> Dict[str, float]:
    total_rows = float(sum(len(df) for df in client_frames.values()))
    out = {}
    for c in columns:
        nonnull = sum(int(df[c].notna().sum()) for df in client_frames.values() if c in df.columns)
        out[c] = float(nonnull / max(1.0, total_rows))
    return out


def build_gx_healthcare_reference(client_frames: Dict[str, pd.DataFrame], preprocessor: Any) -> Dict[str, Any]:
    first = next(iter(client_frames.values()))

    # GX datatype validation is intentionally independent of TADP's model
    # preprocessing type inference.  The preprocessor labels a feature numeric
    # when >=95% of pooled TRAIN non-missing values are parseable; reusing that
    # inferred list inside GX and then requiring 99% parseability per client
    # creates an artificial contradiction.  GX therefore validates datatype
    # consistency only for fields whose numeric meaning is explicit in the
    # Diabetes data schema.
    known_numeric_fields = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]
    gx_numeric_cols = [c for c in known_numeric_fields if c in first.columns]

    return {
        "original_columns": list(first.columns),
        "numeric_cols": gx_numeric_cols,
        "feature_cols": list(preprocessor.feature_cols),
        "nonnull_reference": _aggregate_nonnull_reference(
            client_frames, list(preprocessor.feature_cols)
        ),
    }


def build_gx_healthcare_validation_frame(df: pd.DataFrame, preprocessor: Any) -> pd.DataFrame:
    """GX validation view; raw TRAIN records remain the source of all checks."""
    out = df.copy()
    for c in preprocessor.numeric_cols:
        raw = df[c]
        parsed = pd.to_numeric(raw, errors="coerce")
        type_ok = raw.isna() | parsed.notna()
        out[c] = parsed.astype(float)
        out[f"__gx_type_ok__{c}"] = type_ok.astype(np.int8)
    return out


def build_gx_healthcare_suite(gx, reference: Dict[str, Any]):
    """Simple native GX technical-validation suite for the Diabetes TRAIN shards."""
    gxe = gx.expectations
    suite = gx.ExpectationSuite(name="tadp_healthcare_gx_native_suite")
    specs = []
    cols = reference["original_columns"]

    # 1) Schema.
    _gx_add(
        suite, specs,
        gxe.ExpectTableColumnsToMatchSet(
            column_set=cols,
            exact_match=False,
            severity="critical",
            meta=_gx_meta("Schema", "required_column_set"),
        ),
        "Schema", "required_column_set",
    )

    # 2) Datatype consistency for columns inferred as numeric from TRAIN only.
    for c in reference["numeric_cols"]:
        diag = f"__gx_type_ok__{c}"
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=diag,
                value_set=[1],
                mostly=GX_VALUE_MOSTLY,
                severity="critical",
                meta=_gx_meta("Datatype Consistency", f"{c}_numeric_parseability"),
            ),
            "Datatype Consistency", f"{c}_numeric_parseability",
        )

    # 3) Required-value ranges for known count/duration fields.
    nonnegative_fields = [
        "num_lab_procedures", "num_procedures", "num_medications",
        "number_outpatient", "number_emergency", "number_inpatient",
        "number_diagnoses",
    ]
    for c in nonnegative_fields:
        if c in cols:
            _gx_add(
                suite, specs,
                gxe.ExpectColumnValuesToBeBetween(
                    column=c, min_value=0.0, mostly=GX_VALUE_MOSTLY,
                    severity="critical",
                    meta=_gx_meta("Required Value Ranges", f"{c}_nonnegative"),
                ),
                "Required Value Ranges", f"{c}_nonnegative",
            )
    if "time_in_hospital" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeBetween(
                column="time_in_hospital", min_value=1.0, max_value=14.0,
                mostly=GX_VALUE_MOSTLY,
                severity="critical",
                meta=_gx_meta("Required Value Ranges", "time_in_hospital_range"),
            ),
            "Required Value Ranges", "time_in_hospital_range",
        )

    # 4) Missingness. Only columns that are substantially populated in the frozen
    # TRAIN reference are treated as required-enough for a missingness expectation.
    for c, global_nonnull in reference["nonnull_reference"].items():
        if c not in cols or float(global_nonnull) < GX_NONNULL_REFERENCE_MIN:
            continue
        minimum = max(0.70, float(global_nonnull) - GX_NONNULL_TOLERANCE)
        _gx_add(
            suite, specs,
            gxe.ExpectColumnProportionOfNonNullValuesToBeBetween(
                column=c, min_value=float(minimum), max_value=1.0,
                severity="warning",
                meta=_gx_meta("Missingness", f"{c}_nonnull"),
            ),
            "Missingness", f"{c}_nonnull",
        )

    # 5) Duplicates / unique identifiers.
    if "encounter_id" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeUnique(
                column="encounter_id",
                severity="critical",
                meta=_gx_meta("Duplicates / Unique IDs", "encounter_id_unique"),
            ),
            "Duplicates / Unique IDs", "encounter_id_unique",
        )

    # 6) Labels / domain validity.
    if "_target" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column="_target", value_set=[0, 1, 2],
                severity="critical",
                meta=_gx_meta("Labels / Domain Validity", "target_domain"),
            ),
            "Labels / Domain Validity", "target_domain",
        )

    # 7) Structural integrity.
    if "_row_id" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeUnique(
                column="_row_id",
                severity="critical",
                meta=_gx_meta("Structural Integrity", "row_id_unique"),
            ),
            "Structural Integrity", "row_id_unique",
        )
    _gx_add(
        suite, specs,
        gxe.ExpectTableRowCountToBeBetween(
            min_value=100,
            severity="warning",
            meta=_gx_meta("Structural Integrity", "minimum_client_rows"),
        ),
        "Structural Integrity", "minimum_client_rows",
    )
    return suite, specs


def build_gx_cifar_metadata(X: np.ndarray, y: np.ndarray) -> pd.DataFrame:
    X = np.asarray(X)
    y = np.asarray(y).reshape(-1)
    if X.ndim != 4:
        raise RuntimeError(f"Expected CIFAR tensor [N,H,W,C], got shape={X.shape}")
    finite = np.isfinite(X.astype(np.float32)).reshape(len(X), -1).all(axis=1)
    flat = X.reshape(len(X), -1)
    return pd.DataFrame({
        "sample_id": np.arange(len(X), dtype=np.int64),
        "label": y.astype(np.int32),
        "height": np.full(len(X), X.shape[1], dtype=np.int32),
        "width": np.full(len(X), X.shape[2], dtype=np.int32),
        "channels": np.full(len(X), X.shape[3], dtype=np.int32),
        "dtype_ok": np.full(len(X), int(X.dtype == np.uint8), dtype=np.int8),
        "finite": finite.astype(np.int8),
        "pixel_min": flat.min(axis=1).astype(float),
        "pixel_max": flat.max(axis=1).astype(float),
    })


def build_gx_cifar_reference(client_raw: Dict[str, Tuple[np.ndarray, np.ndarray]]) -> Dict[str, Any]:
    # Native validator uses fixed technical constraints; no distribution reference needed.
    return {}


def build_gx_cifar_suite(gx, reference: Dict[str, Any]):
    """Simple native GX technical-validation suite for CIFAR-10 client metadata."""
    gxe = gx.expectations
    suite = gx.ExpectationSuite(name="tadp_cifar10_gx_native_suite")
    specs = []
    columns = [
        "sample_id", "label", "height", "width", "channels",
        "dtype_ok", "finite", "pixel_min", "pixel_max",
    ]

    _gx_add(
        suite, specs,
        gxe.ExpectTableColumnsToMatchSet(
            column_set=columns, exact_match=True,
            meta=_gx_meta("Schema", "image_metadata_schema"),
        ),
        "Schema", "image_metadata_schema",
    )
    for c in columns:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToNotBeNull(
                column=c,
                severity="warning",
                meta=_gx_meta("Missingness", f"{c}_not_null"),
            ),
            "Missingness", f"{c}_not_null",
        )
    for c, value in [("height", 32), ("width", 32), ("channels", 3), ("dtype_ok", 1), ("finite", 1)]:
        category = "Datatype Consistency" if c in {"dtype_ok", "finite"} else "Structural Integrity"
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=c, value_set=[value],
                meta=_gx_meta(category, f"{c}_constraint"),
            ),
            category, f"{c}_constraint",
        )
    for c in ["pixel_min", "pixel_max"]:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeBetween(
                column=c, min_value=0.0, max_value=255.0,
                meta=_gx_meta("Required Value Ranges", f"{c}_valid_range"),
            ),
            "Required Value Ranges", f"{c}_valid_range",
        )
    _gx_add(
        suite, specs,
        gxe.ExpectColumnValuesToBeInSet(
            column="label", value_set=list(range(10)),
            meta=_gx_meta("Labels / Domain Validity", "label_domain"),
        ),
        "Labels / Domain Validity", "label_domain",
    )
    _gx_add(
        suite, specs,
        gxe.ExpectColumnValuesToBeUnique(
            column="sample_id",
            meta=_gx_meta("Duplicates / Unique IDs", "sample_id_unique"),
        ),
        "Duplicates / Unique IDs", "sample_id_unique",
    )
    _gx_add(
        suite, specs,
        gxe.ExpectTableRowCountToBeBetween(
            min_value=100,
            severity="warning",
            meta=_gx_meta("Structural Integrity", "minimum_client_images"),
        ),
        "Structural Integrity", "minimum_client_images",
    )
    return suite, specs


def ge_governance(
    client_data: Dict[str, Any],
    client_ids: List[str],
    accept_count: Optional[int] = None,  # ignored; retained only for call compatibility
    domain: str = "healthcare",
    preprocessor: Optional[Any] = None,
    dq_scores: Optional[Dict[str, Dict[str, float]]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run native GX suite validation and PASS only clients whose entire suite succeeds."""
    domain = str(domain).lower()
    gx = ensure_great_expectations()
    context = gx.get_context(mode="ephemeral")
    datasource = context.data_sources.add_pandas(name=f"tadp_gx_native_{domain}_datasource")
    asset = datasource.add_dataframe_asset(name=f"tadp_gx_native_{domain}_asset")
    batch_definition = asset.add_batch_definition_whole_dataframe(name="whole_client_train_partition")

    if domain == "healthcare":
        if preprocessor is None:
            raise ValueError("Healthcare GX native baseline requires TRAIN-only preprocessor metadata.")
        reference = build_gx_healthcare_reference(client_data, preprocessor)
        suite, specs = build_gx_healthcare_suite(gx, reference)
        make_frame = lambda cid: build_gx_healthcare_validation_frame(client_data[cid], preprocessor)
    elif domain == "cifar10":
        reference = build_gx_cifar_reference(client_data)
        suite, specs = build_gx_cifar_suite(gx, reference)
        def make_frame(cid):
            X, y = client_data[cid]
            return build_gx_cifar_metadata(X, y)
    else:
        raise ValueError(f"Unsupported GX domain: {domain!r}")

    summary_rows, detail_rows = [], []
    for cid in client_ids:
        frame = make_frame(cid)
        batch = batch_definition.get_batch(batch_parameters={"dataframe": frame})
        validation = batch.validate(suite)
        results = list(validation.results)
        if len(results) != len(specs):
            raise RuntimeError(
                f"GX result/spec mismatch for client {cid}: {len(results)} vs {len(specs)}"
            )

        passed = 0
        critical_failures = 0
        warning_failures = 0
        info_failures = 0
        category_totals, category_passed = {}, {}
        observed_names = []

        for result in results:
            success = bool(result.success)
            passed += int(success)
            cfg = result.expectation_config
            meta = getattr(cfg, "meta", None) or {}
            name = str(meta.get("check_name", getattr(cfg, "type", "GX_EXPECTATION")))
            cat = str(meta.get("dq_category", "Technical Validation"))

            sev_obj = getattr(cfg, "severity", None)
            sev = getattr(sev_obj, "value", sev_obj)
            sev = str(sev if sev is not None else "critical").lower()
            if "." in sev:
                sev = sev.split(".")[-1]
            if sev not in {"critical", "warning", "info"}:
                sev = "critical"

            if not success:
                if sev == "critical":
                    critical_failures += 1
                elif sev == "warning":
                    warning_failures += 1
                else:
                    info_failures += 1

            observed_names.append(name)
            category_totals[cat] = category_totals.get(cat, 0) + 1
            category_passed[cat] = category_passed.get(cat, 0) + int(success)

            detail_rows.append({
                "client": str(cid),
                "domain": domain,
                "gx_version": GX_CORE_VERSION,
                "expectation_name": name,
                "dq_category": cat,
                "severity": sev,
                "success": success,
            })

        expected_names = sorted(str(x["name"]) for x in specs)
        if sorted(observed_names) != expected_names:
            raise RuntimeError(
                f"GX expectation identity mismatch for client {cid}: "
                f"expected={expected_names}, observed={sorted(observed_names)}"
            )

        total = len(specs)
        stats = getattr(validation, "statistics", {}) or {}
        suite_success = bool(validation.success)

        # GX-native severity-aware operational gate.
        # GX itself exposes the maximum failed severity for a Validation Result.
        # We use that native result rather than reconstructing the gate from a
        # custom ranking.  A failed Expectation execution is also treated by GX
        # as CRITICAL.
        max_failed_severity_obj = validation.get_max_severity_failure()
        if max_failed_severity_obj is None:
            max_failed_severity = "none"
        else:
            max_failed_severity = str(
                getattr(max_failed_severity_obj, "value", max_failed_severity_obj)
            ).lower()
            if "." in max_failed_severity:
                max_failed_severity = max_failed_severity.split(".")[-1]

        operational_valid = (max_failed_severity != "critical")

        row = {
            "client": str(cid),
            "gx_version": GX_CORE_VERSION,
            "gx_native_suite_success": suite_success,
            "gx_operational_valid": bool(operational_valid),
            "gx_critical_failures": int(critical_failures),
            "gx_warning_failures": int(warning_failures),
            "gx_info_failures": int(info_failures),
            "gx_max_failed_severity": max_failed_severity,
            "ge_expectations_passed": int(stats.get("successful_expectations", passed)),
            "ge_expectations_total": int(stats.get("evaluated_expectations", total)),
            "ge_pass_rate": float(
                stats.get("success_percent", 100.0 * passed / max(1, total))
            ) / 100.0,
            "ge_native_validation_class": (
                "GX_SUITE_PASS"
                if suite_success
                else (
                    "GX_WARNING_ONLY"
                    if operational_valid
                    else "GX_CRITICAL_FAILURE"
                )
            ),
            "ge_final_action": "ACCEPT" if operational_valid else "REJECT",
            "ge_policy": (
                "GX Core severity-aware validation; ACCEPT requires zero critical "
                "Expectation failures; warning/info failures are reported but do not "
                "exclude; no ranking and no forced-K selection"
            ),
        }

        for cat in sorted(category_totals):
            safe = re.sub(r"[^a-z0-9]+", "_", cat.lower()).strip("_")
            row[f"ge_{safe}_passed"] = int(category_passed.get(cat, 0))
            row[f"ge_{safe}_total"] = int(category_totals[cat])

        summary_rows.append(row)

    return pd.DataFrame(summary_rows), pd.DataFrame(detail_rows)


def score_lower_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [best_upper, score4_upper, score3_upper, score2_upper, score1_upper]
    value <= cuts[0] => 5; ... value <= cuts[4] => 1; else 0.
    """
    v = float(value)
    for score, upper in zip([5, 4, 3, 2, 1], cuts):
        if v <= float(upper):
            return float(score)
    return 0.0


def score_higher_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [score5_lower, score4_lower, score3_lower, score2_lower, score1_lower]
    """
    v = float(value)
    for score, lower in zip([5, 4, 3, 2, 1], cuts):
        if v >= float(lower):
            return float(score)
    return 0.0


def js_divergence(p, q, eps=1e-12) -> float:
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / max(eps, p.sum())
    q = q / max(eps, q.sum())
    m = 0.5 * (p + q)
    kl_pm = np.sum(np.where(p > 0, p * np.log((p + eps) / (m + eps)), 0.0))
    kl_qm = np.sum(np.where(q > 0, q * np.log((q + eps) / (m + eps)), 0.0))
    return float(0.5 * (kl_pm + kl_qm))


# ======================================================================================
# MODEL / METRICS / FL TRAINING
# ======================================================================================

def class_weight_dict(y: np.ndarray) -> Dict[int, float]:
    y = np.asarray(y, dtype=np.int32)
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(w) for c, w in zip(classes, weights)}


def model_parameter_bytes(model: keras.Model) -> int:
    return int(sum(np.asarray(w).nbytes for w in model.get_weights()))


def evaluate_model(
    model: keras.Model, X: np.ndarray, y: np.ndarray, n_classes: int
) -> Dict[str, float]:
    p = model.predict(X, batch_size=512, verbose=0)
    pred = np.argmax(p, axis=1)
    out = {
        "accuracy": float(accuracy_score(y, pred)),
        "precision_macro": float(
            precision_score(y, pred, average="macro", zero_division=0)
        ),
        "recall_macro": float(
            recall_score(y, pred, average="macro", zero_division=0)
        ),
        "f1_macro": float(
            f1_score(y, pred, average="macro", zero_division=0)
        ),
    }
    try:
        out["roc_auc_ovr_macro"] = float(
            roc_auc_score(y, p, multi_class="ovr", average="macro")
        )
    except Exception:
        out["roc_auc_ovr_macro"] = float("nan")
    return out



ROUND_PROGRESS_MONITOR_MAX_SAMPLES = 4096
ROUND_PROGRESS_POWER_W = 12.0


def build_train_monitor_subset(
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    max_samples: int = ROUND_PROGRESS_MONITOR_MAX_SAMPLES,
    seed: int = 99117,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Build a deterministic lightweight monitoring subset from TRAIN client arrays only.

    This subset is used solely for console progress after FL rounds. It never affects
    preprocessing, governance, client selection, optimizer budgets, checkpoint
    decisions, or final reporting. The held-out TEST set remains final-evaluation-only.
    """
    rng = np.random.default_rng(int(seed))
    client_ids = sorted(client_arrays.keys())
    sizes = {cid: int(len(client_arrays[cid][1])) for cid in client_ids}
    total = int(sum(sizes.values()))
    target = int(min(max_samples, total))

    # Proportional allocation across clients, then distribute rounding remainder.
    raw = {cid: target * sizes[cid] / max(1, total) for cid in client_ids}
    take = {cid: min(sizes[cid], int(np.floor(raw[cid]))) for cid in client_ids}

    while sum(take.values()) < target:
        candidates = [c for c in client_ids if take[c] < sizes[c]]
        if not candidates:
            break
        cid = max(candidates, key=lambda c: (raw[c] - take[c], sizes[c], c))
        take[cid] += 1

    xs, ys = [], []
    for cid in client_ids:
        n_take = int(take[cid])
        if n_take <= 0:
            continue
        Xc, yc = client_arrays[cid]
        if n_take >= len(yc):
            idx = np.arange(len(yc))
        else:
            idx = rng.choice(len(yc), size=n_take, replace=False)
        xs.append(np.asarray(Xc[idx]))
        ys.append(np.asarray(yc[idx], dtype=np.int32))

    Xmon = np.concatenate(xs, axis=0)
    ymon = np.concatenate(ys, axis=0)

    # Deterministic shuffle so monitoring batches do not follow client order.
    order = rng.permutation(len(ymon))
    return Xmon[order], ymon[order]


def _fmt_metric(x: float) -> str:
    return "nan" if not np.isfinite(float(x)) else f"{float(x):.4f}"


def print_round_progress(
    progress_context: Optional[Dict[str, Any]],
    round_idx: int,
    round_total: int,
    selected_ids: List[str],
    round_steps: int,
    cumulative_steps: int,
    monitor_metrics: Dict[str, float],
    cumulative_runtime_s: float,
    cumulative_communication_mb: float,
    ram_start_mb: float,
    ram_peak_mb: float,
    extra: str = "",
) -> None:
    """Compact, human-readable progress block after each federated round."""
    ctx = progress_context or {}
    scenario = str(ctx.get("scenario", "Federated scenario"))
    run_idx = int(ctx.get("run_idx", 0))
    run_total = int(ctx.get("run_total", 0))
    scenario_idx = int(ctx.get("scenario_idx", 0))
    scenario_total = int(ctx.get("scenario_total", 0))
    overall_idx = int(ctx.get("overall_idx", 0))
    overall_total = int(ctx.get("overall_total", 0))

    energy_wh = float(
        ROUND_PROGRESS_POWER_W * float(cumulative_runtime_s) / 3600.0
    )
    ram_delta = max(0.0, float(ram_peak_mb) - float(ram_start_mb))

    print("\n" + "-" * 112)
    print(
        f"PROGRESS | overall configuration {overall_idx}/{overall_total} | "
        f"run {run_idx}/{run_total} | scenario {scenario_idx}/{scenario_total}"
    )
    print(f"SCENARIO | {scenario}")
    print(
        f"ROUND    | {round_idx}/{round_total} | "
        f"selected={len(selected_ids)} [{','.join(map(str, selected_ids))}] | "
        f"steps={round_steps} | cumulative_steps={cumulative_steps}"
    )
    if extra:
        print(f"DETAIL   | {extra}")
    print(
        "TRAIN-MONITOR (diagnostic only; TEST untouched) | "
        f"Accuracy={_fmt_metric(monitor_metrics.get('accuracy', np.nan))} | "
        f"F1={_fmt_metric(monitor_metrics.get('f1_macro', np.nan))} | "
        f"AUC={_fmt_metric(monitor_metrics.get('roc_auc_ovr_macro', np.nan))} | "
        f"Precision={_fmt_metric(monitor_metrics.get('precision_macro', np.nan))} | "
        f"Recall={_fmt_metric(monitor_metrics.get('recall_macro', np.nan))}"
    )
    print(
        f"CUMULATIVE OPERATIONAL | runtime={cumulative_runtime_s:.2f}s | "
        f"energy≈{energy_wh:.5f}Wh | communication={cumulative_communication_mb:.3f}MB | "
        f"RAM peak={ram_peak_mb:.1f}MB | RAM Δ={ram_delta:.1f}MB"
    )
    print("-" * 112)


def train_exact_steps(
    model: keras.Model,
    X: np.ndarray,
    y: np.ndarray,
    steps: int,
    batch_size: int,
    seed: int,
    class_weights: Optional[Dict[int, float]] = None,
    prox_reference: Optional[List[np.ndarray]] = None,
    prox_mu: float = 0.0,
):
    """
    Exact mini-batch update count. Used for step-parity audits.
    """
    steps = int(max(1, steps))
    rng = np.random.default_rng(int(seed))
    n = len(y)
    if n == 0:
        raise RuntimeError("Cannot train on an empty dataset.")

    loss_fn = keras.losses.SparseCategoricalCrossentropy(
        reduction=keras.losses.Reduction.NONE
    )

    order = rng.permutation(n)
    cursor = 0

    prox_tensors = None
    if prox_reference is not None and prox_mu > 0:
        prox_tensors = [tf.convert_to_tensor(w) for w in prox_reference]

    for _ in range(steps):
        if cursor + batch_size > n:
            order = rng.permutation(n)
            cursor = 0

        idx = order[cursor:cursor + batch_size]
        cursor += batch_size

        xb = tf.convert_to_tensor(np.asarray(X[idx]), dtype=tf.float32)
        yb_np = np.asarray(y[idx], dtype=np.int32)
        yb = tf.convert_to_tensor(yb_np, dtype=tf.int32)

        with tf.GradientTape() as tape:
            probs = model(xb, training=True)
            per_loss = loss_fn(yb, probs)

            if class_weights:
                sw = np.array(
                    [class_weights.get(int(v), 1.0) for v in yb_np],
                    dtype=np.float32,
                )
                sw_t = tf.convert_to_tensor(sw)
                data_loss = tf.reduce_sum(per_loss * sw_t) / tf.reduce_sum(sw_t)
            else:
                data_loss = tf.reduce_mean(per_loss)

            loss = data_loss

            if prox_tensors is not None:
                prox = tf.constant(0.0, dtype=tf.float32)
                for var, ref in zip(model.trainable_variables, prox_tensors):
                    prox += tf.reduce_sum(tf.square(var - tf.cast(ref, var.dtype)))
                loss = loss + 0.5 * float(prox_mu) * prox

        grads = tape.gradient(loss, model.trainable_variables)
        model.optimizer.apply_gradients(zip(grads, model.trainable_variables))


def natural_steps(n_records: int, batch_size: int, local_epochs: int = 1) -> int:
    return int(max(1, math.ceil(int(n_records) / int(batch_size)) * int(local_epochs)))


def allocate_exact_step_budget(
    selected: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    target_total: int,
    batch_size: int,
    local_epochs: int = 1,
) -> Dict[str, int]:
    natural = {
        cid: natural_steps(len(client_arrays[cid][1]), batch_size, local_epochs)
        for cid in selected
    }
    total_nat = max(1, sum(natural.values()))
    raw = {cid: target_total * natural[cid] / total_nat for cid in selected}
    alloc = {cid: max(1, int(math.floor(raw[cid]))) for cid in selected}

    # Adjust to exact target.
    while sum(alloc.values()) < target_total:
        cid = max(selected, key=lambda c: raw[c] - alloc[c])
        alloc[cid] += 1
    while sum(alloc.values()) > target_total:
        candidates = [c for c in selected if alloc[c] > 1]
        if not candidates:
            break
        cid = min(candidates, key=lambda c: raw[c] - alloc[c])
        alloc[cid] -= 1

    if sum(alloc.values()) != int(target_total):
        raise RuntimeError("Exact step-budget allocation failed.")
    return alloc


def aggregate_weights(
    local_weights: List[List[np.ndarray]],
    sample_sizes: List[int],
    equal_weight: bool = False,
) -> List[np.ndarray]:
    if not local_weights:
        raise RuntimeError("No local weights to aggregate.")
    if equal_weight:
        alpha = np.ones(len(local_weights), dtype=float) / len(local_weights)
    else:
        sizes = np.asarray(sample_sizes, dtype=float)
        alpha = sizes / sizes.sum()

    out = []
    for layer_idx in range(len(local_weights[0])):
        x = sum(alpha[j] * np.asarray(local_weights[j][layer_idx])
                for j in range(len(local_weights)))
        out.append(np.asarray(x))
    return out


def federated_train(
    build_model_fn,
    initial_weights: List[np.ndarray],
    selected_per_round: List[List[str]],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    local_epochs: int,
    class_weights: Dict[int, float],
    run_seed: int,
    exact_step_maps: Optional[List[Dict[str, int]]] = None,
    equal_weight: bool = False,
    fedprox_mu: float = 0.0,
    train_monitor: Optional[Tuple[np.ndarray, np.ndarray]] = None,
    progress_context: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    monitor = RAMMonitor().start()
    start = time.perf_counter()

    global_model = build_model_fn()
    global_model.set_weights([np.array(w, copy=True) for w in initial_weights])

    round_rows = []
    total_steps = 0
    total_selected = 0
    monitor_eval_s = 0.0
    cumulative_comm_raw_b = 0
    param_b = model_parameter_bytes(global_model)

    for r, selected in enumerate(selected_per_round, start=1):
        global_weights = [np.array(w, copy=True) for w in global_model.get_weights()]
        local_weights = []
        local_sizes = []
        round_steps = 0

        if exact_step_maps is None:
            step_map = {
                cid: natural_steps(
                    len(client_arrays[cid][1]), batch_size, local_epochs
                )
                for cid in selected
            }
        else:
            step_map = exact_step_maps[r - 1]

        for j, cid in enumerate(selected):
            Xc, yc = client_arrays[cid]
            local_model = build_model_fn()
            local_model.set_weights(global_weights)
            train_exact_steps(
                local_model, Xc, yc,
                steps=int(step_map[cid]),
                batch_size=batch_size,
                seed=int(run_seed + 1000 * r + 17 * j),
                class_weights=class_weights,
                prox_reference=global_weights if fedprox_mu > 0 else None,
                prox_mu=float(fedprox_mu),
            )
            local_weights.append(
                [np.array(w, copy=True) for w in local_model.get_weights()]
            )
            local_sizes.append(len(yc))
            round_steps += int(step_map[cid])

            del local_model
            tf.keras.backend.clear_session()

        agg = aggregate_weights(
            local_weights, local_sizes, equal_weight=equal_weight
        )
        global_model = build_model_fn()
        global_model.set_weights(agg)

        total_steps += round_steps
        total_selected += len(selected)

        # Cumulative model traffic through this round. Same accounting as the
        # final experiment metric: download + upload + 12% protocol overhead.
        cumulative_comm_raw_b += int(len(selected)) * 2 * int(param_b)
        cumulative_comm_mb = float(
            (cumulative_comm_raw_b * 1.12) / (1024 ** 2)
        )

        # Measure training runtime BEFORE this round's diagnostic evaluation.
        # Previous diagnostic-evaluation time is subtracted so progress printing
        # does not inflate the experiment's runtime metric.
        cumulative_runtime_s = float(
            max(0.0, (time.perf_counter() - start) - monitor_eval_s)
        )

        if train_monitor is not None:
            Xmon, ymon = train_monitor
            t_mon = time.perf_counter()
            monitor_metrics = evaluate_model(global_model, Xmon, ymon, n_classes)
            monitor_eval_s += float(time.perf_counter() - t_mon)
        else:
            monitor_metrics = {
                "accuracy": float("nan"),
                "precision_macro": float("nan"),
                "recall_macro": float("nan"),
                "f1_macro": float("nan"),
                "roc_auc_ovr_macro": float("nan"),
            }

        live_peak = float(monitor.peak_mb)
        round_row = {
            "round": r,
            "selected_clients": len(selected),
            "selected_ids": ";".join(selected),
            "optimizer_steps": int(round_steps),
            "cumulative_optimizer_steps": int(total_steps),
            "train_monitor_accuracy": float(monitor_metrics["accuracy"]),
            "train_monitor_precision_macro": float(monitor_metrics["precision_macro"]),
            "train_monitor_recall_macro": float(monitor_metrics["recall_macro"]),
            "train_monitor_f1_macro": float(monitor_metrics["f1_macro"]),
            "train_monitor_auc_ovr_macro": float(monitor_metrics["roc_auc_ovr_macro"]),
            "cumulative_runtime_s": float(cumulative_runtime_s),
            "cumulative_energy_wh_est": float(
                ROUND_PROGRESS_POWER_W * cumulative_runtime_s / 3600.0
            ),
            "cumulative_communication_mb": float(cumulative_comm_mb),
            "ram_peak_mb_live": float(live_peak),
            "ram_delta_mb_live": float(max(0.0, live_peak - monitor.start_mb)),
        }
        round_rows.append(round_row)

        print_round_progress(
            progress_context=progress_context,
            round_idx=r,
            round_total=len(selected_per_round),
            selected_ids=list(selected),
            round_steps=int(round_steps),
            cumulative_steps=int(total_steps),
            monitor_metrics=monitor_metrics,
            cumulative_runtime_s=cumulative_runtime_s,
            cumulative_communication_mb=cumulative_comm_mb,
            ram_start_mb=float(monitor.start_mb),
            ram_peak_mb=live_peak,
        )

    runtime = float(max(0.0, (time.perf_counter() - start) - monitor_eval_s))
    metrics = evaluate_model(global_model, X_test, y_test, n_classes)
    ram = monitor.stop()

    communication_b = int(cumulative_comm_raw_b * 1.12)

    return {
        "model": global_model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(total_steps),
        "communication_mb": float(communication_b / (1024 ** 2)),
        "round_audit": round_rows,
        "participants_mean_per_round": float(
            total_selected / max(1, len(round_rows))
        ),
        **ram,
    }



def select_dq_only_clients(
    dq_scores: Dict[str, Dict[str, float]],
    k: int,
) -> Tuple[List[str], pd.DataFrame]:
    """Select top-K clients by the machine-measured TADP Data Quality dimension only."""
    rows = []
    for cid, scores in dq_scores.items():
        vals = [float(scores[f]) for f in FACTOR_NAMES["dim2"]]
        rows.append({
            "client": str(cid),
            "dq_only_score": float(np.mean(vals)),
            "dq_factor_count": int(len(vals)),
        })
    audit = pd.DataFrame(rows).sort_values(
        ["dq_only_score", "client"], ascending=[False, True]
    ).reset_index(drop=True)
    audit["dq_only_rank"] = np.arange(1, len(audit) + 1)
    selected = audit.head(int(k))["client"].astype(str).tolist()
    audit["dq_only_selected"] = audit["client"].isin(selected)
    return selected, audit


def local_training_loss(model: keras.Model, X: np.ndarray, y: np.ndarray, batch_size: int = 512) -> float:
    """Mean sparse cross-entropy on client TRAIN data only; used by Power-of-Choice."""
    p = model.predict(X, batch_size=batch_size, verbose=0)
    y = np.asarray(y, dtype=np.int32)
    idx = np.arange(len(y))
    probs = np.clip(p[idx, y], 1e-12, 1.0)
    return float(-np.mean(np.log(probs)))


def federated_train_power_of_choice(
    build_model_fn,
    initial_weights: List[np.ndarray],
    candidate_clients: List[str],
    select_k: int,
    target_steps_per_round: int,
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    local_epochs: int,
    class_weights: Dict[int, float],
    run_seed: int,
    candidate_multiplier: int = 2,
    train_monitor: Optional[Tuple[np.ndarray, np.ndarray]] = None,
    progress_context: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Power-of-Choice baseline: each round samples d candidates and selects the K
    clients with largest current local TRAIN loss. K, rounds, initialisation, and
    total optimizer steps per round are matched to TADP-VR. No TEST information is used.
    """
    monitor = RAMMonitor().start()
    start = time.perf_counter()
    global_model = build_model_fn()
    global_model.set_weights([np.array(w, copy=True) for w in initial_weights])
    rng = np.random.default_rng(int(run_seed) + 880000)
    round_rows = []
    total_steps = 0
    total_selected = 0
    communication_b = 0
    monitor_eval_s = 0.0

    all_candidates = list(candidate_clients)
    if int(select_k) < 1 or int(select_k) > len(all_candidates):
        raise RuntimeError("Invalid Power-of-Choice K.")

    for r in range(1, NUM_ROUNDS_FL + 1):
        d = min(len(all_candidates), max(int(select_k), int(candidate_multiplier) * int(select_k)))
        if d == len(all_candidates):
            candidate_pool = list(all_candidates)
        else:
            # Canonical pow-d samples candidate clients without replacement
            # according to p_k, the client's fraction of total TRAIN data.
            sizes = np.asarray(
                [len(client_arrays[c][1]) for c in all_candidates], dtype=float
            )
            probs = sizes / sizes.sum()
            candidate_pool = rng.choice(
                all_candidates, size=d, replace=False, p=probs
            ).tolist()

        losses = []
        for cid in candidate_pool:
            Xc, yc = client_arrays[cid]
            losses.append((str(cid), local_training_loss(global_model, Xc, yc)))
        losses.sort(key=lambda x: (-x[1], x[0]))
        selected = [cid for cid, _ in losses[:int(select_k)]]
        step_map = allocate_exact_step_budget(
            selected, client_arrays, int(target_steps_per_round), batch_size, local_epochs
        )

        global_weights = [np.array(w, copy=True) for w in global_model.get_weights()]
        local_weights, local_sizes = [], []
        for j, cid in enumerate(selected):
            Xc, yc = client_arrays[cid]
            local_model = build_model_fn()
            local_model.set_weights(global_weights)
            train_exact_steps(
                local_model, Xc, yc,
                steps=int(step_map[cid]), batch_size=batch_size,
                seed=int(run_seed + 1000 * r + 17 * j),
                class_weights=class_weights,
            )
            local_weights.append([np.array(w, copy=True) for w in local_model.get_weights()])
            local_sizes.append(len(yc))
            del local_model
            tf.keras.backend.clear_session()

        agg = aggregate_weights(local_weights, local_sizes, equal_weight=False)
        global_model = build_model_fn()
        global_model.set_weights(agg)
        round_steps = int(sum(step_map.values()))
        total_steps += round_steps
        total_selected += len(selected)
        param_b = model_parameter_bytes(global_model)
        # Candidate clients receive the current model to evaluate local loss;
        # selected clients return one model update. Scalar loss uploads are negligible.
        communication_b += (len(candidate_pool) + len(selected)) * param_b
        cumulative_comm_mb = float(
            (communication_b * 1.12) / (1024 ** 2)
        )
        cumulative_runtime_s = float(
            max(0.0, (time.perf_counter() - start) - monitor_eval_s)
        )

        if train_monitor is not None:
            Xmon, ymon = train_monitor
            t_mon = time.perf_counter()
            monitor_metrics = evaluate_model(global_model, Xmon, ymon, n_classes)
            monitor_eval_s += float(time.perf_counter() - t_mon)
        else:
            monitor_metrics = {
                "accuracy": float("nan"),
                "precision_macro": float("nan"),
                "recall_macro": float("nan"),
                "f1_macro": float("nan"),
                "roc_auc_ovr_macro": float("nan"),
            }

        live_peak = float(monitor.peak_mb)
        round_rows.append({
            "round": r,
            "candidate_count": len(candidate_pool),
            "candidate_ids": ";".join(candidate_pool),
            "selected_clients": len(selected),
            "selected_ids": ";".join(selected),
            "optimizer_steps": round_steps,
            "cumulative_optimizer_steps": int(total_steps),
            "local_losses": json.dumps({cid: loss for cid, loss in losses}, sort_keys=True),
            "train_monitor_accuracy": float(monitor_metrics["accuracy"]),
            "train_monitor_precision_macro": float(monitor_metrics["precision_macro"]),
            "train_monitor_recall_macro": float(monitor_metrics["recall_macro"]),
            "train_monitor_f1_macro": float(monitor_metrics["f1_macro"]),
            "train_monitor_auc_ovr_macro": float(monitor_metrics["roc_auc_ovr_macro"]),
            "cumulative_runtime_s": float(cumulative_runtime_s),
            "cumulative_energy_wh_est": float(
                ROUND_PROGRESS_POWER_W * cumulative_runtime_s / 3600.0
            ),
            "cumulative_communication_mb": float(cumulative_comm_mb),
            "ram_peak_mb_live": float(live_peak),
            "ram_delta_mb_live": float(max(0.0, live_peak - monitor.start_mb)),
        })

        loss_preview = ", ".join(
            f"{cid}:{loss:.3f}" for cid, loss in losses[:min(5, len(losses))]
        )
        print_round_progress(
            progress_context=progress_context,
            round_idx=r,
            round_total=NUM_ROUNDS_FL,
            selected_ids=list(selected),
            round_steps=int(round_steps),
            cumulative_steps=int(total_steps),
            monitor_metrics=monitor_metrics,
            cumulative_runtime_s=cumulative_runtime_s,
            cumulative_communication_mb=cumulative_comm_mb,
            ram_start_mb=float(monitor.start_mb),
            ram_peak_mb=live_peak,
            extra=f"PoC candidates={len(candidate_pool)} | highest TRAIN losses: {loss_preview}",
        )

    runtime = float(max(0.0, (time.perf_counter() - start) - monitor_eval_s))
    metrics = evaluate_model(global_model, X_test, y_test, n_classes)
    ram = monitor.stop()
    communication_b = int(communication_b * 1.12)
    return {
        "model": global_model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(total_steps),
        "communication_mb": float(communication_b / (1024 ** 2)),
        "round_audit": round_rows,
        "participants_mean_per_round": float(total_selected / max(1, NUM_ROUNDS_FL)),
        **ram,
    }


def centralized_train(
    build_model_fn,
    initial_weights: List[np.ndarray],
    selected_clients: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    exact_steps: int,
    class_weights: Dict[int, float],
    seed: int,
) -> Dict[str, Any]:
    if not selected_clients:
        raise RuntimeError("Centralized scenario has no TRAIN clients.")

    monitor = RAMMonitor().start()
    start = time.perf_counter()

    # Pool ACCEPTED TRAIN partitions only. TEST is not present here.
    X = np.concatenate([client_arrays[c][0] for c in selected_clients], axis=0)
    y = np.concatenate([client_arrays[c][1] for c in selected_clients], axis=0)

    model = build_model_fn()
    model.set_weights([np.array(w, copy=True) for w in initial_weights])
    train_exact_steps(
        model, X, y,
        steps=int(exact_steps),
        batch_size=batch_size,
        seed=int(seed),
        class_weights=class_weights,
    )

    runtime = time.perf_counter() - start
    metrics = evaluate_model(model, X_test, y_test, n_classes)
    ram = monitor.stop()

    return {
        "model": model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(exact_steps),
        "communication_mb": 0.0,
        "participants_mean_per_round": float(len(selected_clients)),
        **ram,
    }


def result_row(
    run: int,
    seed: int,
    scenario: str,
    result: Dict[str, Any],
    initial_hash: str,
    power_w: float = 12.0,
) -> Dict[str, Any]:
    row = {
        "run": int(run),
        "seed": int(seed),
        "scenario": str(scenario),
        **result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "participants": float(result.get("participants_mean_per_round", 0.0)),
        "ram_start_mb": float(result.get("ram_start_mb", 0.0)),
        "ram_end_mb": float(result.get("ram_end_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
        "ram_delta_mb": float(result.get("ram_delta_mb", 0.0)),
        "ram_mb": float(result.get("ram_mb", result.get("ram_peak_mb", 0.0))),
        "initial_weights_sha256": initial_hash,
    }
    row["energy_wh"] = float(power_w * row["runtime_s"] / 3600.0)
    row["energy_kwh"] = float(row["energy_wh"] / 1000.0)
    row["co2_kg"] = float(row["energy_kwh"] * 0.430)
    row["energy_cost_usd"] = float(row["energy_kwh"] * 0.20)
    row["communication_cost_usd"] = float(row["communication_mb"] * 0.005)
    row["total_estimated_cost_usd"] = float(
        row["energy_cost_usd"] + row["communication_cost_usd"]
    )
    return row


# ======================================================================================
# LEAKAGE AUDIT
# ======================================================================================

def write_leakage_audit(
    out_dir: Path,
    train_ids,
    test_ids,
    client_train_ids: Dict[str, np.ndarray],
    extra: Optional[Dict[str, Any]] = None,
):
    train_set = set(map(str, train_ids))
    test_set = set(map(str, test_ids))
    overlap = train_set & test_set

    client_union = set()
    duplicates_across_clients = 0
    for cid, ids in client_train_ids.items():
        s = set(map(str, ids))
        duplicates_across_clients += len(client_union & s)
        client_union |= s

    test_in_clients = len(test_set & client_union)
    missing_train = len(train_set - client_union)
    extra_client_rows = len(client_union - train_set)

    row = {
        "train_test_overlap": len(overlap),
        "test_rows_in_any_client": test_in_clients,
        "train_rows_missing_from_clients": missing_train,
        "client_rows_not_in_global_train": extra_client_rows,
        "duplicate_train_rows_across_clients": duplicates_across_clients,
        "pass": (
            len(overlap) == 0
            and test_in_clients == 0
            and missing_train == 0
            and extra_client_rows == 0
            and duplicates_across_clients == 0
        ),
    }
    if extra:
        row.update(extra)

    pd.DataFrame([row]).to_csv(
        Path(out_dir) / "leakage_audit.csv", index=False
    )

    if not bool(row["pass"]):
        raise RuntimeError(f"FAIL-CLOSED leakage audit failed: {row}")
    return row


# ======================================================================================
# DIABETES 130-US — LEAKAGE-SAFE DATA PREPARATION
# ======================================================================================

TARGET = "readmitted"
ID_COLUMNS = ["encounter_id", "patient_nbr"]


def locate_diabetes_csv() -> str:
    candidates = [
        os.environ.get("DIABETES_CSV", ""),
        "/content/diabetes_130US.csv",
        "./diabetes_130US.csv",
        "/content/drive/MyDrive/diabetes_130US.csv",
    ]
    for p in candidates:
        if p and os.path.exists(p):
            return p

    try:
        from google.colab import files as colab_files
        print("Please upload the Diabetes 130-US CSV file.")
        uploaded = colab_files.upload()
        csvs = [name for name in uploaded if str(name).lower().endswith(".csv")]
        if not csvs:
            raise RuntimeError("No CSV file was uploaded.")
        return str(csvs[0])
    except ImportError:
        pass

    raise FileNotFoundError(
        "Diabetes CSV not found. Set DIABETES_CSV or upload the CSV in Colab."
    )


def load_diabetes(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df = df.replace("?", np.nan)

    if TARGET not in df.columns:
        raise RuntimeError(f"Missing target column: {TARGET}")

    valid_target = {"NO": 0, ">30": 1, "<30": 2}
    df = df[df[TARGET].isin(valid_target)].copy()
    df["_target"] = df[TARGET].map(valid_target).astype(np.int32)
    df["_row_id"] = np.arange(len(df), dtype=np.int64)

    return df


def global_patient_grouped_split(
    df: pd.DataFrame, seed: int, test_fraction: float = 0.20
):
    """
    Patient-grouped and stratified whenever patient_nbr is available.
    The split happens before any client partitioning or data-dependent preprocessing.
    """
    if "patient_nbr" in df.columns:
        splitter = StratifiedGroupKFold(
            n_splits=5, shuffle=True, random_state=int(seed)
        )
        train_idx, test_idx = next(
            splitter.split(
                np.zeros(len(df)),
                y=df["_target"].to_numpy(),
                groups=df["patient_nbr"].astype(str).to_numpy(),
            )
        )
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        patient_overlap = len(
            set(train_df["patient_nbr"].astype(str))
            & set(test_df["patient_nbr"].astype(str))
        )
        if patient_overlap != 0:
            raise RuntimeError("Patient leakage detected across TRAIN/TEST.")
    else:
        train_df, test_df = train_test_split(
            df, test_size=float(test_fraction),
            stratify=df["_target"], random_state=int(seed)
        )
        patient_overlap = 0

    return train_df.reset_index(drop=True), test_df.reset_index(drop=True), patient_overlap


def dirichlet_partition_dataframe(
    train_df: pd.DataFrame,
    n_clients: int,
    alpha: float,
    seed: int,
    min_client_records: int = 100,
) -> Dict[str, pd.DataFrame]:
    y = train_df["_target"].to_numpy()
    client_ids = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")[:n_clients]

    for attempt in range(100):
        rng = np.random.default_rng(int(seed + attempt))
        buckets = [[] for _ in range(n_clients)]

        for cls in sorted(np.unique(y)):
            idx = np.where(y == cls)[0]
            rng.shuffle(idx)
            props = rng.dirichlet(np.full(n_clients, float(alpha)))
            cuts = (np.cumsum(props)[:-1] * len(idx)).astype(int)
            splits = np.split(idx, cuts)
            for k, s in enumerate(splits):
                buckets[k].extend(s.tolist())

        sizes = [len(b) for b in buckets]
        if min(sizes) >= int(min_client_records):
            out = {}
            for cid, idxs in zip(client_ids, buckets):
                out[cid] = train_df.iloc[np.array(idxs, dtype=int)].copy()
            return out

    raise RuntimeError("Could not obtain a valid Dirichlet client partition.")


@dataclass
class TabularPreprocessor:
    feature_cols: List[str]
    numeric_cols: List[str]
    categorical_cols: List[str]
    mean: Dict[str, float]
    std: Dict[str, float]
    categories: Dict[str, List[str]]
    encoder: Any

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        pieces = []

        if self.numeric_cols:
            x_num = []
            for c in self.numeric_cols:
                s = pd.to_numeric(df[c], errors="coerce").astype(float)
                a = s.fillna(self.mean[c]).to_numpy(dtype=np.float32)
                a = (a - self.mean[c]) / self.std[c]
                x_num.append(a[:, None])
            pieces.append(np.concatenate(x_num, axis=1).astype(np.float32))

        if self.categorical_cols:
            cat = pd.DataFrame({
                c: df[c].astype("string").fillna("__MISSING__").astype(str)
                for c in self.categorical_cols
            })
            x_cat = self.encoder.transform(cat)
            pieces.append(np.asarray(x_cat, dtype=np.float32))

        if not pieces:
            raise RuntimeError("No predictor columns remained.")
        return np.concatenate(pieces, axis=1).astype(np.float32)


def fit_federated_train_only_preprocessor(
    client_frames: Dict[str, pd.DataFrame]
) -> TabularPreprocessor:
    """
    No raw TRAIN pooling is used to ESTIMATE numeric parameters.
    Numeric mean/std comes from aggregated local count/sum/sum-of-squares.
    Categorical vocabulary comes from union of local TRAIN category sets.
    """
    any_df = next(iter(client_frames.values()))
    feature_cols = [
        c for c in any_df.columns
        if c not in {TARGET, "_target", "_row_id", *ID_COLUMNS}
    ]

    # Infer expected type from TRAIN only.
    numeric_cols = []
    categorical_cols = []
    for c in feature_cols:
        total_nonmissing = 0
        numeric_valid = 0
        for df in client_frames.values():
            raw = df[c]
            nm = raw.notna()
            total_nonmissing += int(nm.sum())
            if nm.any():
                numeric_valid += int(
                    pd.to_numeric(raw[nm], errors="coerce").notna().sum()
                )
        ratio = numeric_valid / max(1, total_nonmissing)
        if ratio >= 0.95:
            numeric_cols.append(c)
        else:
            categorical_cols.append(c)

    mean = {}
    std = {}
    for c in numeric_cols:
        count = 0
        sum_ = 0.0
        sumsq = 0.0
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            count += len(a)
            sum_ += float(a.sum())
            sumsq += float(np.square(a).sum())
        mu = sum_ / max(1, count)
        var = max(1e-12, sumsq / max(1, count) - mu * mu)
        mean[c] = float(mu)
        std[c] = float(math.sqrt(var))

    categories = {}
    for c in categorical_cols:
        values = set()
        for df in client_frames.values():
            s = df[c].astype("string").fillna("__MISSING__").astype(str)
            values.update(s.unique().tolist())
        categories[c] = sorted(values)

    encoder = OneHotEncoder(
        categories=[categories[c] for c in categorical_cols],
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float32,
    )

    # Fit only metadata-shaped dummy rows; the vocabulary is already frozen from TRAIN.
    if categorical_cols:
        max_len = max(len(categories[c]) for c in categorical_cols)
        dummy = {}
        for c in categorical_cols:
            vals = categories[c]
            dummy[c] = [vals[i % len(vals)] for i in range(max_len)]
        encoder.fit(pd.DataFrame(dummy))

    return TabularPreprocessor(
        feature_cols=feature_cols,
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        mean=mean,
        std=std,
        categories=categories,
        encoder=encoder,
    )


def build_tabular_reference(
    client_frames: Dict[str, pd.DataFrame],
    preprocessor: TabularPreprocessor,
) -> Dict[str, Any]:
    """
    TRAIN-only DQ reference built from client-local summaries only.

    Numerical reference histograms are constructed by combining local
    min/max summaries and then summing client-local histogram counts.
    Categorical reference support is the union of local TRAIN category sets.
    Raw client records are not concatenated to construct the reference.
    """
    ref = {"num_hist": {}, "cat_values": {}}

    chosen_num = preprocessor.numeric_cols[:12]
    for c in chosen_num:
        local_min, local_max = [], []
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                local_min.append(float(np.min(a)))
                local_max.append(float(np.max(a)))
        if not local_min:
            continue
        gmin, gmax = float(min(local_min)), float(max(local_max))
        if gmax <= gmin:
            edges = np.array([gmin - 1e-6, gmax + 1e-6], dtype=float)
        else:
            edges = np.linspace(gmin, gmax, 11, dtype=float)
        global_hist = np.zeros(len(edges)-1, dtype=np.float64)
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                h, _ = np.histogram(a, bins=edges)
                global_hist += h.astype(np.float64)
        ref["num_hist"][c] = {
            "edges": edges.tolist(),
            "hist": global_hist.tolist(),
            "construction": "aggregated_client_local_histograms_only",
        }

    for c in preprocessor.categorical_cols:
        values = set()
        for df in client_frames.values():
            values.update(
                df[c].astype("string").fillna("__MISSING__")
                .astype(str).unique().tolist()
            )
        ref["cat_values"][c] = sorted(values)
    return ref

def tabular_dq_scores(
    df: pd.DataFrame,
    preprocessor: TabularPreprocessor,
    reference: Dict[str, Any],
) -> Tuple[Dict[str, float], Dict[str, float]]:
    """
    Eight machine-measured TRAIN-only DQ factors for the healthcare experiment.
    """
    feature_df = df[preprocessor.feature_cols]

    # 1) Completeness.
    missing_fraction = float(feature_df.isna().mean().mean())
    completeness = score_lower_is_better(
        missing_fraction,
        [0.01, 0.05, 0.10, 0.20, 0.50],
    )

    # 2) Duplication rate.
    duplicate_fraction = float(feature_df.duplicated().mean())
    duplication = score_lower_is_better(
        duplicate_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 3) Value validity / error rate.
    bad = 0
    observed = 0

    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())

        if nm.any():
            conv = pd.to_numeric(
                raw[nm], errors="coerce"
            ).to_numpy(dtype=float)
            bad += int(np.sum(~np.isfinite(conv)))

    for c in preprocessor.categorical_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())

        if nm.any():
            bad += int(
                np.sum(raw[nm].astype(str).str.strip().eq(""))
            )

    error_fraction = float(bad / max(1, observed))
    value_validity_error_rate = score_lower_is_better(
        error_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.15],
    )

    # 4) Type consistency.
    type_bad = 0
    type_obs = 0

    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        type_obs += int(nm.sum())

        if nm.any():
            type_bad += int(
                pd.to_numeric(
                    raw[nm], errors="coerce"
                ).isna().sum()
            )

    type_inconsistency = float(type_bad / max(1, type_obs))
    type_consistency = score_lower_is_better(
        type_inconsistency,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 5) Label integrity.
    invalid_label_fraction = float(
        (~df["_target"].isin([0, 1, 2])).mean()
    )
    label_integrity = score_lower_is_better(
        invalid_label_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    # 6) Feature-distribution consistency — TRAIN-only reference.
    jsds = []

    for c, spec in reference["num_hist"].items():
        a = pd.to_numeric(
            df[c], errors="coerce"
        ).to_numpy(dtype=float)
        a = a[np.isfinite(a)]

        if len(a):
            hist, _ = np.histogram(
                a,
                bins=np.array(spec["edges"], dtype=float),
            )
            jsds.append(
                js_divergence(
                    hist,
                    np.array(spec["hist"], dtype=float),
                )
            )

    max_jsd = float(max(jsds)) if jsds else 0.0
    distribution_consistency = score_lower_is_better(
        max_jsd,
        [0.01, 0.025, 0.05, 0.10, 0.20],
    )

    # 7) Feature/category coverage.
    coverage_vals = []

    for c, ref_vals in reference["cat_values"].items():
        ref_set = set(ref_vals)

        if ref_set:
            client_set = set(
                df[c]
                .astype("string")
                .fillna("__MISSING__")
                .astype(str)
                .unique()
            )
            coverage_vals.append(
                len(client_set & ref_set) / len(ref_set)
            )

    mean_coverage = (
        float(np.mean(coverage_vals))
        if coverage_vals else 1.0
    )
    feature_coverage = score_higher_is_better(
        mean_coverage,
        [0.90, 0.825, 0.75, 0.65, 0.50],
    )

    # 8) Structural / constraint integrity.
    # Uses only TRAIN records. It checks:
    #   - key presence / encounter uniqueness;
    #   - non-negative count-like clinical fields;
    #   - finite numeric values where a numeric value is expected.
    n = len(df)
    record_violation = np.zeros(n, dtype=bool)

    if "encounter_id" not in df.columns:
        record_violation[:] = True
    else:
        encounter = df["encounter_id"]
        record_violation |= encounter.isna().to_numpy()
        record_violation |= encounter.duplicated(keep=False).to_numpy()

    nonnegative_fields = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]

    for c in nonnegative_fields:
        if c in df.columns:
            a = pd.to_numeric(
                df[c], errors="coerce"
            ).to_numpy(dtype=float)
            bad_c = (~np.isfinite(a)) | (a < 0)
            record_violation |= bad_c

    structural_violation_fraction = float(
        np.mean(record_violation)
    ) if n else 1.0

    structural_integrity = score_lower_is_better(
        structural_violation_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    scores = {
        "completeness": completeness,
        "duplication_rate": duplication,
        "value_validity_error_rate": value_validity_error_rate,
        "type_consistency": type_consistency,
        "label_integrity": label_integrity,
        "feature_distribution_consistency": distribution_consistency,
        "feature_category_coverage": feature_coverage,
        "structural_constraint_integrity": structural_integrity,
    }

    raw = {
        "missing_fraction": missing_fraction,
        "duplicate_fraction": duplicate_fraction,
        "error_fraction": error_fraction,
        "type_inconsistency_fraction": type_inconsistency,
        "invalid_label_fraction": invalid_label_fraction,
        "max_jsd": max_jsd,
        "mean_category_coverage": mean_coverage,
        "structural_violation_fraction": structural_violation_fraction,
    }

    return scores, raw

def prepare_diabetes_no_leakage(
    csv_path: str,
    split_seed: int,
    partition_seed: int,
    n_clients: int = 10,
    alpha: float = 1.0,
):
    raw = load_diabetes(csv_path)
    train_df, test_df, patient_overlap = global_patient_grouped_split(
        raw, split_seed
    )

    clients = dirichlet_partition_dataframe(
        train_df, n_clients=n_clients, alpha=alpha, seed=partition_seed
    )
    client_ids = list(clients.keys())

    # Hard row-level leakage audit.
    client_train_ids = {
        cid: df["_row_id"].astype(str).to_numpy()
        for cid, df in clients.items()
    }

    pre = fit_federated_train_only_preprocessor(clients)
    reference = build_tabular_reference(clients, pre)

    dq_scores = {}
    dq_raw_rows = []
    client_arrays = {}

    for cid, df in clients.items():
        scores, raw_metrics = tabular_dq_scores(df, pre, reference)
        dq_scores[cid] = scores

        row = {"client": cid, **scores, **raw_metrics}
        dq_raw_rows.append(row)

        X = pre.transform(df)
        y = df["_target"].to_numpy(dtype=np.int32)
        client_arrays[cid] = (X, y)

    X_test = pre.transform(test_df)
    y_test = test_df["_target"].to_numpy(dtype=np.int32)

    y_train_all = np.concatenate(
        [client_arrays[c][1] for c in client_ids]
    )
    cw = class_weight_dict(y_train_all)

    meta = {
        "raw_rows": len(raw),
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "patient_overlap": int(patient_overlap),
        "input_dim": int(X_test.shape[1]),
        "n_clients": int(n_clients),
        "numeric_features": len(pre.numeric_cols),
        "categorical_features": len(pre.categorical_cols),
    }

    return {
        "raw": raw,
        "train_df": train_df,
        "test_df": test_df,
        "clients_raw": clients,
        "client_arrays": client_arrays,
        "client_ids": client_ids,
        "X_test": X_test,
        "y_test": y_test,
        "dq_scores": dq_scores,
        "dq_audit": pd.DataFrame(dq_raw_rows),
        "class_weights": cw,
        "meta": meta,
        "client_train_ids": client_train_ids,
        "global_train_ids": train_df["_row_id"].astype(str).to_numpy(),
        "global_test_ids": test_df["_row_id"].astype(str).to_numpy(),
        "preprocessor": pre,
        "dq_reference": reference,
    }


def build_diabetes_model(input_dim: int, lr: float = 1e-3) -> keras.Model:
    inp = keras.Input(shape=(int(input_dim),), dtype=tf.float32)
    x = layers.Dense(128, activation="relu")(inp)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(3, activation="softmax", dtype=tf.float32)(x)
    model = keras.Model(inp, out)
    model.optimizer = keras.optimizers.Adam(learning_rate=float(lr))
    return model



# ======================================================================================
# FULL EXPERIMENT-A REPORTING, CHECKPOINTING, LEDGER, AND STATISTICS
# ======================================================================================
from datetime import datetime, timezone
import re

# =============================================================================
# MAIN REPEATED TRAINING SET
#
# These eight configurations are genuinely distinct and are repeated over all
# five training seeds for mean ± SD / CI reporting.
#
# NOT repeated here:
#   - Great Expectations Centralized
#   - TADP-AA Centralized
#   - Great Expectations Federated
#   - TADP-AA Federated
# Those all-client equivalences are verified independently by the companion
# one-seed equivalence-audit script.
#
# TADP-SDA is retained as a boundary/stress condition but is run once only,
# outside the five-seed headline statistical comparison.
# =============================================================================
MANUSCRIPT_SCENARIOS = [
    "Naïve Centralized",
    "TADP-VR Centralized",
    "Vanilla FedAvg",
    "FedProx",
    "Random-K",
    "DQ-only Federated",
    "Power-of-Choice",
    "TADP-VR Federated",
]
assert len(MANUSCRIPT_SCENARIOS) == 8

BOUNDARY_SCENARIOS = [
    "TADP-SDA Centralized",
    "TADP-SDA Federated",
]
BOUNDARY_SEED = 42



def print_banner(title: str, width: int = 108):
    print("\n" + "=" * width)
    print(title)
    print("=" * width)


def client_partition_table(clients_raw):
    rows = []
    for cid, df in clients_raw.items():
        counts = df["_target"].value_counts().to_dict()
        rows.append({
            "client": cid,
            "records": int(len(df)),
            "class_0_NO": int(counts.get(0, 0)),
            "class_1_GT30": int(counts.get(1, 0)),
            "class_2_LT30": int(counts.get(2, 0)),
        })
    return pd.DataFrame(rows)


def evidence_assignment_summary(
    evidence_df: pd.DataFrame
) -> pd.DataFrame:
    """
    Summarize the frozen controlled documentary-evidence assignment.

    v16.7 reports the pre-specified governance archetype for each evidence
    bundle. These archetypes are branch-coverage scenarios, not observed
    real-world prevalence classes.
    """
    required = {
        "client",
        "bundle_id",
        "evidence_profile",
        "scenario_role",
        "profile_variant",
        "factor",
        "rubric_score_0_5",
        "meets_factor_adequacy",
        "evidence_seed",
    }

    missing = sorted(
        required - set(evidence_df.columns)
    )

    if missing:
        raise RuntimeError(
            "Controlled documentary-evidence table is missing required "
            f"column(s): {missing}. Available columns: "
            f"{sorted(evidence_df.columns.tolist())}"
        )

    work = evidence_df.copy()
    work["meets_factor_adequacy"] = (
        work["meets_factor_adequacy"]
        .astype(bool)
    )

    summary = (
        work
        .groupby(
            ["client", "bundle_id"],
            as_index=False,
        )
        .agg(
            evidence_profile=(
                "evidence_profile",
                "first",
            ),
            scenario_role=(
                "scenario_role",
                "first",
            ),
            profile_variant=(
                "profile_variant",
                "first",
            ),
            documentary_factor_count=(
                "factor",
                "count",
            ),
            documentary_adequate_factor_count=(
                "meets_factor_adequacy",
                "sum",
            ),
            documentary_mean_score=(
                "rubric_score_0_5",
                "mean",
            ),
            documentary_min_score=(
                "rubric_score_0_5",
                "min",
            ),
            documentary_max_score=(
                "rubric_score_0_5",
                "max",
            ),
            evidence_seed=(
                "evidence_seed",
                "first",
            ),
        )
        .sort_values("client")
        .reset_index(drop=True)
    )

    summary["documentary_adequacy_fraction"] = (
        summary["documentary_adequate_factor_count"]
        / summary["documentary_factor_count"].clip(lower=1)
    )

    expected_documentary_factors = sum(
        len(FACTOR_NAMES[d])
        for d in DOCUMENTARY_DIMS
    )

    count_ok = summary[
        "documentary_factor_count"
    ].eq(
        expected_documentary_factors
    )

    if not count_ok.all():
        bad = summary.loc[
            ~count_ok,
            [
                "client",
                "bundle_id",
                "documentary_factor_count",
            ],
        ]

        raise RuntimeError(
            "Unexpected controlled-evidence factor count. "
            f"Expected {expected_documentary_factors} documentary factors "
            "per client. Offending rows:\n"
            + bad.to_string(index=False)
        )

    return summary

def print_governance_details(
    gov: pd.DataFrame,
    ge: pd.DataFrame,
    dq: pd.DataFrame,
    vr_clients: List[str],
    sda_clients: List[str],
    ge_clients: List[str],
    domain: str,
):
    domain = str(domain).lower()

    print_banner(
        "DOMAIN ADEQUACY POLICY — FULL WAC + CRITICAL WAC"
    )

    policy_rows = []
    for dim in FACTOR_NAMES:
        policy_rows.append({
            "dimension":
                dim,
            "dimension_name":
                DIMENSION_NAMES[dim],
            "n_factors":
                len(FACTOR_NAMES[dim]),
            "minimum_adequacy_ranks":
                ",".join(
                    str(
                        DOMAIN_FACTOR_MINIMA[
                            domain
                        ][dim][f]
                    )
                    for f in FACTOR_NAMES[
                        dim
                    ]
                ),
            "dimension_policy_wac":
                DOMAIN_DIMENSION_WAC[
                    domain
                ][dim],
        })

    print(
        pd.DataFrame(
            policy_rows
        ).to_string(
            index=False
        )
    )

    global_critical_wac = derive_global_critical_wac(
        domain
    )

    print(
        f"\nGLOBAL {domain.upper()} WAC "
        f"(descriptive full-policy summary) = "
        f"{DOMAIN_GLOBAL_WAC[domain]:.6f}"
    )
    print(
        f"GLOBAL {domain.upper()} CRITICAL WAC "
        f"(automated Review threshold) = "
        f"{global_critical_wac:.6f}"
    )

    print_banner(
        "CRITICAL FACTORS — INDIVIDUAL ADEQUACY REQUIREMENTS"
    )

    critical_rows = []

    for dim, factor_list in (
        CRITICAL_FACTORS_BY_DOMAIN[
            domain
        ].items()
    ):
        for factor in factor_list:
            minimum = float(
                DOMAIN_FACTOR_MINIMA[
                    domain
                ][dim][factor]
            )
            critical_rows.append({
                "dimension":
                    dim,
                "factor":
                    factor,
                "individual_adequacy_min_0_5":
                    minimum,
                "normalized_reference":
                    minimum / MAX_FACTOR_SCORE,
                "direct_auto_accept_rule":
                    f"score >= {minimum:.1f}",
            })

    print(
        pd.DataFrame(
            critical_rows
        ).to_string(
            index=False
        )
    )

    print(
        f"\nMinimum dimension floor: EVERY averaged dimension "
        f"must be >= {DIMENSION_MIN_FLOOR:.1f}/5."
    )

    print(
        "Human reviewer role: verify uploaded questionnaire evidence only. "
        "The server makes the admission decision automatically."
    )

    print_banner(
        "TRAIN-ONLY DATA-QUALITY FACTORS — 8 FACTORS"
    )

    dq_cols = [
        "client",
        "completeness",
        "duplication_rate",
        "value_validity_error_rate",
        "type_consistency",
        "label_integrity",
        "feature_distribution_consistency",
        "feature_category_coverage",
        "structural_constraint_integrity",
    ]

    print(
        dq[dq_cols].to_string(
            index=False
        )
    )

    print_banner(
        "FROZEN TADP GOVERNANCE — HPS + CRITICAL WAC"
    )

    display_cols = [
        "client",
        "hps",
        "critical_wac_i",
        "global_critical_wac",
        "critical_wac_margin",
        "all_critical_meet_adequacy",
        "critical_below_adequacy",
        "dimension_floor_failures",
        "dim1_score_0_5",
        "dim2_score_0_5",
        "dim3_score_0_5",
        "dim4_score_0_5",
        "dim5_score_0_5",
        "dim6_score_0_5",
        "decision_path",
        "initial_action",
        "final_action",
        "status",
        "reason",
    ]

    print(
        gov[
            display_cols
        ].to_string(
            index=False
        )
    )

    print(
        "\nTADP v16.7 final decision order:"
    )
    print(
        f"  1) ANY averaged dimension < "
        f"{DIMENSION_MIN_FLOOR:.1f}/5 -> AUTO-REJECT"
    )
    print(
        f"  2) HPS < {GOOD_CUT:.1f} -> AUTO-REJECT"
    )
    print(
        f"  3) HPS >= {HIGH_CUT:.1f}:"
    )
    print(
        "       all critical factors meet their own adequacy minima "
        "-> DIRECT AUTO-ACCEPT"
    )
    print(
        "       otherwise -> AUTOMATED REVIEW fallback"
    )
    print(
        f"  4) {GOOD_CUT:.1f} <= HPS < "
        f"{HIGH_CUT:.1f} -> AUTOMATED REVIEW"
    )
    print(
        "  5) AUTOMATED REVIEW:"
    )
    print(
        f"       Critical WAC_i >= Global Critical WAC "
        f"({global_critical_wac:.6f}) -> ACCEPT AFTER REVIEW"
    )
    print(
        "       otherwise -> AUTO-REJECT"
    )

    print(
        f"\nFrozen TADP-VR cohort: "
        f"{len(vr_clients)}/10 -> "
        f"{vr_clients}"
    )
    print(
        f"Frozen TADP-SDA cohort: "
        f"{len(sda_clients)}/10 -> "
        f"{sda_clients}"
    )

    print_banner("GX CORE NATIVE TRAIN-ONLY VALIDATOR BASELINE")
    gx_base_cols = [
        "client", "gx_native_suite_success", "ge_expectations_passed",
        "ge_expectations_total", "ge_pass_rate", "ge_native_validation_class",
        "ge_final_action",
    ]
    gx_category_cols = [
        c for c in ge.columns
        if c.startswith("ge_") and (c.endswith("_passed") or c.endswith("_total"))
        and c not in {"ge_expectations_passed", "ge_expectations_total"}
    ]
    print(ge[gx_base_cols + sorted(gx_category_cols)].to_string(index=False))
    print(f"\nGX operationally valid clients (zero critical failures): {len(ge_clients)}/{len(ge)} -> {ge_clients}")
    print(
        "GX is used only as a native rule-based data validator. PASS/FAIL is the "
        "suite-level GX result; no ranking, no forced-K selection, and no TADP signal is used."
    )





def build_hash_chained_governance_ledger(
    gov,
    ge,
    output_path,
):
    rows = []
    prev_hash = "GENESIS"
    seq = 0

    for _, r in (
        gov.sort_values(
            "client"
        ).iterrows()
    ):
        seq += 1

        payload = {
            "sequence":
                seq,
            "governance_system":
                "TADP",
            "client":
                str(r["client"]),
            "hps":
                float(r["hps"]),
            "global_domain_wac":
                float(r["global_domain_wac"]),
            "global_critical_wac":
                float(r["global_critical_wac"]),
            "critical_wac_i": (
                float(r["critical_wac_i"])
                if np.isfinite(r["critical_wac_i"])
                else None
            ),
            "critical_wac_margin": (
                float(r["critical_wac_margin"])
                if np.isfinite(r["critical_wac_margin"])
                else None
            ),
            "all_critical_meet_adequacy":
                bool(r["all_critical_meet_adequacy"]),
            "critical_below_adequacy":
                str(r["critical_below_adequacy"]),
            "dimension_floor_failures":
                str(r["dimension_floor_failures"]),
            "decision_path":
                str(r["decision_path"]),
            "initial_action":
                str(r["initial_action"]),
            "final_action":
                str(r["final_action"]),
            "status":
                str(r["status"]),
            "reason":
                str(r["reason"]),
            "previous_hash":
                prev_hash,
        }

        canonical = json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
        )

        entry_hash = hashlib.sha256(
            canonical.encode("utf-8")
        ).hexdigest()

        payload["entry_hash"] = entry_hash

        rows.append(payload)
        prev_hash = entry_hash

    for _, r in (
        ge.sort_values(
            "client"
        ).iterrows()
    ):
        seq += 1

        payload = {
            "sequence":
                seq,
            "governance_system":
                "Great Expectations GX Core native validator",
            "client":
                str(r["client"]),
            "hps":
                None,
            "global_domain_wac":
                None,
            "global_critical_wac":
                None,
            "critical_wac_i":
                None,
            "critical_wac_margin":
                None,
            "all_critical_meet_adequacy":
                None,
            "critical_below_adequacy":
                None,
            "dimension_floor_failures":
                None,
            "decision_path":
                "RULE_BASED_VALIDATION",
            "initial_action":
                "RULE_BASED_VALIDATION",
            "final_action":
                str(r["ge_final_action"]),
            "status":
                "GE_CONTROLLED_TRAIN_ONLY",
            "reason": (
                f"GX native suite {'PASSED' if bool(r['gx_native_suite_success']) else 'FAILED'}; "
                f"{int(r['ge_expectations_passed'])}/{int(r['ge_expectations_total'])} "
                "configured technical Expectations passed"
            ),
            "previous_hash":
                prev_hash,
        }

        canonical = json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
        )

        entry_hash = hashlib.sha256(
            canonical.encode("utf-8")
        ).hexdigest()

        payload["entry_hash"] = entry_hash

        rows.append(payload)
        prev_hash = entry_hash

    out = pd.DataFrame(rows)

    out.to_csv(
        output_path,
        index=False,
    )

    return out

def choose_experiment_root(experiment_name, use_drive=True):
    if use_drive:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            root = Path("/content/drive/MyDrive/TADP_CHECKPOINTS") / experiment_name
            root.mkdir(parents=True, exist_ok=True)
            print(f"Persistent checkpoint root: {root}")
            return root
        except Exception as exc:
            print(f"Google Drive checkpoint mount unavailable: {exc}")
    root = Path("/content") / experiment_name
    root.mkdir(parents=True, exist_ok=True)
    print(f"Local checkpoint root: {root}")
    return root


def atomic_write_json(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")
    tmp.replace(path)


def load_checkpoint_state(path):
    path = Path(path)
    if not path.exists():
        return {"completed": [], "last_completed": None, "updated_utc": None}
    return json.loads(path.read_text(encoding="utf-8"))


def mark_checkpoint_complete(state_path, key, extra=None):
    state = load_checkpoint_state(state_path)
    completed = list(state.get("completed", []))
    if key not in completed:
        completed.append(key)
    state["completed"] = completed
    state["last_completed"] = key
    state["updated_utc"] = datetime.now(timezone.utc).isoformat()
    if extra:
        state.update(extra)
    atomic_write_json(state, state_path)


def upsert_csv(row, path, key_cols):
    path = Path(path)
    new = pd.DataFrame([row])
    if path.exists():
        old = pd.read_csv(path)
        if not old.empty:
            mask = pd.Series(True, index=old.index)
            for c in key_cols:
                mask &= old[c].astype(str).eq(str(row[c]))
            old = old.loc[~mask].copy()
            new = pd.concat([old, new], ignore_index=True)
    new.to_csv(path, index=False)


def write_scenario_checkpoint(root, run_idx, scenario, result):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", scenario).strip("_")
    cdir = ensure_dir(root / "scenario_checkpoints" / f"run_{run_idx:02d}")
    payload = {
        "run": int(run_idx),
        "scenario": scenario,
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "metrics": result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
    }
    atomic_write_json(payload, cdir / f"{safe}.json")
    if "round_audit" in result:
        pd.DataFrame(result["round_audit"]).to_csv(
            cdir / f"{safe}_rounds.csv", index=False
        )


def ci95_mean(values):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan
    if len(x) == 1:
        return float(x[0]), float(x[0])
    mean = float(np.mean(x))
    sd = float(np.std(x, ddof=1))
    try:
        from scipy.stats import t
        crit = float(t.ppf(0.975, df=len(x)-1))
    except Exception:
        crit = 1.96
    half = crit * sd / math.sqrt(len(x))
    return mean-half, mean+half


def summarize_runs_with_ci(perf):
    metrics = [
        "accuracy", "precision_macro", "recall_macro", "f1_macro",
        "roc_auc_ovr_macro", "runtime_s", "energy_wh", "energy_kwh",
        "co2_kg", "communication_mb", "ram_peak_mb", "ram_delta_mb",
        "optimizer_steps", "participants", "energy_cost_usd",
        "communication_cost_usd", "total_estimated_cost_usd",
    ]
    rows = []
    for scenario, d in perf.groupby("scenario", sort=False):
        row = {"scenario": scenario, "n_runs": int(len(d))}
        for metric in metrics:
            if metric not in d.columns:
                continue
            vals = pd.to_numeric(d[metric], errors="coerce")
            vals = vals[np.isfinite(vals)]
            row[f"{metric}_mean"] = float(vals.mean()) if len(vals) else np.nan
            row[f"{metric}_sd"] = float(vals.std(ddof=1)) if len(vals)>1 else 0.0
            lo, hi = ci95_mean(vals)
            row[f"{metric}_ci95_low"] = lo
            row[f"{metric}_ci95_high"] = hi
        rows.append(row)
    return pd.DataFrame(rows)


def paired_vr_randomk_statistics(perf):
    vr = perf[perf["scenario"].eq("TADP-VR Federated")].copy()
    rk = perf[perf["scenario"].eq("Random-K")].copy()
    merged = vr.merge(rk, on=["run", "seed"], suffixes=("_vr", "_randomk"), validate="one_to_one")
    rows = []
    for metric in ["accuracy", "f1_macro", "roc_auc_ovr_macro"]:
        a = merged[f"{metric}_vr"].to_numpy(dtype=float)
        b = merged[f"{metric}_randomk"].to_numpy(dtype=float)
        diff = a-b
        mean = float(np.mean(diff))
        sd = float(np.std(diff, ddof=1)) if len(diff)>1 else 0.0
        lo, hi = ci95_mean(diff)
        t_stat=t_p=wil_stat=wil_p=np.nan
        try:
            from scipy.stats import ttest_rel, wilcoxon
            tr = ttest_rel(a,b,nan_policy="omit")
            t_stat, t_p = float(tr.statistic), float(tr.pvalue)
            if np.any(np.abs(diff)>0):
                wr = wilcoxon(a,b)
                wil_stat, wil_p = float(wr.statistic), float(wr.pvalue)
        except Exception:
            pass
        rows.append({
            "metric": metric,
            "n_pairs": len(diff),
            "mean_difference_vr_minus_randomk": mean,
            "sd_difference": sd,
            "ci95_low": lo,
            "ci95_high": hi,
            "cohens_dz": float(mean/sd) if sd>0 else np.nan,
            "paired_t_stat": t_stat,
            "paired_t_p": t_p,
            "wilcoxon_stat": wil_stat,
            "wilcoxon_p": wil_p,
            "vr_wins": int(np.sum(diff>0)),
            "ties": int(np.sum(np.isclose(diff,0))),
            "vr_losses": int(np.sum(diff<0)),
        })
    return pd.DataFrame(rows)


def scenario_method_table():
    return pd.DataFrame([
        ["Naïve Centralized","centralized","baseline","all clients","5-seed headline"],
        ["Great Expectations Centralized","centralized","GX native validator","GX clients with zero critical failures","equivalence audit only"],
        ["TADP-AA Centralized","centralized","TADP-AA","all clients","equivalence audit only"],
        ["TADP-VR Centralized","centralized","TADP-VR","frozen TADP-VR cohort","5-seed headline"],
        ["TADP-SDA Centralized","centralized","TADP-SDA","frozen best eligible client","one-seed boundary"],
        ["Vanilla FedAvg","federated","FedAvg","all clients","5-seed headline"],
        ["FedProx","federated","FedProx","all clients","5-seed headline"],
        ["Random-K","federated","matched random control","same K/rounds/steps as TADP-VR","5-seed headline"],
        ["Great Expectations Federated","federated","GX native validator","GX clients with zero critical failures","equivalence audit only"],
        ["DQ-only Federated","federated","DQ-only","top-K by machine-measured DQ; same K/rounds/steps as TADP-VR","5-seed headline"],
        ["Power-of-Choice","federated","Power-of-Choice","dynamic loss-based selection; same K/rounds/steps as TADP-VR","5-seed headline"],
        ["TADP-AA Federated","federated","TADP-AA","all clients","equivalence audit only"],
        ["TADP-VR Federated","federated","TADP-VR","frozen TADP-VR cohort","5-seed headline"],
        ["TADP-SDA Federated","federated","TADP-SDA","frozen best eligible client","one-seed boundary"],
    ], columns=["scenario","paradigm","method","participation","execution_role"])



def governance_only_monte_carlo(
    client_ids,
    dq_scores,
    base_seed,
    n_realizations=1000,
    domain="healthcare",
):
    domain = str(domain).lower()
    rows=[]
    for j in range(int(n_realizations)):
        seed=int(base_seed+j)
        evidence,_=generate_controlled_documentary_evidence(
            client_ids,
            seed,
            DOMAIN_FACTOR_MINIMA[domain],
            domain=domain,
        )
        gov=build_tadp_governance(
            client_ids,
            evidence,
            dq_scores,
            run=0,
            evidence_seed=seed,
            domain=domain,
        )
        accepted=accepted_tadp_vr(gov)
        rows.append({
            "realization":j+1,
            "evidence_seed":seed,
            "accepted_count":len(accepted),
            "accepted_clients":";".join(accepted),
            "mean_hps":float(gov["hps"].mean()),
            "mean_critical_wac_i":float(gov["critical_wac_i"].mean()),
            "dimension_floor_rejects":int(gov["status"].eq("AUTO_REJECTED_DIMENSION_FLOOR").sum()),
            "low_hps_rejects":int(gov["status"].eq("AUTO_REJECTED_LOW_HPS").sum()),
            "review_accepts":int(
                gov["status"].eq("ACCEPTED_AFTER_AUTOMATED_REVIEW").sum()
            ),
            "direct_auto_accepts":int(
                gov["status"].eq("DIRECT_AUTO_ACCEPTED").sum()
            ),
        })
    return pd.DataFrame(rows)


# ======================================================================================
# EXPERIMENT B3 — PREDICTIVE UTILITY UNDER CLIENT SCALING
# ======================================================================================
#
# PURPOSE
# -------
# Complement Experiment B1/B2 (governance-only scalability) with downstream
# predictive-utility scalability.
#
# Tested client counts:
#     K = {20, 50, 100}
#
# K=10 is intentionally not rerun here because it is already evaluated in the
# final Experiment-A protocol. For cross-K manuscript plots, use only the
# corresponding 4-round Experiment-A K=10 rows for seeds {42,142,242}.
#
# Three scenarios are intentionally retained:
#
#   1) Vanilla FedAvg
#      - all submitted clients participate
#      - full-participation predictive-utility reference
#
#   2) Random-K
#      - same number of participating clients as TADP-VR
#      - exact same total optimizer-step budget per round as TADP-VR
#      - frozen random cohort within each K, across all rounds and seeds
#
#   3) TADP-VR Federated
#      - clients admitted by the frozen TADP governance policy
#      - natural one-local-epoch step budget
#
# FAIRNESS / INTERPRETATION
# -------------------------
# TADP-VR vs Random-K is the matched-compute selection comparison:
#   same K_selected, same FL rounds, same total optimizer steps per round,
#   same model architecture, same initial weights within seed, same batch size,
#   same optimizer, same global TEST set.
#
# Vanilla FedAvg is NOT step-matched to TADP-VR because it is the intended
# full-participation utility reference. Its larger compute/communication budget
# is reported rather than hidden.
#
# The experiment tests whether TADP-VR retains useful predictive performance
# as the submitted contributor population grows, while using fewer clients
# than full FedAvg and while being compared fairly against a matched random
# subset.
#
# The held-out TEST set is used only after training for final evaluation.
# It is never used for governance, preprocessing, client selection, step-budget
# construction, or checkpoint decisions.
# ======================================================================================

import platform
import shutil
from datetime import datetime, timezone

EXPERIMENT_VERSION = "TADP-B3-v16.9-PREDICTIVE-SCALABILITY-K20-50-100-3SEED-4ROUND"

CLIENT_COUNTS = [20, 50, 100]
TRAINING_RUN_SEEDS = [42, 142, 242]

NUM_ROUNDS_FL = 4
LOCAL_EPOCHS = 1
BATCH_SIZE = 64
LEARNING_RATE = 1e-3

DIRICHLET_ALPHA = 1.0
MIN_CLIENT_RECORDS = 30

GLOBAL_SPLIT_SEED = 7001
PARTITION_BASE_SEED = 9101
EVIDENCE_BASE_SEED = 12042
RANDOMK_BASE_SEED = 22042

DOMAIN = "healthcare"
USE_GOOGLE_DRIVE_CHECKPOINTS = True

SCENARIOS = [
    "Vanilla FedAvg",
    "Random-K",
    "TADP-VR Federated",
]

EXPERIMENT_ROOT = choose_experiment_root(
    "TADP_EXPERIMENT_B3_" + EXPERIMENT_VERSION,
    use_drive=USE_GOOGLE_DRIVE_CHECKPOINTS,
)
CHECKPOINT_STATE = EXPERIMENT_ROOT / "checkpoint_state.json"
PERF_CHECKPOINT = EXPERIMENT_ROOT / "performance_metrics_checkpoint.csv"


# ======================================================================================
# SCALABLE CLIENT PARTITIONING / CONTROLLED EVIDENCE
# ======================================================================================

def scalable_client_ids(n_clients: int) -> List[str]:
    return [f"C{i:03d}" for i in range(1, int(n_clients) + 1)]


def dirichlet_partition_dataframe_scalable(
    train_df: pd.DataFrame,
    n_clients: int,
    alpha: float,
    seed: int,
    min_client_records: int = 30,
    max_attempts: int = 500,
) -> Dict[str, pd.DataFrame]:
    """
    Label-wise Dirichlet partition that supports K > 26 while preserving every
    global TRAIN row exactly once.
    """
    y = train_df["_target"].to_numpy()
    client_ids = scalable_client_ids(n_clients)

    for attempt in range(int(max_attempts)):
        rng = np.random.default_rng(int(seed + attempt))
        buckets = [[] for _ in range(int(n_clients))]

        for cls in sorted(np.unique(y)):
            idx = np.where(y == cls)[0]
            rng.shuffle(idx)
            props = rng.dirichlet(np.full(int(n_clients), float(alpha)))
            cuts = (np.cumsum(props)[:-1] * len(idx)).astype(int)
            splits = np.split(idx, cuts)
            for j, split in enumerate(splits):
                buckets[j].extend(split.tolist())

        sizes = [len(b) for b in buckets]
        if min(sizes) >= int(min_client_records):
            return {
                cid: train_df.iloc[np.asarray(idxs, dtype=int)].copy()
                for cid, idxs in zip(client_ids, buckets)
            }

    raise RuntimeError(
        f"Could not obtain valid K={n_clients} partition after {max_attempts} attempts "
        f"with min_client_records={min_client_records}."
    )


def partition_integrity_audit(
    train_df: pd.DataFrame,
    clients: Dict[str, pd.DataFrame],
) -> Dict[str, Any]:
    global_ids = set(train_df["_row_id"].astype(int).tolist())
    all_ids = []
    for df in clients.values():
        all_ids.extend(df["_row_id"].astype(int).tolist())
    client_set = set(all_ids)

    result = {
        "global_train_rows": int(len(train_df)),
        "client_rows_total": int(len(all_ids)),
        "unique_client_rows": int(len(client_set)),
        "rows_missing_from_clients": int(len(global_ids - client_set)),
        "rows_not_in_global_train": int(len(client_set - global_ids)),
        "duplicate_rows_across_clients": int(len(all_ids) - len(client_set)),
    }
    result["pass"] = bool(
        len(all_ids) == len(train_df)
        and client_set == global_ids
        and len(all_ids) == len(client_set)
    )
    return result


def _evidence_factor_signature(factors: Dict[str, Dict[str, float]]) -> Tuple[float, ...]:
    """Stable factor-level signature used to guarantee 20 distinct evidence bundles."""
    return tuple(
        float(factors[dim][factor])
        for dim in DOCUMENTARY_DIMS
        for factor in FACTOR_NAMES[dim]
    )


def build_20_profile_documentary_evidence_library(
    library_seed: int,
    domain: str = "healthcare",
) -> Tuple[List[Dict[str, Any]], pd.DataFrame]:
    """
    Build exactly 20 DISTINCT documentary-evidence bundles before any client is
    assigned an evidence profile.

    The library deliberately covers a broad but policy-valid range of evidence
    strengths. It contains 12 profiles expected to be governance-eligible under
    typical TRAIN-measured DQ values and 8 weaker profiles. This gives an expected
    admission tendency near 60% without forcing the realized K=20 admission rate.

    IMPORTANT:
      * The 20 bundles are generated independently of client identity, TEST data,
        model loss, and downstream predictive performance.
      * DQ (dim2) is never synthesized here; it remains machine-measured from each
        client's TRAIN shard.
      * The library is frozen by ``library_seed`` and then reused unchanged by
        both RQ5 and RQ6.
    """
    domain = str(domain).lower()
    if domain not in DOMAIN_FACTOR_MINIMA:
        raise ValueError(f"Unsupported domain: {domain!r}")

    rng = np.random.default_rng(int(library_seed))
    factor_minima = DOMAIN_FACTOR_MINIMA[domain]
    critical_policy = CRITICAL_FACTORS_BY_DOMAIN[domain]
    critical_sequence = [
        (dim, factor)
        for dim, factor_list in critical_policy.items()
        for factor in factor_list
    ]
    critical_keys = set(critical_sequence)

    # 20 distinct profiles. The role mix is specified before client assignment.
    # 7 + 5 = 12 profiles are designed around the accepted/review-recoverable
    # region; the remaining 8 exercise rejection paths. Because clients sample
    # WITH REPLACEMENT, the realized admitted percentage is not forced to 60%.
    profile_specs = (
        [("DIRECT_STRONG", v) for v in range(7)]
        + [("REVIEW_RECOVERABLE", v) for v in range(5)]
        + [("REVIEW_LIMITED", v) for v in range(3)]
        + [("LOW_HPS_WEAK", v) for v in range(3)]
        + [("DIMENSION_FLOOR_WEAK", v) for v in range(2)]
    )
    if len(profile_specs) != 20:
        raise AssertionError("The evidence library must contain exactly 20 profiles")

    def make_profile(role: str, variant: int) -> Dict[str, Dict[str, float]]:
        factors = {dim: {} for dim in DOCUMENTARY_DIMS}

        if role == "DIRECT_STRONG":
            # Strong evidence with factor-level variation. Critical factors always
            # remain at or above their declared adequacy minima.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    minimum = float(factor_minima[dim][factor])
                    low = int(max(4.0, minimum))
                    score = low if low >= 5 else int(rng.integers(low, 6))
                    factors[dim][factor] = float(score)

        elif role == "REVIEW_RECOVERABLE":
            # Moderate evidence. At least one critical factor is set below its
            # individual minimum, while another is strengthened so that the
            # compensatory CWAC review path is meaningfully exercised.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = float(rng.choice([3, 3, 3, 4]))

            for dim, factor in critical_sequence:
                factors[dim][factor] = float(factor_minima[dim][factor])

            high_min = [
                key for key in critical_sequence
                if float(factor_minima[key[0]][key[1]]) >= 4.0
            ]
            compensators = [
                key for key in critical_sequence
                if float(factor_minima[key[0]][key[1]]) <= 3.0
            ]
            weak_key = high_min[variant % len(high_min)]
            comp_key = compensators[(variant + 1) % len(compensators)]
            weak_min = float(factor_minima[weak_key[0]][weak_key[1]])
            comp_min = float(factor_minima[comp_key[0]][comp_key[1]])
            factors[weak_key[0]][weak_key[1]] = float(max(0.0, weak_min - 1.0))
            factors[comp_key[0]][comp_key[1]] = float(min(5.0, comp_min + 2.0))

        elif role == "REVIEW_LIMITED":
            # Dimensions remain around the review region, but critical evidence is
            # deliberately weaker than the domain adequacy reference.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    base = 4 if (dim, factor) not in critical_keys and rng.random() < 0.45 else 3
                    factors[dim][factor] = float(base)
            for dim, factor in critical_sequence:
                minimum = float(factor_minima[dim][factor])
                drop = 1.0 + (1.0 if ((variant + len(factor)) % 3 == 0) else 0.0)
                factors[dim][factor] = float(max(2.0, minimum - drop))

        elif role == "LOW_HPS_WEAK":
            # Every documentary dimension stays at or slightly above the 2.5
            # dimension floor, but the overall documentary evidence remains weak.
            for dim in DOCUMENTARY_DIMS:
                names = list(FACTOR_NAMES[dim])
                n_three = (len(names) + 1) // 2
                values = [3.0] * n_three + [2.0] * (len(names) - n_three)
                rng.shuffle(values)
                # Variant-specific small change that preserves the weak region.
                if variant == 1 and len(values) >= 4:
                    values[0], values[-1] = values[-1], values[0]
                elif variant == 2 and len(values) >= 3:
                    values = values[1:] + values[:1]
                for factor, value in zip(names, values):
                    factors[dim][factor] = float(value)

        elif role == "DIMENSION_FLOOR_WEAK":
            # Keep most evidence moderate but deliberately push one documentary
            # dimension below the 2.5 floor. Two variants use different dimensions.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = float(rng.choice([3, 3, 4]))
            weak_dim = ["dim4", "dim3"][variant % 2]
            weak_names = list(FACTOR_NAMES[weak_dim])
            for j, factor in enumerate(weak_names):
                factors[weak_dim][factor] = float(1.0 if j == (variant % len(weak_names)) else 2.0)

        else:
            raise ValueError(f"Unknown evidence role: {role!r}")

        return factors

    library: List[Dict[str, Any]] = []
    seen = set()

    for idx, (role, variant) in enumerate(profile_specs, start=1):
        # Very large discrete space; collisions are unlikely, but fail closed and
        # regenerate if one occurs so the library really contains 20 profiles.
        factors = None
        signature = None
        for _attempt in range(200):
            candidate = make_profile(role, int(variant))
            sig = _evidence_factor_signature(candidate)
            if sig not in seen:
                factors, signature = candidate, sig
                break
        if factors is None:
            raise RuntimeError("Could not generate 20 unique documentary evidence profiles")
        seen.add(signature)
        library.append({
            "library_profile_id": f"P{idx:02d}",
            "profile_role": role,
            "profile_variant": int(variant),
            "factors": factors,
        })

    if len(library) != 20 or len(seen) != 20:
        raise RuntimeError("Evidence-library uniqueness audit failed")

    rows = []
    for entry in library:
        for dim in DOCUMENTARY_DIMS:
            for factor, score in entry["factors"][dim].items():
                min_rank = float(factor_minima[dim][factor])
                rows.append({
                    "library_profile_id": entry["library_profile_id"],
                    "evidence_profile": entry["profile_role"],
                    "scenario_role": entry["profile_role"],
                    "profile_variant": int(entry["profile_variant"]),
                    "dimension": dim,
                    "dimension_name": DIMENSION_NAMES[dim],
                    "factor": factor,
                    "rubric_score_0_5": float(score),
                    "rubric_descriptor": rubric_descriptor(dim, factor, score),
                    "adequacy_min_rank": min_rank,
                    "meets_factor_adequacy": bool(float(score) >= min_rank),
                    "evidence_library_seed": int(library_seed),
                    "domain": domain,
                })

    library_df = pd.DataFrame(rows)
    return library, library_df


def generate_random_library_controlled_evidence(
    client_ids: List[str],
    evidence_seed: int,
    library_seed: int,
    domain: str = "healthcare",
) -> Tuple[Dict[str, Dict[str, Dict[str, float]]], pd.DataFrame, pd.DataFrame]:
    """
    Assign one of 20 distinct documentary evidence profiles independently to
    every client WITH REPLACEMENT.

    Therefore:
      * two or more clients may receive the same documentary evidence profile;
      * some of the 20 profiles may not be sampled in a particular realization;
      * the admitted-client percentage is an observed result, not a fixed design
        target or a repeated 10-client block.

    The sampled assignment is frozen before training and reused by RQ5 and RQ6.
    """
    library, library_df = build_20_profile_documentary_evidence_library(
        library_seed=library_seed,
        domain=domain,
    )
    if len(library) != 20:
        raise RuntimeError("Expected exactly 20 evidence profiles")

    rng = np.random.default_rng(int(evidence_seed))
    draw_idx = rng.integers(0, len(library), size=len(client_ids))

    profile_ids = [library[int(i)]["library_profile_id"] for i in draw_idx]
    usage_counts = pd.Series(profile_ids).value_counts().to_dict()

    evidence_by_client: Dict[str, Dict[str, Dict[str, float]]] = {}
    rows = []

    for draw_position, (cid, lib_idx) in enumerate(zip(client_ids, draw_idx), start=1):
        entry = library[int(lib_idx)]
        evidence_by_client[str(cid)] = {
            dim: dict(entry["factors"][dim])
            for dim in DOCUMENTARY_DIMS
        }

        for dim in DOCUMENTARY_DIMS:
            for factor, score in entry["factors"][dim].items():
                min_rank = float(DOMAIN_FACTOR_MINIMA[domain][dim][factor])
                rows.append({
                    "client": str(cid),
                    "assignment_draw_position": int(draw_position),
                    "library_profile_id": entry["library_profile_id"],
                    "bundle_id": entry["library_profile_id"],
                    "evidence_profile": entry["profile_role"],
                    "scenario_role": entry["profile_role"],
                    "profile_variant": int(entry["profile_variant"]),
                    "times_profile_sampled_in_realization": int(usage_counts[entry["library_profile_id"]]),
                    "assignment_sampling": "UNIFORM_WITH_REPLACEMENT",
                    "dimension": dim,
                    "dimension_name": DIMENSION_NAMES[dim],
                    "factor": factor,
                    "evidence_source_type": "CONTROLLED_DOCUMENTARY_EVIDENCE_20_PROFILE_LIBRARY",
                    "verified_rubric_level_0_5": float(score),
                    "rubric_score_0_5": float(score),
                    "server_mapped_score_0_5": float(score),
                    "rubric_descriptor": rubric_descriptor(dim, factor, score),
                    "human_role": "VERIFY_EVIDENCE_ONLY",
                    "score_assignment": "DETERMINISTIC_SERVER_MAPPING_FROM_VERIFIED_RUBRIC_LEVEL",
                    "admission_decision_by": "SERVER_POLICY",
                    "adequacy_min_rank": min_rank,
                    "adequacy_min_normalized": min_rank / MAX_FACTOR_SCORE,
                    "meets_factor_adequacy": bool(float(score) >= min_rank),
                    "evidence_artifact_id": f"{cid}-{entry['library_profile_id']}-{dim}-{factor}",
                    "validation_status": "CONTROLLED_RANDOM_LIBRARY_EVIDENCE",
                    "evidence_seed": int(evidence_seed),
                    "evidence_library_seed": int(library_seed),
                    "domain": domain,
                })

    assignment_df = pd.DataFrame(rows)

    # Fail-closed assignment audit: exactly one sampled profile per client.
    one = assignment_df[["client", "library_profile_id"]].drop_duplicates()
    per_client = one.groupby("client")["library_profile_id"].nunique()
    if len(per_client) != len(client_ids) or not (per_client == 1).all():
        raise RuntimeError("Evidence assignment audit failed: each client must receive exactly one profile")

    return evidence_by_client, assignment_df, library_df


def summarize_random_library_assignment(evidence_df: pd.DataFrame) -> pd.DataFrame:
    """One-row-per-client audit of the frozen with-replacement evidence draw."""
    cols = [
        "client", "library_profile_id", "evidence_profile", "profile_variant",
        "times_profile_sampled_in_realization", "assignment_sampling",
        "evidence_seed", "evidence_library_seed",
    ]
    out = evidence_df[cols].drop_duplicates("client").sort_values("client").reset_index(drop=True)
    return out


def prepare_k_data(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    k: int,
    partition_seed: int,
):
    clients = dirichlet_partition_dataframe_scalable(
        train_df=train_df,
        n_clients=k,
        alpha=DIRICHLET_ALPHA,
        seed=partition_seed,
        min_client_records=MIN_CLIENT_RECORDS,
    )

    integrity = partition_integrity_audit(train_df, clients)
    if not integrity["pass"]:
        raise RuntimeError(f"Partition integrity failed at K={k}: {integrity}")

    pre = fit_federated_train_only_preprocessor(clients)
    reference = build_tabular_reference(clients, pre)

    dq_scores = {}
    dq_rows = []
    client_arrays = {}

    for cid, df in clients.items():
        scores, raw_metrics = tabular_dq_scores(df, pre, reference)
        dq_scores[cid] = scores
        dq_rows.append({"client": cid, **scores, **raw_metrics})

        Xc = pre.transform(df)
        yc = df["_target"].to_numpy(dtype=np.int32)
        client_arrays[cid] = (Xc, yc)

    X_test = pre.transform(test_df)
    y_test = test_df["_target"].to_numpy(dtype=np.int32)

    y_train_all = np.concatenate(
        [client_arrays[cid][1] for cid in sorted(client_arrays)]
    )
    class_weights = class_weight_dict(y_train_all)

    return {
        "clients_raw": clients,
        "client_ids": list(clients.keys()),
        "client_arrays": client_arrays,
        "preprocessor": pre,
        "dq_reference": reference,
        "dq_scores": dq_scores,
        "dq_audit": pd.DataFrame(dq_rows),
        "X_test": X_test,
        "y_test": y_test,
        "class_weights": class_weights,
        "input_dim": int(X_test.shape[1]),
        "integrity": integrity,
    }


# ======================================================================================
# SUMMARIES / CHECKPOINTS
# ======================================================================================

def summarize_b3(perf: pd.DataFrame) -> pd.DataFrame:
    metrics = [
        "accuracy",
        "precision_macro",
        "recall_macro",
        "f1_macro",
        "roc_auc_ovr_macro",
        "runtime_s",
        "communication_mb",
        "optimizer_steps",
        "participants",
        "energy_wh",
        "ram_peak_mb",
        "ram_delta_mb",
    ]
    rows = []
    for (k, scenario), d in perf.groupby(
        ["k_submissions", "scenario"], sort=True
    ):
        row = {
            "k_submissions": int(k),
            "scenario": str(scenario),
            "n_runs": int(len(d)),
        }
        for metric in metrics:
            if metric not in d.columns:
                continue
            vals = pd.to_numeric(d[metric], errors="coerce").to_numpy(dtype=float)
            vals = vals[np.isfinite(vals)]
            row[f"{metric}_mean"] = (
                float(np.mean(vals)) if len(vals) else np.nan
            )
            row[f"{metric}_sd"] = (
                float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            )
            lo, hi = ci95_mean(vals)
            row[f"{metric}_ci95_low"] = lo
            row[f"{metric}_ci95_high"] = hi
        rows.append(row)
    return pd.DataFrame(rows)


def paired_b3(perf: pd.DataFrame) -> pd.DataFrame:
    """
    Paired seed-wise differences.

    TADP-VR vs Random-K:
        matched K and matched optimizer-step budget.

    TADP-VR vs Vanilla FedAvg:
        utility comparison against full participation; NOT compute-matched.
    """
    rows = []
    comparisons = [
        ("Random-K", "MATCHED_SELECTION_CONTROL"),
        ("Vanilla FedAvg", "FULL_PARTICIPATION_REFERENCE"),
    ]

    for k in sorted(perf["k_submissions"].unique()):
        tadp = perf[
            (perf["k_submissions"].eq(k))
            & (perf["scenario"].eq("TADP-VR Federated"))
        ].copy()

        for baseline, role in comparisons:
            base = perf[
                (perf["k_submissions"].eq(k))
                & (perf["scenario"].eq(baseline))
            ].copy()

            merged = tadp.merge(
                base,
                on=["k_submissions", "seed"],
                suffixes=("_tadp", "_baseline"),
                validate="one_to_one",
            )

            for metric in [
                "accuracy",
                "precision_macro",
                "recall_macro",
                "f1_macro",
                "roc_auc_ovr_macro",
            ]:
                a = merged[f"{metric}_tadp"].to_numpy(dtype=float)
                b = merged[f"{metric}_baseline"].to_numpy(dtype=float)
                diff = a - b
                lo, hi = ci95_mean(diff)

                rows.append({
                    "k_submissions": int(k),
                    "baseline": baseline,
                    "baseline_role": role,
                    "metric": metric,
                    "n_pairs": int(len(diff)),
                    "mean_difference_tadp_minus_baseline":
                        float(np.mean(diff)),
                    "sd_difference":
                        float(np.std(diff, ddof=1)) if len(diff) > 1 else 0.0,
                    "ci95_low": lo,
                    "ci95_high": hi,
                    "tadp_wins": int(np.sum(diff > 0)),
                    "ties": int(np.sum(np.isclose(diff, 0.0))),
                    "tadp_losses": int(np.sum(diff < 0)),
                })

    return pd.DataFrame(rows)


def utility_retention_table(summary: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for k in sorted(summary["k_submissions"].unique()):
        d = summary[summary["k_submissions"].eq(k)].set_index("scenario")
        if "TADP-VR Federated" not in d.index or "Vanilla FedAvg" not in d.index:
            continue

        row = {"k_submissions": int(k)}
        for metric in ["accuracy", "f1_macro", "roc_auc_ovr_macro"]:
            tadp = float(d.loc["TADP-VR Federated", f"{metric}_mean"])
            full = float(d.loc["Vanilla FedAvg", f"{metric}_mean"])
            rand = float(d.loc["Random-K", f"{metric}_mean"])

            row[f"tadp_{metric}_mean"] = tadp
            row[f"fedavg_{metric}_mean"] = full
            row[f"randomk_{metric}_mean"] = rand
            row[f"tadp_minus_fedavg_{metric}"] = tadp - full
            row[f"tadp_minus_randomk_{metric}"] = tadp - rand
            row[f"tadp_retention_pct_of_fedavg_{metric}"] = (
                100.0 * tadp / full if np.isfinite(full) and full != 0 else np.nan
            )

        row["tadp_participants_mean"] = float(
            d.loc["TADP-VR Federated", "participants_mean"]
        )
        row["fedavg_participants_mean"] = float(
            d.loc["Vanilla FedAvg", "participants_mean"]
        )
        row["randomk_participants_mean"] = float(
            d.loc["Random-K", "participants_mean"]
        )
        row["tadp_communication_mb_mean"] = float(
            d.loc["TADP-VR Federated", "communication_mb_mean"]
        )
        row["fedavg_communication_mb_mean"] = float(
            d.loc["Vanilla FedAvg", "communication_mb_mean"]
        )
        row["communication_reduction_pct_vs_fedavg"] = (
            100.0
            * (
                float(d.loc["Vanilla FedAvg", "communication_mb_mean"])
                - float(d.loc["TADP-VR Federated", "communication_mb_mean"])
            )
            / max(
                float(d.loc["Vanilla FedAvg", "communication_mb_mean"]),
                1e-12,
            )
        )
        rows.append(row)

    return pd.DataFrame(rows)


def write_b3_scenario_checkpoint(
    root: Path,
    k: int,
    run_idx: int,
    scenario: str,
    result: Dict[str, Any],
):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", scenario).strip("_")
    cdir = ensure_dir(
        root
        / "scenario_checkpoints"
        / f"K_{int(k):03d}"
        / f"run_{int(run_idx):02d}"
    )
    payload = {
        "k_submissions": int(k),
        "run": int(run_idx),
        "scenario": str(scenario),
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "metrics": result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
    }
    atomic_write_json(payload, cdir / f"{safe}.json")
    if "round_audit" in result:
        pd.DataFrame(result["round_audit"]).to_csv(
            cdir / f"{safe}_rounds.csv",
            index=False,
        )


def save_environment_metadata():
    meta = {
        "experiment_version": EXPERIMENT_VERSION,
        "python": sys.version,
        "platform": platform.platform(),
        "tensorflow": tf.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "client_counts": CLIENT_COUNTS,
        "training_seeds": TRAINING_RUN_SEEDS,
        "fl_rounds": NUM_ROUNDS_FL,
        "local_epochs": LOCAL_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "dirichlet_alpha": DIRICHLET_ALPHA,
        "global_split_seed": GLOBAL_SPLIT_SEED,
        "partition_seed_formula": "PARTITION_BASE_SEED + 101*K",
        "evidence_seed_formula": "EVIDENCE_BASE_SEED + 103*K",
        "randomk_seed_formula": "RANDOMK_BASE_SEED + 107*K",
        "test_semantics":
            "final evaluation only; never used for governance, selection, "
            "preprocessing, step budgets, or checkpoint decisions",
        "scenario_roles": {
            "Vanilla FedAvg":
                "full-participation predictive-utility reference; not compute matched",
            "Random-K":
                "same selected-client count and exact total optimizer-step budget per round as TADP-VR",
            "TADP-VR Federated":
                "governance-selected cohort",
        },
        "k10_note":
            "K=10 intentionally omitted here; use final Experiment-A 4-round rows "
            "for seeds 42,142,242 when constructing the manuscript K=10/20/50/100 plot.",
    }
    atomic_write_json(
        meta,
        EXPERIMENT_ROOT / "b3_experiment_design.json",
    )


# ======================================================================================
# RQ6 — THRESHOLD + HPS-WEIGHT SENSITIVITY (K=20, 20-profile evidence library, with replacement)
# ======================================================================================
#
# Research question:
#   How sensitive are TADP-VR admission decisions and downstream utility to
#   controlled changes in HPS decision thresholds and HPS dimension weights?
#
# Frozen reference setting:
#   20 distinct documentary evidence profiles are generated first; each client
#   samples one profile independently WITH REPLACEMENT using a frozen seed.
#   K=20, assignment seed=14102, library seed=17220, partition seed=11121, split seed=7001,
#   training seeds=[42,142,242,342,442], 4 FL rounds, 1 local epoch, batch=64.
#
# Threshold perturbation families (each tested separately):
#   1) LOWER_ONLY  : lower threshold x (1 ± 10/20/30%), upper fixed at 3.5
#   2) UPPER_ONLY  : upper threshold x (1 ± 10/20/30%), lower fixed at 3.0
#   3) BOTH        : both thresholds x the same factor
# Invalid policies with lower >= upper are recorded and not executed.
#
# Weight sensitivity:
#   Each HPS dimension is perturbed ONE AT A TIME by ±10%, ±20%, ±30% around its
#   reference weight. The other five weights are rescaled proportionally so that
#   the full vector still sums to 1. This is necessary because multiplying every
#   weight by the same factor and renormalizing would reproduce the original vector
#   and therefore would not be a meaningful sensitivity test.
#
# For every valid configuration the code reports:
#   - admitted client count and admitted percentage;
#   - percentage-point and relative gain/loss versus the frozen reference;
#   - client decision flips and admitted-set Jaccard similarity;
#   - predictive utility (macro ROC-AUC, macro-F1, accuracy) over 5 training seeds;
#   - paired seed-level differences versus the frozen reference.
#
# This is sensitivity analysis, NOT policy tuning. No configuration is selected
# using held-out TEST performance.
# ======================================================================================

import matplotlib.pyplot as plt

EXPERIMENT_VERSION = "TADP-RQ6-v17.3-K20-20EVIDENCE-WR-THRESHOLD-WEIGHT-SENSITIVITY-5SEED-4ROUND"

K_SUBMISSIONS = 20
TRAINING_RUN_SEEDS = [42, 142, 242, 342, 442]
NUM_ROUNDS_FL = 4
LOCAL_EPOCHS = 1
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
DIRICHLET_ALPHA = 1.0

GLOBAL_SPLIT_SEED = 7001
CLIENT_PARTITION_SEED = 11121
FROZEN_EVIDENCE_ASSIGNMENT_SEED = 14102
EVIDENCE_LIBRARY_SEED = 17220
EVIDENCE_LIBRARY_SIZE = 20
EVIDENCE_ASSIGNMENT_SAMPLING = "UNIFORM_WITH_REPLACEMENT"
DOMAIN = "healthcare"

REFERENCE_LOWER_CUT = 3.0
REFERENCE_UPPER_CUT = 3.5
REFERENCE_DIMENSION_FLOOR = 2.5
PERTURBATION_PCTS = [-30, -20, -10, 10, 20, 30]
DIMS = [f"dim{i}" for i in range(1, 7)]

USE_GOOGLE_DRIVE_CHECKPOINTS = True
EXPERIMENT_ROOT = choose_experiment_root(
    "TADP_EXPERIMENT_RQ6_" + EXPERIMENT_VERSION,
    use_drive=USE_GOOGLE_DRIVE_CHECKPOINTS,
)
CHECKPOINT_STATE = EXPERIMENT_ROOT / "checkpoint_state.json"
UNIQUE_COHORT_PERF = EXPERIMENT_ROOT / "unique_cohort_performance_checkpoint.csv"


def _bool_series_rq6(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s
    return s.astype(str).str.lower().isin(["true", "1", "yes"])


def _jaccard_rq6(a, b) -> float:
    a = set(map(str, a)); b = set(map(str, b)); u = a | b
    return 1.0 if not u else float(len(a & b) / len(u))


def build_policy_tables_rq6(full_factor_df: pd.DataFrame):
    f = full_factor_df.copy()
    f["client"] = f["client"].astype(str)
    f["dimension"] = f["dimension"].astype(str)
    f["rubric_score_0_5"] = pd.to_numeric(f["rubric_score_0_5"], errors="raise")
    dim_scores = (
        f.groupby(["client", "dimension"])["rubric_score_0_5"]
        .mean().unstack("dimension").sort_index().reindex(columns=DIMS)
    )
    crit = f.loc[_bool_series_rq6(f["is_critical_factor"])].copy()
    crit["critical_adequacy_min_rank"] = pd.to_numeric(
        crit["critical_adequacy_min_rank"], errors="coerce"
    )
    crit = crit.loc[crit["critical_adequacy_min_rank"].notna()].copy()
    return f, dim_scores, crit


def perturb_one_weight(target_dim: str, pct: int) -> dict:
    """
    Change one HPS dimension weight by pct% and proportionally rescale the other
    five weights so the final vector sums exactly to 1 while preserving their
    relative proportions.
    """
    if target_dim not in DIMS:
        raise ValueError(target_dim)
    base = {d: float(WEIGHTS_PSCORE_DEFAULT[d]) for d in DIMS}
    factor = 1.0 + float(pct) / 100.0
    target_new = base[target_dim] * factor
    if not (0.0 < target_new < 1.0):
        raise ValueError(f"Invalid perturbed weight for {target_dim}: {target_new}")
    remaining_old = 1.0 - base[target_dim]
    remaining_new = 1.0 - target_new
    scale = remaining_new / remaining_old
    out = {}
    for d in DIMS:
        out[d] = target_new if d == target_dim else base[d] * scale
    # Numerical cleanup.
    total = float(sum(out.values()))
    out = {d: float(w / total) for d, w in out.items()}
    if abs(sum(out.values()) - 1.0) > 1e-12:
        raise RuntimeError("Weight normalization failed")
    if any(w <= 0.0 for w in out.values()):
        raise RuntimeError("Weight perturbation produced non-positive weight")
    return out


def evaluate_rq6_policy(
    dim_scores: pd.DataFrame,
    crit: pd.DataFrame,
    lower_cut: float,
    upper_cut: float,
    weights: dict,
) -> pd.DataFrame:
    lower_cut = float(lower_cut); upper_cut = float(upper_cut)
    if not (0.0 <= lower_cut <= 5.0 and 0.0 <= upper_cut <= 5.0 and lower_cut < upper_cut):
        raise ValueError(f"Invalid thresholds: lower={lower_cut}, upper={upper_cut}")
    if set(weights) != set(DIMS):
        raise ValueError("Weight vector must contain dim1..dim6")
    if abs(sum(float(weights[d]) for d in DIMS) - 1.0) > 1e-10:
        raise ValueError("Weight vector must sum to 1")

    active_crit = crit.copy()
    if active_crit.empty:
        global_critical_wac = 0.0
    else:
        global_critical_wac = float(np.mean(
            active_crit["critical_adequacy_min_rank"].astype(float).to_numpy()
            / MAX_FACTOR_SCORE
        ))

    rows = []
    for client in dim_scores.index.astype(str):
        ds = dim_scores.loc[client]
        hps = float(sum(float(ds[d]) * float(weights[d]) for d in DIMS))
        floor_failures = [
            d for d in DIMS
            if (not np.isfinite(float(ds[d]))) or float(ds[d]) < REFERENCE_DIMENSION_FLOOR
        ]

        cdf = active_crit.loc[active_crit["client"].astype(str).eq(client)].copy()
        if cdf.empty:
            critical_wac_i = 1.0
            all_critical_meet = True
            below = []
        else:
            scores = cdf["rubric_score_0_5"].astype(float).to_numpy()
            minima = cdf["critical_adequacy_min_rank"].astype(float).to_numpy()
            critical_wac_i = float(np.mean(scores / MAX_FACTOR_SCORE))
            all_critical_meet = bool(np.all(scores >= minima))
            below = [
                f"{r.dimension}.{r.factor}:{float(r.rubric_score_0_5):.1f}"
                f"<{float(r.critical_adequacy_min_rank):.1f}"
                for r in cdf.itertuples()
                if float(r.rubric_score_0_5) < float(r.critical_adequacy_min_rank)
            ]

        if floor_failures:
            action = "REJECT"; path = "DIMENSION_FLOOR"
        elif hps < lower_cut:
            action = "REJECT"; path = "LOW_HPS"
        elif hps >= upper_cut and all_critical_meet:
            action = "ACCEPT"; path = "DIRECT_AUTO_ACCEPT"
        elif critical_wac_i >= global_critical_wac:
            action = "ACCEPT"; path = "ACCEPTED_AFTER_AUTOMATED_REVIEW"
        else:
            action = "REJECT"; path = "AUTO_REJECTED_REVIEW_CRITICAL_WAC"

        rows.append({
            "client": client, "hps": hps,
            "final_action": action, "decision_path": path,
            "dimension_floor_failures": ";".join(floor_failures),
            "critical_wac_i": critical_wac_i,
            "global_critical_wac": global_critical_wac,
            "all_critical_meet_adequacy": bool(all_critical_meet),
            "critical_below_adequacy": ";".join(below),
            "lower_cut": lower_cut, "upper_cut": upper_cut,
        })
    return pd.DataFrame(rows)


def build_rq6_configurations() -> pd.DataFrame:
    rows = [{
        "configuration": "REFERENCE",
        "family": "REFERENCE",
        "perturbation_pct": 0,
        "target_dimension": "",
        "lower_cut": REFERENCE_LOWER_CUT,
        "upper_cut": REFERENCE_UPPER_CUT,
        "valid_policy": True,
        "invalid_reason": "",
        "weights_json": json.dumps(WEIGHTS_PSCORE_DEFAULT, sort_keys=True),
    }]

    # Threshold families.
    for family in ["LOWER_ONLY", "UPPER_ONLY", "BOTH"]:
        for pct in PERTURBATION_PCTS:
            factor = 1.0 + float(pct)/100.0
            lower = REFERENCE_LOWER_CUT
            upper = REFERENCE_UPPER_CUT
            if family == "LOWER_ONLY":
                lower = REFERENCE_LOWER_CUT * factor
            elif family == "UPPER_ONLY":
                upper = REFERENCE_UPPER_CUT * factor
            else:
                lower = REFERENCE_LOWER_CUT * factor
                upper = REFERENCE_UPPER_CUT * factor
            valid = bool(0.0 <= lower <= 5.0 and 0.0 <= upper <= 5.0 and lower < upper)
            reason = "" if valid else (
                "Invalid threshold ordering/range: lower must remain strictly below upper within [0,5]."
            )
            rows.append({
                "configuration": f"{family}_{pct:+d}pct",
                "family": family,
                "perturbation_pct": int(pct),
                "target_dimension": "",
                "lower_cut": float(lower), "upper_cut": float(upper),
                "valid_policy": valid, "invalid_reason": reason,
                "weights_json": json.dumps(WEIGHTS_PSCORE_DEFAULT, sort_keys=True),
            })

    # HPS weight sensitivity: one dimension at a time; thresholds stay frozen.
    for dim in DIMS:
        for pct in PERTURBATION_PCTS:
            weights = perturb_one_weight(dim, pct)
            rows.append({
                "configuration": f"WEIGHT_{dim}_{pct:+d}pct",
                "family": "WEIGHT_ONE_DIM",
                "perturbation_pct": int(pct),
                "target_dimension": dim,
                "lower_cut": REFERENCE_LOWER_CUT,
                "upper_cut": REFERENCE_UPPER_CUT,
                "valid_policy": True,
                "invalid_reason": "",
                "weights_json": json.dumps(weights, sort_keys=True),
            })
    return pd.DataFrame(rows)


def summarize_rq6_governance(configs, dim_scores, crit):
    ref_weights = {d: float(WEIGHTS_PSCORE_DEFAULT[d]) for d in DIMS}
    ref = evaluate_rq6_policy(
        dim_scores, crit, REFERENCE_LOWER_CUT, REFERENCE_UPPER_CUT, ref_weights
    ).sort_values("client").reset_index(drop=True)
    ref_accept = ref.loc[ref["final_action"].eq("ACCEPT"), "client"].astype(str).tolist()
    ref_rate = len(ref_accept) / len(ref)

    rows = []; gov_by_config = {}
    for cfg in configs.itertuples(index=False):
        base = {
            "configuration": cfg.configuration,
            "family": cfg.family,
            "perturbation_pct": int(cfg.perturbation_pct),
            "target_dimension": str(cfg.target_dimension),
            "lower_cut": float(cfg.lower_cut), "upper_cut": float(cfg.upper_cut),
            "valid_policy": bool(cfg.valid_policy), "invalid_reason": str(cfg.invalid_reason),
            "weights_json": str(cfg.weights_json),
        }
        if not cfg.valid_policy:
            rows.append({
                **base,
                "accepted_count": np.nan, "accepted_rate": np.nan, "accepted_rate_pct": np.nan,
                "accepted_count_change_vs_reference": np.nan,
                "accepted_rate_change_pp_vs_reference": np.nan,
                "accepted_rate_relative_change_pct_vs_reference": np.nan,
                "decision_flip_count": np.nan, "decision_flip_rate": np.nan,
                "accepted_set_jaccard_vs_reference": np.nan,
                "mean_abs_hps_change": np.nan, "admitted_clients": "", "cohort_key": "",
            })
            continue

        weights = {k: float(v) for k, v in json.loads(cfg.weights_json).items()}
        cur = evaluate_rq6_policy(dim_scores, crit, cfg.lower_cut, cfg.upper_cut, weights)
        cur = cur.sort_values("client").reset_index(drop=True)
        gov_by_config[cfg.configuration] = cur
        admitted = cur.loc[cur["final_action"].eq("ACCEPT"), "client"].astype(str).tolist()
        rate = len(admitted)/len(cur)
        comp = ref[["client","final_action","hps"]].merge(
            cur[["client","final_action","hps"]], on="client",
            suffixes=("_reference","_current"), validate="one_to_one"
        )
        flips = comp["final_action_reference"].ne(comp["final_action_current"])
        rows.append({
            **base,
            "accepted_count": int(len(admitted)),
            "accepted_rate": float(rate), "accepted_rate_pct": float(100.0*rate),
            "accepted_count_change_vs_reference": int(len(admitted)-len(ref_accept)),
            "accepted_rate_change_pp_vs_reference": float(100.0*(rate-ref_rate)),
            "accepted_rate_relative_change_pct_vs_reference": (
                float(100.0*(rate-ref_rate)/ref_rate) if ref_rate>0 else np.nan
            ),
            "decision_flip_count": int(flips.sum()), "decision_flip_rate": float(flips.mean()),
            "accepted_set_jaccard_vs_reference": _jaccard_rq6(ref_accept, admitted),
            "mean_abs_hps_change": float(np.mean(np.abs(
                comp["hps_current"].to_numpy(float)-comp["hps_reference"].to_numpy(float)
            ))),
            "admitted_clients": ";".join(admitted), "cohort_key": ";".join(sorted(admitted)),
        })
    return pd.DataFrame(rows), gov_by_config, ref


def assign_rq6_cohort_ids(gov_summary: pd.DataFrame):
    valid = gov_summary[gov_summary["valid_policy"].eq(True) & gov_summary["cohort_key"].astype(str).ne("")]
    keys = sorted(valid["cohort_key"].unique().tolist())
    mapping = {key: f"COHORT_{i+1:02d}" for i, key in enumerate(keys)}
    out = gov_summary.copy(); out["cohort_id"] = out["cohort_key"].map(mapping).fillna("")
    return out, mapping


def train_rq6_unique_cohorts(cohort_mapping, gov_summary, data):
    cohort_clients = {
        cohort_id: [x for x in str(key).split(";") if x]
        for key, cohort_id in cohort_mapping.items()
    }
    train_monitor = build_train_monitor_subset(
        data["client_arrays"], max_samples=ROUND_PROGRESS_MONITOR_MAX_SAMPLES, seed=99137
    )
    build_model_fn = lambda: build_diabetes_model(data["meta"]["input_dim"], lr=LEARNING_RATE)
    completed = set(load_checkpoint_state(CHECKPOINT_STATE).get("completed", []))
    total_jobs = len(TRAINING_RUN_SEEDS)*len(cohort_clients); job_idx = 0

    for training_seed in TRAINING_RUN_SEEDS:
        seed_everything(training_seed)
        base_model = build_model_fn()
        initial_weights = [np.array(w, copy=True) for w in base_model.get_weights()]
        initial_hash = sha256_weights(initial_weights)
        del base_model; tf.keras.backend.clear_session(); gc.collect()

        for cohort_id, selected in cohort_clients.items():
            job_idx += 1; key = f"seed{training_seed}|{cohort_id}"
            if key in completed:
                print(f"CHECKPOINT FOUND -> SKIP: {key}"); continue
            if not selected:
                raise RuntimeError(f"Empty cohort {cohort_id}")
            print_banner(f"RQ6 UNIQUE COHORT {job_idx}/{total_jobs} | seed={training_seed} | {cohort_id}")
            print(f"Selected clients ({len(selected)}): {selected}")

            natural_map = {
                cid: natural_steps(len(data["client_arrays"][cid][1]), BATCH_SIZE, LOCAL_EPOCHS)
                for cid in selected
            }
            result = federated_train(
                build_model_fn=build_model_fn, initial_weights=initial_weights,
                selected_per_round=[list(selected) for _ in range(NUM_ROUNDS_FL)],
                client_arrays=data["client_arrays"], X_test=data["X_test"], y_test=data["y_test"],
                n_classes=3, batch_size=BATCH_SIZE, local_epochs=LOCAL_EPOCHS,
                class_weights=data["class_weights"], run_seed=training_seed,
                exact_step_maps=[dict(natural_map) for _ in range(NUM_ROUNDS_FL)],
                equal_weight=False, fedprox_mu=0.0, train_monitor=train_monitor,
                progress_context={
                    "scenario": f"RQ6 | {cohort_id}",
                    "run_idx": TRAINING_RUN_SEEDS.index(training_seed)+1,
                    "run_total": len(TRAINING_RUN_SEEDS),
                    "scenario_idx": job_idx, "scenario_total": total_jobs,
                    "overall_idx": job_idx, "overall_total": total_jobs,
                },
            )
            row = result_row(
                run=TRAINING_RUN_SEEDS.index(training_seed)+1, seed=training_seed,
                scenario=cohort_id, result=result, initial_hash=initial_hash,
            )
            row.update({
                "training_seed": int(training_seed), "cohort_id": cohort_id,
                "selected_k": int(len(selected)), "selected_clients": ";".join(selected),
                "steps_per_round": int(sum(natural_map.values())), "fl_rounds": NUM_ROUNDS_FL,
            })
            upsert_csv(row, UNIQUE_COHORT_PERF, ["training_seed","cohort_id"])
            mark_checkpoint_complete(CHECKPOINT_STATE, key, extra={"last_job": key})
            completed.add(key)
            del result; tf.keras.backend.clear_session(); gc.collect()

    perf = pd.read_csv(UNIQUE_COHORT_PERF)
    expected = len(TRAINING_RUN_SEEDS)*len(cohort_clients)
    if len(perf) != expected:
        raise RuntimeError(f"Expected {expected} unique-cohort rows; found {len(perf)}")

    audit_rows=[]
    for seed in TRAINING_RUN_SEEDS:
        d=perf[perf["training_seed"].eq(seed)]
        hashes=d["initial_weights_sha256"].astype(str).unique(); passed=len(hashes)==1
        audit_rows.append({
            "training_seed":seed,"n_unique_cohorts":len(d),"unique_W0_hashes":len(hashes),
            "same_W0_across_cohorts":passed,"initial_weights_sha256":hashes[0] if len(hashes) else "",
        })
        if not passed: raise RuntimeError(f"W0 parity failed for seed {seed}")
    pd.DataFrame(audit_rows).to_csv(EXPERIMENT_ROOT/"rq6_W0_parity_audit.csv",index=False)
    return perf


def summarize_rq6_performance(gov_summary, cohort_perf):
    valid = gov_summary[gov_summary["valid_policy"].eq(True) & gov_summary["cohort_id"].astype(str).ne("")]
    expanded = valid.merge(cohort_perf, on="cohort_id", how="left", validate="many_to_many")
    expanded.to_csv(EXPERIMENT_ROOT/"rq6_performance_all_seeds.csv",index=False)
    metrics=[
        "roc_auc_ovr_macro","f1_macro","accuracy","precision_macro","recall_macro",
        "runtime_s","communication_mb","optimizer_steps","participants","energy_wh",
    ]
    rows=[]
    for cfg,d in expanded.groupby("configuration",sort=False):
        g=valid[valid["configuration"].eq(cfg)].iloc[0]
        row={
            "configuration":cfg,"family":g["family"],"perturbation_pct":int(g["perturbation_pct"]),
            "target_dimension":g["target_dimension"],"lower_cut":float(g["lower_cut"]),
            "upper_cut":float(g["upper_cut"]),"weights_json":g["weights_json"],
            "cohort_id":g["cohort_id"],"accepted_count":int(g["accepted_count"]),
            "accepted_rate_pct":float(g["accepted_rate_pct"]),
            "accepted_count_change_vs_reference":int(g["accepted_count_change_vs_reference"]),
            "accepted_rate_change_pp_vs_reference":float(g["accepted_rate_change_pp_vs_reference"]),
            "accepted_rate_relative_change_pct_vs_reference":float(g["accepted_rate_relative_change_pct_vs_reference"]),
            "decision_flip_count":int(g["decision_flip_count"]),
            "accepted_set_jaccard_vs_reference":float(g["accepted_set_jaccard_vs_reference"]),
            "admitted_clients":g["admitted_clients"],"n_training_seeds":len(d),
        }
        for metric in metrics:
            if metric not in d.columns: continue
            x=pd.to_numeric(d[metric],errors="coerce").dropna().to_numpy(float)
            row[f"{metric}_mean"]=float(np.mean(x)) if len(x) else np.nan
            row[f"{metric}_sd"]=float(np.std(x,ddof=1)) if len(x)>1 else 0.0
            lo,hi=ci95_mean(x); row[f"{metric}_ci95_low"]=lo; row[f"{metric}_ci95_high"]=hi
        rows.append(row)
    summary=pd.DataFrame(rows)
    ref=summary[summary["configuration"].eq("REFERENCE")].iloc[0]
    for metric in ["roc_auc_ovr_macro","f1_macro","accuracy"]:
        base=float(ref[f"{metric}_mean"])
        summary[f"{metric}_delta_vs_reference"]=summary[f"{metric}_mean"]-base
        summary[f"{metric}_change_pct_vs_reference"]=100.0*summary[f"{metric}_delta_vs_reference"]/base
    summary.to_csv(EXPERIMENT_ROOT/"rq6_sensitivity_performance_summary.csv",index=False)

    # Paired seed-level differences.
    ref_seed=expanded[expanded["configuration"].eq("REFERENCE")][
        ["training_seed","roc_auc_ovr_macro","f1_macro","accuracy"]
    ].copy()
    paired=[]
    for cfg,d in expanded.groupby("configuration",sort=False):
        if cfg=="REFERENCE": continue
        m=d.merge(ref_seed,on="training_seed",suffixes=("_config","_reference"),validate="one_to_one")
        for metric in ["roc_auc_ovr_macro","f1_macro","accuracy"]:
            diff=m[f"{metric}_config"].to_numpy(float)-m[f"{metric}_reference"].to_numpy(float)
            lo,hi=ci95_mean(diff)
            paired.append({
                "configuration":cfg,"family":d["family"].iloc[0],"target_dimension":d["target_dimension"].iloc[0],
                "perturbation_pct":int(d["perturbation_pct"].iloc[0]),"metric":metric,"n_pairs":len(diff),
                "mean_difference":float(np.mean(diff)),"sd_difference":float(np.std(diff,ddof=1)) if len(diff)>1 else 0.0,
                "ci95_low":lo,"ci95_high":hi,"wins":int(np.sum(diff>0)),"ties":int(np.sum(np.isclose(diff,0))),
                "losses":int(np.sum(diff<0)),
            })
    pd.DataFrame(paired).to_csv(EXPERIMENT_ROOT/"rq6_paired_differences_vs_reference.csv",index=False)
    return expanded,summary


def save_rq6_split_outputs(gov_summary, perf_summary):
    threshold_families=["LOWER_ONLY","UPPER_ONLY","BOTH"]
    gov_summary[gov_summary["family"].isin(threshold_families)].to_csv(
        EXPERIMENT_ROOT/"rq6_threshold_governance_summary.csv",index=False
    )
    perf_summary[perf_summary["family"].isin(threshold_families)].to_csv(
        EXPERIMENT_ROOT/"rq6_threshold_performance_summary.csv",index=False
    )
    gov_summary[gov_summary["family"].eq("WEIGHT_ONE_DIM")].to_csv(
        EXPERIMENT_ROOT/"rq6_weight_governance_summary.csv",index=False
    )
    perf_summary[perf_summary["family"].eq("WEIGHT_ONE_DIM")].to_csv(
        EXPERIMENT_ROOT/"rq6_weight_performance_summary.csv",index=False
    )
    invalid=gov_summary[gov_summary["valid_policy"].eq(False)]
    invalid.to_csv(EXPERIMENT_ROOT/"rq6_invalid_threshold_configurations.csv",index=False)


def create_rq6_threshold_figure(perf_summary):
    d=perf_summary[perf_summary["family"].isin(["LOWER_ONLY","UPPER_ONLY","BOTH"])].copy()
    families=[("LOWER_ONLY","Lower only"),("UPPER_ONLY","Upper only"),("BOTH","Both")]
    pcts=PERTURBATION_PCTS; x=np.arange(len(pcts),dtype=float); width=0.24
    fig=plt.figure(figsize=(14,10)); gs=fig.add_gridspec(2,2,hspace=0.38,wspace=0.28)
    panels=[
        (gs[0,0],"accepted_rate_change_pp_vs_reference","A. Change in admitted-client percentage","Percentage-point change"),
        (gs[0,1],"roc_auc_ovr_macro_delta_vs_reference","B. Change in Macro ROC-AUC","Δ Macro ROC-AUC"),
        (gs[1,0],"f1_macro_delta_vs_reference","C. Change in Macro-F1","Δ Macro-F1"),
        (gs[1,1],"accuracy_delta_vs_reference","D. Change in Accuracy","Δ Accuracy"),
    ]
    for cell,col,title,ylabel in panels:
        ax=fig.add_subplot(cell)
        for i,(fam,label) in enumerate(families):
            vals=[]
            for pct in pcts:
                z=d[d["family"].eq(fam)&d["perturbation_pct"].eq(pct)]
                vals.append(float(z[col].iloc[0]) if len(z) else np.nan)
            ax.bar(x+(i-1)*width,vals,width,label=label)
        ax.axhline(0,linewidth=0.8); ax.set_xticks(x); ax.set_xticklabels([f"{p:+d}%" for p in pcts])
        ax.set_xlabel("Threshold perturbation"); ax.set_ylabel(ylabel); ax.set_title(title,fontweight="bold")
        if cell==gs[0,0]: ax.legend(fontsize=8)
    fig.suptitle("RQ6 — HPS Threshold Sensitivity under Frozen K=20 TADP-VR",fontsize=14,fontweight="bold")
    out=EXPERIMENT_ROOT/"RQ6_threshold_sensitivity_figure.png"; fig.savefig(out,dpi=300,bbox_inches="tight"); plt.close(fig)
    return out


def create_rq6_weight_figure(perf_summary):
    d=perf_summary[perf_summary["family"].eq("WEIGHT_ONE_DIM")].copy()
    dims=DIMS; pcts=PERTURBATION_PCTS
    panels=[
        ("accepted_rate_change_pp_vs_reference","A. Admitted-client percentage change","Percentage points"),
        ("roc_auc_ovr_macro_delta_vs_reference","B. Macro ROC-AUC change","Δ Macro ROC-AUC"),
        ("f1_macro_delta_vs_reference","C. Macro-F1 change","Δ Macro-F1"),
        ("accuracy_delta_vs_reference","D. Accuracy change","Δ Accuracy"),
    ]
    fig=plt.figure(figsize=(15,10)); gs=fig.add_gridspec(2,2,hspace=0.32,wspace=0.28)
    for idx,(col,title,cbar_label) in enumerate(panels):
        ax=fig.add_subplot(gs[idx//2,idx%2]); matrix=np.full((len(dims),len(pcts)),np.nan)
        for i,dim in enumerate(dims):
            for j,pct in enumerate(pcts):
                z=d[d["target_dimension"].eq(dim)&d["perturbation_pct"].eq(pct)]
                if len(z): matrix[i,j]=float(z[col].iloc[0])
        im=ax.imshow(matrix,aspect="auto")
        ax.set_xticks(np.arange(len(pcts))); ax.set_xticklabels([f"{p:+d}%" for p in pcts])
        ax.set_yticks(np.arange(len(dims))); ax.set_yticklabels([DIMENSION_NAMES[x] for x in dims])
        ax.set_xlabel("Single-dimension weight perturbation"); ax.set_title(title,fontweight="bold")
        for i in range(matrix.shape[0]):
            for j in range(matrix.shape[1]):
                if np.isfinite(matrix[i,j]):
                    ax.text(j,i,f"{matrix[i,j]:.3f}",ha="center",va="center",fontsize=7)
        cb=fig.colorbar(im,ax=ax,fraction=0.046,pad=0.04); cb.set_label(cbar_label)
    fig.suptitle("RQ6 — HPS Weight Sensitivity under Frozen K=20 TADP-VR",fontsize=14,fontweight="bold")
    out=EXPERIMENT_ROOT/"RQ6_weight_sensitivity_figure.png"; fig.savefig(out,dpi=300,bbox_inches="tight"); plt.close(fig)
    return out


def main():
    print_banner(EXPERIMENT_VERSION)
    print("Threshold families: lower-only, upper-only, both")
    print("Weight sensitivity: one HPS dimension at a time; other weights rescaled proportionally")
    print(f"Perturbations: {PERTURBATION_PCTS}")
    print(f"K={K_SUBMISSIONS} | seeds={TRAINING_RUN_SEEDS} | rounds={NUM_ROUNDS_FL}")

    configs=build_rq6_configurations()
    configs.to_csv(EXPERIMENT_ROOT/"rq6_requested_configurations.csv",index=False)
    weight_vectors=configs[configs["family"].eq("WEIGHT_ONE_DIM")][
        ["configuration","target_dimension","perturbation_pct","weights_json"]
    ].copy()
    weight_vectors.to_csv(EXPERIMENT_ROOT/"rq6_weight_vectors.csv",index=False)

    atomic_write_json({
        "experiment_version":EXPERIMENT_VERSION,
        "research_question":"How sensitive are TADP-VR admission decisions and downstream utility to HPS thresholds and dimension weights?",
        "k_submissions":K_SUBMISSIONS,"training_seeds":TRAINING_RUN_SEEDS,"fl_rounds":NUM_ROUNDS_FL,
        "local_epochs":LOCAL_EPOCHS,"batch_size":BATCH_SIZE,"learning_rate":LEARNING_RATE,
        "dirichlet_alpha":DIRICHLET_ALPHA,"global_split_seed":GLOBAL_SPLIT_SEED,
        "client_partition_seed":CLIENT_PARTITION_SEED,"frozen_evidence_assignment_seed":FROZEN_EVIDENCE_ASSIGNMENT_SEED,
        "evidence_library_seed":EVIDENCE_LIBRARY_SEED,
        "evidence_library_size":EVIDENCE_LIBRARY_SIZE,
        "evidence_assignment_sampling":EVIDENCE_ASSIGNMENT_SAMPLING,
        "evidence_assignment_note":"Twenty distinct documentary evidence profiles are created before client assignment; each of the 20 clients samples one uniformly with replacement. Duplicate client assignments are allowed and the realized admission rate is not forced.",
        "reference_lower_cut":REFERENCE_LOWER_CUT,"reference_upper_cut":REFERENCE_UPPER_CUT,
        "reference_dimension_floor":REFERENCE_DIMENSION_FLOOR,"reference_weights":WEIGHTS_PSCORE_DEFAULT,
        "perturbation_pcts":PERTURBATION_PCTS,
        "threshold_families":["LOWER_ONLY","UPPER_ONLY","BOTH"],
        "weight_rule":"Perturb one target dimension by ±10/20/30%; rescale the other five proportionally so weights sum to 1.",
        "why_not_all_weights_together":"Common scaling of all weights followed by normalization leaves the original normalized vector unchanged.",
        "interpretation":"Sensitivity analysis only; no configuration is selected using held-out TEST performance.",
    },EXPERIMENT_ROOT/"rq6_experiment_design.json")

    csv_path=locate_diabetes_csv()
    data=prepare_diabetes_no_leakage(
        csv_path=csv_path,split_seed=GLOBAL_SPLIT_SEED,partition_seed=CLIENT_PARTITION_SEED,
        n_clients=K_SUBMISSIONS,alpha=DIRICHLET_ALPHA,
    )
    leakage=write_leakage_audit(
        EXPERIMENT_ROOT,data["global_train_ids"],data["global_test_ids"],data["client_train_ids"],
        extra={
            "patient_overlap":data["meta"]["patient_overlap"],"global_holdout_before_client_partition":True,
            "preprocessing_train_only":True,"dq_tadp_train_only":True,"class_weights_train_only":True,
            "test_used_for_final_evaluation_only":True,"rq6_frozen_k20_setting":True,
            "evidence_library_size":EVIDENCE_LIBRARY_SIZE,
            "evidence_assignment_sampling":EVIDENCE_ASSIGNMENT_SAMPLING,
        },
    )
    if not bool(leakage.get("pass",False)): raise RuntimeError(f"No-leakage audit failed: {leakage}")

    client_ids=list(data["client_ids"])
    expected_client_ids=list("ABCDEFGHIJKLMNOPQRST")
    if client_ids!=expected_client_ids: raise RuntimeError(f"Expected A-T clients for K=20; got {client_ids}")

    documentary,evidence_df,evidence_library_df=generate_random_library_controlled_evidence(
        client_ids=client_ids,
        evidence_seed=FROZEN_EVIDENCE_ASSIGNMENT_SEED,
        library_seed=EVIDENCE_LIBRARY_SEED,
        domain=DOMAIN,
    )
    evidence_library_df.to_csv(EXPERIMENT_ROOT/"rq6_20_profile_evidence_library.csv",index=False)
    evidence_df.to_csv(EXPERIMENT_ROOT/"rq6_frozen_controlled_evidence.csv",index=False)
    assignment_summary=summarize_random_library_assignment(evidence_df)
    assignment_summary.to_csv(EXPERIMENT_ROOT/"rq6_frozen_evidence_assignment_map.csv",index=False)
    usage_summary=(assignment_summary.groupby(["library_profile_id","evidence_profile"])
                   .size().reset_index(name="clients_receiving_profile")
                   .sort_values(["clients_receiving_profile","library_profile_id"],ascending=[False,True]))
    usage_summary.to_csv(EXPERIMENT_ROOT/"rq6_evidence_profile_usage_summary.csv",index=False)
    print_banner("RQ6 FROZEN 20-PROFILE EVIDENCE ASSIGNMENT (WITH REPLACEMENT)")
    print(assignment_summary.to_string(index=False))
    data["dq_audit"].to_csv(EXPERIMENT_ROOT/"rq6_frozen_DQ_audit.csv",index=False)

    original_gov=build_tadp_governance(
        client_ids,documentary,data["dq_scores"],run=0,evidence_seed=FROZEN_EVIDENCE_ASSIGNMENT_SEED,domain=DOMAIN
    ).sort_values("client").reset_index(drop=True)
    original_gov.to_csv(EXPERIMENT_ROOT/"rq6_original_frozen_governance.csv",index=False)

    full_factor_df=build_full_factor_evidence_table(
        client_ids,documentary,data["dq_scores"],data["dq_audit"],FROZEN_EVIDENCE_ASSIGNMENT_SEED,DOMAIN
    )
    full_factor_df.to_csv(EXPERIMENT_ROOT/"rq6_frozen_all_28_factor_evidence.csv",index=False)
    _,dim_scores,crit=build_policy_tables_rq6(full_factor_df)

    # Fail closed: reproduce exact reference policy first.
    ref_recomputed=evaluate_rq6_policy(
        dim_scores,crit,REFERENCE_LOWER_CUT,REFERENCE_UPPER_CUT,
        {d:float(WEIGHTS_PSCORE_DEFAULT[d]) for d in DIMS}
    ).sort_values("client").reset_index(drop=True)
    audit=ref_recomputed[["client","hps","final_action"]].merge(
        original_gov[["client","hps","final_action"]],on="client",suffixes=("_recomputed","_original"),validate="one_to_one"
    )
    audit["decision_match"]=audit["final_action_recomputed"].eq(audit["final_action_original"])
    audit["hps_abs_diff"]=np.abs(audit["hps_recomputed"]-audit["hps_original"])
    audit.to_csv(EXPERIMENT_ROOT/"rq6_reference_policy_reproduction_audit.csv",index=False)
    if not audit["decision_match"].all() or float(audit["hps_abs_diff"].max())>1e-10:
        raise RuntimeError("RQ6 reference-policy reproduction failed")

    gov_summary,gov_by_config,_=summarize_rq6_governance(configs,dim_scores,crit)
    gov_summary,cohort_mapping=assign_rq6_cohort_ids(gov_summary)
    gov_summary.to_csv(EXPERIMENT_ROOT/"rq6_governance_sensitivity_summary.csv",index=False)

    frames=[]
    for cfg,df in gov_by_config.items():
        x=df.copy();x.insert(0,"configuration",cfg);frames.append(x)
    pd.concat(frames,ignore_index=True).to_csv(EXPERIMENT_ROOT/"rq6_governance_client_level.csv",index=False)

    print_banner("RQ6 GOVERNANCE SENSITIVITY SUMMARY")
    show=[
        "configuration","family","target_dimension","perturbation_pct","lower_cut","upper_cut","valid_policy",
        "accepted_count","accepted_rate_pct","accepted_rate_change_pp_vs_reference","decision_flip_count",
        "accepted_set_jaccard_vs_reference","admitted_clients"
    ]
    print(gov_summary[[c for c in show if c in gov_summary.columns]].to_string(index=False))

    cohort_perf=train_rq6_unique_cohorts(cohort_mapping,gov_summary,data)
    _,perf_summary=summarize_rq6_performance(gov_summary,cohort_perf)
    save_rq6_split_outputs(gov_summary,perf_summary)
    fig_threshold=create_rq6_threshold_figure(perf_summary)
    fig_weight=create_rq6_weight_figure(perf_summary)

    print_banner("RQ6 PERFORMANCE SUMMARY")
    print(perf_summary[[
        "configuration","family","target_dimension","perturbation_pct","accepted_count","accepted_rate_pct",
        "accepted_rate_change_pp_vs_reference","roc_auc_ovr_macro_mean","roc_auc_ovr_macro_delta_vs_reference",
        "f1_macro_mean","f1_macro_delta_vs_reference","accuracy_mean","accuracy_delta_vs_reference"
    ]].to_string(index=False))

    n_threshold=int(configs["family"].isin(["LOWER_ONLY","UPPER_ONLY","BOTH"]).sum())
    n_weight=int(configs["family"].eq("WEIGHT_ONE_DIM").sum())
    n_invalid=int((configs["valid_policy"].eq(False)).sum())
    atomic_write_json({
        "reference_policy_reproduction":"PASS","no_leakage":"PASS","same_W0_within_seed":"PASS",
        "requested_threshold_configs":n_threshold,"requested_weight_configs":n_weight,
        "invalid_threshold_configs":n_invalid,"valid_configs_including_reference":int(gov_summary["valid_policy"].eq(True).sum()),
        "n_unique_admitted_cohorts":int(len(cohort_mapping)),
        "actual_model_training_runs":int(len(cohort_mapping)*len(TRAINING_RUN_SEEDS)),
        "threshold_figure":str(fig_threshold),"weight_figure":str(fig_weight),
    },EXPERIMENT_ROOT/"rq6_final_audit_summary.json")

    return EXPERIMENT_ROOT


if __name__=="__main__":
    finished_root=main()
    package_and_download_results(finished_root,EXPERIMENT_VERSION)


Google Drive checkpoint mount unavailable: mount failed
Local checkpoint root: /content/TADP_EXPERIMENT_B3_TADP-B3-v16.9-PREDICTIVE-SCALABILITY-K20-50-100-3SEED-4ROUND
Google Drive checkpoint mount unavailable: mount failed
Local checkpoint root: /content/TADP_EXPERIMENT_RQ6_TADP-RQ6-v17.3-K20-20EVIDENCE-WR-THRESHOLD-WEIGHT-SENSITIVITY-5SEED-4ROUND

TADP-RQ6-v17.3-K20-20EVIDENCE-WR-THRESHOLD-WEIGHT-SENSITIVITY-5SEED-4ROUND
Threshold families: lower-only, upper-only, both
Weight sensitivity: one HPS dimension at a time; other weights rescaled proportionally
Perturbations: [-30, -20, -10, 10, 20, 30]
K=20 | seeds=[42, 142, 242, 342, 442] | rounds=4

RQ6 FROZEN 20-PROFILE EVIDENCE ASSIGNMENT (WITH REPLACEMENT)
client library_profile_id     evidence_profile  profile_variant  times_profile_sampled_in_realization      assignment_sampling  evidence_seed  evidence_library_seed
     A                P19 DIMENSION_FLOOR_WEAK                0                                     4 UNIFORM_WITH_REP

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
"""
Publication-style RQ6 combined sensitivity figure.

One grouped bar chart only.
Positive bars = gains relative to the frozen reference policy.
Negative bars = losses relative to the frozen reference policy.

The chart reports mean signed RELATIVE percentage changes for:
  - admitted-client rate
  - macro ROC-AUC
  - macro-F1
  - accuracy

Threshold categories:
  lower threshold only
  upper threshold only
  both thresholds together

Weight categories:
  each HPS dimension perturbed one at a time

Means are calculated across valid tested perturbations.
Invalid threshold orderings are excluded.
"""

from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. INPUT
# ============================================================

ZIP_NAME = (
    "TADP_EXPERIMENT_RQ6_"
    "TADP-RQ6-v17.3-K20-20EVIDENCE-WR-THRESHOLD-WEIGHT-"
    "SENSITIVITY-5SEED-4ROUND_RESULTS.zip"
)

# Colab default:
ZIP_PATH = Path("/content") / ZIP_NAME

# Change ZIP_PATH if necessary.
if not ZIP_PATH.exists():
    candidates = list(Path.cwd().glob(f"**/{ZIP_NAME}"))
    if not candidates:
        raise FileNotFoundError(
            f"Could not find {ZIP_NAME}. "
            "Upload the results ZIP to /content or update ZIP_PATH."
        )
    ZIP_PATH = candidates[0]

OUTPUT_PNG = ZIP_PATH.parent / "RQ6_combined_sensitivity_figure.png"
OUTPUT_PDF = ZIP_PATH.parent / "RQ6_combined_sensitivity_figure.pdf"

print("Using:", ZIP_PATH)

# ============================================================
# 2. READ RQ6 SUMMARIES
# ============================================================

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    with zf.open("rq6_threshold_performance_summary.csv") as f:
        threshold = pd.read_csv(f)

    with zf.open("rq6_weight_performance_summary.csv") as f:
        weight = pd.read_csv(f)

# ============================================================
# 3. COLUMNS USED
#    All are RELATIVE percentage changes vs the reference.
# ============================================================

metric_cols = {
    "Admission": "accepted_rate_relative_change_pct_vs_reference",
    "Macro ROC-AUC": "roc_auc_ovr_macro_change_pct_vs_reference",
    "Macro-F1": "f1_macro_change_pct_vs_reference",
    "Accuracy": "accuracy_change_pct_vs_reference",
}

required = list(metric_cols.values())

for name, df in [("threshold", threshold), ("weight", weight)]:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"{name} summary missing columns: {missing}")

# ============================================================
# 4. MEAN SIGNED IMPACT BY POLICY-ADJUSTMENT TYPE
# ============================================================

threshold_order = ["LOWER_ONLY", "UPPER_ONLY", "BOTH"]
threshold_labels = {
    "LOWER_ONLY": "Lower\nthreshold",
    "UPPER_ONLY": "Upper\nthreshold",
    "BOTH": "Both\nthresholds",
}

thr_mean = (
    threshold
    .groupby("family")[required]
    .mean()
    .reindex(threshold_order)
)

dim_order = ["dim1", "dim2", "dim3", "dim4", "dim5", "dim6"]
dim_labels = {
    "dim1": "SR\nweight",
    "dim2": "DQ\nweight",
    "dim3": "DOC\nweight",
    "dim4": "TIME\nweight",
    "dim5": "REG\nweight",
    "dim6": "CTX\nweight",
}

wgt_mean = (
    weight
    .groupby("target_dimension")[required]
    .mean()
    .reindex(dim_order)
)

summary = pd.concat([thr_mean, wgt_mean], axis=0)
summary.index = (
    [threshold_labels[x] for x in threshold_order]
    + [dim_labels[x] for x in dim_order]
)
summary = summary.rename(columns={v: k for k, v in metric_cols.items()})

print("\nMean signed impact used in the plot:")
print(summary.round(4).to_string())

# ============================================================
# 5. PLOT
# ============================================================

categories = summary.index.tolist()
metrics = ["Admission", "Macro ROC-AUC", "Macro-F1", "Accuracy"]

x = np.arange(len(categories))
width = 0.18
offsets = np.array([-1.5, -0.5, 0.5, 1.5]) * width

fig, ax = plt.subplots(figsize=(14.5, 6.7))

bar_sets = []

for i, metric in enumerate(metrics):
    values = summary[metric].to_numpy(dtype=float)
    bars = ax.bar(
        x + offsets[i],
        values,
        width,
        label=metric,
    )
    bar_sets.append((bars, values))

# Zero line: positive = gain, negative = loss
ax.axhline(0, linewidth=1.1)

# Visual separation between threshold and weight perturbations
ax.axvline(2.5, linewidth=0.9, linestyle="--")

# Labels
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel("Mean signed change vs reference (%)")
ax.set_title(
    "RQ6 Sensitivity of TADP-VR to Threshold and HPS-Weight Perturbations\n"
    "K=20 | five training seeds | perturbations = ±10%, ±20%, ±30%",
    pad=38,
)

# Legend outside the plotting area
ax.legend(
    ncol=4,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.12),
    frameon=False,
)

# Leave some positive room above zero for annotations
current_min = min(-10.0, float(np.nanmin(summary.to_numpy())) - 0.6)
ax.set_ylim(current_min, 1.25)

# Annotate only non-zero bars
for bars, values in bar_sets:
    for bar, value in zip(bars, values):
        if np.isfinite(value) and abs(value) > 1e-8:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                value - 0.18 if value < 0 else value + 0.18,
                f"{value:+.2f}%",
                ha="center",
                va="top" if value < 0 else "bottom",
                fontsize=8.5,
            )

# Explain the otherwise-flat regions directly on the figure
ax.text(
    0.5,
    0.55,
    "Mean change = 0%\\nfor all valid one-sided threshold perturbations",
    ha="center",
    va="center",
    fontsize=9,
)

ax.text(
    5.5,
    0.55,
    "Mean change = 0% across all 36\\none-dimension weight perturbations",
    ha="center",
    va="center",
    fontsize=9,
)

ax.text(
    2.0,
    -7.0,
    "Only joint +20% and +30% threshold increases\\nchanged the admitted cohort (55% → 40%)",
    ha="center",
    va="center",
    fontsize=9,
)

# Keep layout clean; put methodological details in the manuscript caption.
plt.tight_layout()

# ============================================================
# 6. SAVE
# ============================================================

fig.savefig(OUTPUT_PNG, dpi=400, bbox_inches="tight")
fig.savefig(OUTPUT_PDF, bbox_inches="tight")
plt.close(fig)

print("\nSaved:")
print(OUTPUT_PNG)
print(OUTPUT_PDF)


Using: /content/TADP_EXPERIMENT_RQ6_TADP-RQ6-v17.3-K20-20EVIDENCE-WR-THRESHOLD-WEIGHT-SENSITIVITY-5SEED-4ROUND_RESULTS.zip

Mean signed impact used in the plot:
                  Admission  Macro ROC-AUC  Macro-F1  Accuracy
Lower\nthreshold     0.0000         0.0000    0.0000    0.0000
Upper\nthreshold     0.0000         0.0000    0.0000    0.0000
Both\nthresholds    -9.0909        -0.2381   -4.8871   -0.7597
SR\nweight           0.0000         0.0000    0.0000    0.0000
DQ\nweight           0.0000         0.0000    0.0000    0.0000
DOC\nweight          0.0000         0.0000    0.0000    0.0000
TIME\nweight         0.0000         0.0000    0.0000    0.0000
REG\nweight          0.0000         0.0000    0.0000    0.0000
CTX\nweight          0.0000         0.0000    0.0000    0.0000

Saved:
/content/RQ6_combined_sensitivity_figure.png
/content/RQ6_combined_sensitivity_figure.pdf
